# Analysis Notebook: Compute-Matched Encoders vs Frozen Pathology Foundation Model

Produces paper-ready figures, LaTeX tables, and a headline-findings summary from the completed LOMO, ablation and cross-validation runs.

## Contents

| Section | Purpose |
|---|---|
| **1. Setup** | Config, run log (every printed result also goes to `analysis_outputs/run_log.txt`, so the bundle ZIP keeps it), imports, run loading (energy sign + in-distribution reference), bootstrap CI, palette |
| **2. Figures** | Fig 1–8: accuracy, calibration, compute, risk–coverage, BACH means, reliability, temperature |
| **2b. Image-based** | Fig 9–11 at held-out 400× (confusion, sample predictions, Grad-CAM++) |
| **2c. XAI** | Fig 11b Attention Rollout (Phikon), Fig 12 UMAP at 400× |
| **2d. Part C** | Fig 13 SS2D ablation, Fig 14 patient-level cross-validation (accuracy), Fig 14b the same cross-validation for calibration and selective risk |
| **2e. Dataset + divergence** | Fig 15 dataset composition and LOMO splits, Fig 30–31 and T13 the preprocessing pipeline with the exact training transforms plus colour statistics per magnification and for BACH (CPU; needs BreakHis attached), Fig 16 accuracy vs reliability (3D) |
| **2f. Every magnification** | Fig 9b confusion, Fig 10b sample predictions, Fig 11c Grad-CAM++ / Rollout at 40×, 100×, 200×, 400× |
| **2g. RNE + OOD** | Reference-Normalized Energy, per-image AUROC/AUPR/FPR95 vs MSP, Mahalanobis, kNN; pooled and fixed-threshold analysis; Fig 17, Fig 18, Fig 12b UMAP at every magnification |
| **2h. 3D interpretability** | Fig 19 saliency layer stack, Fig 20 radial attribution profile, Fig 21 one patient across magnifications, Fig 22 selective-risk landscape, Fig 23 attention zones marked on the tile with a nuclei proxy, Fig 24 nucleus focus by depth, Fig 25 the relied-on region marked on the slice with its area, Fig 26 measurements taken from those regions |
| **2i. Capability summary** | Fig 27 capability radar: accuracy, calibration, selective risk, cross-institution detection, compute and seed stability on one figure, every axis scaled between a floor and ceiling fixed in advance |
| **2j. Ranking statistics** | T5b E-AURC (AURC without its accuracy part), Fig 28 per-run accuracy against cross-institution AUROC, Fig 29 critical-difference diagrams (Friedman + Nemenyi) |
| **3. Tables** | T3–T10, T11/T11b per-run OOD separability, T12/T12b pooled and fixed-threshold (T5b is written in 2j) |
| **4. Findings** | Headline findings, plus the RNE / OOD summary |
| **4b. Subtype extension** | The two-stage pipeline (benign vs malignant, then the 8 subtypes) re-scored from `subtype_hierarchical.ipynb`'s stored logits: Fig 32 accuracy and reliability per held-out magnification, Fig 33 error sources, Fig 34 per-subtype F1, Fig 35 8 × 8 confusion, Fig 36 accuracy vs reliability ranks, Fig 37 SS2D ablation, T14–T17, headline findings (CPU; skipped if the subtype outputs are not attached) |
| **4c. Proposed model** | MagScan-Mamba (frozen Phikon patch tokens read by a Mamba-scan head) against the four encoders on the same splits: Fig 38 every held-out magnification, Fig 39 mean ranks with Friedman/Nemenyi and Holm-corrected Wilcoxon tests, Fig 40 ablations (token mixer, CLS probe, zoom views), Fig 41 compute and the reliability gate, T18–T23 (CPU; skipped if its output is not attached) |
| **4d. Subtype in 3D** | The binary 3D figures redrawn for the subtype pipeline with all five encoders: Fig 42 accuracy and ECE as 3D bars, Fig 43 selective-risk landscape, Fig 44 error towers (confusion without the diagonal, by error source), Fig 45 capability radar, Fig 46 saliency layer stack of the malignant-subtype models on a lobular carcinoma tile (GPU + BreakHis; MagScan-Mamba via token Grad-CAM on its Mamba layers), Fig 47 one patient across the four magnifications |
| **5. Bundle** | Archive `analysis_outputs/` into a ZIP |

## How to run

1. **Settings → Accelerator → GPU T4** (Grad-CAM++, MedMamba and section 2g need CUDA).
2. **Add Input**: every LOMO and Part C training output, `ambarish/breakhis`, and `truthisneverlinear/bach-breast-cancer-histology-images` (new: section 2g re-scores BACH per image). For section 4b also every output of `subtype_hierarchical.ipynb` (including the MagScan-Mamba session); if that exceeds Kaggle's input limit, put all subtype outputs into one dataset (each session's folder as it is) and attach that. For section 4c the output of `magscan_mamba_full.ipynb`. Binary, subtype and MagScan-Mamba outputs are told apart automatically.
3. **Run All**. Section 2g takes roughly 20–40 min the first time; it caches per-run arrays in `/kaggle/working/ood_cache/`. To skip that work in a later version, add this notebook's previous output as an input — the cache is found and reused.

Cells that lack an input print why and skip; nothing downstream crashes.

## 1. Setup

In [ ]:
# ---------- User configuration ----------
# Run outputs are found automatically: every folder under /kaggle/input that contains
# runs/<run_id>/record.json is merged. When the LOMO magnifications run on several Kaggle
# accounts, each attached output mounts under a different username and notebook slug, so
# hard-coded paths break. Attach every training output with Add Input and leave
# INPUT_PATHS = None.
#
# To pin inputs by hand, set INPUT_PATHS to a list of folders or .zip files. Records are
# deduped by run_id and auxiliary CSVs are read from the first input that has them, so
# auto-discovery lists LOMO outputs before Part C (Part C's from_scratch_compute_report.csv
# does not cover every LOMO encoder).
INPUT_PATHS = None
SUBTYPE_INPUT_PATHS = None      # subtype_hierarchical.ipynb outputs (section 4b); None = auto-discover
MAGSCAN_INPUT_PATHS = None      # magscan_mamba_full.ipynb output (section 4c); None = auto-discover
OUTPUT_DIR = '/kaggle/working/analysis_outputs'
BOOTSTRAP_ITERATIONS = 1000

from pathlib import Path as _P

SUBTYPE_RUN_PREFIXES = ('detect_', 'benign_subtype_', 'malignant_subtype_')
MAGSCAN_RUN_PREFIX = 'magscan_'      # magscan_mamba_full.ipynb and its pilots

def output_kind(folder):
    """'subtype', 'magscan' or 'binary', from the run ids in <folder>/runs."""
    names = [d.name for d in (_P(folder) / 'runs').iterdir()]
    if any(n.startswith(SUBTYPE_RUN_PREFIXES) for n in names):
        return 'subtype'
    if any(n.startswith(MAGSCAN_RUN_PREFIX) for n in names):
        return 'magscan'
    return 'binary'

def is_subtype_output(folder):
    return output_kind(folder) == 'subtype'

def discover_run_outputs(search_root='/kaggle/input', subtype=False, kind=None):
    """Binary-study outputs by default; kind='subtype' (or subtype=True) the subtype notebook's
    outputs, kind='magscan' the MagScan-Mamba notebook's outputs."""
    kind = kind or ('subtype' if subtype else 'binary')
    root = _P(search_root)
    if not root.exists():
        return []
    found = {rec.parent.parent.parent for rec in root.rglob('record.json')
             if rec.parent.parent.name == 'runs'}
    found = {p for p in found if output_kind(p) == kind}
    def is_partc(folder):
        return any(d.name.startswith(('ablation_', 'cv_', 'baselineref_'))
                   for d in (folder / 'runs').iterdir())
    return [str(p) for p in sorted(found, key=lambda p: (is_partc(p), str(p)))]

if INPUT_PATHS is None:
    INPUT_PATHS = discover_run_outputs()
    print(f'Auto-discovered {len(INPUT_PATHS)} run output(s) under /kaggle/input')
elif isinstance(INPUT_PATHS, str):
    INPUT_PATHS = [INPUT_PATHS]
if SUBTYPE_INPUT_PATHS is None:
    SUBTYPE_INPUT_PATHS = discover_run_outputs(subtype=True)
    print(f'Auto-discovered {len(SUBTYPE_INPUT_PATHS)} subtype output(s) (section 4b)')
elif isinstance(SUBTYPE_INPUT_PATHS, str):
    SUBTYPE_INPUT_PATHS = [SUBTYPE_INPUT_PATHS]
if MAGSCAN_INPUT_PATHS is None:
    MAGSCAN_INPUT_PATHS = discover_run_outputs(kind='magscan')
    print(f'Auto-discovered {len(MAGSCAN_INPUT_PATHS)} MagScan-Mamba output(s) (section 4c)')
elif isinstance(MAGSCAN_INPUT_PATHS, str):
    MAGSCAN_INPUT_PATHS = [MAGSCAN_INPUT_PATHS]
assert INPUT_PATHS, ('No run outputs found. Attach each training notebook output with '
                     'Add Input, or set INPUT_PATHS by hand.')
INPUT_PATH = next((p for p in INPUT_PATHS if _P(p).exists()), INPUT_PATHS[0])

for _i, _p in enumerate(INPUT_PATHS):
    if not _P(_p).exists():
        print(f'INPUT_PATHS[{_i}] = {_p}   (NOT FOUND - will be skipped)')
    elif _P(_p).is_dir():
        print(f'INPUT_PATHS[{_i}] = {_p}   ({len(list(_P(_p).glob("runs/*/record.json")))} finished runs)')
    else:
        print(f'INPUT_PATHS[{_i}] = {_p}   (archive)')
print(f'OUTPUT_DIR  = {OUTPUT_DIR}')
print(f'BOOTSTRAP N = {BOOTSTRAP_ITERATIONS}')


In [ ]:
# ---------- Run log: everything printed from here on is also written to analysis_outputs/run_log.txt ----------
# Figures and tables are saved as files, but many results are only printed: the headline findings, the
# rank and Spearman tables, the CV-minus-LOMO deltas, the per-axis leaders of Fig 27. The bundle ZIP holds
# OUTPUT_DIR only, so without this those numbers would live nowhere but the notebook viewer. The log is
# line-buffered, so it is complete up to the moment the bundle cell zips OUTPUT_DIR. Download progress
# bars are left out. A pre-run hook re-attaches the log if the kernel ever swaps sys.stdout between cells,
# and a failed log write is swallowed, so logging can never break a cell.
import sys as _sys
import time as _time
from pathlib import Path as _Path

RUN_LOG = _Path(OUTPUT_DIR) / 'run_log.txt'


class _TeeToLog:
    """Forward every write to the notebook's own stream and append it to the run log."""
    _is_run_log_tee = True        # a marker, not isinstance: re-running this cell defines a new class

    def __init__(self, stream, log_file):
        self._stream, self._log, self._skip_newline = stream, log_file, False

    def write(self, text):
        out = self._stream.write(text)
        try:
            if '━━' in text:                    # pip / hub download progress bar
                self._skip_newline = True                  # ... and the newline print() sends after it
            elif self._skip_newline and text == '\n':
                self._skip_newline = False
            else:
                self._skip_newline = False
                self._log.write(text)
        except Exception:
            pass
        return out

    def flush(self):
        self._stream.flush()
        try:
            self._log.flush()
        except Exception:
            pass

    def __getattr__(self, name):                      # set_parent, encoding, isatty, fileno -> real stream
        return getattr(self._stream, name)


def _attach_run_log(*_):
    if not getattr(_sys.stdout, '_is_run_log_tee', False):
        _sys.stdout = _TeeToLog(_sys.stdout, _RUN_LOG_FILE)


if globals().get('_RUN_LOG_FILE') is None:           # first run of this cell in this kernel
    _Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    _RUN_LOG_FILE = open(RUN_LOG, 'w', encoding='utf-8', buffering=1)
    _RUN_LOG_FILE.write(f'# analysis run log, started {_time.strftime("%Y-%m-%d %H:%M:%S")}\n')
    _attach_run_log()
    try:
        get_ipython().events.register('pre_run_cell', _attach_run_log)
    except NameError:                                  # plain Python, not a notebook kernel
        pass
print(f'Printed output is also logged to {RUN_LOG}')

In [ ]:
# ---------- Imports ----------
import os, sys, json, io, zipfile, shutil, subprocess
from pathlib import Path

# Standard scientific stack (all preinstalled on Kaggle).
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams

# scikit-learn: install-if-missing (harmless on Kaggle).
try:
    from sklearn.metrics import roc_auc_score, confusion_matrix
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn'], check=False)
    from sklearn.metrics import roc_auc_score, confusion_matrix

# Publication-quality style: 300 DPI, serif fonts, colorblind-safe.
rcParams.update({
    'font.family': 'serif',
    'font.serif': ['DejaVu Serif', 'Times New Roman', 'Times'],
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.fontsize': 9,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 13,
    'axes.linewidth': 0.8,
    'grid.alpha': 0.25,
    'lines.linewidth': 2.0,
    'lines.markersize': 8,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.08,
})
sns.set_theme(style='whitegrid', context='paper', font_scale=1.0, rc={'font.family': 'serif'})
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/figures', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/tables', exist_ok=True)

def save_fig(fig, name, dpi=300):
    """Save both PNG (raster) and PDF (vector) — PDF for LaTeX, PNG for slides."""
    fig.savefig(f'{OUTPUT_DIR}/figures/{name}.png', dpi=dpi)
    fig.savefig(f'{OUTPUT_DIR}/figures/{name}.pdf')

print('Imports OK. Publication style applied.')


In [ ]:
# ---------- Load run archive(s): zip OR extracted folder, single OR multiple ----------
class ArchiveReader:
    """Uniform accessor for either a zip archive or an extracted folder."""
    def __init__(self, path, kind):
        self.path, self.kind = path, kind
        if kind == 'zip':
            self._zip = zipfile.ZipFile(path); self._names = set(self._zip.namelist())
        else:
            self._zip = None; self._names = None
    def exists(self, member):
        return (member in self._names) if self.kind == 'zip' else (Path(self.path) / member).exists()
    def open_bytes(self, member):
        if self.kind == 'zip':
            with self._zip.open(member) as f: return f.read()
        return (Path(self.path) / member).read_bytes()
    def open_text(self, member): return self.open_bytes(member).decode('utf-8')
    def list_matching(self, suffix):
        if self.kind == 'zip':
            return sorted(n for n in self._names if n.endswith(suffix))
        return sorted(str(p.relative_to(self.path)).replace('\\', '/')
                       for p in Path(self.path).rglob('*') if str(p).endswith(suffix))

class MergedReader:
    """Presents a unified reader-like interface over a list of ArchiveReaders.
    For every method, checks each underlying reader in order and returns from the first that has it."""
    def __init__(self, readers): self._readers = readers
    def exists(self, member): return any(r.exists(member) for r in self._readers)
    def open_bytes(self, member):
        for r in self._readers:
            if r.exists(member): return r.open_bytes(member)
        raise KeyError(member)
    def open_text(self, member): return self.open_bytes(member).decode('utf-8')
    def list_matching(self, suffix):
        seen = set(); out = []
        for r in self._readers:
            for n in r.list_matching(suffix):
                if n not in seen: seen.add(n); out.append(n)
        return sorted(out)

def detect_input_kind(path_str):
    p = Path(path_str)
    if not p.exists(): return None, None
    if p.is_file() and str(p).endswith('.zip'): return 'zip', path_str
    if p.is_dir():
        if any((p / n).exists() for n in ('runs', 'raw_runs.csv', 'config.json')):
            return 'folder', path_str
        zips_inside = list(p.rglob('*.zip'))
        if zips_inside: return 'zip', str(zips_inside[0])
        for candidate in sorted(p.iterdir()):
            if candidate.is_dir() and any((candidate / n).exists() for n in ('runs', 'raw_runs.csv', 'config.json')):
                return 'folder', str(candidate)
    return None, None

# Build a reader per INPUT_PATHS entry.
readers = []
for ip in INPUT_PATHS:
    kind, resolved = detect_input_kind(ip)
    if kind is None:
        print('  WARNING: could not identify runs artifact at', ip, '- skipping')
        continue
    print('  input:', resolved, '  (kind:', kind + ')')
    readers.append(ArchiveReader(resolved, kind))
assert readers, 'No valid inputs found. Update INPUT_PATHS in the config cell above.'

# `reader` is a MergedReader that searches ALL inputs — used by every figure cell downstream.
reader = MergedReader(readers)

# Rebuild raw_runs from per-run record.json files across ALL readers.
records = []
seen_run_ids = set()
for r in readers:
    for name in r.list_matching('record.json'):
        try:
            rec = json.loads(r.open_text(name))
            rid = rec.get('run_id')
            if rid in seen_run_ids: continue
            if str(rec.get('branch', '')).startswith(('subtype_', 'magscan_')): continue   # sections 4b / 4c
            seen_run_ids.add(rid); records.append(rec)
        except Exception as exc:
            print(f'  skipped {name}: {exc}')

# Load auxiliary files (each optional) from the first reader that has them.
compute_report = bach_ood = config = environment = None
for target, is_csv in [('from_scratch_compute_report.csv', True),
                        ('bach_ood_scores.csv', True),
                        ('config.json', False),
                        ('environment.json', False)]:
    if not reader.exists(target):
        print(f'  aux missing across all readers: {target}'); continue
    data = reader.open_bytes(target)
    if is_csv: globals()[target.replace('.csv', '')] = pd.read_csv(io.BytesIO(data))
    else: globals()[target.replace('.json', '')] = json.loads(data.decode('utf-8'))

# Merge bach_ood_scores.csv from all readers so ho=400 rows from Part B are included.
bach_frames = []
for r in readers:
    if r.exists('bach_ood_scores.csv'):
        try:
            data = r.open_bytes('bach_ood_scores.csv')
            bach_frames.append(pd.read_csv(io.BytesIO(data)))
        except Exception: pass
if bach_frames:
    bach_ood = pd.concat(bach_frames, ignore_index=True)
    if 'run_id' in bach_ood.columns:
        bach_ood = bach_ood.drop_duplicates(subset=['run_id'])
compute_report = globals().get('from_scratch_compute_report')

# Build master DataFrame.
DISPLAY = {'convnext_tiny':'ConvNeXt-T','swin_tiny':'Swin-T','medmamba':'MedMamba',
           'dsamamba':'DSA-Mamba','mambavision_tiny_1k':'MambaVision',
           'owkin/phikon':'Phikon (frozen FM)'}
raw = pd.DataFrame(records)
assert 'model' in raw.columns, 'record.json files missing the "model" field - check the archive.'
raw['method'] = raw['model'].map(DISPLAY).fillna(raw['model'])

# ---- Split LOMO results from Part C's ablation / CV branches ----
# Part C's CV and baseline-reference runs both record model='medmamba', which DISPLAY
# maps to 'MedMamba'. Merged in, that gives THREE summary rows for (MedMamba, 400) --
# branch='from_scratch' (Part B), 'baseline_ref' and 'cv' (Part C). Figures 2 and 4
# pivot on (method, held_out_mag) and would raise "Index contains duplicate entries";
# worse, cell-findings averages groupby('method') with no branch filter and would
# silently blend LOMO with the CV folds.
#
# So `raw` keeps only the branches the existing figures were written against, and
# `raw_all` carries everything for the Part C cells (Fig 13/14, T9/T10) to read.
LOMO_BRANCHES = ('from_scratch', 'fm_linear_probe')
PARTC_BRANCHES = ('baseline_ref', 'ablation', 'cv')

raw_all = raw.copy()
raw_all.to_csv(f'{OUTPUT_DIR}/raw_runs_clean.csv', index=False)

if 'branch' in raw_all.columns:
    _n_partc = int(raw_all.branch.isin(PARTC_BRANCHES).sum())
    raw = raw_all[raw_all.branch.isin(LOMO_BRANCHES)].reset_index(drop=True)
    _unknown = sorted(set(raw_all.branch.dropna().unique())
                      - set(LOMO_BRANCHES) - set(PARTC_BRANCHES))
    if _unknown:
        print(f'  NOTE: unrecognised branch value(s) {_unknown} were excluded from the '
              f'main figures. Add them to LOMO_BRANCHES or PARTC_BRANCHES above.')
else:
    _n_partc = 0
    _unknown = []

print(f'\nLoaded {len(raw_all)} unique runs from {len(readers)} input source(s)')
if _n_partc:
    print(f'  {len(raw)} LOMO runs  ->  `raw`      (Fig 1-12, T3-T8)')
    print(f'  {_n_partc} Part C runs ->  `raw_all`  (ablation + CV cells only)')
    for _b in PARTC_BRANCHES:
        _sub = raw_all[raw_all.branch == _b]
        if len(_sub):
            print(f'      branch={_b:<13} {len(_sub):>2} runs, models={sorted(_sub.model.unique())}')
else:
    print(f'  all {len(raw)} are LOMO runs (Part C not merged yet)')
print(f'  methods: {sorted(raw.method.unique())}')
print(f'  held-out mags: {sorted(raw.held_out_mag.unique())}')
print(f'  seeds: {sorted(raw.seed.unique())}')

# Guard: the pivots in Fig 2 / Fig 4 require (method, held_out_mag) to be unique.
_dupes = raw.groupby(['method', 'held_out_mag']).branch.nunique()
_dupes = _dupes[_dupes > 1]
assert _dupes.empty, (
    f'(method, held_out_mag) maps to multiple branches, which will break the Fig 2/4 '
    f'pivots:\n{_dupes}\nCheck LOMO_BRANCHES.')

# ---- Split the BACH OOD scores the same way ----
# bach_ood_scores.csv is concatenated across every reader, and Fig 6 / T6 / cell-findings
# group it by method. Part C's baseline_ref and cv rows carry model='medmamba', so
# without this split MedMamba's OOD number at ho=400 would blend Part A/B's LOMO runs
# with Part C's. Part A/B's CSV predates the branch column and only ever scored
# from_scratch checkpoints, so those rows are labelled accordingly.
bach_ood_all = None
if bach_ood is not None and len(bach_ood):
    bach_ood_all = bach_ood.copy()
    if 'branch' not in bach_ood_all.columns:
        bach_ood_all['branch'] = 'from_scratch'
    else:
        bach_ood_all['branch'] = bach_ood_all['branch'].fillna('from_scratch')
    bach_ood = bach_ood_all[bach_ood_all.branch.isin(LOMO_BRANCHES)].reset_index(drop=True)
    _n_pc_ood = int(bach_ood_all.branch.isin(PARTC_BRANCHES).sum())
    print(f'  BACH OOD: {len(bach_ood)} LOMO rows -> `bach_ood`'
          + (f', {_n_pc_ood} Part C rows -> `bach_ood_all`' if _n_pc_ood else ''))

# ---- Energy sign and an in-distribution reference for the BACH scores ----
# The training notebooks store energy_score() = T*logsumexp(logits/T), the NEGATIVE of the energy
# of Liu et al. (2020), E(x) = -T*logsumexp(f(x)/T). It is converted here so every figure and table
# uses the standard sign: lower energy = more in-distribution-like, so HIGHER energy on BACH means
# the model is less confident on the shifted data.
# A BACH mean alone cannot separate a shift-aware model from one that is unconfident everywhere,
# so each run's own BreakHis test set (test_outputs.npz, same raw logits) is scored the same way:
#   energy_gap = E(BACH) - E(test)       > 0: BACH looks less in-distribution than the test set
#   msp_gap    = MSP(test) - MSP(BACH)   > 0: less confident on BACH than on the test set
def _logsumexp_rows(logits):
    logits = np.asarray(logits, dtype=np.float64)
    m = logits.max(axis=1, keepdims=True)
    return (m + np.log(np.exp(logits - m).sum(axis=1, keepdims=True))).ravel()

def _add_id_reference(frame):
    frame = frame.copy()
    frame['energy_mean'] = -frame['energy_mean']
    id_energy, id_msp = [], []
    for rid in frame['run_id']:
        npz_path = f'runs/{rid}/test_outputs.npz'
        if 'reader' in globals() and reader.exists(npz_path):
            logits = np.load(io.BytesIO(reader.open_bytes(npz_path)))['logits']
            p = np.exp(logits - logits.max(axis=1, keepdims=True))
            p = p / p.sum(axis=1, keepdims=True)
            id_energy.append(float(-_logsumexp_rows(logits).mean()))
            id_msp.append(float(p.max(axis=1).mean()))
        else:
            id_energy.append(np.nan)
            id_msp.append(np.nan)
    frame['id_energy_mean'] = id_energy
    frame['id_msp_mean'] = id_msp
    frame['energy_gap'] = frame['energy_mean'] - frame['id_energy_mean']
    frame['msp_gap'] = frame['id_msp_mean'] - frame['msp_mean']
    return frame

if bach_ood_all is not None and len(bach_ood_all):
    bach_ood_all = _add_id_reference(bach_ood_all)
    bach_ood = bach_ood_all[bach_ood_all.branch.isin(LOMO_BRANCHES)].reset_index(drop=True)
    print(f'  BACH energy converted to the standard sign (Liu et al. 2020); in-distribution '
          f'reference found for {int(bach_ood_all.energy_gap.notna().sum())}/{len(bach_ood_all)} runs')


In [ ]:
# ---------- Bootstrap 95% CI summary ----------
def bootstrap_ci(values, n_iter=BOOTSTRAP_ITERATIONS, alpha=0.05, seed=0):
    values = np.asarray(values, dtype=float); values = values[~np.isnan(values)]
    if len(values) == 0: return (np.nan, np.nan, np.nan)
    if len(values) == 1: return (values[0], values[0], values[0])
    rng = np.random.default_rng(seed)
    boot = np.stack([rng.choice(values, size=len(values), replace=True) for _ in range(n_iter)])
    m = boot.mean(axis=1)
    return float(np.mean(values)), float(np.percentile(m, 100*alpha/2)), float(np.percentile(m, 100*(1-alpha/2)))

METRICS = ['test_accuracy','test_balanced_accuracy','test_macro_f1','test_auc',
           'test_ece','test_brier','test_risk_coverage_auc',
           'test_risk_at_coverage_80','test_risk_at_coverage_90','test_risk_at_coverage_95']

summary_rows = []
for (branch, method, model, ho), sub in raw.groupby(['branch','method','model','held_out_mag']):
    row = {'branch':branch, 'method':method, 'model':model, 'held_out_mag':ho, 'n_seeds':len(sub)}
    for metric in METRICS:
        if metric in sub.columns:
            mean, lo, hi = bootstrap_ci(sub[metric].values)
            row[f'{metric}_mean'] = mean
            row[f'{metric}_ci_low'] = lo
            row[f'{metric}_ci_high'] = hi
            row[f'{metric}_sd'] = sub[metric].std() if len(sub) > 1 else 0.0
    summary_rows.append(row)
summary = pd.DataFrame(summary_rows).sort_values(['branch','method','held_out_mag']).reset_index(drop=True)
summary.to_csv(f'{OUTPUT_DIR}/summary_bootstrap.csv', index=False)
print(f'summary_bootstrap.csv: {len(summary)} rows')
summary[['branch','method','held_out_mag','n_seeds',
         'test_balanced_accuracy_mean','test_balanced_accuracy_ci_low','test_balanced_accuracy_ci_high',
         'test_ece_mean','test_macro_f1_mean']].round(4)

In [ ]:
# ---------- Color palette + method order + magnifications ----------
# Colorblind-safe palette (Wong 2011, Nature Methods)
COLORS = {
    'Phikon (frozen FM)': '#D55E00',   # vermillion
    'ConvNeXt-T': '#0072B2',            # blue
    'Swin-T': '#009E73',                # bluish green
    'MedMamba': '#CC79A7',              # reddish purple
    'DSA-Mamba': '#F0E442',             # yellow
    'MambaVision': '#56B4E9',           # sky blue
}
MARKERS = {
    'Phikon (frozen FM)': 'D',
    'ConvNeXt-T': 'o',
    'Swin-T': 's',
    'MedMamba': '^',
    'DSA-Mamba': 'v',
    'MambaVision': 'P',
}
ORDER = [m for m in ['Phikon (frozen FM)', 'ConvNeXt-T', 'Swin-T', 'MedMamba', 'DSA-Mamba', 'MambaVision']
         if m in raw.method.unique()]
MAGS = sorted(raw.held_out_mag.unique())
SEEDS = sorted(raw.seed.unique())

# Prefix mapping for locating per-run test_outputs.npz files.
PREFIX_MAP = {'Phikon (frozen FM)': 'fm_owkin_phikon',
              'ConvNeXt-T': 'scratch_convnext_tiny', 'Swin-T': 'scratch_swin_tiny',
              'MedMamba': 'scratch_medmamba', 'DSA-Mamba': 'scratch_dsamamba',
              'MambaVision': 'scratch_mambavision_tiny_1k'}

print(f'ORDER: {ORDER}')
print(f'MAGS:  {MAGS}')
print(f'SEEDS: {SEEDS}')


## 2. Figures

Each cell displays a figure inline and saves it to `analysis_outputs/figures/` at 150 DPI.

In [ ]:
# ---------- Fig 1: LOMO accuracy / macro-F1 / AUC by held-out magnification ----------
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2))
for ax, metric, title in zip(axes,
    ['test_balanced_accuracy', 'test_macro_f1', 'test_auc'],
    ['Balanced Accuracy', 'Macro-F1', 'AUC']):
    for method in ORDER:
        sub = summary[summary.method == method].sort_values('held_out_mag')
        if len(sub) == 0: continue
        m = sub[f'{metric}_mean']; lo = sub[f'{metric}_ci_low']; hi = sub[f'{metric}_ci_high']
        ax.plot(sub.held_out_mag, m, marker=MARKERS[method], color=COLORS[method],
                label=method, mec='black', mew=0.5)
        ax.fill_between(sub.held_out_mag, lo, hi, alpha=0.18, color=COLORS[method], linewidth=0)
    ax.set_xlabel('Held-out magnification'); ax.set_ylabel(title); ax.set_title(title, fontweight='bold')
    ax.set_xticks(MAGS); ax.set_ylim(0.4, 1.03)
    ax.grid(True, linewidth=0.4)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
axes[0].legend(loc='lower left', frameon=True, framealpha=0.95, edgecolor='0.7')
plt.suptitle('Cross-magnification LOMO on BreakHis  (mean $\\pm$ 95% bootstrap CI)',
             y=1.02, fontweight='bold')
plt.tight_layout()
save_fig(fig, 'fig1_accuracy_by_magnification')
plt.show()


In [ ]:
# ---------- Fig 2: Calibration comparison (ECE + Brier) ----------
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, metric, title, ylim in zip(axes,
    ['test_ece', 'test_brier'],
    ['Expected Calibration Error (lower is better)', 'Brier Score (lower is better)'],
    [(0, 0.20), (0, 0.30)]):
    piv = summary.pivot(index='method', columns='held_out_mag', values=f'{metric}_mean')
    piv = piv.reindex([m for m in ORDER if m in piv.index])
    piv.plot(kind='bar', ax=ax, width=0.78,
             color=['#4C72B0', '#55A868', '#C44E52', '#8172B3'],
             edgecolor='black', linewidth=0.5)
    ax.set_xlabel(''); ax.set_ylabel(title); ax.set_ylim(*ylim)
    ax.legend(title='held-out mag', frameon=True, framealpha=0.95, edgecolor='0.7')
    ax.set_xticklabels([t.get_text() for t in ax.get_xticklabels()], rotation=12, ha='right')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.suptitle('Calibration under magnification shift', y=1.02, fontweight='bold')
plt.tight_layout()
save_fig(fig, 'fig2_calibration')
plt.show()


In [ ]:
# ---------- Fig 3: accuracy against inference compute (FLOPs vs macro-F1) ----------
# Not compute-matched (FLOPs differ up to 8x), and not all trained from scratch: ConvNeXt-T and Swin-T
# start from ImageNet weights, MedMamba from random ones. Hence "task-trained" in the title.
FIG3_LABEL_POS = {'Swin-T': (16, 8, 'left'), 'ConvNeXt-T': (16, -16, 'left'),   # (dx, dy, ha) in points;
                  'MedMamba': (0, -30, 'center')}                               # keeps labels off markers
fig, ax = plt.subplots(figsize=(9.5, 6))
if compute_report is not None:
    compute_report['method'] = compute_report['model'].map(DISPLAY)
    fscratch = summary[summary.branch == 'from_scratch'][['method', 'model', 'test_macro_f1_mean']]
    fscratch_avg = fscratch.groupby(['method', 'model']).test_macro_f1_mean.mean().reset_index()
    merged = fscratch_avg.merge(compute_report[['model', 'parameters', 'flops_g_at_224']], on='model', how='left')
    for _, row in merged.iterrows():
        ax.scatter(row['flops_g_at_224'], row['test_macro_f1_mean'], s=380,
                   color=COLORS.get(row['method'], '#666'), edgecolor='black', linewidth=1.2,
                   label=row['method'], alpha=0.9, zorder=3)
        _dx, _dy, _ha = FIG3_LABEL_POS.get(row['method'], (16, 7, 'left'))
        ax.annotate(row['method'], (row['flops_g_at_224'], row['test_macro_f1_mean']),
                    xytext=(_dx, _dy), textcoords='offset points', fontsize=10, ha=_ha, va='center')
    best_scratch = merged.test_macro_f1_mean.max()
else:
    best_scratch = None

if 'Phikon (frozen FM)' in ORDER:
    phikon_score = summary[summary.method == 'Phikon (frozen FM)'].test_macro_f1_mean.mean()
    ax.scatter(17.6, phikon_score, s=650, color=COLORS['Phikon (frozen FM)'],
               edgecolor='black', linewidth=1.5, label='Phikon (frozen FM)', alpha=0.95, marker='D', zorder=4)
    ax.annotate(f'Phikon (frozen FM)\nmacro-F1 = {phikon_score:.3f}', (17.6, phikon_score),
                xytext=(15, -20), textcoords='offset points', fontsize=10, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='0.7', alpha=0.95))
    if best_scratch is not None:
        ax.axhline(phikon_score, color=COLORS['Phikon (frozen FM)'], linestyle='--', linewidth=1, alpha=0.4)
        ax.axhline(best_scratch, color='0.5', linestyle=':', linewidth=1, alpha=0.5)
        gap = phikon_score - best_scratch
        ax.text(0.5, (phikon_score + best_scratch) / 2, f'$\\Delta$ = {gap:+.3f}',
                fontsize=11, color='0.3', ha='left', va='center', style='italic')

ax.set_xlabel('Inference FLOPs @ 224$\\times$224 (GFLOPs)')
ax.set_ylabel('Macro-F1 (mean across mags and seeds)')
ax.set_title('Accuracy against inference compute: task-trained encoders vs a frozen pathology FM',
             fontweight='bold', pad=15)
ax.grid(True, linewidth=0.4); ax.set_ylim(0.5, 1.0); ax.set_xlim(0, 22)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
handles, labels = ax.get_legend_handles_labels()
ax.legend(dict(zip(labels, handles)).values(), dict(zip(labels, handles)).keys(),
          loc='lower right', frameon=True, framealpha=0.95, edgecolor='0.7', markerscale=0.35,
          labelspacing=1.0)
plt.tight_layout()
save_fig(fig, 'fig3_accuracy_vs_compute')
plt.show()


In [ ]:
# ---------- Fig 4: Method x magnification heatmap of macro-F1 ----------
piv = summary.pivot(index='method', columns='held_out_mag', values='test_macro_f1_mean')
piv = piv.reindex([m for m in ORDER if m in piv.index])
fig, ax = plt.subplots(figsize=(8, 3.5))
sns.heatmap(piv, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0.4, vmax=1.0,
            cbar_kws={'label': 'Macro-F1'}, ax=ax, linewidth=0.5, linecolor='white',
            annot_kws={'fontweight': 'bold', 'fontsize': 11})
ax.set_xlabel('Held-out magnification'); ax.set_ylabel('')
ax.set_title('Macro-F1 across methods and held-out magnifications', fontweight='bold', pad=10)
plt.tight_layout()
save_fig(fig, 'fig4_heatmap_macro_f1')
plt.show()


In [ ]:
# ---------- Fig 5: Risk-coverage curves per held-out magnification ----------
fig, axes = plt.subplots(1, len(MAGS), figsize=(4.5 * len(MAGS), 4.2), squeeze=False, sharey=True)
axes = axes[0]
for ax, ho in zip(axes, MAGS):
    for method_name in ORDER:
        prefix = PREFIX_MAP.get(method_name)
        if not prefix: continue
        all_risks = []
        for seed in SEEDS:
            npz_path = f'runs/{prefix}_ho{ho}_frac1.0_seed{seed}/test_outputs.npz'
            if not reader.exists(npz_path): continue
            data = np.load(io.BytesIO(reader.open_bytes(npz_path)))
            probs, labels = data['probs'], data['labels']
            confs = probs.max(axis=1); preds = probs.argmax(axis=1)
            order = np.argsort(-confs); errs = (preds != labels).astype(float)[order]
            risk = np.cumsum(errs) / np.arange(1, len(order) + 1)
            all_risks.append(risk)
        if not all_risks: continue
        min_len = min(len(r) for r in all_risks)
        trimmed = np.stack([r[:min_len] for r in all_risks])
        cov_axis = np.linspace(1 / min_len, 1, min_len)
        mean_r = trimmed.mean(0); std_r = trimmed.std(0)
        ax.plot(cov_axis, mean_r, label=method_name, color=COLORS[method_name],
                marker=MARKERS[method_name], markersize=5,
                markevery=max(1, min_len // 15), mec='black', mew=0.4)
        ax.fill_between(cov_axis, mean_r - std_r, mean_r + std_r,
                        alpha=0.15, color=COLORS[method_name], linewidth=0)
    ax.set_xlabel('Coverage'); ax.set_title(f'Held-out $= {ho}\\times$', fontweight='bold')
    ax.set_xlim(0, 1); ax.set_ylim(0, 0.5); ax.grid(True, linewidth=0.4)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
axes[0].set_ylabel('Cumulative selective risk')
axes[0].legend(loc='upper left', frameon=True, framealpha=0.95, edgecolor='0.7')
plt.suptitle('Risk-Coverage curves: how selective prediction reduces risk',
             y=1.02, fontweight='bold')
plt.tight_layout()
save_fig(fig, 'fig5_risk_coverage')
plt.show()


In [ ]:
# ---------- Fig 6: BACH as OOD probe ----------
if bach_ood is None or len(bach_ood) == 0:
    print('BACH OOD scores not available; skipping Fig 6.')
else:
    bach_ood['method'] = bach_ood['model'].map(DISPLAY).fillna(bach_ood['model'])
    _f6 = [('msp_mean', 'Mean MSP on BACH', 'MSP (higher = more confident)'),
           ('energy_mean', 'Mean energy on BACH', 'Energy (higher = less confident)')]
    if 'energy_gap' in bach_ood.columns and bach_ood['energy_gap'].notna().any():
        _f6.append(('energy_gap', 'Energy gap: BACH minus own test set', 'Energy gap (higher = shift detected)'))
    fig, axes = plt.subplots(1, len(_f6), figsize=(5.5 * len(_f6), 4.2), squeeze=False)
    for ax, (metric, ylabel, title) in zip(axes[0], _f6):
        agg = bach_ood.groupby(['method', 'held_out_mag'])[metric].mean().reset_index()
        piv = agg.pivot(index='method', columns='held_out_mag', values=metric)
        piv = piv.reindex([m for m in ORDER if m in piv.index])
        piv.plot(kind='bar', ax=ax, width=0.78, edgecolor='black', linewidth=0.5,
                 color=['#4C72B0', '#55A868', '#C44E52', '#8172B3'])
        ax.set_ylabel(ylabel)
        ax.set_title(title, fontweight='bold')
        ax.set_xticklabels([t.get_text() for t in ax.get_xticklabels()], rotation=12, ha='right')
        ax.legend(title='held-out mag', frameon=True, framealpha=0.95, edgecolor='0.7')
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.suptitle('BACH as an out-of-distribution probe', y=1.02, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, 'fig6_bach_ood')
    plt.show()


In [ ]:
# ---------- Fig 7: Reliability diagrams grid ----------
N_BINS = 10
fig, axes = plt.subplots(len(MAGS), len(ORDER), figsize=(3.0 * len(ORDER), 2.7 * len(MAGS)),
                          squeeze=False, sharex=True, sharey=True)
for row_idx, ho in enumerate(MAGS):
    for col_idx, method_name in enumerate(ORDER):
        ax = axes[row_idx, col_idx]
        prefix = PREFIX_MAP.get(method_name)
        if not prefix: ax.axis('off'); continue
        all_confs, all_correct = [], []
        for seed in SEEDS:
            npz_path = f'runs/{prefix}_ho{ho}_frac1.0_seed{seed}/test_outputs.npz'
            if not reader.exists(npz_path): continue
            data = np.load(io.BytesIO(reader.open_bytes(npz_path)))
            probs, labels = data['probs'], data['labels']
            confs = probs.max(axis=1); preds = probs.argmax(axis=1)
            all_confs.append(confs); all_correct.append((preds == labels).astype(float))
        if not all_confs: ax.axis('off'); continue
        confs = np.concatenate(all_confs); correct = np.concatenate(all_correct)
        bins = np.linspace(0, 1, N_BINS + 1); bin_conf, bin_acc, bin_count = [], [], []
        for lo, hi in zip(bins[:-1], bins[1:]):
            mask = (confs > lo) & (confs <= hi)
            if mask.sum() == 0: continue
            bin_conf.append(confs[mask].mean()); bin_acc.append(correct[mask].mean()); bin_count.append(mask.sum())
        ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.6)
        if bin_conf:
            sizes = np.array(bin_count) / max(bin_count) * 200 + 25
            ax.scatter(bin_conf, bin_acc, s=sizes, color=COLORS[method_name],
                       edgecolor='black', linewidth=0.6, alpha=0.85, zorder=3)
            ax.plot(bin_conf, bin_acc, color=COLORS[method_name], linewidth=1.5, alpha=0.7)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.grid(True, linewidth=0.4)
        if col_idx == 0: ax.set_ylabel(f'ho $= {ho}\\times$\naccuracy')
        if row_idx == len(MAGS) - 1: ax.set_xlabel('predicted confidence')
        if row_idx == 0: ax.set_title(method_name, fontsize=10, fontweight='bold')
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.suptitle('Reliability diagrams:  perfectly calibrated $\\Rightarrow$ points on $y=x$',
             y=1.005, fontweight='bold')
plt.tight_layout()
save_fig(fig, 'fig7_reliability_diagrams')
plt.show()


In [ ]:
# ---------- Fig 8: Temperature scaling effect ----------
temp_data = raw[raw.test_temperature.notna()].groupby(['method', 'held_out_mag'])['test_temperature'].mean().reset_index()
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
piv_temp = temp_data.pivot(index='method', columns='held_out_mag', values='test_temperature')
piv_temp = piv_temp.reindex([m for m in ORDER if m in piv_temp.index])
piv_temp.plot(kind='bar', ax=axes[0], width=0.78, edgecolor='black', linewidth=0.5,
              color=['#4C72B0', '#55A868', '#C44E52', '#8172B3'])
axes[0].axhline(1.0, color='red', linestyle='--', linewidth=1, alpha=0.7, label='$T=1$ (no scaling)')
axes[0].set_ylabel('Learned temperature $T$')
axes[0].set_title('Temperature scaling per method', fontweight='bold')
axes[0].set_xticklabels([t.get_text() for t in axes[0].get_xticklabels()], rotation=12, ha='right')
axes[0].legend(title='held-out mag', frameon=True, framealpha=0.95, edgecolor='0.7')
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

ece_mean = summary.groupby('method').test_ece_mean.mean().reindex([m for m in ORDER if m in summary.method.unique()])
ece_ranked = ece_mean.sort_values()
colors_bar = [COLORS.get(m, '#666') for m in ece_ranked.index]
ece_ranked.plot(kind='bar', ax=axes[1], color=colors_bar, edgecolor='black', linewidth=0.5, width=0.78)
axes[1].set_ylabel('Mean ECE (post-temperature scaling)')
axes[1].set_title('Calibration ranking (lower is better)', fontweight='bold')
axes[1].set_xticklabels([t.get_text() for t in axes[1].get_xticklabels()], rotation=12, ha='right')
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)
plt.suptitle('Temperature scaling parameter and post-scaling calibration',
             y=1.02, fontweight='bold')
plt.tight_layout()
save_fig(fig, 'fig8_temperature_effect')
plt.show()


## 2b. Image-based visualizations (Fig 9–11)

**Prerequisites for Fig 10 and Fig 11:**
- Attach `ambarish/breakhis` dataset to the analysis notebook (right sidebar > Add Input)
- For Grad-CAM (Fig 11): switch to **GPU T4** accelerator
- MedMamba Grad-CAM also needs `mamba_ssm` (auto-install attempted)

If BreakHis is not attached, Fig 10 and Fig 11 print a clear message and skip cleanly. Fig 9 (confusion matrices) needs no images — always runs.

In [ ]:
# ---------- Image-based prerequisites: locate BreakHis + rebuild the exact training test set ----------
BREAKHIS_ROOT = None

def _discover_breakhis(search_root='/kaggle/input'):
    if not Path(search_root).exists():
        print(f'  {search_root} does not exist'); return None
    best = None; best_count = 0
    for top in sorted(Path(search_root).iterdir()):
        if not top.is_dir(): continue
        try: count = sum(1 for _ in Path(top).rglob('SOB_*.png'))
        except (PermissionError, OSError): continue
        print(f'  scanned {top.name}: {count} SOB files')
        if count > best_count: best = top; best_count = count
    if best_count == 0:
        print('  No SOB_*.png files found anywhere under', search_root); return None
    print(f'Chose BreakHis root: {best}  ({best_count} images)')
    return str(best)

BREAKHIS_ROOT = _discover_breakhis()
HAS_IMAGES = BREAKHIS_ROOT is not None
if not HAS_IMAGES:
    print('BreakHis NOT attached to this notebook. Fig 10 and Fig 11 will be skipped.')
    print('To enable them: Add Input > ambarish/breakhis, then re-run this cell.')

# The image cells must use the test set the stored predictions (test_outputs.npz) were computed on.
# It is rebuilt from the breakhis_manifest.csv the training notebooks saved, with their
# patient_lomo_split() copied verbatim; rows keep manifest order, which is the order of the
# shuffle=False test loader. Paths are re-rooted by file name onto this session's BreakHis mount.
# An earlier version re-scanned the images with the pre-fix filename pattern (dropping alphanumeric
# slide ids) and a simplified split, so its "test" images were not the evaluated test set.
_TRAIN_MAGNIFICATIONS = [40, 100, 200, 400]
_MANIFEST_CACHE = {}

def patient_lomo_split(frame, held_out_mag, fraction=1.0, seed=2025,
                       val_ratio=0.15, test_ratio=0.20, val_on_held_out=False):
    """Patient-disjoint LOMO split with PROPER per-class stratification.
    Both benign and malignant patients are proportionally represented in every split
    (train, val, test) to prevent single-class val/test collapse.
    """
    frame = frame.copy()
    patient_class = frame.groupby('patient_id').coarse_label.first().reset_index()
    rng = np.random.default_rng(seed)
    train_patients, val_patients, test_patients = set(), set(), set()
    for cls in sorted(patient_class.coarse_label.unique()):
        group = patient_class.loc[patient_class.coarse_label == cls, 'patient_id'].to_numpy()
        perm = rng.permutation(group)
        n = len(perm)
        n_test = max(1, int(round(test_ratio * n)))
        n_val = max(1, int(round(val_ratio * n)))
        assert n_test + n_val < n, 'Class ' + str(cls) + ' has only ' + str(n) + ' patients; val+test ratios too high.'
        test_patients.update(perm[:n_test].tolist())
        val_patients.update(perm[n_test:n_test + n_val].tolist())
        remain = perm[n_test + n_val:].tolist()
        n_keep = max(1, int(round(fraction * len(remain))))
        train_patients.update(remain[:n_keep])
    train_mags = [m for m in _TRAIN_MAGNIFICATIONS if m != held_out_mag]
    frame['split'] = 'drop'
    frame.loc[frame.patient_id.isin(train_patients) & frame.magnification.isin(train_mags), 'split'] = 'train'
    if val_on_held_out:
        frame.loc[frame.patient_id.isin(val_patients) & (frame.magnification == held_out_mag), 'split'] = 'val'
    else:
        frame.loc[frame.patient_id.isin(val_patients) & frame.magnification.isin(train_mags), 'split'] = 'val'
    frame.loc[frame.patient_id.isin(test_patients) & (frame.magnification == held_out_mag), 'split'] = 'test'
    subset = frame.loc[frame.split != 'drop'].copy()
    leak = subset.groupby('patient_id').split.nunique().gt(1).any()
    assert not leak, 'Patient leakage detected.'
    for name in ('train', 'val', 'test'):
        rows = subset[subset.split == name]
        if len(rows) == 0:
            raise RuntimeError('Split ' + name + ' empty (held_out=' + str(held_out_mag) + ', fraction=' + str(fraction) + ', seed=' + str(seed) + ').')
        counts = rows.coarse_label.value_counts().to_dict()
        if len(counts) < 2:
            raise RuntimeError('Split ' + name + ' has only one class ' + str(counts) + ' -- stratification failed.')
    return subset

def _training_manifest():
    if 'frame' not in _MANIFEST_CACHE:
        if not ('reader' in globals() and reader.exists('breakhis_manifest.csv')):
            print('  breakhis_manifest.csv not found in the attached training outputs.')
            return None
        frame = pd.read_csv(io.BytesIO(reader.open_bytes('breakhis_manifest.csv')))
        local = {p.name: str(p) for p in Path(BREAKHIS_ROOT).rglob('SOB_*.png')}
        frame['path'] = [local.get(Path(str(p)).name, str(p)) for p in frame['path']]
        missing = int(sum(Path(str(p)).name not in local for p in frame['path']))
        print(f'  training manifest: {len(frame)} images, {frame.patient_id.nunique()} patients'
              + (f'; WARNING {missing} files not found under BREAKHIS_ROOT' if missing else ''))
        _MANIFEST_CACHE['frame'] = frame
    return _MANIFEST_CACHE['frame']

def reconstruct_test_paths(held_out_mag, seed, val_ratio=0.15, test_ratio=0.20):
    """Test rows for (held_out_mag, seed) in the order the training test loader produced them."""
    if not HAS_IMAGES:
        return None
    frame = _training_manifest()
    if frame is None:
        return None
    subset = patient_lomo_split(frame, int(held_out_mag), fraction=1.0, seed=int(seed),
                                val_ratio=val_ratio, test_ratio=test_ratio)
    return subset[subset.split == 'test'].reset_index(drop=True)

print('Image-based prerequisites ready.')


In [ ]:
# ---------- Fig 9: Confusion matrices per method (no BreakHis attach needed) ----------
CLASS_NAMES = ['Benign', 'Malignant']
USE_HO = MAGS[-1] if MAGS else 400

fig, axes = plt.subplots(1, len(ORDER), figsize=(3.2 * len(ORDER), 3.4), squeeze=False)
axes = axes[0]
for ax, method_name in zip(axes, ORDER):
    prefix = PREFIX_MAP.get(method_name)
    if not prefix: ax.axis('off'); continue
    all_labels, all_preds = [], []
    for seed in SEEDS:
        npz_path = f'runs/{prefix}_ho{USE_HO}_frac1.0_seed{seed}/test_outputs.npz'
        if not reader.exists(npz_path): continue
        d = np.load(io.BytesIO(reader.open_bytes(npz_path)))
        all_labels.append(d['labels']); all_preds.append(d['probs'].argmax(axis=1))
    if not all_labels: ax.axis('off'); continue
    y_true = np.concatenate(all_labels); y_pred = np.concatenate(all_preds)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    annot = [[f'{cm[i,j]}\n({cm_norm[i,j]:.2f})' for j in range(2)] for i in range(2)]
    sns.heatmap(cm_norm, annot=annot, fmt='', cmap='Blues', vmin=0, vmax=1, cbar=False,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax,
                linewidth=0.5, linecolor='white',
                annot_kws={'fontweight': 'bold', 'fontsize': 10})
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'{method_name}\nho $= {USE_HO}\\times$, $n = {len(y_true)}$', fontsize=10, fontweight='bold')
plt.suptitle(f'Confusion matrices at held-out magnification $= {USE_HO}\\times$ (row-normalized)',
             y=1.03, fontweight='bold')
plt.tight_layout()
save_fig(fig, 'fig9_confusion_matrices')
plt.show()


In [ ]:
# ---------- Fig 10: Sample predictions with images (TP/TN/FP/FN quadrants per method) ----------
from PIL import Image as PILImage
if not HAS_IMAGES:
    print('Skipped Fig 10 — BreakHis not attached.')
else:
    USE_SEED = SEEDS[0]
    test_paths_frame = reconstruct_test_paths(USE_HO, USE_SEED)
    if test_paths_frame is None or len(test_paths_frame) == 0:
        print('Skipped Fig 10 — reconstruct_test_paths returned empty.')
    else:
        print(f'Reconstructed test set: {len(test_paths_frame)} images at ho={USE_HO}x seed={USE_SEED}')

    fig, axes = plt.subplots(len(ORDER), 4, figsize=(14, 3.2*len(ORDER)), squeeze=False)
    quadrant_labels = ['True Positive', 'True Negative', 'False Positive', 'False Negative']
    for row_idx, method_name in enumerate(ORDER):
        prefix = PREFIX_MAP.get(method_name)
        if not prefix:
            for ax in axes[row_idx]: ax.axis('off'); continue
        npz_path = f'runs/{prefix}_ho{USE_HO}_frac1.0_seed{USE_SEED}/test_outputs.npz'
        if not reader.exists(npz_path):
            for ax in axes[row_idx]: ax.axis('off')
            axes[row_idx, 0].set_title(f'{method_name} — no data')
            continue
        d = np.load(io.BytesIO(reader.open_bytes(npz_path)))
        probs, labels = d['probs'], d['labels']
        preds = probs.argmax(axis=1); confs = probs.max(axis=1)
        if len(labels) != len(test_paths_frame):
            print(f'  {method_name}: WARNING - {len(labels)} preds vs {len(test_paths_frame)} paths (proceeding on shortest)')
            common = min(len(labels), len(test_paths_frame))
            labels = labels[:common]; preds = preds[:common]; confs = confs[:common]
            _paths = test_paths_frame.iloc[:common]
        else:
            _paths = test_paths_frame
        # Verify the positional pairing actually holds before trusting it: the stored
        # labels must match the reconstructed frame's labels element-wise. Equal lengths
        # alone do not prove equal ORDER, and a mismatch would put the wrong image under
        # a TP/TN/FP/FN caption.
        if not np.array_equal(labels, _paths.coarse_label.to_numpy()):
            _n_mis = int((labels != _paths.coarse_label.to_numpy()).sum())
            for ax in axes[row_idx]: ax.axis('off')
            axes[row_idx, 0].set_title(
                f'{method_name} - row order mismatch, skipped', fontsize=9, color='#d62728')
            print(f'  {method_name}: SKIPPED - stored labels disagree with reconstructed '
                  f'path order at {_n_mis}/{len(labels)} positions.')
            print(f'    The rebuilt test set does not match the stored predictions; check that the '
                  f'attached LOMO outputs and their breakhis_manifest.csv come from the same runs.')
            continue
        quadrants = {
            'TP': (labels == 1) & (preds == 1),
            'TN': (labels == 0) & (preds == 0),
            'FP': (labels == 0) & (preds == 1),
            'FN': (labels == 1) & (preds == 0),
        }
        for col_idx, (qname, mask) in enumerate(quadrants.items()):
            ax = axes[row_idx, col_idx]
            if mask.sum() == 0:
                ax.text(0.5, 0.5, f'{qname}\
(no examples)', ha='center', va='center', transform=ax.transAxes)
                ax.axis('off'); continue
            idxs = np.where(mask)[0]
            best_idx = idxs[np.argmax(confs[idxs])]
            img_path = _paths.iloc[best_idx]['path']
            try:
                img = PILImage.open(img_path).convert('RGB')
                ax.imshow(img)
            except Exception as exc:
                ax.text(0.5, 0.5, f'load failed: {exc}', ha='center', va='center', transform=ax.transAxes)
            ax.set_xticks([]); ax.set_yticks([])
            true_name = CLASS_NAMES[labels[best_idx]]; pred_name = CLASS_NAMES[preds[best_idx]]
            ax.set_title(f'{quadrant_labels[col_idx]}\
true={true_name}, pred={pred_name}, conf={confs[best_idx]:.2f}', fontsize=9)
        axes[row_idx, 0].set_ylabel(method_name, fontsize=11, rotation=90, labelpad=10)
    plt.suptitle(f'Sample predictions per quadrant (held-out mag={USE_HO}x, seed={USE_SEED})', fontsize=13, y=1.005)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/figures/fig10_sample_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ---------- Fig 11: Publication-quality Grad-CAM++ comparison ----------
# Rows = samples (2 benign + 2 malignant); Cols = [Input | Grad-CAM++ per method].
# Green border = correct prediction, red = wrong. Higher fonts, PDF+PNG at 200 DPI.
GRADCAM_HELD_OUT  = MAGS[-1] if MAGS else 400
GRADCAM_SEED      = SEEDS[0] if SEEDS else 13
GRADCAM_N_SAMPLES = 4   # 2 benign + 2 malignant

if not HAS_IMAGES:
    print('Skipped Fig 11 (Grad-CAM++) - BreakHis not attached.')
else:
    import subprocess, torch
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if not torch.cuda.is_available():
        print('Warning: GPU not available. Grad-CAM++ will run on CPU (slow, ~5-10 min).')

    try:
        from pytorch_grad_cam import GradCAMPlusPlus
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'grad-cam', 'timm'], check=False)
        from pytorch_grad_cam import GradCAMPlusPlus
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    import timm
    from torchvision import transforms as _tf

    MAMBA_OK = False
    try: import mamba_ssm; MAMBA_OK = True
    except ImportError:
        for spec in [['mamba-ssm','causal-conv1d'], ['mamba-ssm==2.2.4','causal-conv1d==1.5.0.post8']]:
            try:
                subprocess.run([sys.executable,'-m','pip','install','-q','--no-build-isolation']+spec, check=True)
                import mamba_ssm; MAMBA_OK = True; break
            except Exception: continue

    IMAGE_TRANSFORM = _tf.Compose([_tf.Resize((224, 224)), _tf.ToTensor(),
                                    _tf.Normalize((.485,.456,.406),(.229,.224,.225))])

    def build_from_scratch(name, num_classes=2):
        if name == 'convnext_tiny':
            return timm.create_model('convnext_tiny', pretrained=False, num_classes=num_classes)
        if name == 'swin_tiny':
            return timm.create_model('swin_tiny_patch4_window7_224', pretrained=False, num_classes=num_classes)
        if name == 'mambavision_tiny_1k':
            return timm.create_model('mambavision_tiny_1k', pretrained=False, num_classes=num_classes)
        if name == 'medmamba' and MAMBA_OK:
            src = Path('/kaggle/working/gradcam_medmamba')
            if not src.exists():
                subprocess.run(['git','clone','https://github.com/YubiaoYue/MedMamba.git', str(src)], check=True)
                subprocess.run(['git','-C',str(src),'checkout','--detach','104beb3d641d67cef6f09b94ac299083d91bc417'], check=True)
            source = Path(src, 'MedMamba.py').read_text().split('\nmedmamba_t = VSSM', 1)[0]
            safe = Path('/kaggle/working/medmamba_safe.py'); safe.write_text(source)
            import importlib.util
            spec = importlib.util.spec_from_file_location('medmamba_safe', safe)
            mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
            return mod.VSSM(depths=[2,2,4,2], dims=[96,192,384,768], num_classes=num_classes)
        if name == 'dsamamba' and MAMBA_OK:
            # Rebuild the exact DSAmamba.py revision the checkpoints were trained with, when
            # the training run recorded it, and restore the cross_attention.py that the
            # upstream repo no longer ships (author's commit 94c5042).
            src = Path('/kaggle/working/gradcam_dsamamba')
            commit = '7af166288e5e0b710481788c15af79326017b37b'
            try:
                if reader.exists('source_manifest.json'):
                    commit = (json.loads(reader.open_text('source_manifest.json'))
                              .get('DSA_Mamba_commit') or commit).strip()
            except Exception:
                pass
            if not src.exists():
                subprocess.run(['git', 'clone', 'https://github.com/First-Ronin/DSA-Mamba.git', str(src)], check=True)
            subprocess.run(['git', '-C', str(src), 'checkout', '--detach', commit], check=True)
            ca = Path(src, 'model', 'cross_attention.py')
            if not ca.exists():
                import urllib.request
                with urllib.request.urlopen('https://raw.githubusercontent.com/First-Ronin/DSA-Mamba/'
                                            '94c5042/model/cross_attention.py', timeout=60) as resp:
                    ca.write_bytes(resp.read())
            if str(src) not in sys.path:
                sys.path.insert(0, str(src))
            for _m in [m for m in list(sys.modules) if m == 'model' or m.startswith('model.')]:
                del sys.modules[_m]
            from model.DSAmamba import VSSM as _DSAMambaVSSM
            return _DSAMambaVSSM(in_chans=3, num_classes=num_classes)
        raise KeyError(name)

    def get_target_layer(model, name):
        if name == 'convnext_tiny': return [model.stages[-1].blocks[-1].conv_dw]
        if name == 'swin_tiny':     return [model.layers[-1].blocks[-1].norm1]
        if name == 'mambavision_tiny_1k':
            try:    return [model.levels[-1].blocks[-1]]
            except AttributeError: return [model.stages[-1].blocks[-1]]
        if name == 'medmamba': return [model.layers[-1]]
        # DSA-Mamba is encoder-decoder; the last decoder stage feeds the pooled head.
        if name == 'dsamamba': return [model.Decoder_layers[-1]]
        raise KeyError(name)

    def to_nchw(tensor, height=None, width=None):
        """Return target-layer activations as (B, C, H, W), the layout pytorch_grad_cam assumes.

        The encoders disagree: ConvNeXt emits (B, C, H, W); DSA-Mamba, MedMamba and Swin in
        timm >= 0.9 emit (B, H, W, C); older Swin emitted tokens (B, N, C). Passing
        (B, H, W, C) through unchanged makes Grad-CAM treat a spatial axis as the channel
        axis and yields a spatially meaningless map. The layout is read from the shape:
        on a square feature map the two spatial sizes are equal and differ from C.
        """
        if tensor.dim() == 3:
            b, n, c = tensor.shape
            h = w = int(round(n ** 0.5))
            return tensor.reshape(b, h, w, c).permute(0, 3, 1, 2)
        if tensor.dim() == 4 and tensor.shape[1] == tensor.shape[2] and tensor.shape[3] != tensor.shape[1]:
            return tensor.permute(0, 3, 1, 2)
        return tensor

    # Old name kept: other cells reference it.
    swin_reshape_transform = to_nchw

    test_paths_frame = reconstruct_test_paths(GRADCAM_HELD_OUT, GRADCAM_SEED)
    if test_paths_frame is None or len(test_paths_frame) == 0:
        print('Skipped Fig 11 - reconstruct_test_paths returned empty.')
    else:
        # Balanced sample: 2 benign + 2 malignant
        sample_indices = []
        for label in [0, 1]:
            idxs = test_paths_frame.index[test_paths_frame.coarse_label == label].tolist()
            if idxs: sample_indices.extend(idxs[:GRADCAM_N_SAMPLES // 2])
        sample_indices = sample_indices[:GRADCAM_N_SAMPLES]

        cam_methods = [m for m in ORDER if m != 'Phikon (frozen FM)']
        n_rows, n_cols = len(sample_indices), len(cam_methods) + 1

        # ---- Precompute overlays one model at a time (GPU-memory friendly) ----
        reverse_display = {v: k for k, v in DISPLAY.items()}
        cam_results = {m: [None] * n_rows for m in cam_methods}

        for method_name in cam_methods:
            model_name = reverse_display.get(method_name)
            prefix     = PREFIX_MAP.get(method_name)
            ckpt_path  = f'runs/{prefix}_ho{GRADCAM_HELD_OUT}_frac1.0_seed{GRADCAM_SEED}/best.pt'
            if not reader.exists(ckpt_path):
                print(f'  {method_name}: checkpoint missing - column will be blank'); continue
            try:
                model = build_from_scratch(model_name)
                sd = torch.load(io.BytesIO(reader.open_bytes(ckpt_path)), map_location=device)
                model.load_state_dict(sd); model = model.to(device).eval()
                target_layers = get_target_layer(model, model_name)
                reshape_fn    = to_nchw   # every model: layout is detected from the shape
                cam = GradCAMPlusPlus(model=model, target_layers=target_layers, reshape_transform=reshape_fn)
                for r, idx in enumerate(sample_indices):
                    img_path = test_paths_frame.iloc[idx]['path']
                    img = PILImage.open(img_path).convert('RGB').resize((224, 224))
                    img_np = np.asarray(img, dtype=np.float32) / 255.0
                    tensor = IMAGE_TRANSFORM(img).unsqueeze(0).to(device)
                    with torch.no_grad():
                        probs = torch.softmax(model(tensor), dim=1)
                        pred_class = int(probs.argmax(1).item())
                        pred_conf  = float(probs[0, pred_class].item())
                    gcam = cam(input_tensor=tensor, targets=[ClassifierOutputTarget(pred_class)])[0]
                    overlay = show_cam_on_image(img_np, gcam, use_rgb=True)
                    cam_results[method_name][r] = (overlay, pred_class, pred_conf)
                del cam, model
                if torch.cuda.is_available(): torch.cuda.empty_cache()
                print(f'  {method_name}: Grad-CAM++ done')
            except Exception as e:
                print(f'  {method_name}: Grad-CAM++ FAILED - {e}')

        # ---- Render the publication-quality grid ----
        fig, axes = plt.subplots(n_rows, n_cols,
                                  figsize=(2.6 * n_cols, 2.8 * n_rows),
                                  squeeze=False,
                                  gridspec_kw={'wspace': 0.06, 'hspace': 0.32})

        GREEN, RED = '#2ca02c', '#d62728'
        for r, idx in enumerate(sample_indices):
            row = test_paths_frame.iloc[idx]
            true_label = int(row['coarse_label']); true_name = CLASS_NAMES[true_label]
            img = PILImage.open(row['path']).convert('RGB').resize((224, 224))

            # Col 0: Input image
            ax = axes[r, 0]
            ax.imshow(img); ax.set_xticks([]); ax.set_yticks([])
            for s in ax.spines.values():
                s.set_visible(True); s.set_linewidth(1.5); s.set_edgecolor('0.3')
            if r == 0:
                ax.set_title('Input Image', fontsize=12, fontweight='bold', pad=10)
            ax.set_ylabel(f'True: {true_name}', fontsize=11, fontweight='bold',
                          rotation=90, labelpad=14)

            # Cols 1..N: Grad-CAM++ overlays per method
            for c, method_name in enumerate(cam_methods, start=1):
                ax = axes[r, c]
                if r == 0:
                    ax.set_title(f'{method_name}\nGrad-CAM++', fontsize=11, fontweight='bold', pad=10)
                result = cam_results[method_name][r]
                if result is None:
                    ax.imshow(np.ones((224, 224, 3), dtype=np.uint8) * 240)
                    ax.text(0.5, 0.5, 'no checkpoint', ha='center', va='center',
                            transform=ax.transAxes, fontsize=9, color='0.5')
                    ax.set_xticks([]); ax.set_yticks([])
                    continue
                overlay, pred_class, pred_conf = result
                ax.imshow(overlay); ax.set_xticks([]); ax.set_yticks([])
                correct = (pred_class == true_label)
                bcol = GREEN if correct else RED
                for s in ax.spines.values():
                    s.set_visible(True); s.set_linewidth(2.4); s.set_edgecolor(bcol)
                mark = '[correct]' if correct else '[wrong]'
                ax.set_xlabel(f'{mark} pred: {CLASS_NAMES[pred_class]}  ({pred_conf:.2f})',
                              fontsize=10, color=bcol, fontweight='bold', labelpad=4)

        plt.suptitle(
            f'Grad-CAM++ class-discriminative regions per encoder\n'
            f'(BreakHis, held-out magnification = {GRADCAM_HELD_OUT}$\\times$, seed = {GRADCAM_SEED})',
            fontsize=13, fontweight='bold', y=1.005)
        plt.savefig(f'{OUTPUT_DIR}/figures/fig11_gradcam_plusplus.png', dpi=200, bbox_inches='tight')
        plt.savefig(f'{OUTPUT_DIR}/figures/fig11_gradcam_plusplus.pdf', bbox_inches='tight')
        plt.show()


## 2c. Advanced XAI: Attention Rollout & Feature-space UMAP

- **Fig 11b** — Attention Rollout on frozen Phikon (Grad-CAM is unreliable on ViT).
- **Fig 12** — UMAP of each encoder's penultimate features on the held-out test set.
- **T8** — Grad-CAM++ faithfulness via Deletion / Insertion AUC (produced in section 3).

In [ ]:
# ---------- Fig 11b: Attention Rollout for frozen Phikon (Grad-CAM alternative) ----------
# Grad-CAM is unreliable on ViT patch tokens; Attention Rollout (Abnar & Zuidema, 2020)
# is the standard XAI for transformer FMs. Fills the Phikon gap left by Fig 11.
#
# NOTE: the FM branch of Part A/B (`train_fm_linear_probe`) saves only test_outputs.npz
# and record.json -- it never writes a best.pt. That is fine here: the rollout depends
# ONLY on the frozen backbone's attention, not on the classifier head. Predictions are
# read back from test_outputs.npz, and the row ordering is verified against the
# reconstructed frame's labels before any prediction text is drawn.
ROLL_HELD_OUT  = MAGS[-1] if MAGS else 400
ROLL_SEED      = SEEDS[0] if SEEDS else 13
ROLL_N_SAMPLES = 4

if 'Phikon (frozen FM)' not in ORDER:
    print('Skipped Fig 11b - Phikon not present in ORDER.')
elif not HAS_IMAGES:
    print('Skipped Fig 11b - BreakHis not attached.')
else:
    try:
        import torch, cv2
        from transformers import AutoModel
        from torchvision import transforms as _tf
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        test_paths_frame = reconstruct_test_paths(ROLL_HELD_OUT, ROLL_SEED)
        if test_paths_frame is None or len(test_paths_frame) == 0:
            raise RuntimeError('reconstruct_test_paths returned empty')

        # ---- Try to recover stored predictions; verify ordering before trusting them ----
        preds = confs = None
        npz_path = f'runs/fm_owkin_phikon_ho{ROLL_HELD_OUT}_frac1.0_seed{ROLL_SEED}/test_outputs.npz'
        if reader.exists(npz_path):
            d = np.load(io.BytesIO(reader.open_bytes(npz_path)))
            npz_labels = d['labels']
            frame_labels = test_paths_frame.coarse_label.to_numpy()
            if len(npz_labels) == len(frame_labels) and np.array_equal(npz_labels, frame_labels):
                preds = d['probs'].argmax(axis=1)
                confs = d['probs'].max(axis=1)
                print(f'  Predictions recovered from {npz_path}')
                print(f'  Row ordering verified: {len(npz_labels)} labels match position-by-position.')
            else:
                print(f'  Stored predictions NOT usable: '
                      f'{len(npz_labels)} npz rows vs {len(frame_labels)} reconstructed rows, '
                      f'labels aligned={len(npz_labels) == len(frame_labels) and np.array_equal(npz_labels, frame_labels)}.')
                print('  -> rendering rollout WITHOUT prediction labels (rollout itself is unaffected).')
        else:
            print(f'  {npz_path} not found -> rendering rollout without prediction labels.')

        # ---- Frozen backbone only; no classifier head needed for rollout ----
        # SDPA attention (the default in recent transformers) returns no attention weights, which
        # left out.attentions = None; the eager implementation returns them.
        try:
            fm = AutoModel.from_pretrained('owkin/phikon', add_pooling_layer=False,
                                           attn_implementation='eager').to(device).eval()
        except TypeError:
            fm = AutoModel.from_pretrained('owkin/phikon', add_pooling_layer=False).to(device).eval()
        for p in fm.parameters():
            p.requires_grad_(False)

        TF = _tf.Compose([_tf.Resize((224, 224)), _tf.ToTensor(),
                          _tf.Normalize((.485, .456, .406), (.229, .224, .225))])

        def attention_rollout(attentions):
            """Rollout = product of (0.5*A + 0.5*I) across layers, accounting for residuals.
            attentions: tuple of L tensors (1, H, N, N) with N = 1 (CLS) + 196 patches."""
            result = None
            for A in attentions:
                A = A.mean(dim=1)[0]                                  # average heads -> (N, N)
                A = 0.5 * A + 0.5 * torch.eye(A.size(-1), device=A.device)
                A = A / A.sum(dim=-1, keepdim=True)
                result = A if result is None else A @ result
            mask = result[0, 1:].detach().cpu().numpy()                # CLS row -> patch tokens
            grid = int(round(mask.shape[0] ** 0.5))
            mask = mask[:grid * grid].reshape(grid, grid)
            return (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)

        # Balanced sample: 2 benign + 2 malignant
        sample_indices = []
        for label in (0, 1):
            idxs = test_paths_frame.index[test_paths_frame.coarse_label == label].tolist()
            sample_indices.extend(idxs[:ROLL_N_SAMPLES // 2])
        sample_indices = sample_indices[:ROLL_N_SAMPLES]

        fig, axes = plt.subplots(len(sample_indices), 2,
                                  figsize=(5.6, 2.9 * len(sample_indices)), squeeze=False,
                                  gridspec_kw={'wspace': 0.06, 'hspace': 0.32})
        GREEN, RED, NEUTRAL = '#2ca02c', '#d62728', '0.35'

        for r, idx in enumerate(sample_indices):
            row = test_paths_frame.iloc[idx]
            true_label = int(row['coarse_label'])
            img = PILImage.open(row['path']).convert('RGB').resize((224, 224))
            img_np = np.asarray(img, dtype=np.float32) / 255.0
            tensor = TF(img).unsqueeze(0).to(device)

            with torch.no_grad():
                out = fm(tensor, output_attentions=True)
            if not getattr(out, 'attentions', None):
                raise RuntimeError('Phikon returned no attention weights; load it with attn_implementation="eager".')
            mask = attention_rollout(out.attentions)
            heat = cv2.resize(mask, (224, 224), interpolation=cv2.INTER_LINEAR)
            cm = cv2.applyColorMap(np.uint8(255 * heat), cv2.COLORMAP_JET)
            cm = cv2.cvtColor(cm, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
            overlay = np.clip(0.5 * img_np + 0.5 * cm, 0, 1)

            # Col 0: input
            ax = axes[r, 0]
            ax.imshow(img); ax.set_xticks([]); ax.set_yticks([])
            for s in ax.spines.values():
                s.set_visible(True); s.set_linewidth(1.5); s.set_edgecolor('0.3')
            if r == 0:
                ax.set_title('Input Image', fontsize=12, fontweight='bold', pad=10)
            ax.set_ylabel(f'True: {CLASS_NAMES[true_label]}', fontsize=11,
                          fontweight='bold', rotation=90, labelpad=14)

            # Col 1: rollout overlay
            ax = axes[r, 1]
            ax.imshow(overlay); ax.set_xticks([]); ax.set_yticks([])
            if r == 0:
                ax.set_title('Phikon (frozen FM)\nAttention Rollout',
                              fontsize=11, fontweight='bold', pad=10)
            if preds is not None:
                pc = int(preds[idx]); pconf = float(confs[idx])
                correct = (pc == true_label)
                bcol = GREEN if correct else RED
                mark = '[correct]' if correct else '[wrong]'
                ax.set_xlabel(f'{mark} pred: {CLASS_NAMES[pc]}  ({pconf:.2f})',
                              fontsize=10, color=bcol, fontweight='bold', labelpad=4)
            else:
                bcol = NEUTRAL
                ax.set_xlabel('attention only (no stored prediction)',
                              fontsize=9, color=bcol, labelpad=4)
            for s in ax.spines.values():
                s.set_visible(True); s.set_linewidth(2.4); s.set_edgecolor(bcol)

        plt.suptitle(f'Attention Rollout on frozen Phikon\n'
                     f'(BreakHis, held-out mag = {ROLL_HELD_OUT}$\\times$, seed = {ROLL_SEED})',
                     fontsize=13, fontweight='bold', y=1.005)
        plt.savefig(f'{OUTPUT_DIR}/figures/fig11b_attention_rollout_phikon.png',
                    dpi=200, bbox_inches='tight')
        plt.savefig(f'{OUTPUT_DIR}/figures/fig11b_attention_rollout_phikon.pdf',
                    bbox_inches='tight')
        plt.show()
        del fm
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception as e:
        print(f'Fig 11b failed: {repr(e)[:300]}')


In [ ]:
# ---------- Fig 12: UMAP of penultimate features across all encoders (incl. Phikon) ----------
# Explains generalization: how well does each encoder separate Benign vs Malignant in learned space?
import subprocess
try:
    import umap
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'umap-learn'], check=False)
    try: import umap
    except Exception: umap = None

if umap is None:
    print('Skipped Fig 12 - umap-learn not available.')
elif not HAS_IMAGES:
    print('Skipped Fig 12 - BreakHis not attached.')
else:
    import torch
    from torchvision import transforms as _tf
    from sklearn.metrics import silhouette_score
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    UMAP_HELD_OUT = MAGS[-1] if MAGS else 400
    UMAP_SEED     = SEEDS[0] if SEEDS else 13
    UMAP_MAX_N    = 400

    test_paths_frame = reconstruct_test_paths(UMAP_HELD_OUT, UMAP_SEED)
    if test_paths_frame is None or len(test_paths_frame) == 0:
        print('Skipped Fig 12 - no test paths.')
    else:
        rng = np.random.default_rng(UMAP_SEED)
        keep = []
        for lab in [0, 1]:
            idxs = test_paths_frame.index[test_paths_frame.coarse_label == lab].tolist()
            n_take = min(len(idxs), UMAP_MAX_N // 2)
            keep.extend(rng.choice(idxs, size=n_take, replace=False).tolist())
        umap_frame = test_paths_frame.loc[sorted(keep)].reset_index(drop=True)
        print(f'  UMAP will use {len(umap_frame)} samples '
              f'({(umap_frame.coarse_label==0).sum()} B + {(umap_frame.coarse_label==1).sum()} M)')

        TF = _tf.Compose([_tf.Resize((224,224)), _tf.ToTensor(),
                          _tf.Normalize((.485,.456,.406),(.229,.224,.225))])
        stack_imgs = torch.stack([TF(PILImage.open(p).convert('RGB').resize((224, 224)))
                                   for p in umap_frame['path']])
        stack_labels = umap_frame['coarse_label'].to_numpy()

        def hooked_features(model, hook_layer, batch_size=32):
            store = {}
            def hook(m, i, o): store['x'] = o
            h = hook_layer.register_forward_hook(hook)
            feats = []
            try:
                with torch.no_grad():
                    for i in range(0, len(stack_imgs), batch_size):
                        b = stack_imgs[i:i+batch_size].to(device)
                        _ = model(b)
                        x = store['x']
                        if x.dim() == 4:
                            # Channels-last stages (Swin in timm>=0.9, MedMamba, DSA-Mamba) must
                            # be permuted first, or the mean runs over a spatial axis and the
                            # 'feature' is a length-H vector instead of C channels.
                            _t = globals().get('to_nchw')
                            x = (_t(x) if _t else x).mean(dim=[2, 3])
                        elif x.dim() == 3: x = x.mean(dim=1)
                        feats.append(x.detach().cpu().numpy())
            finally:
                h.remove()
            return np.concatenate(feats, axis=0)

        reverse_display = {v: k for k, v in DISPLAY.items()}
        embeddings = {}

        for method_name in ORDER:
            model_name = reverse_display.get(method_name)
            try:
                if method_name == 'Phikon (frozen FM)':
                    from transformers import AutoModel
                    fm = AutoModel.from_pretrained('owkin/phikon', add_pooling_layer=False).to(device).eval()
                    feats = []
                    with torch.no_grad():
                        for i in range(0, len(stack_imgs), 32):
                            b = stack_imgs[i:i+32].to(device)
                            out = fm(b)
                            feats.append(out.last_hidden_state[:, 0].cpu().numpy())
                    X = np.concatenate(feats, axis=0)
                    del fm
                else:
                    prefix = PREFIX_MAP.get(method_name)
                    ckpt_path = f'runs/{prefix}_ho{UMAP_HELD_OUT}_frac1.0_seed{UMAP_SEED}/best.pt'
                    if not reader.exists(ckpt_path):
                        print(f'  {method_name}: no checkpoint; skipping'); continue
                    if 'build_from_scratch' not in globals():
                        print(f'  {method_name}: build_from_scratch missing (run Fig 11 cell first); skipping'); continue
                    model = build_from_scratch(model_name)
                    sd = torch.load(io.BytesIO(reader.open_bytes(ckpt_path)), map_location=device)
                    model.load_state_dict(sd); model = model.to(device).eval()
                    if model_name == 'convnext_tiny':          hook_layer = model.stages[-1]
                    elif model_name == 'swin_tiny':            hook_layer = model.layers[-1]
                    elif model_name == 'mambavision_tiny_1k':  hook_layer = (model.levels[-1] if hasattr(model,'levels') else model.stages[-1])
                    elif model_name == 'medmamba':             hook_layer = model.layers[-1]
                    elif model_name == 'dsamamba':             hook_layer = model.Decoder_layers[-1]
                    else:
                        print(f'  {method_name}: no known hook layer; skipping'); del model; continue
                    X = hooked_features(model, hook_layer)
                    del model
                if torch.cuda.is_available(): torch.cuda.empty_cache()
                print(f'  {method_name}: features shape = {X.shape}')

                reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                                    metric='cosine', random_state=UMAP_SEED)
                embeddings[method_name] = reducer.fit_transform(X)
            except Exception as e:
                print(f'  {method_name}: FAILED - {e}')

        if not embeddings:
            print('No embeddings computed; skipping Fig 12 render.')
        else:
            n = len(embeddings); ncols = 3; nrows = (n + ncols - 1) // ncols
            fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 4.3 * nrows), squeeze=False)
            axes_flat = axes.flatten()
            for i, (method_name, emb) in enumerate(embeddings.items()):
                ax = axes_flat[i]
                for lab, cname, col in [(0, 'Benign', '#4C72B0'), (1, 'Malignant', '#C44E52')]:
                    m = stack_labels == lab
                    ax.scatter(emb[m, 0], emb[m, 1], s=22, c=col, alpha=0.75,
                                edgecolor='white', linewidth=0.35, label=cname)
                try:    sil = silhouette_score(emb, stack_labels)
                except Exception: sil = float('nan')
                ax.set_title(f'{method_name}\n(silhouette = {sil:.3f})',
                              fontsize=11, fontweight='bold', pad=6)
                ax.set_xticks([]); ax.set_yticks([])
                ax.set_xlabel('UMAP-1', fontsize=9); ax.set_ylabel('UMAP-2', fontsize=9)
                if i == 0:
                    ax.legend(loc='best', fontsize=9, frameon=True, framealpha=0.95, edgecolor='0.7')
                ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
            for j in range(n, len(axes_flat)):
                axes_flat[j].axis('off')
            plt.suptitle(f'Learned feature space (UMAP of penultimate embeddings, cosine metric)  '
                         f'\u2014 held-out mag = {UMAP_HELD_OUT}$\\times$',
                         fontsize=13, fontweight='bold', y=1.005)
            plt.tight_layout()
            plt.savefig(f'{OUTPUT_DIR}/figures/fig12_umap_feature_space.png', dpi=200, bbox_inches='tight')
            plt.savefig(f'{OUTPUT_DIR}/figures/fig12_umap_feature_space.pdf', bbox_inches='tight')
            plt.show()


## 2d. Part C: ablation and cross-validation (Fig 13-14b, T9-T10b)

Reads the Part C sessions, whose runs live in `raw_all` under branch values `ablation` and `cv` (and `baseline_ref` if a same-session baseline was trained). They are kept out of `raw`, so they cannot enter the LOMO figures above. CV folds run on different accounts are merged by `run_id`.

- **Fig 13 / T9** — MedMamba without SS2D (SS2D → depthwise conv) against MedMamba's own LOMO runs at the same held-out magnification, seeds and split.
- **Fig 14 / T10** — 5-fold patient-level CV next to the LOMO seeds for every (model, held-out magnification) with CV folds: MedMamba at 400× and the frozen Phikon probe at all four. The means should agree; the two SDs measure different sources of variation.
- **Fig 14b / T10b** — the same comparison for reliability rather than accuracy: ECE, Brier, risk at 80% coverage and AURC, recomputed from the stored per-image logits of the CV folds and of the LOMO seeds through one code path, under each run’s own fitted temperature and again at $T=1$. No GPU and no new input: it reuses `runs/<run_id>/test_outputs.npz`, which the CV runs kept even though they stored no checkpoint.

Each cell skips with a message when its Part C output is not attached.

In [ ]:
# ---------- Fig 13: component ablation at the hardest held-out magnification ----------
# Part C's ablated variants against the unmodified model at the same held-out magnification,
# seeds and patient split. The baseline is the model's own LOMO runs (branch 'from_scratch');
# if Part C also retrained a same-session baseline (branch 'baseline_ref'), that is used
# instead and the LOMO runs become a dotted reference line.
ABL_LABELS = {
    'dsamamba':           'DSA-Mamba\n(baseline)',
    'dsamamba_no_ssm':    'no-SSM\n(SS2D$\\rightarrow$DWConv)',
    'dsamamba_no_decomp': 'no trend\n(decomp$\\rightarrow$identity)',
    'medmamba':           'MedMamba\n(baseline)',
    'medmamba_no_ssm':    'no-SSM\n(SS2D$\\rightarrow$DWConv)',
    'medmamba_shallow':   'shallow\n(depths 1,1,2,1)',
    'medmamba_narrow':    'narrow\n(dims 48-384)',
}
ABL_MODEL_NAMES = {'dsamamba': 'DSA-Mamba', 'medmamba': 'MedMamba'}

def _envs_with_runs(prefix):
    """environment.json of every attached output holding a run_id that starts with prefix."""
    found = []
    for r in (readers if 'readers' in globals() else []):
        try:
            if any(f'runs/{prefix}' in n for n in r.list_matching('record.json')) and r.exists('environment.json'):
                found.append(json.loads(r.open_text('environment.json')))
        except Exception:
            pass
    return found

_has_branch = 'raw_all' in globals() and 'branch' in raw_all.columns
_abl = raw_all[raw_all.branch == 'ablation'] if _has_branch else None

if _abl is None or len(_abl) == 0:
    print('Skipped Fig 13 - no runs with branch="ablation". Attach the Part C (ablation) output.')
else:
    ABL_HO = int(_abl.held_out_mag.mode().iloc[0])
    _bases = sorted({v.split('_')[0] for v in _abl.model.unique()})
    ABL_BASE = _bases[0]
    if len(_bases) > 1:
        print(f'  NOTE: ablation runs for several base models {_bases}; plotting {ABL_BASE} only.')
    _abl = _abl[(_abl.held_out_mag == ABL_HO) & _abl.model.str.startswith(ABL_BASE + '_')]
    _variants = ([k for k in ABL_LABELS if k in set(_abl.model)]
                 + sorted(set(_abl.model) - set(ABL_LABELS)))

    _ref = raw_all[(raw_all.branch == 'baseline_ref') & (raw_all.model == ABL_BASE)
                   & (raw_all.held_out_mag == ABL_HO)]
    _lomo = raw[(raw.model == ABL_BASE) & (raw.held_out_mag == ABL_HO)]
    if len(_ref):
        _base, _base_label, _dotted = _ref, 'same-session baseline', _lomo
    else:
        _base, _base_label, _dotted = _lomo, 'LOMO baseline', None

    if len(_base) == 0:
        print(f'Skipped Fig 13 - no {ABL_BASE} runs at ho={ABL_HO} to measure the deltas against.')
        print('  Attach the LOMO output for that held-out magnification, or train a baseline in Part C.')
    else:
        _seeds_b = sorted(int(s) for s in _base.seed.unique())
        _seeds_v = sorted(int(s) for s in _abl.seed.unique())
        if _seeds_b != _seeds_v:
            print(f'  WARNING: baseline seeds {_seeds_b} differ from ablation seeds {_seeds_v}, so the '
                  f'deltas also contain split differences.')

        # (metric, title, subtitle, higher_is_better, source)
        _panels = [
            ('test_macro_f1',            'Macro-F1',                 'higher is better', True,  'runs'),
            ('test_ece',                 'ECE (post-temp. scaling)', 'lower is better',  False, 'runs'),
            ('test_risk_at_coverage_80', 'Risk @ 80% coverage',      'lower is better',  False, 'runs'),
        ]
        _panels = [p for p in _panels if p[0] in raw_all.columns]

        # BACH energy panel (standard sign, converted in the load cell). The gap to each run's own
        # BreakHis test set is plotted when available: a HIGHER gap means BACH looks less
        # in-distribution to the model than its own test set, i.e. the shift is detected.
        _ood_var = _ood_base = _ood_dotted = None
        if 'bach_ood_all' in globals() and bach_ood_all is not None and len(bach_ood_all):
            _at = bach_ood_all[bach_ood_all.held_out_mag == ABL_HO]
            _ood_var = _at[_at.branch == 'ablation']
            _ood_lomo = _at[(_at.branch == 'from_scratch') & (_at.model == ABL_BASE)]
            _ood_base = _at[(_at.branch == 'baseline_ref') & (_at.model == ABL_BASE)] if len(_ref) else _ood_lomo
            _ood_dotted = _ood_lomo if len(_ref) else None
            if len(_ood_var) and len(_ood_base):
                if 'energy_gap' in _at.columns and _at['energy_gap'].notna().any():
                    _panels.append(('energy_gap', 'BACH energy gap (OOD probe)',
                                    'BACH minus own test set; higher = shift detected', True, 'ood'))
                else:
                    _panels.append(('energy_mean', 'BACH energy (OOD probe)',
                                    'higher = less confident on BACH', True, 'ood'))
                print(f'  BACH OOD panel: {len(_ood_var)} variant rows, {len(_ood_base)} baseline rows')
            else:
                print('  BACH scores missing for the variants or the baseline; OOD panel omitted.')

        fig, axes = plt.subplots(1, len(_panels), figsize=(5.0 * len(_panels), 4.8), squeeze=False)
        axes = axes[0]
        BASE_C, ABL_C = '#CC79A7', '#0072B2'

        for ax, (metric, title, direction, higher_better, source) in zip(axes, _panels):
            rows, xs = [], []
            for mdl in [ABL_BASE] + _variants:
                if source == 'ood':
                    sub = _ood_base if mdl == ABL_BASE else _ood_var[_ood_var.model == mdl]
                else:
                    sub = _base if mdl == ABL_BASE else _abl[_abl.model == mdl]
                if len(sub) == 0 or metric not in sub.columns:
                    continue
                mean, lo, hi = bootstrap_ci(sub[metric].values)
                rows.append({'model': mdl, 'mean': mean, 'lo': lo, 'hi': hi, 'n': len(sub)})
                xs.append(ABL_LABELS.get(mdl, mdl))
            if not rows or rows[0]['model'] != ABL_BASE:
                ax.axis('off'); continue
            frame = pd.DataFrame(rows)
            base_mean = frame['mean'].iloc[0]
            colors = [BASE_C if r['model'] == ABL_BASE else ABL_C for r in rows]
            pos = np.arange(len(frame))

            # Reference lines are collected before drawing so the y-limits include them; a
            # reference line that silently leaves the axes is worse than none.
            ref_lines = [base_mean]
            _dot_src = _ood_dotted if source == 'ood' else _dotted
            b_mean = None
            if _dot_src is not None and len(_dot_src) and metric in _dot_src.columns:
                b_mean = _dot_src[metric].mean()
                ref_lines.append(b_mean)

            _lo = min(frame['lo'].min(), min(ref_lines))
            _hi = max(frame['hi'].max(), max(ref_lines))
            _pad = max(_hi - _lo, 1e-6)
            # Clamp to zero only for metrics that cannot be negative; BACH energy and its gap can be.
            _floor = max(0.0, _lo - 0.18 * _pad) if _lo >= 0 else _lo - 0.18 * _pad
            _ceil = _hi + 0.22 * _pad

            if source == 'ood':
                # Energy has no meaningful zero, so markers rather than bars drawn from zero.
                ax.errorbar(pos, frame['mean'],
                            yerr=[frame['mean'] - frame['lo'], frame['hi'] - frame['mean']],
                            fmt='none', ecolor='black', elinewidth=1.2, capsize=5, zorder=2)
                ax.scatter(pos, frame['mean'], s=180, c=colors, marker='o',
                           edgecolor='black', linewidth=0.9, zorder=3)
            else:
                ax.bar(pos, frame['mean'] - _floor, bottom=_floor, width=0.66, color=colors,
                       edgecolor='black', linewidth=0.6, zorder=2)
                ax.errorbar(pos, frame['mean'],
                            yerr=[frame['mean'] - frame['lo'], frame['hi'] - frame['mean']],
                            fmt='none', ecolor='black', elinewidth=1.2, capsize=4, zorder=3)

            ax.axhline(base_mean, color=BASE_C, linestyle='--', linewidth=1.3, alpha=0.8,
                       zorder=1, label=_base_label)
            if b_mean is not None:
                ax.axhline(b_mean, color='0.35', linestyle=':', linewidth=1.3, alpha=0.9,
                           zorder=1, label='LOMO baseline (reference)')

            for i, r in enumerate(rows):
                if r['model'] == ABL_BASE:
                    continue
                d = r['mean'] - base_mean
                worse = (d < 0) if higher_better else (d > 0)
                ax.annotate(f'{d:+.3f}' if abs(d) < 10 else f'{d:+.2f}', (i, r['hi']),
                            textcoords='offset points', xytext=(0, 8), ha='center', fontsize=10,
                            fontweight='bold', color=('#d62728' if worse else '#2ca02c'))

            ax.set_xticks(pos); ax.set_xticklabels(xs, fontsize=8.5)
            ax.set_ylabel(title)
            ax.set_title(f'{title}\n({direction})', fontweight='bold', fontsize=11, pad=8)
            ax.set_ylim(_floor, _ceil)
            ax.grid(True, axis='y', linewidth=0.4); ax.set_axisbelow(True)
            ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

        # 'best' rather than a fixed corner: which corner is free depends on the data.
        axes[0].legend(loc='best', fontsize=8, frameon=True, framealpha=0.95, edgecolor='0.7')
        plt.suptitle(f'{ABL_MODEL_NAMES.get(ABL_BASE, ABL_BASE)} component ablation at held-out '
                     f'magnification $= {ABL_HO}\\times$  (mean $\\pm$ 95% bootstrap CI over '
                     f'{len(_seeds_v)} seeds)', y=1.02, fontweight='bold')
        plt.tight_layout()
        save_fig(fig, 'fig13_ablation')
        plt.show()

        print(f'\nBaseline: {ABL_MODEL_NAMES.get(ABL_BASE, ABL_BASE)} {_base_label}, '
              f'n={len(_base)}, seeds {_seeds_b}')
        _e_var = _envs_with_runs('ablation_')
        _e_base = _envs_with_runs('baselineref_' if len(_ref) else f'scratch_{ABL_BASE}_ho{ABL_HO}_')
        if _e_var and _e_base:
            # Session-specific fields (clock, nvidia-smi snapshot with temperatures) always differ.
            _volatile = ('timestamp', 'nvidia_smi', 'hostname', 'uptime')
            _keys = sorted(k for k in set(_e_var[0]) & set(_e_base[0])
                           if not any(v in k.lower() for v in _volatile))
            _diff = {k: (_e_base[0][k], _e_var[0][k]) for k in _keys if _e_base[0][k] != _e_var[0][k]}
            if _diff:
                print('  WARNING: the ablation and baseline outputs ran in different environments:')
                for k, (b, v) in _diff.items():
                    print(f'    {k}: baseline={b}  ablation={v}')
                print('  The deltas then include that difference; set INCLUDE_BASELINE_REFERENCE = True in')
                print('  Part C and retrain the baseline alongside the variants if it matters.')
            else:
                print(f'  Environment check: ablation and baseline outputs match on {len(_keys)} recorded keys.')
        else:
            print('  Environment check skipped: environment.json not found in both outputs.')
        if len(_ref) and len(_lomo):
            print(f'  Same-session baseline minus LOMO baseline, macro-F1: '
                  f'{_ref.test_macro_f1.mean() - _lomo.test_macro_f1.mean():+.4f}')

        if 'reader' in globals() and reader.exists('dsamamba_code_audit.json'):
            _audit = json.loads(reader.open_text('dsamamba_code_audit.json'))
            print('\nDSA-Mamba code audit (from Part C, no training):')
            print(f"  cross-attention: {_audit.get('cross_attention_params', 0)/1e6:.2f}M params; "
                  f"tensors receiving gradient {_audit.get('cross_attention_tensors_with_gradient')}"
                  f"/{_audit.get('cross_attention_param_tensors')}")
            print(f"  max |output diff| with cross-attention bypassed: "
                  f"{_audit.get('max_abs_output_diff_without_cross_attention')}")
            print(f"  series_decomp returns the raw input: {_audit.get('decomp_first_output_is_raw_input')}; "
                  f"the residual: {_audit.get('decomp_first_output_is_residual')}")


In [ ]:
# ---------- Fig 14: 5-fold patient-level CV vs LOMO seeds, per model and held-out magnification ----------
# For every (model, held-out magnification) with CV folds, the fold results sit next to the
# LOMO seed results for the same model and magnification. The two answer different questions
# and are not competing estimates of one quantity:
#   fold spread (CV)   -> how much the result moves when the TEST PATIENTS change
#   seed spread (LOMO) -> how much it moves when only the initialisation changes
# The means should agree, because patient_kfold_lomo_split is sized to match
# patient_lomo_split (65/15/20). A LOMO mean outside its CV fold range -- for example a
# threshold effect that shows up for one patient split only -- is what this figure exposes.
# Balanced accuracy and AUC are shown together because a low balanced accuracy with a high
# AUC is the signature of a decision-threshold effect rather than weak features.
FIG14_METRICS = [('test_balanced_accuracy', 'Balanced accuracy'), ('test_auc', 'AUC')]
CV_DISPLAY = {'owkin/phikon': 'Phikon (frozen FM)', 'dsamamba': 'DSA-Mamba', 'medmamba': 'MedMamba',
              'convnext_tiny': 'ConvNeXt-T', 'swin_tiny': 'Swin-T'}
CV_MODEL_ORDER = ['owkin/phikon', 'dsamamba', 'medmamba', 'convnext_tiny', 'swin_tiny']

_has_branch = 'raw_all' in globals() and 'branch' in raw_all.columns
_cv = raw_all[raw_all.branch == 'cv'] if _has_branch else None

if _cv is None or len(_cv) == 0:
    print('Skipped Fig 14 - no runs with branch="cv". Attach the Part C (cv) output.')
else:
    _groups = []
    for (mdl, ho), sub in _cv.groupby(['model', 'held_out_mag']):
        lomo = raw[(raw.model == mdl) & (raw.held_out_mag == ho)]
        _groups.append((mdl, int(ho), sub, lomo))
    _rank = {m: i for i, m in enumerate(CV_MODEL_ORDER)}
    _groups.sort(key=lambda g: (_rank.get(g[0], len(_rank)), g[1]))
    _metrics = [(c, l) for c, l in FIG14_METRICS if c in _cv.columns]
    _k = int(_cv['fold_idx'].nunique()) if 'fold_idx' in _cv.columns else int(_cv.groupby(['model', 'held_out_mag']).size().max())

    fig, axes = plt.subplots(len(_metrics), len(_groups),
                             figsize=(2.9 * len(_groups) + 0.6, 3.3 * len(_metrics)),
                             squeeze=False, sharey='row')
    CV_C, LOMO_C = '#009E73', '#0072B2'
    _jit = np.random.default_rng(0)
    for r, (col, label) in enumerate(_metrics):
        for c, (mdl, ho, sub, lomo) in enumerate(_groups):
            ax = axes[r, c]
            for i, (frame, color) in enumerate([(sub, CV_C), (lomo, LOMO_C)]):
                v = frame[col].dropna().to_numpy(dtype=float) if len(frame) else np.array([])
                if len(v) == 0:
                    ax.text(i, 0.5, 'n/a', ha='center', va='center', color='0.5', fontsize=9,
                            transform=ax.get_xaxis_transform())
                    continue
                # n is 3-5: every point is shown; a box or violin would imply a shape the data
                # cannot support.
                ax.scatter(np.full(len(v), i) + _jit.uniform(-0.08, 0.08, len(v)), v, s=46, color=color,
                           edgecolor='black', linewidth=0.6, alpha=0.9, zorder=3)
                ax.hlines(v.mean(), i - 0.28, i + 0.28, color=color, linewidth=2.4, zorder=4)
                if len(v) > 1:
                    ax.errorbar(i, v.mean(), yerr=v.std(ddof=1), fmt='none', ecolor=color,
                                elinewidth=1.4, capsize=6, alpha=0.85, zorder=2)
            ax.set_xticks([0, 1])
            ax.set_xticklabels([f'CV\n{len(sub)} folds', f'LOMO\n{len(lomo)} seeds'], fontsize=8.5)
            ax.set_xlim(-0.6, 1.6)
            if r == 0:
                ax.set_title(f'{CV_DISPLAY.get(mdl, mdl)}\nheld-out {ho}$\\times$',
                             fontsize=10, fontweight='bold', pad=6)
            if c == 0:
                ax.set_ylabel(label)
            ax.grid(True, axis='y', linewidth=0.4); ax.set_axisbelow(True)
            ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.suptitle(f'Patient-level {_k}-fold cross-validation vs LOMO seeds '
                 f'(points: individual runs; bar: mean $\\pm$1 SD)', y=1.02, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, 'fig14_cross_validation')
    plt.show()

    print(f'\nCV fold mean vs LOMO seed mean:')
    for mdl, ho, sub, lomo in _groups:
        parts = []
        for col, label in _metrics:
            cv_m, cv_sd = sub[col].mean(), sub[col].std(ddof=1)
            if len(lomo):
                lo_m = lomo[col].mean()
                flag = '' if sub[col].min() <= lo_m <= sub[col].max() else '  <- LOMO mean outside the CV fold range'
                parts.append(f'{label} CV {cv_m:.3f}+/-{cv_sd:.3f} | LOMO {lo_m:.3f}{flag}')
            else:
                parts.append(f'{label} CV {cv_m:.3f}+/-{cv_sd:.3f} | LOMO n/a')
        print(f'  {CV_DISPLAY.get(mdl, mdl):<20} {ho:>3}x   ' + ';   '.join(parts))
    print('  Report the means as a consistency check; the two SDs measure different sources of variation.')


In [ ]:
# ---------- Fig 14b / T10b: patient-level cross-validation for calibration and selective prediction ----------
# Fig 14 and T10 check accuracy under patient-level CV. The CV runs also stored per-image logits, so
# the reliability claims can be checked the same way at no GPU cost: ECE, Brier, risk at 80% coverage
# and AURC, recomputed through one code path for the CV folds and the LOMO seeds alike.
# Two temperature conventions are reported, because the fitted temperature turned out to be unstable
# across seeds (0.06 to 6.0 in this study, every value below 0.5 belonging to one seed):
#   scaled : each run's own fitted temperature, exactly what Table IV and Fig 2 report
#   T = 1  : the raw logits, no scaling
# If the reliability ordering holds under patient-level CV and under both conventions, the paper's
# reliability claim does not rest on three seeds or on one temperature fit.
CVREL_METRICS = [('ece', 'ECE (15 bins)'), ('risk80', 'Risk @ 80% coverage'),
                 ('brier', 'Brier score'), ('aurc', 'AURC')]
_trapz_cvrel = getattr(np, 'trapezoid', None) or np.trapz


def _probs_at(logits, temperature=1.0):
    z = np.asarray(logits, dtype=np.float64) / float(temperature)          # (N, C)
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)                                # (N, C)


def _reliability_from_probs(probs, labels, n_bins=15):
    """ECE, Brier, risk at 80/90/95% coverage and AURC - the same definitions the training code used."""
    probs = np.asarray(probs, dtype=np.float64)                            # (N, C)
    labels = np.asarray(labels).astype(int)                                # (N,)
    conf, pred = probs.max(axis=1), probs.argmax(axis=1)
    acc = (pred == labels).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum():
            ece += m.mean() * abs(acc[m].mean() - conf[m].mean())
    onehot = np.eye(probs.shape[1])[labels]
    brier = float(((probs - onehot) ** 2).sum(axis=1).mean())
    order = np.argsort(-conf)
    err = (1.0 - acc)[order]
    risk = np.cumsum(err) / np.arange(1, len(err) + 1)                     # (N,)
    coverage = np.arange(1, len(err) + 1) / len(err)                       # (N,)
    out = {'ece': float(ece), 'brier': brier, 'aurc': float(_trapz_cvrel(risk, coverage)),
           'accuracy': float(acc.mean())}
    for target in (0.80, 0.90, 0.95):
        idx = int(target * len(coverage)) - 1
        out[f'risk{int(target * 100)}'] = float(risk[idx]) if idx >= 0 else float('nan')
    return out


_cvrel_source = raw_all if ('raw_all' in globals() and 'branch' in raw_all.columns) else None
if _cvrel_source is None or not (_cvrel_source.branch == 'cv').any():
    print('Skipped Fig 14b - no runs with branch="cv". Attach the Part C (cv) output.')
    cv_reliability = None
else:
    _rows, _missing = [], []
    _wanted = _cvrel_source[_cvrel_source.branch.isin(['cv', 'from_scratch', 'fm_linear_probe'])]
    for _, run in _wanted.iterrows():
        npz_path = f"runs/{run['run_id']}/test_outputs.npz"
        if not reader.exists(npz_path):
            _missing.append(run['run_id']); continue
        d = np.load(io.BytesIO(reader.open_bytes(npz_path)))
        logits, labels = d['logits'], d['labels'].astype(int)
        temperature = float(run['test_temperature']) if pd.notna(run.get('test_temperature')) else 1.0
        for convention, probs in (('scaled', _probs_at(logits, temperature)), ('T=1', _probs_at(logits, 1.0))):
            _rows.append({'run_id': run['run_id'], 'method': run['method'], 'model': run['model'],
                          'branch': run['branch'], 'held_out_mag': int(run['held_out_mag']),
                          'seed': int(run['seed']), 'temperature': temperature, 'convention': convention,
                          'design': 'CV fold' if run['branch'] == 'cv' else 'LOMO seed',
                          **_reliability_from_probs(probs, labels)})
    cv_reliability = pd.DataFrame(_rows)
    if _missing:
        print(f'  {len(_missing)} runs had no test_outputs.npz and were skipped, e.g. {_missing[:3]}')
    if len(cv_reliability) == 0:
        print('Skipped Fig 14b - no per-image outputs could be read.')
    else:
        cv_reliability.to_csv(f'{OUTPUT_DIR}/tables/cv_reliability_per_run.csv', index=False)
        # A pair exists only where the same (model, held-out magnification) has both designs.
        _pairs = [(m, h) for (m, h), g in cv_reliability.groupby(['model', 'held_out_mag'])
                  if g.design.nunique() == 2]
        _rank = {m: i for i, m in enumerate(CV_MODEL_ORDER)} if 'CV_MODEL_ORDER' in globals() else {}
        _pairs.sort(key=lambda p: (_rank.get(p[0], len(_rank)), p[1]))
        print(f'{len(_pairs)} (model, held-out magnification) cells have both CV folds and LOMO seeds')

        _scaled = cv_reliability[cv_reliability.convention == 'scaled']
        fig, axes = plt.subplots(len(CVREL_METRICS), len(_pairs),
                                 figsize=(2.9 * len(_pairs) + 0.6, 2.9 * len(CVREL_METRICS)),
                                 squeeze=False, sharey='row')
        CV_C, LOMO_C = '#009E73', '#0072B2'
        _jit = np.random.default_rng(0)
        for r, (col, label) in enumerate(CVREL_METRICS):
            for c, (mdl, ho) in enumerate(_pairs):
                ax = axes[r, c]
                cell = _scaled[(_scaled.model == mdl) & (_scaled.held_out_mag == ho)]
                for i, (design, colour) in enumerate([('CV fold', CV_C), ('LOMO seed', LOMO_C)]):
                    v = cell.loc[cell.design == design, col].to_numpy(dtype=float)
                    if len(v) == 0:
                        continue
                    ax.scatter(np.full(len(v), i) + _jit.uniform(-0.08, 0.08, len(v)), v, s=42, color=colour,
                               edgecolor='black', linewidth=0.6, alpha=0.9, zorder=3)
                    ax.hlines(v.mean(), i - 0.28, i + 0.28, color=colour, linewidth=2.4, zorder=4)
                    if len(v) > 1:
                        ax.errorbar(i, v.mean(), yerr=v.std(ddof=1), fmt='none', ecolor=colour,
                                    elinewidth=1.4, capsize=6, alpha=0.85, zorder=2)
                ax.set_xticks([0, 1])
                ax.set_xticklabels([f"CV\n{int((cell.design == 'CV fold').sum())} folds",
                                    f"LOMO\n{int((cell.design == 'LOMO seed').sum())} seeds"], fontsize=8.5)
                ax.set_xlim(-0.6, 1.6)
                if r == 0:
                    name = (CV_DISPLAY.get(mdl, mdl) if 'CV_DISPLAY' in globals() else DISPLAY.get(mdl, mdl))
                    ax.set_title(f'{name}\nheld-out {ho}$\\times$', fontsize=10, fontweight='bold')
                if c == 0:
                    ax.set_ylabel(label)
                ax.grid(True, axis='y', linewidth=0.4); ax.set_axisbelow(True)
                ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        plt.suptitle('Reliability under patient-level cross-validation (points: individual runs; bar: mean $\\pm$1 SD; '
                     "each run's own fitted temperature)", y=1.01, fontweight='bold')
        plt.tight_layout()
        save_fig(fig, 'fig14b_cv_reliability')
        plt.show()

        _agg = (cv_reliability.groupby(['method', 'held_out_mag', 'convention', 'design'])
                [[c for c, _ in CVREL_METRICS]].agg(['mean', 'std']))
        _agg.to_csv(f'{OUTPUT_DIR}/tables/cv_reliability_summary.csv')
        for convention in ('scaled', 'T=1'):
            sub = cv_reliability[cv_reliability.convention == convention]
            print(f'\nCV vs LOMO within each encoder, temperature convention = {convention}')
            print('(read along a row only: each encoder is averaged over its own CV magnifications)')
            keep = sub[[(m, h) in _pairs for m, h in zip(sub.model, sub.held_out_mag)]]
            print(keep.pivot_table(index='method', columns='design',
                                   values=[c for c, _ in CVREL_METRICS]).round(3).to_string())
        # Across encoders the comparison is only like-for-like at a magnification where more than one
        # encoder has CV folds, so it is printed separately for exactly those magnifications.
        _cv_only = cv_reliability[cv_reliability.design == 'CV fold']
        for ho in sorted(h for h, g in _cv_only.groupby('held_out_mag') if g.method.nunique() > 1):
            sub = cv_reliability[[(m, h) in _pairs and h == ho
                                  for m, h in zip(cv_reliability.model, cv_reliability.held_out_mag)]]
            print(f'\nAcross encoders at held-out {ho}x (the matched comparison):')
            print(sub.pivot_table(index=['convention', 'design'], columns='method',
                                  values=[c for c, _ in CVREL_METRICS]).round(3).to_string())
        print('\nPer cell, CV minus LOMO (scaled convention; positive = worse under CV):')
        _paired_only = _scaled[[(m, h) in _pairs for m, h in zip(_scaled.model, _scaled.held_out_mag)]]
        _delta = (_paired_only.pivot_table(index=['method', 'held_out_mag'], columns='design',
                                           values=[c for c, _ in CVREL_METRICS]))
        for col, _ in CVREL_METRICS:
            if (col, 'CV fold') in _delta.columns and (col, 'LOMO seed') in _delta.columns:
                _delta[(col, 'CV - LOMO')] = _delta[(col, 'CV fold')] - _delta[(col, 'LOMO seed')]
        print(_delta[[(c, 'CV - LOMO') for c, _ in CVREL_METRICS
                      if (c, 'CV - LOMO') in _delta.columns]].round(3).to_string())

## 2e. Dataset composition and the accuracy–reliability divergence (Fig 15–16)

- **Fig 15** — images per magnification and class, images per split for every held-out magnification (mean over the LOMO seeds, whiskers = min–max), and patients per split and class. Counts are read from the `breakhis_manifest.csv` the training runs saved and re-derived with the same `patient_lomo_split()` (cell *Image-based prerequisites*), so the figure cannot drift from the data the models were trained on. No images or GPU needed.
- **Fig 16** — macro-F1 and post-scaling ECE for every encoder × held-out magnification as paired 3D bars, plus the rank of each encoder by accuracy and by reliability. Bars rather than an interpolated surface: the encoder axis is categorical, so a surface between two encoders would show values that do not exist. Bars start at zero so their heights are comparable.

In [ ]:
# ---------- Fig 15: dataset composition and the patient-disjoint LOMO split ----------
# (a) images per magnification and class, (b) images per split for every held-out magnification
# (mean over the LOMO seeds, whiskers = min-max over seeds), (c) patients per split and class.
# The split is recomputed with patient_lomo_split() from the image-prerequisites cell, i.e. the
# function the training notebooks used, on the manifest they saved.
FIG15_CLASS_COLORS = {0: '#4C72B0', 1: '#C44E52'}          # benign / malignant, as in Fig 12
FIG15_SPLIT_COLORS = {'train': '#C9CED6', 'val': '#7A8591', 'test': '#253341'}
FIG15_SPLITS = ['train', 'val', 'test']

if not reader.exists('breakhis_manifest.csv'):
    print('Skipped Fig 15 - breakhis_manifest.csv not found in the attached training outputs.')
elif 'patient_lomo_split' not in globals():
    print('Skipped Fig 15 - run the image-prerequisites cell first (it defines patient_lomo_split).')
else:
    _man15 = pd.read_csv(io.BytesIO(reader.open_bytes('breakhis_manifest.csv')))
    _missing = {'patient_id', 'magnification', 'coarse_label'} - set(_man15.columns)
    assert not _missing, f'breakhis_manifest.csv lacks columns {_missing}'
    _man15['magnification'] = _man15['magnification'].astype(int)
    _man15['coarse_label'] = _man15['coarse_label'].astype(int)
    _mags15 = sorted(_man15.magnification.unique())
    _patient_class = _man15.groupby('patient_id').coarse_label.first()
    assert (_man15.groupby('patient_id').coarse_label.nunique() == 1).all(), 'a patient carries both labels'

    # (a) images per magnification x class
    _img_counts = (_man15.groupby(['magnification', 'coarse_label']).size()
                   .unstack(fill_value=0).reindex(index=_mags15, columns=[0, 1], fill_value=0))

    # (b)/(c) split sizes for every (held-out magnification, seed) the LOMO runs used
    _split_rows = []
    for ho in MAGS:
        for seed in SEEDS:
            _sub = patient_lomo_split(_man15, int(ho), fraction=1.0, seed=int(seed))
            for split in FIG15_SPLITS:
                _rows = _sub[_sub.split == split]
                _pc = _rows.groupby('patient_id').coarse_label.first().value_counts()
                _split_rows.append({'held_out_mag': int(ho), 'seed': int(seed), 'split': split,
                                    'images': len(_rows),
                                    'images_benign': int((_rows.coarse_label == 0).sum()),
                                    'images_malignant': int((_rows.coarse_label == 1).sum()),
                                    'patients_benign': int(_pc.get(0, 0)),
                                    'patients_malignant': int(_pc.get(1, 0))})
    split_table = pd.DataFrame(_split_rows)
    split_table.to_csv(f'{OUTPUT_DIR}/tables/dataset_split_counts.csv', index=False)

    # Patient counts depend only on the class sizes, so they must be identical for every run.
    _pat = split_table.groupby('split')[['patients_benign', 'patients_malignant']].nunique()
    PATIENT_COUNTS_CONSTANT = bool((_pat == 1).all().all())
    _pat_first = split_table.drop_duplicates('split').set_index('split').loc[FIG15_SPLITS]
    if not PATIENT_COUNTS_CONSTANT:
        print('  WARNING: patients per split differ across (held-out magnification, seed); '
              'panel (c) shows the first combination only.')

    fig = plt.figure(figsize=(15.5, 4.6))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.0, 1.25, 0.85], wspace=0.28)

    # ---- (a) ----
    ax = fig.add_subplot(gs[0, 0])
    _x = np.arange(len(_mags15))
    ax.bar(_x, _img_counts[0], color=FIG15_CLASS_COLORS[0], edgecolor='black', linewidth=0.5, label='Benign')
    ax.bar(_x, _img_counts[1], bottom=_img_counts[0], color=FIG15_CLASS_COLORS[1],
           edgecolor='black', linewidth=0.5, label='Malignant')
    for i, m in enumerate(_mags15):
        b, mal = int(_img_counts.loc[m, 0]), int(_img_counts.loc[m, 1])
        ax.text(i, b / 2, f'{b:,}', ha='center', va='center', color='white', fontsize=9, fontweight='bold')
        ax.text(i, b + mal / 2, f'{mal:,}', ha='center', va='center', color='white', fontsize=9, fontweight='bold')
        ax.text(i, b + mal + 25, f'{b + mal:,}', ha='center', va='bottom', fontsize=9)
    ax.set_xticks(_x); ax.set_xticklabels([f'{m}$\\times$' for m in _mags15])
    ax.set_ylabel('Images'); ax.set_ylim(0, _img_counts.sum(axis=1).max() * 1.28)
    ax.set_title(f'(a) BreakHis: {len(_man15):,} images, {_patient_class.size} patients\n'
                 f'({int((_patient_class == 0).sum())} benign, {int((_patient_class == 1).sum())} malignant)',
                 fontweight='bold', fontsize=11)
    ax.legend(frameon=False, loc='upper center', ncol=2, fontsize=9)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    # ---- (b) ----
    ax = fig.add_subplot(gs[0, 1])
    _agg = split_table.groupby(['held_out_mag', 'split']).images.agg(['mean', 'min', 'max']).reset_index()
    _w = 0.26
    for k, split in enumerate(FIG15_SPLITS):
        _s = _agg[_agg.split == split].set_index('held_out_mag').reindex(MAGS)
        _xs = np.arange(len(MAGS)) + (k - 1) * _w
        ax.bar(_xs, _s['mean'], width=_w, color=FIG15_SPLIT_COLORS[split], edgecolor='black', linewidth=0.5,
               label={'train': 'Train', 'val': 'Validation', 'test': 'Test'}[split])
        ax.errorbar(_xs, _s['mean'], yerr=[_s['mean'] - _s['min'], _s['max'] - _s['mean']],
                    fmt='none', ecolor='black', elinewidth=0.9, capsize=3)
        for xx, v, top in zip(_xs, _s['mean'], _s['max']):
            ax.text(xx, top + 45, f'{v:,.0f}', ha='center', va='bottom', fontsize=7)
    ax.set_xticks(np.arange(len(MAGS))); ax.set_xticklabels([f'{m}$\\times$' for m in MAGS])
    ax.set_xlabel('Held-out magnification'); ax.set_ylabel('Images per split')
    ax.set_ylim(0, _agg['max'].max() * 1.30)
    ax.set_title(f'(b) Images per split, mean over {len(SEEDS)} seeds\n'
                 '(whiskers = min-max; train/val from the 3 training mags)', fontweight='bold', fontsize=11)
    ax.legend(frameon=False, fontsize=9, loc='upper center', ncol=3)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    # ---- (c) ----
    ax = fig.add_subplot(gs[0, 2])
    _xc = np.arange(len(FIG15_SPLITS))
    _pb = _pat_first['patients_benign'].to_numpy(); _pm = _pat_first['patients_malignant'].to_numpy()
    ax.bar(_xc, _pb, color=FIG15_CLASS_COLORS[0], edgecolor='black', linewidth=0.5, label='Benign')
    ax.bar(_xc, _pm, bottom=_pb, color=FIG15_CLASS_COLORS[1], edgecolor='black', linewidth=0.5, label='Malignant')
    for i in range(len(FIG15_SPLITS)):
        ax.text(i, _pb[i] / 2, str(_pb[i]), ha='center', va='center', color='white', fontsize=9, fontweight='bold')
        ax.text(i, _pb[i] + _pm[i] / 2, str(_pm[i]), ha='center', va='center', color='white', fontsize=9, fontweight='bold')
        ax.text(i, _pb[i] + _pm[i] + 0.8, str(_pb[i] + _pm[i]), ha='center', va='bottom', fontsize=9)
    ax.set_xticks(_xc); ax.set_xticklabels(['Train', 'Validation', 'Test'])
    ax.set_ylabel('Patients'); ax.set_ylim(0, (_pb + _pm).max() * 1.15)
    ax.set_title('(c) Patients per split (65/15/20% per class)\n'
                 + ('identical for every held-out mag and seed' if PATIENT_COUNTS_CONSTANT else 'first run shown'),
                 fontweight='bold', fontsize=11)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    plt.suptitle('Dataset composition and the patient-disjoint LOMO protocol', y=1.04, fontweight='bold')
    save_fig(fig, 'fig15_dataset_composition')
    plt.show()

    print('Images per magnification and class:')
    print(_img_counts.rename(columns={0: 'benign', 1: 'malignant'}).assign(total=lambda d: d.sum(axis=1)).to_string())
    print('\nImages per split (mean over seeds):')
    print(_agg.pivot(index='held_out_mag', columns='split', values='mean').reindex(columns=FIG15_SPLITS).round(1).to_string())
    print('\nPatients per split:', {s: int(_pb[i] + _pm[i]) for i, s in enumerate(FIG15_SPLITS)},
          '(constant across runs)' if PATIENT_COUNTS_CONSTANT else '(varies across runs)')

In [ ]:
# ---------- Fig 30-31 / T13: the preprocessing pipeline, shown with the exact training transforms ----------
# Training (cell 7 of the training notebooks):
#   Resize((224, 224)) -> RandomHorizontalFlip -> RandomVerticalFlip -> RandomRotation(15)
#   -> ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05) -> ToTensor -> Normalize(ImageNet)
# Validation, test and BACH: Resize((224, 224)) -> ToTensor -> Normalize(ImageNet).
# The same torchvision objects are built here, so every panel is what the model received.
# How often the augmentation is drawn differs by encoder (training notebook cells 10 and 12):
#   * ConvNeXt-T, Swin-T and MedMamba draw a new random view of every training image in every epoch.
#   * The frozen Phikon probe encodes ONE random view of each training image once (features are cached)
#     and reuses those features for all 200 probe epochs.
# Phikon's Hugging Face image processor is loaded but never used: Phikon receives the same 224 x 224,
# ImageNet-normalised tensors as the other encoders.
# Two properties are shown rather than hidden:
#   * Resize((224, 224)) does not keep the aspect ratio. A 700 x 460 BreakHis image is scaled by 0.32
#     horizontally and 0.49 vertically, and a 2048 x 1536 BACH image by 0.11 and 0.15, so tissue appears
#     squeezed, and BACH is shrunk about 3x more than BreakHis.
#   * RandomRotation keeps the canvas size and fills the uncovered corners with black (fill = 0,
#     nearest-neighbour interpolation, torchvision defaults).
# Fig 31 measures per-image colour statistics per held-out magnification and for BACH, before and after
# the ImageNet normalisation, which was not re-estimated on histology.
import os
import re as _re30
from pathlib import Path as _P30
from PIL import Image as _Img30

PREP_IMAGE_SIZE = 224
PREP_MEAN, PREP_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
PREP_JITTER = dict(brightness=0.1, contrast=0.1, saturation=0.05)
PREP_ROTATION = 15
PREP_STATS_PER_GROUP = 200          # images per magnification (and from BACH) for the colour statistics
PREP_SEED = 0
_SOB_RE = _re30.compile(r'SOB_([BM])_([A-Za-z]+)-(\d+)-([0-9A-Za-z]+)-(\d+)-(\d+)\.png$')

try:
    import torch as _torch30
    from torchvision import transforms as _T30
    from torchvision.transforms import functional as _TF30
    _have_tv30 = True
except ImportError:
    _have_tv30 = False


def _find_breakhis_root30():
    root = globals().get('BREAKHIS_ROOT')
    if root and _P30(root).exists():
        return _P30(root)
    base = _P30('/kaggle/input')
    return base if base.exists() else None


def _find_bach_root30(search_root='/kaggle/input'):
    best, best_n = None, 0
    if not _P30(search_root).exists():
        return None
    for dirpath, dirnames, _ in os.walk(search_root):
        if {'Normal', 'Benign', 'InSitu', 'Invasive'}.issubset(set(dirnames)):
            n = sum(1 for ext in ('*.tif', '*.png', '*.jpg') for _ in _P30(dirpath).rglob(ext))
            if n > best_n:
                best, best_n = _P30(dirpath), n
    return best


_bh_root30 = _find_breakhis_root30()
if not _have_tv30 or _bh_root30 is None:
    print('Skipped Fig 30-31 - ' + ('torchvision is not installed.' if not _have_tv30
                                     else 'BreakHis is not attached (Add Input -> ambarish/breakhis).'))
    preprocessing_stats = None
else:
    # ---- index BreakHis by patient and magnification from the file names ----
    _files30 = sorted(p for p in _bh_root30.rglob('SOB_*.png') if _SOB_RE.search(p.name))
    _idx30 = []
    for p in _files30:
        cls, sub, yr, slide, mag, seq = _SOB_RE.search(p.name).groups()
        _idx30.append({'path': str(p), 'name': p.name, 'label': cls, 'patient': f'{cls}_{sub}-{yr}-{slide}',
                       'mag': int(mag), 'seq': int(seq)})
    _idx30 = pd.DataFrame(_idx30).drop_duplicates('name')          # a dataset can hold the same file twice
    print(f'BreakHis files found: {len(_idx30)} from {_idx30.patient.nunique()} patients')
    _mags30 = sorted(_idx30.mag.unique())

    # one malignant patient seen at every magnification, chosen deterministically
    _full30 = _idx30.groupby('patient').mag.nunique()
    _cands30 = sorted(p for p in _full30[_full30 == len(_mags30)].index if p.startswith('M_'))
    _patient30 = _cands30[0] if _cands30 else sorted(_full30.index)[0]
    _examples30 = [(f'{m}×', _idx30[(_idx30.patient == _patient30) & (_idx30.mag == m)]
                    .sort_values('seq').path.iloc[0]) for m in _mags30
                   if len(_idx30[(_idx30.patient == _patient30) & (_idx30.mag == m)])]
    _bach_root30 = _find_bach_root30()
    _bach_files30 = (sorted(p for ext in ('*.tif', '*.png', '*.jpg') for p in _bach_root30.rglob(ext))
                     if _bach_root30 is not None else [])
    _bach_inv30 = [p for p in _bach_files30 if p.parent.name == 'Invasive']
    if _bach_inv30:
        _examples30.append(('BACH', str(_bach_inv30[0])))
    print(f'Fig 30 example patient: {_patient30}; rows: {[r for r, _ in _examples30]}')

    # ---- the exact torchvision transforms ----
    _resize30 = _T30.Resize((PREP_IMAGE_SIZE, PREP_IMAGE_SIZE))
    _train_aug30 = _T30.Compose([_resize30, _T30.RandomHorizontalFlip(), _T30.RandomVerticalFlip(),
                                 _T30.RandomRotation(PREP_ROTATION), _T30.ColorJitter(**PREP_JITTER)])
    _to_input30 = _T30.Compose([_T30.ToTensor(), _T30.Normalize(PREP_MEAN, PREP_STD)])
    _rot_defaults30 = _T30.RandomRotation(PREP_ROTATION)

    def _jitter_fixed30(img, sign):
        f = 1 + sign * np.array([PREP_JITTER['brightness'], PREP_JITTER['contrast'], PREP_JITTER['saturation']])
        img = _TF30.adjust_brightness(img, float(f[0]))
        img = _TF30.adjust_contrast(img, float(f[1]))
        return _TF30.adjust_saturation(img, float(f[2]))

    _cols30 = ['Original', f'Resize {PREP_IMAGE_SIZE}×{PREP_IMAGE_SIZE}\n(val / test / BACH input)',
               'Horizontal flip\n(p = 0.5)', 'Vertical flip\n(p = 0.5)',
               f'Rotation +{PREP_ROTATION}°\n(range ±{PREP_ROTATION}°)',
               'Colour jitter, low end\n(×0.90 / 0.90 / 0.95)', 'Colour jitter, high end\n(×1.10 / 1.10 / 1.05)',
               'One random training\nsample (all steps)', 'Normalised input\n(per-channel values)']
    fig, axes = plt.subplots(len(_examples30), len(_cols30),
                             figsize=(2.05 * len(_cols30), 2.15 * len(_examples30) + 0.7), squeeze=False)
    _torch30.manual_seed(PREP_SEED)
    for r, (row_name, path) in enumerate(_examples30):
        with _Img30.open(path) as im:
            orig = im.convert('RGB')
        w, h = orig.size
        base = _resize30(orig)
        is_bach = row_name == 'BACH'
        panels = [orig, base, _TF30.hflip(base), _TF30.vflip(base),
                  _TF30.rotate(base, PREP_ROTATION, interpolation=_rot_defaults30.interpolation,
                               fill=_rot_defaults30.fill),
                  _jitter_fixed30(base, -1), _jitter_fixed30(base, +1), _train_aug30(orig)]
        for c, img in enumerate(panels):
            ax = axes[r, c]
            ax.imshow(np.asarray(img)); ax.set_xticks([]); ax.set_yticks([])
            if is_bach and 2 <= c <= 7:          # BACH is never augmented: grey the panels out
                ax.imshow(np.full((8, 8, 3), 1.0), alpha=0.72, extent=ax.get_xlim() + ax.get_ylim())
                ax.text(0.5, 0.5, 'not applied\n(external test set)', transform=ax.transAxes,
                        ha='center', va='center', fontsize=8, color='0.25')
            if c == 0:
                ax.set_ylabel(row_name, fontsize=12, fontweight='bold')
                ax.set_xlabel(f'{w}×{h} px', fontsize=8.5)
            if c == 1:
                ax.set_xlabel(f'scale ×{PREP_IMAGE_SIZE / w:.2f} (width), ×{PREP_IMAGE_SIZE / h:.2f} (height)',
                              fontsize=8)
        # last column: what the network actually receives, as per-channel value histograms
        x = _to_input30(base).numpy()                                            # (3, 224, 224)
        ax = axes[r, -1]
        for ch, colour in enumerate(('#d62728', '#2ca02c', '#1f77b4')):
            ax.hist(x[ch].ravel(), bins=60, range=(-2.5, 2.8), histtype='step', color=colour, linewidth=1.2)
        ax.axvline(0, color='0.3', linewidth=0.8, linestyle='--')
        ax.set_yticks([]); ax.tick_params(axis='x', labelsize=7)
        ax.set_xlabel('z (ImageNet mean/std)', fontsize=8)
    for c, title in enumerate(_cols30):
        axes[0, c].set_title(title, fontsize=9, fontweight='bold')
    fig.suptitle(f'Preprocessing as applied in training (patient {_patient30}; exact torchvision transforms; '
                 f'last column: red / green / blue channel values after normalisation)',
                 fontsize=11.5, fontweight='bold', y=1.0)
    fig.text(0.5, 0.004,
             f'Columns 3-7 show each step at a fixed setting (rotation at its +{PREP_ROTATION}° limit, colour jitter at both '
             'limits); in training each is drawn at random, as in column 8. ConvNeXt-T, Swin-T and MedMamba get a new draw '
             'every epoch; the frozen Phikon probe was trained on one cached draw per image.',
             ha='center', va='bottom', fontsize=9, color='0.3')
    fig.tight_layout(rect=(0, 0.025, 1, 1))
    save_fig(fig, 'fig30_preprocessing_pipeline')
    plt.show()

    # ---- Fig 31: colour statistics per magnification and for BACH, before and after normalisation ----
    _rng30 = np.random.default_rng(PREP_SEED)
    _groups30 = []
    for m in _mags30:
        pool = _idx30[_idx30.mag == m].path.to_numpy()
        _groups30.append((f'{m}×', _rng30.choice(pool, size=min(PREP_STATS_PER_GROUP, len(pool)), replace=False)))
    if _bach_files30:
        _groups30.append(('BACH', _rng30.choice(np.array([str(p) for p in _bach_files30]),
                                                size=min(PREP_STATS_PER_GROUP, len(_bach_files30)), replace=False)))
    _rows30 = []
    for name, paths in _groups30:
        for p in paths:
            with _Img30.open(p) as im:
                img = im.convert('RGB'); size = img.size
            a = np.asarray(_resize30(img), dtype=np.float64) / 255.0              # (224, 224, 3) in [0, 1]
            z = (a - np.array(PREP_MEAN)) / np.array(PREP_STD)                  # (224, 224, 3)
            _rows30.append({'group': name, 'width': size[0], 'height': size[1],
                            **{f'raw_{c}': a[..., i].mean() for i, c in enumerate('RGB')},
                            **{f'z_{c}': z[..., i].mean() for i, c in enumerate('RGB')},
                            'brightness': a.mean(), 'contrast': a.std()})
    preprocessing_stats = pd.DataFrame(_rows30)
    preprocessing_stats.to_csv(f'{OUTPUT_DIR}/tables/preprocessing_colour_stats_per_image.csv', index=False)
    _names30 = [g for g, _ in _groups30]

    fig, axes = plt.subplots(2, 3, figsize=(15, 7.6), sharex=True)
    for j, (ch, colour) in enumerate(zip('RGB', ('#d62728', '#2ca02c', '#1f77b4'))):
        for i, (prefix, ref, ylabel) in enumerate([('raw', PREP_MEAN[j], 'mean intensity after resize (0-1)'),
                                                   ('z', 0.0, 'mean after ImageNet normalisation (z)')]):
            ax = axes[i, j]
            data = [preprocessing_stats.loc[preprocessing_stats.group == g, f'{prefix}_{ch}'].to_numpy()
                    for g in _names30]
            parts = ax.violinplot(data, showmedians=True, widths=0.8)
            for body in parts['bodies']:
                body.set_facecolor(colour); body.set_alpha(0.35); body.set_edgecolor('black')
            for key in ('cmedians', 'cmins', 'cmaxes', 'cbars'):
                parts[key].set_color('0.2')
            ax.axhline(ref, color='0.3', linestyle='--', linewidth=1)
            ax.set_xticks(range(1, len(_names30) + 1))
            ax.set_xticklabels([f'{g}\n(n = {len(d)})' for g, d in zip(_names30, data)], fontsize=9)
            ax.set_title(f'{ch} channel', fontsize=11, fontweight='bold')
            if j == 0:
                ax.set_ylabel(ylabel)
            ax.grid(True, axis='y', linewidth=0.4, alpha=0.6)
    fig.suptitle(f'Per-image colour statistics by held-out magnification and for BACH '
                 f'(up to {PREP_STATS_PER_GROUP} random images per group, after Resize {PREP_IMAGE_SIZE}×{PREP_IMAGE_SIZE})',
                 fontsize=12, fontweight='bold')
    fig.text(0.5, 0.005, 'Violins: per-image channel means (bars: median and range). Dashed line: the ImageNet '
                         'channel mean used for normalisation (top row) and z = 0 (bottom row); a histology image '
                         'is ImageNet-centred only if it sits on the line.', ha='center', fontsize=9, color='0.3')
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    save_fig(fig, 'fig31_colour_statistics')
    plt.show()

    _summ30 = (preprocessing_stats.groupby('group', sort=False)
               .agg(n_images=('group', 'size'), width=('width', lambda s: int(s.mode().iat[0])),
                    height=('height', lambda s: int(s.mode().iat[0])),
                    **{f'raw_{c}_mean': (f'raw_{c}', 'mean') for c in 'RGB'},
                    **{f'z_{c}_mean': (f'z_{c}', 'mean') for c in 'RGB'},
                    brightness_mean=('brightness', 'mean'), contrast_mean=('contrast', 'mean'))
               .reindex(_names30))
    _summ30.to_csv(f'{OUTPUT_DIR}/tables/preprocessing_colour_stats_summary.csv')
    print('\nColour statistics per group (means over images; z = after ImageNet normalisation):')
    print(_summ30.round(3).to_string())
    _zoff30 = _summ30[[f'z_{c}_mean' for c in 'RGB']].abs().max(axis=1)
    print('largest |mean z| per group (0 would mean ImageNet statistics fit):', _zoff30.round(2).to_dict())

    # ---- T13: the preprocessing, as a table ----
    _t13 = [
        ('Resize', f'{PREP_IMAGE_SIZE}$\\times${PREP_IMAGE_SIZE}, bilinear, aspect ratio not kept', 'all'),
        ('Horizontal flip', '$p = 0.5$', 'train$^{*}$'),
        ('Vertical flip', '$p = 0.5$', 'train$^{*}$'),
        ('Rotation', f'uniform in $\\pm{PREP_ROTATION}^\\circ$, nearest-neighbour, corners filled black', 'train$^{*}$'),
        ('Colour jitter', 'brightness and contrast $\\times[0.9, 1.1]$, saturation $\\times[0.95, 1.05]$, no hue', 'train$^{*}$'),
        ('Normalisation', 'ImageNet mean (0.485, 0.456, 0.406), s.d. (0.229, 0.224, 0.225)', 'all'),
    ]
    _l13 = [r'\begin{tabular}{lll}', r'\toprule', r'Step & Setting & Applied to \\', r'\midrule']
    _l13 += [f'{a} & {b} & {c}' + r' \\' for a, b, c in _t13]
    _l13 += [r'\midrule',
             r'\multicolumn{3}{l}{Native size: BreakHis ' +
             ', '.join(sorted({f"{int(_summ30.loc[g, 'width'])}$\\times${int(_summ30.loc[g, 'height'])}"
                              for g in _names30 if g != 'BACH'})) +
             (f"; BACH {int(_summ30.loc['BACH', 'width'])}$\\times${int(_summ30.loc['BACH', 'height'])}"
              if 'BACH' in _summ30.index else '') + r'} \\',
             r'\multicolumn{3}{l}{$^{*}$ConvNeXt-T, Swin-T, MedMamba: a new random draw every epoch. Phikon probe: '
             r'one draw per image,} \\',
             r'\multicolumn{3}{l}{\phantom{$^{*}$}encoded once and cached for all 200 epochs. Phikon receives '
             r'the same tensors (its own image processor is not used).} \\',
             r'\bottomrule', r'\end{tabular}']
    _P30(f'{OUTPUT_DIR}/tables/T13_preprocessing.tex').write_text('\n'.join(_l13) + '\n', encoding='utf-8')
    print('  wrote tables/T13_preprocessing.tex, preprocessing_colour_stats_summary.csv, '
          'preprocessing_colour_stats_per_image.csv')

In [ ]:
# ---------- Fig 16: accuracy and reliability for every encoder x held-out magnification ----------
# (a) macro-F1 and (b) post-temperature-scaling ECE as 3D bars over the encoder x held-out
# magnification grid (means over seeds, from `summary`); (c) the rank of each encoder by
# accuracy (macro-F1, higher is better) and by reliability (ECE, lower is better), averaged over
# magnifications. Bars, not a surface: the encoder axis is categorical, so interpolating between
# encoders would draw values no model produced. Bars start at zero so heights are comparable.
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the '3d' projection)
from scipy.stats import spearmanr

SHORT_NAME = {'Phikon (frozen FM)': 'Phikon', 'ConvNeXt-T': 'ConvNeXt-T', 'Swin-T': 'Swin-T',
              'MedMamba': 'MedMamba', 'DSA-Mamba': 'DSA-Mamba', 'MambaVision': 'MambaVision'}

_f16 = summary[summary.method.isin(ORDER)]
F16_F1 = _f16.pivot(index='method', columns='held_out_mag', values='test_macro_f1_mean').reindex(index=ORDER, columns=MAGS)
F16_ECE = _f16.pivot(index='method', columns='held_out_mag', values='test_ece_mean').reindex(index=ORDER, columns=MAGS)
if F16_F1.isna().any().any() or F16_ECE.isna().any().any():
    print('Skipped Fig 16 - some (encoder, held-out magnification) cells have no summary row:')
    print(F16_F1.isna().stack()[lambda s: s].index.tolist())
else:
    def _bars3d(ax, table, zlabel, zmax, title):
        n_enc, n_mag = table.shape
        xs, ys = np.meshgrid(np.arange(n_enc), np.arange(n_mag), indexing='ij')     # (n_enc, n_mag) each
        xs, ys = xs.ravel(), ys.ravel()                                               # (n_enc*n_mag,)
        dz = table.to_numpy(dtype=float).ravel()                                      # (n_enc*n_mag,)
        colors = [COLORS[ORDER[i]] for i in xs]
        # One bar3d call: matplotlib depth-sorts the bars of a single collection together.
        ax.bar3d(xs - 0.28, ys - 0.28, np.zeros_like(dz), 0.56, 0.56, dz,
                 color=colors, edgecolor='black', linewidth=0.3, alpha=0.92, shade=True)
        ax.set_xticks(np.arange(n_enc)); ax.set_xticklabels([SHORT_NAME.get(m, m) for m in ORDER],
                                                            rotation=-42, ha='left', va='top', fontsize=7.5)
        ax.set_yticks(np.arange(n_mag)); ax.set_yticklabels([f'{m}×' for m in MAGS], fontsize=8.5)
        ax.set_zlim(0, zmax); ax.set_zlabel(zlabel, labelpad=6)
        ax.set_ylabel('Held-out magnification', labelpad=8)
        ax.view_init(elev=24, azim=-50)
        ax.set_title(title, fontweight='bold', fontsize=11, pad=2)

    fig = plt.figure(figsize=(18.5, 5.6))
    _gs16 = fig.add_gridspec(1, 3, width_ratios=[1.0, 1.0, 0.85], wspace=0.22)
    ax_f1 = fig.add_subplot(_gs16[0, 0], projection='3d')
    _bars3d(ax_f1, F16_F1, 'Macro-F1 (higher is better)', 1.0, '(a) Accuracy: macro-F1')
    ax_ece = fig.add_subplot(_gs16[0, 1], projection='3d')
    _bars3d(ax_ece, F16_ECE, 'ECE (lower is better)', float(np.ceil(F16_ECE.to_numpy().max() * 1.15 * 50) / 50),
            '(b) Reliability: ECE after temperature scaling')

    # ---- (c) rank by accuracy vs rank by reliability (mean over magnifications) ----
    ax = fig.add_subplot(_gs16[0, 2])
    f1_mean, ece_mean = F16_F1.mean(axis=1), F16_ECE.mean(axis=1)
    rank_acc = f1_mean.rank(ascending=False, method='min').astype(int)     # 1 = highest macro-F1
    rank_rel = ece_mean.rank(ascending=True, method='min').astype(int)     # 1 = lowest ECE
    for m in ORDER:
        ax.plot([0, 1], [rank_acc[m], rank_rel[m]], color=COLORS[m], linewidth=2.6,
                marker=MARKERS[m], markersize=9, markeredgecolor='black', markeredgewidth=0.6)
        ax.text(-0.12, rank_acc[m], f'#{rank_acc[m]}  {SHORT_NAME.get(m, m)}  {f1_mean[m]:.3f}',
                ha='right', va='center', fontsize=9)
        ax.text(1.12, rank_rel[m], f'{ece_mean[m]:.3f}  {SHORT_NAME.get(m, m)}  #{rank_rel[m]}',
                ha='left', va='center', fontsize=9)
    ax.set_xlim(-1.3, 2.3); ax.set_ylim(len(ORDER) + 0.5, 0.5)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['Rank by\nmacro-F1', 'Rank by\nECE'], fontsize=10)
    ax.set_yticks([]); ax.set_ylabel('')
    ax.grid(False)
    for side in ('top', 'right', 'bottom', 'left'):
        ax.spines[side].set_visible(False)
    ax.set_title('(c) Accuracy rank vs reliability rank\n(mean over held-out magnifications)',
                 fontweight='bold', fontsize=11)

    plt.suptitle('Accuracy and reliability across encoders and held-out magnifications', y=1.02, fontweight='bold')
    save_fig(fig, 'fig16_accuracy_reliability_3d')
    plt.show()

    # Across the 16 (encoder, magnification) cells: does higher accuracy go with lower ECE?
    rho, p_rho = spearmanr(F16_F1.to_numpy().ravel(), F16_ECE.to_numpy().ravel())
    per_mag_winner = pd.DataFrame({'best macro-F1': F16_F1.idxmax(axis=0), 'lowest ECE': F16_ECE.idxmin(axis=0)})
    per_mag_winner['same encoder'] = per_mag_winner['best macro-F1'] == per_mag_winner['lowest ECE']
    print('Rank by macro-F1 :', rank_acc.sort_values().to_dict())
    print('Rank by ECE      :', rank_rel.sort_values().to_dict())
    print(f'Spearman rho(macro-F1, ECE) over {F16_F1.size} cells = {rho:+.3f} (p = {p_rho:.3f}); '
          'negative means more accurate cells tend to be better calibrated.')
    print('\nPer held-out magnification:')
    print(per_mag_winner.to_string())

## 2f. Outputs at every held-out magnification (Fig 9b, 10b, 11c)

Fig 9–11 above show held-out 400× only. These cells repeat them for **40×, 100×, 200× and 400×**:

- **Fig 9b** — confusion matrices, rows = held-out magnification, columns = encoder, summed over the three seeds. Reads `test_outputs.npz` only; no images or GPU.
- **Fig 10b** — one figure per held-out magnification: the most confident TP / TN / FP / FN test image of every encoder (seed 13). The stored labels are checked position by position against the rebuilt test set before any image is captioned.
- **Fig 11c** — one figure per held-out magnification: Grad-CAM++ for ConvNeXt-T, Swin-T and MedMamba and Attention Rollout for the frozen Phikon (Grad-CAM is not meaningful on ViT patch tokens), on the first two benign and first two malignant test images of the rebuilt test set, i.e. not hand-picked. Images go through the exact training transform (`Resize((224, 224))` on the full image). Needs the Fig 11 cell to have run (it defines the model builders and installs `grad-cam` / `mamba_ssm`).

Treat 10b and 11c as illustrations: four images per magnification support no quantitative claim (see T8 for the faithfulness test).

In [ ]:
# ---------- Fig 9b: confusion matrices at every held-out magnification (summed over seeds) ----------
CLASS_NAMES = ['Benign', 'Malignant']
_cm_rows = []
fig, axes = plt.subplots(len(MAGS), len(ORDER), figsize=(2.95 * len(ORDER), 2.75 * len(MAGS)), squeeze=False)
for r, ho in enumerate(MAGS):
    for c, method_name in enumerate(ORDER):
        ax = axes[r, c]
        prefix = PREFIX_MAP.get(method_name)
        y_parts, p_parts = [], []
        for seed in SEEDS:
            npz_path = f'runs/{prefix}_ho{ho}_frac1.0_seed{seed}/test_outputs.npz'
            if prefix and reader.exists(npz_path):
                d = np.load(io.BytesIO(reader.open_bytes(npz_path)))
                y_parts.append(d['labels'].astype(int))            # (N_seed,)
                p_parts.append(d['probs'].argmax(axis=1))            # (N_seed,) temperature does not move argmax
        if not y_parts:
            ax.axis('off'); ax.set_title(f'{method_name}\n{ho}× - no data', fontsize=9); continue
        y_true, y_pred = np.concatenate(y_parts), np.concatenate(p_parts)
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])                      # (2, 2) rows = true
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)    # row-normalised recall
        annot = [[f'{cm[i, j]}\n({cm_norm[i, j]:.2f})' for j in range(2)] for i in range(2)]
        sns.heatmap(cm_norm, annot=annot, fmt='', cmap='Blues', vmin=0, vmax=1, cbar=False,
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax, linewidth=0.5, linecolor='white',
                    annot_kws={'fontweight': 'bold', 'fontsize': 9})
        ax.set_xlabel('Predicted' if r == len(MAGS) - 1 else '')
        ax.set_ylabel('True' if c == 0 else '')
        if r == 0:
            ax.set_title(method_name, fontsize=10.5, fontweight='bold')
        _cm_rows.append({'method': method_name, 'held_out_mag': ho, 'n': len(y_true), 'seeds': len(y_parts),
                         'benign_recall': cm_norm[0, 0], 'malignant_recall': cm_norm[1, 1],
                         'false_negatives': int(cm[1, 0]), 'false_positives': int(cm[0, 1])})
plt.suptitle('Confusion matrices at every held-out magnification (row-normalised; summed over seeds)',
             y=1.01, fontweight='bold')
plt.tight_layout()
# Row labels carry the number of test images; it is the same for every encoder when all seeds are present.
_n_by_mag = pd.DataFrame(_cm_rows).groupby('held_out_mag').n.agg(['min', 'max']) if _cm_rows else None
for r, ho in enumerate(MAGS):
    if _n_by_mag is None or ho not in _n_by_mag.index:
        continue
    lo_n, hi_n = int(_n_by_mag.loc[ho, 'min']), int(_n_by_mag.loc[ho, 'max'])
    n_txt = f'n = {lo_n:,}' if lo_n == hi_n else f'n = {lo_n:,}-{hi_n:,}'
    axes[r, 0].annotate(f'held-out {ho}×\n{n_txt}', xy=(0, 0.5), xycoords='axes fraction', xytext=(-58, 0),
                        textcoords='offset points', ha='right', va='center', fontsize=10.5, fontweight='bold')
save_fig(fig, 'fig9b_confusion_matrices_all_mags')
plt.show()

confusion_by_mag = pd.DataFrame(_cm_rows)
confusion_by_mag.to_csv(f'{OUTPUT_DIR}/tables/confusion_by_magnification.csv', index=False)
print(confusion_by_mag.pivot(index='method', columns='held_out_mag', values='false_negatives')
      .reindex(ORDER).rename_axis(columns='missed malignant images (FN) by held-out mag').to_string())
print()
print(confusion_by_mag.pivot(index='method', columns='held_out_mag', values='benign_recall')
      .reindex(ORDER).round(2).rename_axis(columns='benign recall by held-out mag').to_string())

In [ ]:
# ---------- Fig 10b: most confident TP / TN / FP / FN per encoder, one figure per held-out magnification ----------
from PIL import Image as PILImage
SAMPLES_SEED = SEEDS[0] if SEEDS else 13
QUADRANTS = [('TP', 'True positive', 1, 1), ('TN', 'True negative', 0, 0),
             ('FP', 'False positive', 0, 1), ('FN', 'False negative', 1, 0)]


def render_sample_predictions(ho, seed):
    frame = reconstruct_test_paths(ho, seed)
    if frame is None or len(frame) == 0:
        print(f'  ho={ho}: test set could not be rebuilt - skipped'); return
    fig, axes = plt.subplots(len(ORDER), 4, figsize=(13.2, 3.05 * len(ORDER)), squeeze=False)
    for row_idx, method_name in enumerate(ORDER):
        prefix = PREFIX_MAP.get(method_name)
        npz_path = f'runs/{prefix}_ho{ho}_frac1.0_seed{seed}/test_outputs.npz'
        if not (prefix and reader.exists(npz_path)):
            for ax in axes[row_idx]: ax.axis('off')
            axes[row_idx, 0].set_title(f'{method_name}: no stored predictions', fontsize=9); continue
        d = np.load(io.BytesIO(reader.open_bytes(npz_path)))
        labels, probs = d['labels'].astype(int), d['probs']                  # (N,), (N, 2)
        preds, confs = probs.argmax(axis=1), probs.max(axis=1)                # (N,), (N,)
        # Positional pairing is only trusted if every stored label matches the rebuilt frame.
        if len(labels) != len(frame) or not np.array_equal(labels, frame.coarse_label.to_numpy()):
            for ax in axes[row_idx]: ax.axis('off')
            axes[row_idx, 0].set_title(f'{method_name}: stored rows do not match the rebuilt test set - skipped',
                                       fontsize=9, color='#d62728')
            print(f'  ho={ho} {method_name}: SKIPPED, {len(labels)} stored rows vs {len(frame)} rebuilt rows '
                  f'or label order differs'); continue
        for col_idx, (code, title, y_val, p_val) in enumerate(QUADRANTS):
            ax = axes[row_idx, col_idx]
            mask = (labels == y_val) & (preds == p_val)                       # (N,)
            ax.set_xticks([]); ax.set_yticks([])
            if row_idx == 0:
                ax.set_title(f'{title} ({code})', fontsize=11, fontweight='bold')
            if not mask.any():
                ax.text(0.5, 0.5, f'no {code}\n(0 of {len(labels)})', ha='center', va='center',
                        transform=ax.transAxes, fontsize=10, color='0.4')
                continue
            idx = np.flatnonzero(mask)[np.argmax(confs[mask])]
            ax.imshow(PILImage.open(frame.iloc[idx]['path']).convert('RGB'))
            wrong = code in ('FP', 'FN')
            for s in ax.spines.values():
                s.set_visible(True); s.set_linewidth(2.2); s.set_edgecolor('#d62728' if wrong else '#2ca02c')
            ax.set_xlabel(f'p = {confs[idx]:.2f}   ({int(mask.sum())} {code} in {len(labels)})', fontsize=9)
        axes[row_idx, 0].set_ylabel(method_name, fontsize=11, fontweight='bold')
    plt.suptitle(f'Most confident prediction per outcome - held-out magnification {ho}×, seed {seed}\n'
                 f'(p = temperature-scaled confidence of the predicted class; illustrative, not a sample of typical cases)',
                 y=1.01, fontsize=12, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, f'fig10b_sample_predictions_ho{ho}', dpi=200)
    plt.show()


if not HAS_IMAGES:
    print('Skipped Fig 10b - BreakHis not attached.')
else:
    for _ho in MAGS:
        render_sample_predictions(int(_ho), int(SAMPLES_SEED))

In [ ]:
# ---------- Fig 11c: Grad-CAM++ (task-trained encoders) + Attention Rollout (Phikon), every held-out magnification ----------
# Rows = the first two benign and first two malignant images of the rebuilt test set (manifest
# order, not hand-picked); columns = input + one map per encoder. Every image goes through the
# exact training transform, and the confidence shown is the run's temperature-scaled softmax.
# Reuses the model builders defined by the Fig 11 cell (build_from_scratch, get_target_layer,
# to_nchw, IMAGE_TRANSFORM and the pytorch_grad_cam imports).
SALIENCY_SEED = SEEDS[0] if SEEDS else 13
SALIENCY_PER_CLASS = 2
_F11C_NEEDS = ['build_from_scratch', 'get_target_layer', 'to_nchw', 'IMAGE_TRANSFORM',
               'GradCAMPlusPlus', 'ClassifierOutputTarget', 'show_cam_on_image']


def _run_temperature(run_id):
    row = raw[raw.run_id == run_id]
    if len(row) and 'test_temperature' in row.columns and pd.notna(row.iloc[0]['test_temperature']):
        return float(row.iloc[0]['test_temperature'])
    return 1.0


if not HAS_IMAGES:
    print('Skipped Fig 11c - BreakHis not attached.')
elif [n for n in _F11C_NEEDS if n not in globals()]:
    print('Skipped Fig 11c - run the Fig 11 (Grad-CAM++) cell first; missing:',
          [n for n in _F11C_NEEDS if n not in globals()])
else:
    import torch
    import torch.nn.functional as F
    from torchvision import transforms as _tf
    _dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    _display_resize = _tf.Resize((224, 224))
    _reverse_display = {v: k for k, v in DISPLAY.items()}
    GREEN, RED = '#2ca02c', '#d62728'

    def _attention_rollout(attentions):
        """Abnar & Zuidema (2020). attentions: L tensors (1, H, N, N), N = 1 CLS + 196 patches."""
        result = None
        for A in attentions:
            A = A.mean(dim=1)[0]                                            # (N, N) mean over heads
            A = 0.5 * A + 0.5 * torch.eye(A.size(-1), device=A.device)      # (N, N) residual path
            A = A / A.sum(dim=-1, keepdim=True)                             # (N, N) row-stochastic
            result = A if result is None else A @ result                    # (N, N)
        mask = result[0, 1:]                                                # (N-1,) CLS -> patches
        g = int(round(mask.numel() ** 0.5))
        mask = mask[: g * g].reshape(1, 1, g, g)                            # (1, 1, 14, 14)
        mask = F.interpolate(mask, size=(224, 224), mode='bilinear', align_corners=False)[0, 0]  # (224, 224)
        mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)
        return mask.detach().float().cpu().numpy()

    _rollout_fm = None
    if 'Phikon (frozen FM)' in ORDER:
        try:
            from transformers import AutoModel
            try:
                _rollout_fm = AutoModel.from_pretrained('owkin/phikon', add_pooling_layer=False,
                                                        attn_implementation='eager').to(_dev).eval()
            except TypeError:
                _rollout_fm = AutoModel.from_pretrained('owkin/phikon', add_pooling_layer=False).to(_dev).eval()
            for p in _rollout_fm.parameters():
                p.requires_grad_(False)
        except Exception as exc:
            print('  Phikon could not be loaded for Attention Rollout:', repr(exc)[:200])

    for ho in MAGS:
        ho = int(ho)
        frame = reconstruct_test_paths(ho, SALIENCY_SEED)
        if frame is None or len(frame) == 0:
            print(f'  ho={ho}: test set could not be rebuilt - skipped'); continue
        picks = []
        for lab in (0, 1):
            picks.extend(frame.index[frame.coarse_label == lab].tolist()[:SALIENCY_PER_CLASS])
        pil_imgs = [PILImage.open(frame.loc[i, 'path']).convert('RGB') for i in picks]
        x_all = torch.stack([IMAGE_TRANSFORM(im) for im in pil_imgs]).to(_dev)            # (S, 3, 224, 224)
        show = [np.asarray(_display_resize(im), dtype=np.float32) / 255.0 for im in pil_imgs]  # S x (224, 224, 3)
        true_labels = [int(frame.loc[i, 'coarse_label']) for i in picks]
        maps = {m: [None] * len(picks) for m in ORDER}

        for method_name in ORDER:
            prefix = PREFIX_MAP.get(method_name)
            run_id = f'{prefix}_ho{ho}_frac1.0_seed{SALIENCY_SEED}'
            temperature = _run_temperature(run_id)
            try:
                if method_name == 'Phikon (frozen FM)':
                    probe_path = f'runs/{run_id}/probe.pt'
                    if _rollout_fm is None or not reader.exists(probe_path):
                        print(f'  ho={ho} Phikon: backbone or probe.pt missing - column blank'); continue
                    saved = torch.load(io.BytesIO(reader.open_bytes(probe_path)), map_location=_dev)
                    head = torch.nn.Linear(int(saved['feature_dim']), int(saved['num_classes'])).to(_dev).eval()
                    head.load_state_dict({k.replace('head.', ''): v for k, v in saved['state_dict'].items()})
                    for s_i in range(len(picks)):
                        with torch.no_grad():
                            out = _rollout_fm(pixel_values=x_all[s_i:s_i + 1], output_attentions=True)
                            logits = head(out.last_hidden_state[:, 0])                              # (1, 2)
                            probs = torch.softmax(logits / temperature, dim=1)[0]                   # (2,)
                        if not getattr(out, 'attentions', None):
                            raise RuntimeError('no attention weights returned (needs attn_implementation="eager")')
                        heat = _attention_rollout(out.attentions)
                        pred = int(probs.argmax())
                        maps[method_name][s_i] = (show_cam_on_image(show[s_i], heat, use_rgb=True), pred, float(probs[pred]))
                else:
                    ckpt_path = f'runs/{run_id}/best.pt'
                    model_name = _reverse_display.get(method_name)
                    if not reader.exists(ckpt_path):
                        print(f'  ho={ho} {method_name}: best.pt missing - column blank'); continue
                    model = build_from_scratch(model_name)
                    model.load_state_dict(torch.load(io.BytesIO(reader.open_bytes(ckpt_path)), map_location=_dev))
                    model = model.to(_dev).eval()
                    cam = GradCAMPlusPlus(model=model, target_layers=get_target_layer(model, model_name),
                                          reshape_transform=to_nchw)
                    for s_i in range(len(picks)):
                        x = x_all[s_i:s_i + 1]                                                      # (1, 3, 224, 224)
                        with torch.no_grad():
                            probs = torch.softmax(model(x) / temperature, dim=1)[0]                 # (2,)
                        pred = int(probs.argmax())
                        heat = cam(input_tensor=x, targets=[ClassifierOutputTarget(pred)])[0]      # (224, 224)
                        maps[method_name][s_i] = (show_cam_on_image(show[s_i], heat, use_rgb=True), pred, float(probs[pred]))
                    del cam, model
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            except Exception as exc:
                print(f'  ho={ho} {method_name}: FAILED - {repr(exc)[:200]}')

        n_rows, n_cols = len(picks), len(ORDER) + 1
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.55 * n_cols, 2.75 * n_rows), squeeze=False,
                                 gridspec_kw={'wspace': 0.06, 'hspace': 0.34})
        for r in range(n_rows):
            ax = axes[r, 0]
            ax.imshow(show[r]); ax.set_xticks([]); ax.set_yticks([])
            ax.set_ylabel(f'True: {CLASS_NAMES[true_labels[r]]}', fontsize=10.5, fontweight='bold')
            if r == 0:
                ax.set_title('Input', fontsize=11, fontweight='bold')
            for c, method_name in enumerate(ORDER, start=1):
                ax = axes[r, c]; ax.set_xticks([]); ax.set_yticks([])
                if r == 0:
                    kind = 'Attention Rollout' if method_name == 'Phikon (frozen FM)' else 'Grad-CAM++'
                    ax.set_title(f'{SHORT_NAME.get(method_name, method_name) if "SHORT_NAME" in globals() else method_name}\n{kind}',
                                 fontsize=10.5, fontweight='bold')
                item = maps[method_name][r]
                if item is None:
                    ax.imshow(np.full((224, 224, 3), 0.94)); ax.text(0.5, 0.5, 'not available', ha='center',
                                                                       va='center', transform=ax.transAxes, color='0.5')
                    continue
                overlay, pred, conf = item
                ax.imshow(overlay)
                col = GREEN if pred == true_labels[r] else RED
                for s in ax.spines.values():
                    s.set_visible(True); s.set_linewidth(2.2); s.set_edgecolor(col)
                ax.set_xlabel(f'pred {CLASS_NAMES[pred]} ({conf:.2f})', fontsize=9, color=col, fontweight='bold')
        plt.suptitle(f'Class-discriminative regions - held-out magnification {ho}×, seed {SALIENCY_SEED}\n'
                     '(first 2 benign + first 2 malignant test images; green = correct, red = wrong)',
                     fontsize=12, fontweight='bold', y=1.01)
        save_fig(fig, f'fig11c_saliency_ho{ho}', dpi=200)
        plt.show()

    if _rollout_fm is not None:
        del _rollout_fm
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## 2g. Reference-Normalized Energy (RNE) and per-image OOD separability (Fig 17–18, Fig 12b, T11–T12)

**Definition.** For a run with classifier logits $f(x)$, the energy is $E(x) = -\log\sum_k \exp f_k(x)$ (Liu et al., 2020; higher = less in-distribution). Its **Reference-Normalized Energy** is

$$\mathrm{RNE}(x) = \frac{E(x) - \mu_{\mathrm{ref}}}{\sigma_{\mathrm{ref}}},$$

where $\mu_{\mathrm{ref}}, \sigma_{\mathrm{ref}}$ are the mean and SD of $E$ over the run's **validation set** — in-distribution patients at the three training magnifications, already used for model selection and temperature scaling, and never the test set. RNE expresses how unusual an input is in units of the model's own in-distribution spread, so a model that is unconfident on every input no longer looks shift-aware.

**What RNE can and cannot change.** Within one run RNE is an increasing affine map of $E$, so any rank-based metric computed inside that run (AUROC, AUPR, FPR95) is **identical** for RNE and raw energy — the scoring cell asserts this rather than reporting two equal rows. RNE matters wherever scores from *different* runs meet: (i) pooled AUROC across encoders or magnifications, (ii) one fixed threshold deployed across encoders, and (iii) run-level gaps compared between encoders. Those are measured in the pooled cell. Reference normalization is applied the same way to MSP, Mahalanobis and kNN scores, so the effect of the normalization is separated from the effect of the score.

**Baselines.** Maximum softmax probability (Hendrycks & Gimpel, 2017); feature-space Mahalanobis distance with class-conditional means and a Ledoit–Wolf tied covariance fitted on the run's training features (Lee et al., 2018, penultimate layer only, no input preprocessing); and the distance to the $k$-th nearest L2-normalized training feature (Sun et al., 2022; $k = 10$ fixed in advance, $k = 1$ and $50$ reported as sensitivity). Features are the input of each model's final linear classifier, verified by reproducing the logits from them.

**Evaluation.** ID = the run's own held-out-magnification test set, OOD = the 400 BACH images (another institution). AUROC and AUPR-Out treat BACH as positive; FPR95 is the fraction of BACH images accepted when 95% of ID test images are accepted. A secondary pair (validation vs held-out test) measures whether the score flags the unseen magnification itself.

**Requirements and runtime.** GPU T4; `ambarish/breakhis` **and** `truthisneverlinear/bach-breast-cancer-histology-images` attached; the Fig 11 cell must have run (model builders, `mamba_ssm`). Training kept only per-run BACH means, so every checkpoint is re-scored: roughly 20–40 min for 48 runs on a T4. Per-run arrays are cached to `/kaggle/working/ood_cache/`; attach this notebook's output as an input to a later version and the cache is reused instead of recomputed. Every run is checked against its stored `test_outputs.npz` (labels identical, logits within tolerance) before it enters the analysis.

In [ ]:
# ---------- OOD scoring library + synthetic self-test (CPU, ~20 s) ----------
# Everything the RNE / OOD cells compute goes through these functions, and the self-test below
# checks each one against a case with a known answer before any GPU time is spent.
import json as _json
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from sklearn.covariance import LedoitWolf
from scipy.stats import norm as _norm

OOD_CACHE_DIR = Path('/kaggle/working/ood_cache')
OOD_CACHE_VERSION = 1
OOD_ARRAY_KEYS = ('train_feats', 'train_labels', 'val_logits', 'val_feats', 'val_labels',
                  'test_logits', 'test_feats', 'test_labels', 'bach_logits', 'bach_feats', 'bach_class')
KNN_K = 10                      # fixed before looking at results
KNN_K_SENSITIVITY = (1, 50)     # reported, never used to pick the main result
SIGMA_FLOOR = 1e-8              # guards the reference SD in the normalisation
SCORE_LABELS = {'msp': 'MSP', 'energy': 'Energy', 'rne': 'RNE', 'maha': 'Mahalanobis', 'knn': f'kNN (k={KNN_K})'}


# ---- logit-based scores (higher = more out-of-distribution) ----
def stable_softmax(logits):
    z = np.asarray(logits, dtype=np.float64)                  # (N, C)
    z = z - z.max(axis=1, keepdims=True)                      # (N, C) max-shift: no overflow
    e = np.exp(z)                                             # (N, C) in (0, 1]
    return e / e.sum(axis=1, keepdims=True)                   # (N, C) rows sum to 1


def energy_from_logits(logits):
    """E(x) = -logsumexp_k f_k(x), T = 1 (Liu et al., 2020). Same sign as energy_gap in the load cell."""
    z = np.asarray(logits, dtype=np.float64)                  # (N, C)
    m = z.max(axis=1, keepdims=True)                          # (N, 1)
    return -(m[:, 0] + np.log(np.exp(z - m).sum(axis=1)))     # (N,)


def neg_msp(logits):
    return -stable_softmax(logits).max(axis=1)                # (N,) in [-1, -1/C]


def reference_normalize(scores, reference_scores):
    """(s - mean_ref) / sd_ref. Returns the normalised scores and (mean_ref, sd_ref)."""
    ref = np.asarray(reference_scores, dtype=np.float64)      # (N_ref,)
    mu, sd = float(ref.mean()), float(ref.std(ddof=1))
    if not np.isfinite(sd) or sd < SIGMA_FLOOR:
        print(f'  WARNING: reference SD {sd:.3g} below floor; using {SIGMA_FLOOR}')
        sd = SIGMA_FLOOR
    return (np.asarray(scores, dtype=np.float64) - mu) / sd, (mu, sd)


# ---- feature-space baselines ----
def fit_mahalanobis(train_feats, train_labels):
    """Class-conditional means + one Ledoit-Wolf tied covariance on class-centred training features."""
    X = np.asarray(train_feats, dtype=np.float64)             # (N_tr, D)
    y = np.asarray(train_labels).astype(int)                  # (N_tr,)
    classes = np.unique(y)
    means = np.stack([X[y == c].mean(axis=0) for c in classes])            # (C, D)
    centred = X - means[np.searchsorted(classes, y)]                       # (N_tr, D)
    precision = LedoitWolf(assume_centered=True).fit(centred).precision_   # (D, D) symmetric PD
    return means, precision


def mahalanobis_score(feats, means, precision):
    X = np.asarray(feats, dtype=np.float64)                   # (N, D)
    d2 = []
    for mu in means:                                          # mu: (D,)
        diff = X - mu                                         # (N, D)
        d2.append(np.einsum('nd,nd->n', diff @ precision, diff))  # (N,) squared distance
    return np.min(np.stack(d2, axis=1), axis=1)               # (N,) distance to the closest class


def _l2_normalise(a):
    a = np.asarray(a, dtype=np.float64)                       # (N, D)
    return a / np.maximum(np.linalg.norm(a, axis=1, keepdims=True), 1e-12)


def knn_score(feats, train_feats, k=KNN_K, chunk=2048):
    """Distance to the k-th nearest L2-normalised training feature (Sun et al., 2022)."""
    bank = _l2_normalise(train_feats)                         # (N_tr, D)
    q = _l2_normalise(feats)                                  # (N, D)
    k = int(min(k, len(bank)))
    out = np.empty(len(q))
    for s in range(0, len(q), chunk):
        sims = q[s:s + chunk] @ bank.T                        # (n, N_tr) cosine similarity
        kth = -np.partition(-sims, k - 1, axis=1)[:, k - 1]   # (n,) k-th largest similarity
        out[s:s + chunk] = np.sqrt(np.clip(2.0 - 2.0 * kth, 0.0, None))  # (n,) unit-sphere distance
    return out


# ---- detection metrics: ID = negative, OOD = positive, higher score = more OOD ----
def ood_metrics(id_scores, ood_scores):
    s_id = np.asarray(id_scores, dtype=np.float64)
    s_ood = np.asarray(ood_scores, dtype=np.float64)
    y = np.r_[np.zeros(len(s_id)), np.ones(len(s_ood))]
    s = np.r_[s_id, s_ood]
    # FPR95: accept an input as ID when -s is high; the threshold accepts 95% of ID inputs.
    fpr, tpr, _ = roc_curve(1 - y, -s)                        # ID treated as the positive class here
    fpr95 = float(fpr[min(np.searchsorted(tpr, 0.95, side='left'), len(fpr) - 1)])
    return {'auroc': float(roc_auc_score(y, s)), 'aupr_out': float(average_precision_score(y, s)),
            'fpr95': fpr95}


# ---- per-run cache: one .npz per run, written by the extraction cell ----
def save_ood_cache(path, run_id, arrays, meta):
    missing = [k for k in OOD_ARRAY_KEYS if k not in arrays]
    assert not missing, f'{run_id}: cache arrays missing {missing}'
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    np.savez(path, cache_version=np.int64(OOD_CACHE_VERSION), run_id=np.str_(run_id),
             meta_json=np.str_(_json.dumps(meta)), **{k: np.asarray(arrays[k]) for k in OOD_ARRAY_KEYS})


def load_ood_cache(path):
    with np.load(path, allow_pickle=False) as z:
        if 'cache_version' not in z.files or int(z['cache_version']) != OOD_CACHE_VERSION:
            return None
        if any(k not in z.files for k in OOD_ARRAY_KEYS):
            return None
        data = {k: z[k] for k in OOD_ARRAY_KEYS}
        data['run_id'] = str(z['run_id'])
        data['meta'] = _json.loads(str(z['meta_json']))
    for split in ('val', 'test', 'bach'):
        n = len(data[f'{split}_logits'])
        assert data[f'{split}_feats'].shape[0] == n, f"{data['run_id']}: {split} feats/logits length differ"
    assert len(data['train_feats']) == len(data['train_labels']), f"{data['run_id']}: train feats/labels differ"
    return data


def read_ood_meta(path):
    """Meta dict of a cache file without loading its arrays; None if the file is stale or incomplete."""
    with np.load(path, allow_pickle=False) as z:
        if 'cache_version' not in z.files or int(z['cache_version']) != OOD_CACHE_VERSION:
            return None
        if any(k not in z.files for k in OOD_ARRAY_KEYS) or 'meta_json' not in z.files:
            return None
        return _json.loads(str(z['meta_json']))


def ood_cache_dirs(search_root='/kaggle/input'):
    dirs = [OOD_CACHE_DIR]
    root = Path(search_root)
    if root.exists():
        for dirpath, dirnames, _files in os.walk(root):
            if Path(dirpath).name == 'ood_cache':
                dirs.append(Path(dirpath)); dirnames[:] = []
    return dirs


def lomo_run_table(frame):
    t = frame[frame.method.isin(ORDER)].copy()
    if 'patient_fraction' in t.columns:
        t = t[t.patient_fraction.astype(float) == 1.0]
    if 'error' in t.columns:
        t = t[t['error'].isna()]
    return (t[['run_id', 'method', 'model', 'branch', 'held_out_mag', 'seed']]
            .drop_duplicates('run_id').sort_values(['method', 'held_out_mag', 'seed']).reset_index(drop=True))


# ======================== self-test (known answers) ========================
def _ood_selftest():
    rng = np.random.default_rng(20260917)
    # 1. logit scores: formulas, stability at extreme logits, and the sign used by the load cell
    z = rng.normal(0, 3, size=(64, 2))
    assert np.allclose(energy_from_logits(z), -np.log(np.exp(z).sum(axis=1)))
    assert np.allclose(-neg_msp(z), np.exp(z).max(axis=1) / np.exp(z).sum(axis=1))
    big = np.array([[1e4, -1e4], [-5e3, 5e3], [0.0, 0.0]])
    assert np.all(np.isfinite(energy_from_logits(big))) and np.all(np.isfinite(neg_msp(big)))
    if '_logsumexp_rows' in globals():
        assert np.allclose(energy_from_logits(z), -_logsumexp_rows(z)), 'sign differs from the load cell'

    # 2. AUROC / AUPR / FPR95 are unchanged by any increasing affine map (why RNE = energy per run)
    s_id, s_ood = rng.normal(0, 1, 3000), rng.normal(0.7, 1.3, 2500)
    base = ood_metrics(s_id, s_ood)
    for a, b in [(3.7, -12.0), (0.01, 5.0)]:
        other = ood_metrics(a * s_id + b, a * s_ood + b)
        assert all(abs(base[k] - other[k]) < 1e-12 for k in base), (base, other)
    rn_id, (mu, sd) = reference_normalize(s_id, rng.normal(0, 1, 1000))
    rn_ood, _ = reference_normalize(s_ood, rng.normal(0, 1, 1000))
    assert rn_id.shape == s_id.shape
    rn = ood_metrics((s_id - mu) / sd, (s_ood - mu) / sd)
    assert all(abs(base[k] - rn[k]) < 1e-12 for k in base)

    # 3. analytic values: ID N(0,1), OOD N(3,1) -> AUROC = Phi(3/sqrt 2), FPR95 = Phi(1.645 - 3)
    m = ood_metrics(rng.normal(0, 1, 200_000), rng.normal(3, 1, 200_000))
    assert abs(m['auroc'] - _norm.cdf(3 / np.sqrt(2))) < 3e-3, m
    assert abs(m['fpr95'] - _norm.cdf(_norm.ppf(0.95) - 3)) < 6e-3, m

    # 4. pooling: per-run AUROC equal, pooled AUROC recovered only after reference normalisation
    runs = [(0.0, 1.0), (8.0, 3.0)]                           # (offset, scale) of two "encoders"
    raw_id, raw_ood, rn_id, rn_ood = [], [], [], []
    for off, sc in runs:
        ref = off + sc * rng.normal(0, 1, 2000)
        idr, oodr = off + sc * rng.normal(0.3, 1, 1500), off + sc * rng.normal(2.0, 1, 1500)
        raw_id.append(idr); raw_ood.append(oodr)
        rn_id.append(reference_normalize(idr, ref)[0]); rn_ood.append(reference_normalize(oodr, ref)[0])
    pooled_raw = ood_metrics(np.concatenate(raw_id), np.concatenate(raw_ood))['auroc']
    pooled_rn = ood_metrics(np.concatenate(rn_id), np.concatenate(rn_ood))['auroc']
    assert pooled_rn > pooled_raw + 0.10, (pooled_raw, pooled_rn)

    # 5. Mahalanobis and kNN: detect a shifted cluster, stay at chance on identical data
    D = 16
    scale = rng.uniform(0.5, 2.0, D)
    mus = np.stack([np.r_[2.0, np.zeros(D - 1)], np.r_[-2.0, np.zeros(D - 1)]])
    def draw(n, cls, shift=0.0):
        x = mus[cls] + rng.normal(0, 1, (n, D)) * scale
        x[:, 1] += shift
        return x
    tr = np.vstack([draw(1500, 0), draw(1500, 1)]); ytr = np.r_[np.zeros(1500), np.ones(1500)]
    te_id = np.vstack([draw(250, 0), draw(250, 1)])
    te_same = np.vstack([draw(250, 0), draw(250, 1)])
    te_shift = np.vstack([draw(250, 0, 6.0), draw(250, 1, 6.0)])
    means, prec = fit_mahalanobis(tr, ytr)
    assert means.shape == (2, D) and prec.shape == (D, D)
    assert np.all(np.linalg.eigvalsh((prec + prec.T) / 2) > 0), 'precision not positive definite'
    maha_shift = ood_metrics(mahalanobis_score(te_id, means, prec), mahalanobis_score(te_shift, means, prec))['auroc']
    maha_same = ood_metrics(mahalanobis_score(te_id, means, prec), mahalanobis_score(te_same, means, prec))['auroc']
    knn_shift = ood_metrics(knn_score(te_id, tr), knn_score(te_shift, tr))['auroc']
    knn_same = ood_metrics(knn_score(te_id, tr), knn_score(te_same, tr))['auroc']
    assert maha_shift > 0.95 and abs(maha_same - 0.5) < 0.06, (maha_shift, maha_same)
    assert knn_shift > 0.90 and abs(knn_same - 0.5) < 0.06, (knn_shift, knn_same)

    # 6. cache round trip
    tmp = Path(OUTPUT_DIR) / '_selftest_cache.npz'
    arrays = {k: rng.normal(size=(5, 3)).astype(np.float32) for k in OOD_ARRAY_KEYS}
    for k in ('train_labels', 'val_labels', 'test_labels', 'bach_class'):
        arrays[k] = np.arange(5, dtype=np.int64)
    for k in ('val_logits', 'test_logits', 'bach_logits'):
        arrays[k] = rng.normal(size=(5, 2)).astype(np.float32)
    save_ood_cache(tmp, 'selftest_run', arrays, {'note': 'self-test'})
    back = load_ood_cache(tmp)
    assert read_ood_meta(tmp) == {'note': 'self-test'}
    assert back['run_id'] == 'selftest_run' and back['meta']['note'] == 'self-test'
    assert all(np.array_equal(arrays[k], back[k]) for k in OOD_ARRAY_KEYS)
    tmp.unlink()

    # 7. penultimate-feature hook: the input of the final Linear reproduces the logits
    try:
        import torch
        net = torch.nn.Sequential(torch.nn.Conv2d(3, 4, 3), torch.nn.AdaptiveAvgPool2d(1), torch.nn.Flatten(),
                                  torch.nn.Linear(4, 8), torch.nn.GELU(), torch.nn.Linear(8, 2)).eval()
        store = {}
        lin = [m for m in net.modules() if isinstance(m, torch.nn.Linear) and m.out_features == 2][0]
        h = lin.register_forward_pre_hook(lambda mod, inp: store.__setitem__('f', inp[0]))
        with torch.no_grad():
            out = net(torch.randn(6, 3, 16, 16))                 # (6, 2)
        h.remove()
        assert store['f'].shape == (6, 8) and torch.allclose(lin(store['f']), out, atol=1e-6)
        hook_msg = 'hook reproduces logits'
    except ImportError:
        hook_msg = 'hook check skipped (torch not installed)'

    return {'pooled_auroc_raw': pooled_raw, 'pooled_auroc_rne': pooled_rn, 'maha_shift': maha_shift,
            'knn_shift': knn_shift, 'hook': hook_msg}


_st = _ood_selftest()
print('OOD library self-test PASSED')
print(f"  invariance: AUROC/AUPR/FPR95 identical under affine maps (so RNE = energy inside a run)")
print(f"  analytic AUROC and FPR95 reproduced; Mahalanobis AUROC on shifted data {_st['maha_shift']:.3f}, "
      f"kNN {_st['knn_shift']:.3f}; both at chance on identical data")
print(f"  pooling demo: raw pooled AUROC {_st['pooled_auroc_raw']:.3f} -> reference-normalised {_st['pooled_auroc_rne']:.3f}")
print(f"  {_st['hook']}; cache round trip OK")

In [ ]:
# ---------- Per-image logits + penultimate features for every LOMO run (GPU; cached per run) ----------
# For each of the 48 LOMO runs this cell recomputes, with the exact training preprocessing:
#   train split -> penultimate features            (Mahalanobis / kNN reference)
#   val split   -> logits + features               (RNE reference; ID patients, training mags)
#   test split  -> logits + features               (ID evaluation set, held-out magnification)
#   BACH        -> logits + features               (OOD evaluation set, another institution)
# Nothing enters the analysis unless the recomputed test labels equal the stored ones position by
# position and the recomputed test logits match test_outputs.npz within FIDELITY_TOL.
import time
OOD_BATCH = 128                  # task-trained encoders, fp32 inference
OOD_FM_BATCH = 96                # Phikon ViT-B/16 backbone
FIDELITY_TOL = 1e-2              # max |recomputed - stored| test logit (cuDNN / batch-size noise is ~1e-4)
OOD_DECODE_WORKERS = max(2, min(4, os.cpu_count() or 2))

OOD_RUN_TABLE = lomo_run_table(raw)
OOD_CACHE = {}                   # run_id -> Path of a cache file this session can read
OOD_READY = False


def _discover_bach_root(search_root='/kaggle/input'):
    """Same rule as the training notebook: the folder holding Normal/Benign/InSitu/Invasive with most images."""
    best, best_count = None, 0
    if not Path(search_root).exists():
        return None
    for dirpath, dirnames, _files in os.walk(search_root):
        if {'Normal', 'Benign', 'InSitu', 'Invasive'}.issubset(set(dirnames)):
            count = sum(1 for ext in ('*.tif', '*.png', '*.jpg') for _ in Path(dirpath).rglob(ext))
            if count > best_count:
                best, best_count = Path(dirpath), count
    return best


def _bach_manifest(root):
    rows = []
    for cls, cls_id in {'Normal': 0, 'Benign': 1, 'InSitu': 2, 'Invasive': 3}.items():
        for suffix in ('*.tif', '*.png', '*.jpg'):
            for path in sorted(Path(root, cls).glob(suffix)):
                rows.append({'path': str(path), 'bach_class': cls_id})
    return pd.DataFrame(rows)


# ---- 1. reuse caches from this session or from an attached earlier version ----
_cache_dirs = ood_cache_dirs()
for run_id in OOD_RUN_TABLE.run_id:
    for d in _cache_dirs:
        p = Path(d) / f'{run_id}.npz'
        if p.exists():
            try:
                if read_ood_meta(p) is not None:
                    if Path(d) != OOD_CACHE_DIR:                      # keep this version's output self-contained
                        OOD_CACHE_DIR.mkdir(parents=True, exist_ok=True)
                        shutil.copy2(p, OOD_CACHE_DIR / p.name)
                        p = OOD_CACHE_DIR / p.name
                    OOD_CACHE[run_id] = p
                    break
            except Exception as exc:
                print(f'  unreadable cache {p}: {repr(exc)[:120]}')
_todo = OOD_RUN_TABLE[~OOD_RUN_TABLE.run_id.isin(OOD_CACHE)].reset_index(drop=True)
print(f'LOMO runs: {len(OOD_RUN_TABLE)}   cached: {len(OOD_CACHE)}   to compute: {len(_todo)}')

# ---- 2. preconditions for the runs that still need the GPU ----
_problems = []
if len(_todo):
    try:
        import torch
    except ImportError:
        _problems.append('torch not installed')
    else:
        if not torch.cuda.is_available():
            _problems.append('no CUDA GPU (Settings -> Accelerator -> GPU T4); MedMamba cannot run on CPU')
    if not HAS_IMAGES:
        _problems.append('BreakHis not attached (Add Input -> ambarish/breakhis)')
    if 'build_from_scratch' not in globals():
        _problems.append('run the Fig 11 (Grad-CAM++) cell first: it defines build_from_scratch and installs mamba_ssm')
    BACH_ROOT_ANALYSIS = _discover_bach_root()
    if BACH_ROOT_ANALYSIS is None:
        _problems.append('BACH not attached (Add Input -> truthisneverlinear/bach-breast-cancer-histology-images)')
if _problems:
    print('RNE / OOD extraction SKIPPED for the uncached runs:')
    for p in _problems:
        print('  -', p)

elif len(_todo):
    import torch
    from torchvision import transforms as _tf
    from PIL import Image as PILImage
    _dev = torch.device('cuda')
    torch.backends.cudnn.benchmark = False
    _MEAN = torch.tensor([0.485, 0.456, 0.406], device=_dev).view(1, 3, 1, 1)
    _STD = torch.tensor([0.229, 0.224, 0.225], device=_dev).view(1, 3, 1, 1)

    # ---- 3. decode every image once with the training transform (Resize((224, 224)) on the PIL image) ----
    class _U8Images(torch.utils.data.Dataset):
        def __init__(self, paths):
            self.paths, self.resize = list(paths), _tf.Resize((224, 224))
        def __len__(self):
            return len(self.paths)
        def __getitem__(self, i):
            with PILImage.open(self.paths[i]) as im:
                arr = np.asarray(self.resize(im.convert('RGB')), dtype=np.uint8)          # (224, 224, 3)
            return torch.from_numpy(np.ascontiguousarray(arr.transpose(2, 0, 1))), i      # (3, 224, 224)

    def _decode_all(paths, name):
        t0 = time.perf_counter()
        out = torch.empty((len(paths), 3, 224, 224), dtype=torch.uint8)                    # (N, 3, 224, 224)
        loader = torch.utils.data.DataLoader(_U8Images(paths), batch_size=64, shuffle=False,
                                             num_workers=OOD_DECODE_WORKERS)
        for imgs, idx in loader:
            out[idx] = imgs
        print(f'  decoded {len(paths)} {name} images in {time.perf_counter() - t0:.0f} s')
        return out

    def _normalise_u8(batch_u8):
        x = batch_u8.to(_dev, non_blocking=True).float().div_(255.0)                        # (B, 3, 224, 224) in [0, 1]
        return (x - _MEAN) / _STD                                                           # (B, 3, 224, 224)

    def _classifier_linear(model, num_classes=2):
        found = [(n, m) for n, m in model.named_modules()
                 if isinstance(m, torch.nn.Linear) and m.out_features == num_classes]
        if len(found) != 1:
            raise RuntimeError(f'expected one Linear with out_features={num_classes}, found {[n for n, _ in found]}')
        return found[0]

    @torch.inference_mode()
    def _logits_and_features(model, lin, images_u8, positions, verify):
        store = {}
        handle = lin.register_forward_pre_hook(lambda mod, inp: store.__setitem__('f', inp[0]))
        zs, fs = [], []
        try:
            for s in range(0, len(positions), OOD_BATCH):
                pos = torch.as_tensor(positions[s:s + OOD_BATCH], dtype=torch.long)       # (B,)
                z = model(_normalise_u8(images_u8[pos]))                                    # (B, 2)
                f = store.pop('f')                                                          # (B, D)
                if f.dim() != 2:
                    raise RuntimeError(f'classifier input has shape {tuple(f.shape)}; expected (B, D)')
                if verify and s == 0 and not torch.allclose(lin(f), z, atol=1e-4, rtol=1e-4):
                    raise RuntimeError('the final Linear input does not reproduce the logits')
                zs.append(z.float().cpu()); fs.append(f.float().cpu())
        finally:
            handle.remove()
        return torch.cat(zs).numpy(), torch.cat(fs).numpy()                                 # (N, 2), (N, D)

    manifest_ood = _training_manifest()
    assert manifest_ood is not None, 'breakhis_manifest.csv not found in the attached training outputs'
    manifest_ood = manifest_ood.reset_index(drop=True)                                      # positions = row labels
    _absent = [p for p in manifest_ood.path if not Path(p).exists()]
    assert not _absent, f'{len(_absent)} manifest images not found under BREAKHIS_ROOT, e.g. {_absent[:2]}'
    bach_frame_ood = _bach_manifest(BACH_ROOT_ANALYSIS)
    print(f'  BACH root: {BACH_ROOT_ANALYSIS}  ({len(bach_frame_ood)} images, '
          f'{bach_frame_ood.bach_class.value_counts().sort_index().tolist()} per class)')
    assert len(bach_frame_ood) > 0, 'BACH manifest is empty'

    BREAKHIS_U8 = _decode_all(manifest_ood.path, 'BreakHis')                                # (7909, 3, 224, 224)
    BACH_U8 = _decode_all(bach_frame_ood.path, 'BACH')                                      # (400, 3, 224, 224)
    bach_positions = np.arange(len(BACH_U8))
    labels_all = manifest_ood.coarse_label.to_numpy().astype(np.int64)                      # (7909,)

    # ---- 4. frozen Phikon: CLS features for every image, once (the probe heads differ per run) ----
    FM_FEATS = FM_BACH_FEATS = None
    if (_todo.model == 'owkin/phikon').any():
        from transformers import AutoModel
        _fm = AutoModel.from_pretrained('owkin/phikon').to(_dev).eval()                     # as in training
        @torch.inference_mode()
        def _fm_cls(images_u8):
            out = []
            for s in range(0, len(images_u8), OOD_FM_BATCH):
                h = _fm(pixel_values=_normalise_u8(images_u8[s:s + OOD_FM_BATCH])).last_hidden_state  # (B, 197, 768)
                out.append(h[:, 0].float().cpu())                                               # (B, 768) CLS token
            return torch.cat(out).numpy()
        t0 = time.perf_counter()
        FM_FEATS, FM_BACH_FEATS = _fm_cls(BREAKHIS_U8), _fm_cls(BACH_U8)                     # (7909, 768), (400, 768)
        print(f'  Phikon CLS features: {FM_FEATS.shape} + {FM_BACH_FEATS.shape} in {time.perf_counter() - t0:.0f} s')
        del _fm
        torch.cuda.empty_cache()

    # ---- 5. every uncached run ----
    _reverse_display = {v: k for k, v in DISPLAY.items()}
    _t_start = time.perf_counter()
    for i, run in _todo.iterrows():
        t0 = time.perf_counter()
        run_id, ho, seed = run.run_id, int(run.held_out_mag), int(run.seed)
        try:
            subset = patient_lomo_split(manifest_ood, ho, fraction=1.0, seed=seed)
            pos = {s: subset.index[subset['split'] == s].to_numpy() for s in ('train', 'val', 'test')}  # manifest order
            stored = np.load(io.BytesIO(reader.open_bytes(f'runs/{run_id}/test_outputs.npz')))
            stored_logits, stored_labels = stored['logits'], stored['labels'].astype(np.int64)

            if run.model == 'owkin/phikon':
                saved = torch.load(io.BytesIO(reader.open_bytes(f'runs/{run_id}/probe.pt')), map_location=_dev)
                head = torch.nn.Linear(int(saved['feature_dim']), int(saved['num_classes'])).to(_dev).eval()
                head.load_state_dict({k.replace('head.', ''): v for k, v in saved['state_dict'].items()})
                feats = {s: FM_FEATS[pos[s]] for s in pos}
                feats['bach'] = FM_BACH_FEATS
                with torch.inference_mode():
                    logits = {s: head(torch.from_numpy(feats[s]).to(_dev)).float().cpu().numpy()
                              for s in ('val', 'test', 'bach')}                                   # each (N, 2)
                del head
            else:
                model_name = _reverse_display.get(run.method, run.model)
                model = build_from_scratch(model_name)
                model.load_state_dict(torch.load(io.BytesIO(reader.open_bytes(f'runs/{run_id}/best.pt')),
                                                 map_location='cpu'))
                model = model.to(_dev).eval()
                _, lin = _classifier_linear(model)
                logits, feats = {}, {}
                for s, images, positions in (('val', BREAKHIS_U8, pos['val']), ('test', BREAKHIS_U8, pos['test']),
                                             ('bach', BACH_U8, bach_positions), ('train', BREAKHIS_U8, pos['train'])):
                    logits[s], feats[s] = _logits_and_features(model, lin, images, positions, verify=(s == 'val'))
                del model
                torch.cuda.empty_cache()

            test_labels = labels_all[pos['test']]
            labels_equal = len(test_labels) == len(stored_labels) and np.array_equal(test_labels, stored_labels)
            max_diff = float(np.abs(logits['test'] - stored_logits).max()) if logits['test'].shape == stored_logits.shape else float('inf')
            fidelity_ok = bool(labels_equal and max_diff <= FIDELITY_TOL)
            meta = {'run_id': run_id, 'method': run.method, 'model': run.model, 'held_out_mag': ho, 'seed': seed,
                    'labels_equal': bool(labels_equal), 'max_abs_test_logit_diff': max_diff,
                    'fidelity_ok': fidelity_ok, 'feature_dim': int(feats['test'].shape[1]),
                    'n_train': int(len(pos['train'])), 'n_val': int(len(pos['val'])),
                    'n_test': int(len(pos['test'])), 'n_bach': int(len(bach_positions)),
                    'seconds': round(time.perf_counter() - t0, 1)}
            arrays = {'train_feats': feats['train'].astype(np.float32), 'train_labels': labels_all[pos['train']],
                      'val_logits': logits['val'].astype(np.float32), 'val_feats': feats['val'].astype(np.float32),
                      'val_labels': labels_all[pos['val']],
                      'test_logits': logits['test'].astype(np.float32), 'test_feats': feats['test'].astype(np.float32),
                      'test_labels': test_labels,
                      'bach_logits': logits['bach'].astype(np.float32), 'bach_feats': feats['bach'].astype(np.float32),
                      'bach_class': bach_frame_ood.bach_class.to_numpy().astype(np.int64)}
            cache_path = OOD_CACHE_DIR / f'{run_id}.npz'
            save_ood_cache(cache_path, run_id, arrays, meta)
            OOD_CACHE[run_id] = cache_path
            done = i + 1
            eta = (time.perf_counter() - _t_start) / done * (len(_todo) - done)
            flag = 'OK ' if fidelity_ok else 'FIDELITY FAILED'
            print(f'  [{done:2d}/{len(_todo)}] {flag} {run_id:<42} D={meta["feature_dim"]:<4} '
                  f'max|dlogit|={max_diff:.1e} labels_equal={labels_equal}  {meta["seconds"]:.0f} s  ETA {eta / 60:.0f} min')
        except Exception as exc:
            print(f'  [{i + 1:2d}/{len(_todo)}] FAILED {run_id}: {repr(exc)[:240]}')

    del BREAKHIS_U8, BACH_U8
    torch.cuda.empty_cache()
    print(f'Extraction finished in {(time.perf_counter() - _t_start) / 60:.1f} min')

# ---- 6. what the analysis cells will use ----
_fid = []
for run_id, p in OOD_CACHE.items():
    m = read_ood_meta(p) or {}
    _fid.append({k: m.get(k) for k in ('run_id', 'method', 'held_out_mag', 'seed', 'fidelity_ok',
                                         'labels_equal', 'max_abs_test_logit_diff', 'feature_dim')})
OOD_FIDELITY = pd.DataFrame(_fid)
if len(OOD_FIDELITY):
    OOD_FIDELITY.to_csv(f'{OUTPUT_DIR}/tables/ood_cache_fidelity.csv', index=False)
    OOD_FIDELITY['fidelity_ok'] = OOD_FIDELITY['fidelity_ok'].fillna(False).astype(bool)
    _bad = OOD_FIDELITY[~OOD_FIDELITY.fidelity_ok]
    OOD_READY = bool(OOD_FIDELITY.fidelity_ok.any())
    print(f'\nCached runs usable: {int(OOD_FIDELITY.fidelity_ok.sum())}/{len(OOD_RUN_TABLE)}; '
          f'largest test-logit difference among them: '
          f'{OOD_FIDELITY.loc[OOD_FIDELITY.fidelity_ok, "max_abs_test_logit_diff"].max():.1e}')
    if len(_bad):
        print('EXCLUDED (recomputed outputs do not match the stored test_outputs.npz):')
        print(_bad.to_string(index=False))
else:
    print('No cached runs available; the RNE / OOD cells below will skip.')

In [ ]:
# ---------- Per-run OOD scores and detection metrics (CPU) ----------
# Scores (higher = more out-of-distribution) for every usable cached run:
#   msp    = -max softmax probability             energy = -logsumexp(logits)
#   maha   = Mahalanobis distance to the closest class mean (Ledoit-Wolf tied covariance, training features)
#   knn    = distance to the k-th nearest L2-normalised training feature
# and their reference-normalised versions (reference = the run's validation set): rne for energy,
# msp_rn, maha_rn, knn_rn. Pairs evaluated per run:
#   bach_vs_test : ID = held-out-magnification test set, OOD = BACH   (institution shift; the main result)
#   test_vs_val  : ID = validation set (training mags), OOD = test set (does the score flag the new magnification?)
OOD_SCORES = {}                       # run_id -> {score: {'val', 'test', 'bach': (N,) array}}; used by the pooled cell
BASE_SCORES = ['msp', 'energy', 'maha', 'knn']
RN_NAME = {'msp': 'msp_rn', 'energy': 'rne', 'maha': 'maha_rn', 'knn': 'knn_rn'}
PAIRS = {'bach_vs_test': ('test', 'bach'), 'test_vs_val': ('val', 'test')}

if not globals().get('OOD_READY', False):
    print('Skipped per-run OOD metrics - no usable cached runs (see the extraction cell).')
    ood_per_run = ood_summary = ood_gaps = None
else:
    _rows, _gap_rows = [], []
    _bach_csv = bach_ood.set_index('run_id') if (bach_ood is not None and len(bach_ood)) else None
    _usable = OOD_FIDELITY[OOD_FIDELITY.fidelity_ok].run_id.tolist()
    for _, run in OOD_RUN_TABLE[OOD_RUN_TABLE.run_id.isin(_usable)].iterrows():
        c = load_ood_cache(OOD_CACHE[run.run_id])
        sc = {'msp': {s: neg_msp(c[f'{s}_logits']) for s in ('val', 'test', 'bach')},
              'energy': {s: energy_from_logits(c[f'{s}_logits']) for s in ('val', 'test', 'bach')}}
        means, prec = fit_mahalanobis(c['train_feats'], c['train_labels'])
        sc['maha'] = {s: mahalanobis_score(c[f'{s}_feats'], means, prec) for s in ('val', 'test', 'bach')}
        sc['knn'] = {s: knn_score(c[f'{s}_feats'], c['train_feats'], KNN_K) for s in ('val', 'test', 'bach')}
        for base in BASE_SCORES:
            sc[RN_NAME[base]] = {s: reference_normalize(sc[base][s], sc[base]['val'])[0] for s in ('val', 'test', 'bach')}
        OOD_SCORES[run.run_id] = sc

        meta = dict(run_id=run.run_id, method=run.method, held_out_mag=int(run.held_out_mag), seed=int(run.seed))
        per_score = {}
        for name, splits in sc.items():
            for pair, (id_split, ood_split) in PAIRS.items():
                m = ood_metrics(splits[id_split], splits[ood_split])
                per_score[(name, pair)] = m
                _rows.append({**meta, 'score': name, 'pair': pair, **m})
        # Within a run a reference-normalised score is an increasing affine map of its base score,
        # so every rank metric must agree exactly. Checked here instead of reported as extra rows.
        for base in BASE_SCORES:
            for pair in PAIRS:
                a, b = per_score[(base, pair)], per_score[(RN_NAME[base], pair)]
                # 1e-6, not 0: floating-point rounding can merge two adjacent doubles into a tie,
                # which moves AUROC by ~1e-6 at most; a genuine bug moves it by orders of magnitude more.
                assert all(abs(a[k] - b[k]) < 1e-6 for k in a), f'{run.run_id} {base}: {a} vs {b}'
        for k in KNN_K_SENSITIVITY:
            m = ood_metrics(knn_score(c['test_feats'], c['train_feats'], k), knn_score(c['bach_feats'], c['train_feats'], k))
            _rows.append({**meta, 'score': f'knn_k{k}', 'pair': 'bach_vs_test', **m})

        e_test, e_bach = sc['energy']['test'], sc['energy']['bach']
        gap = {**meta,
               'energy_gap': float(e_bach.mean() - e_test.mean()),               # the paper's Delta E (unnormalised)
               'rne_mean_test': float(sc['rne']['test'].mean()),                 # magnification shift, in reference SDs
               'rne_mean_bach': float(sc['rne']['bach'].mean()),                 # institution shift, in reference SDs
               'rne_gap': float(sc['rne']['bach'].mean() - sc['rne']['test'].mean()),
               'val_energy_sd': float(sc['energy']['val'].std(ddof=1))}
        if _bach_csv is not None and run.run_id in _bach_csv.index:
            # Training scored BACH itself; its per-run mean must agree with this recomputation.
            gap['bach_energy_mean_training'] = float(_bach_csv.loc[run.run_id, 'energy_mean'])
            gap['bach_energy_mean_recomputed'] = float(e_bach.mean())
        _gap_rows.append(gap)

    ood_per_run = pd.DataFrame(_rows)
    ood_gaps = pd.DataFrame(_gap_rows)
    ood_per_run.to_csv(f'{OUTPUT_DIR}/tables/ood_per_run_metrics.csv', index=False)
    ood_gaps.to_csv(f'{OUTPUT_DIR}/tables/ood_run_level_gaps.csv', index=False)

    _summ = []
    for (method, ho, score, pair), sub in ood_per_run.groupby(['method', 'held_out_mag', 'score', 'pair']):
        row = {'method': method, 'held_out_mag': ho, 'score': score, 'pair': pair, 'n_runs': len(sub)}
        for metric in ('auroc', 'aupr_out', 'fpr95'):
            mean, lo, hi = bootstrap_ci(sub[metric].to_numpy())
            row.update({f'{metric}_mean': mean, f'{metric}_ci_low': lo, f'{metric}_ci_high': hi,
                        f'{metric}_min': float(sub[metric].min()), f'{metric}_max': float(sub[metric].max())})
        _summ.append(row)
    ood_summary = pd.DataFrame(_summ)
    ood_summary.to_csv(f'{OUTPUT_DIR}/tables/ood_summary.csv', index=False)

    print(f'Per-run OOD metrics for {len(OOD_SCORES)} runs  '
          f'(invariance check passed: every reference-normalised score matches its base score inside each run)\n')
    _main = ood_summary[(ood_summary.pair == 'bach_vs_test') & ood_summary.score.isin(BASE_SCORES)]
    print('AUROC, BACH vs own held-out test set (mean over seeds; 0.5 = chance; energy row = RNE row):')
    print(_main.pivot_table(index=['method', 'held_out_mag'], columns='score', values='auroc_mean')
          .reindex(columns=BASE_SCORES).reindex(ORDER, level=0).round(3).to_string())
    print('\nAUROC averaged over held-out magnifications:')
    print(_main.groupby(['method', 'score']).auroc_mean.mean().unstack().reindex(index=ORDER, columns=BASE_SCORES).round(3).to_string())
    _shift = ood_summary[(ood_summary.pair == 'test_vs_val') & ood_summary.score.isin(BASE_SCORES)]
    print('\nAUROC, held-out magnification vs validation (does the score notice the unseen magnification?):')
    print(_shift.groupby(['method', 'score']).auroc_mean.mean().unstack().reindex(index=ORDER, columns=BASE_SCORES).round(3).to_string())
    _sens = ood_summary[(ood_summary.pair == 'bach_vs_test') & ood_summary.score.isin(['knn'] + [f'knn_k{k}' for k in KNN_K_SENSITIVITY])]
    print(f'\nkNN sensitivity to k (AUROC averaged over magnifications; k={KNN_K} is the pre-specified main result):')
    print(_sens.groupby(['method', 'score']).auroc_mean.mean().unstack().reindex(ORDER).round(3).to_string())

    print('\nRun-level shift in reference SDs (mean over seeds): RNE on test = magnification shift, RNE on BACH = institution shift')
    print(ood_gaps.groupby(['method', 'held_out_mag'])[['rne_mean_test', 'rne_mean_bach', 'rne_gap', 'energy_gap']]
          .mean().reindex(ORDER, level=0).round(3).to_string())
    if 'bach_energy_mean_training' in ood_gaps.columns:
        _d = (ood_gaps.bach_energy_mean_training - ood_gaps.bach_energy_mean_recomputed).abs()
        print(f'\nBACH preprocessing check: |training - recomputed| mean BACH energy, max over runs = {_d.max():.2e}'
              + ('   <- WARNING: BACH scoring differs from training' if _d.max() > 0.05 else '   (consistent)'))

In [ ]:
# ---------- Where reference normalisation can matter: pooled AUROC and one fixed threshold across runs ----------
# (1) Pooled AUROC: the test (ID) and BACH (OOD) scores of several runs are ranked together, as when
#     one detector score is compared across encoders or magnifications. Raw scores live on
#     run-specific scales; reference-normalised scores share one unit (the run's validation SD).
# (2) One fixed threshold for all runs of a pool: tau = 95th percentile of the pooled validation
#     scores. Per run we record the false-alarm rate on its own held-out test set and the share of
#     BACH images flagged. A threshold that transfers should give similar false-alarm rates in every run.
POOL_PAIRS = [(base, RN_NAME[base]) for base in BASE_SCORES]

if not globals().get('OOD_SCORES'):
    print('Skipped pooled OOD analysis - run the per-run OOD cell first.')
    ood_pooled = ood_threshold = None
else:
    _meta = OOD_RUN_TABLE.set_index('run_id').loc[list(OOD_SCORES)]
    POOLS = {'All encoders, all magnifications': list(OOD_SCORES)}
    for m in ORDER:
        ids = _meta.index[_meta.method == m].tolist()
        if ids:
            POOLS[f'{m}, all magnifications'] = ids
    for ho in MAGS:
        ids = _meta.index[_meta.held_out_mag == ho].tolist()
        if ids:
            POOLS[f'All encoders, held-out {ho}×'] = ids

    _pool_rows = []
    for pool, run_ids in POOLS.items():
        row = {'pool': pool, 'n_runs': len(run_ids)}
        for base, rn in POOL_PAIRS:
            for name in (base, rn):
                id_s = np.concatenate([OOD_SCORES[r][name]['test'] for r in run_ids])
                ood_s = np.concatenate([OOD_SCORES[r][name]['bach'] for r in run_ids])
                m = ood_metrics(id_s, ood_s)
                row[f'{name}_auroc'], row[f'{name}_fpr95'] = m['auroc'], m['fpr95']
        _pool_rows.append(row)
    ood_pooled = pd.DataFrame(_pool_rows)
    ood_pooled.to_csv(f'{OUTPUT_DIR}/tables/ood_pooled.csv', index=False)

    _thr_rows = []
    for pool in ['All encoders, all magnifications'] + [p for p in POOLS if p.startswith('All encoders, held-out')]:
        run_ids = POOLS[pool]
        for base, rn in POOL_PAIRS:
            for name in (base, rn):
                tau = float(np.quantile(np.concatenate([OOD_SCORES[r][name]['val'] for r in run_ids]), 0.95))
                fpr_test = np.array([np.mean(OOD_SCORES[r][name]['test'] > tau) for r in run_ids])   # (n_runs,)
                fpr_val = np.array([np.mean(OOD_SCORES[r][name]['val'] > tau) for r in run_ids])     # (n_runs,)
                tpr_bach = np.array([np.mean(OOD_SCORES[r][name]['bach'] > tau) for r in run_ids])   # (n_runs,)
                _thr_rows.append({'pool': pool, 'score': name, 'base': base, 'normalised': name != base,
                                  'threshold': tau, 'n_runs': len(run_ids),
                                  'fpr_val_mean': fpr_val.mean(), 'fpr_val_sd': fpr_val.std(ddof=1),
                                  'fpr_test_mean': fpr_test.mean(), 'fpr_test_sd': fpr_test.std(ddof=1),
                                  'fpr_test_min': fpr_test.min(), 'fpr_test_max': fpr_test.max(),
                                  'tpr_bach_mean': tpr_bach.mean(), 'tpr_bach_sd': tpr_bach.std(ddof=1),
                                  'tpr_bach_min': tpr_bach.min(), 'tpr_bach_max': tpr_bach.max()})
    ood_threshold = pd.DataFrame(_thr_rows)
    ood_threshold.to_csv(f'{OUTPUT_DIR}/tables/ood_fixed_threshold.csv', index=False)

    _cols = [f'{n}_auroc' for pair in POOL_PAIRS for n in pair]
    print('Pooled AUROC, BACH vs held-out test set (raw score | reference-normalised score):')
    print(ood_pooled.set_index('pool')[_cols].round(3).to_string())
    _all = ood_pooled.iloc[0]
    print(f'\nChange from reference normalisation, all {int(_all.n_runs)} runs pooled:')
    for base, rn in POOL_PAIRS:
        print(f'  {SCORE_LABELS.get(base, base):<14} {_all[f"{base}_auroc"]:.3f} -> {_all[f"{rn}_auroc"]:.3f} '
              f'({_all[f"{rn}_auroc"] - _all[f"{base}_auroc"]:+.3f})')
    print('\nOne threshold for all runs (95th percentile of pooled validation scores):')
    _t = ood_threshold[ood_threshold.pool == 'All encoders, all magnifications']
    for _, r in _t.iterrows():
        print(f"  {r.score:<8} false alarms on held-out test {r.fpr_test_mean:.3f} +/- {r.fpr_test_sd:.3f} "
              f"[{r.fpr_test_min:.3f}, {r.fpr_test_max:.3f}]   BACH flagged {r.tpr_bach_mean:.3f} +/- {r.tpr_bach_sd:.3f} "
              f"[{r.tpr_bach_min:.3f}, {r.tpr_bach_max:.3f}]")
    print('  (a threshold that transfers across runs shows a small SD and a narrow range of false alarms)')

In [ ]:
# ---------- Fig 17: BACH-vs-test AUROC per score, encoder and held-out magnification ----------
# Bars = mean over seeds, dots = the individual seeds (n = 3, so every run is shown). Energy and RNE
# share one group because inside a run their AUROC is identical by construction (checked above).
FIG17_GROUPS = [('msp', 'MSP'), ('energy', 'Energy = RNE\n(per run)'), ('maha', 'Mahalanobis'), ('knn', f'kNN (k={KNN_K})')]

if globals().get('ood_per_run') is None:
    print('Skipped Fig 17 - per-run OOD metrics not available.')
else:
    _d17 = ood_per_run[(ood_per_run.pair == 'bach_vs_test') & ood_per_run.score.isin([g for g, _ in FIG17_GROUPS])]
    fig, axes = plt.subplots(1, len(MAGS), figsize=(4.6 * len(MAGS), 4.3), sharey=True, squeeze=False)
    width = 0.8 / max(1, len(ORDER))
    for ax, ho in zip(axes[0], MAGS):
        for gi, (score, _label) in enumerate(FIG17_GROUPS):
            for mi, method in enumerate(ORDER):
                v = _d17[(_d17.held_out_mag == ho) & (_d17.score == score) & (_d17.method == method)].auroc.to_numpy()
                if len(v) == 0:
                    continue
                x = gi - 0.4 + width * (mi + 0.5)
                ax.bar(x, v.mean(), width=width * 0.92, color=COLORS[method], edgecolor='black', linewidth=0.4,
                       label=method if gi == 0 else None)
                ax.scatter(np.full(len(v), x), v, s=9, color='black', zorder=3, linewidths=0)
        ax.axhline(0.5, color='0.25', linestyle='--', linewidth=0.9)
        ax.set_xticks(range(len(FIG17_GROUPS))); ax.set_xticklabels([l for _, l in FIG17_GROUPS], fontsize=9)
        ax.set_ylim(0, 1.0); ax.set_title(f'Held-out {ho}×', fontweight='bold')
        ax.grid(True, axis='y', linewidth=0.4); ax.set_axisbelow(True)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    axes[0, 0].set_ylabel('AUROC, BACH vs own test set\n(dashed line = chance)')
    plt.tight_layout()
    fig.legend(*axes[0, 0].get_legend_handles_labels(), loc='upper center', ncol=len(ORDER), frameon=False,
               bbox_to_anchor=(0.5, 1.06), fontsize=10)
    plt.suptitle('Separating another institution (BACH) from the held-out BreakHis test set, per held-out magnification '
                 '(bars = mean, dots = seeds)', y=1.12, fontweight='bold')
    save_fig(fig, 'fig17_ood_auroc_by_magnification')
    plt.show()

In [ ]:
# ---------- Fig 18: RNE distributions at every held-out magnification ----------
# Rows = held-out magnification, columns = encoder; seeds pooled. Grey = validation set (the
# reference: ID patients at the training magnifications, mean 0 and SD 1 by construction),
# navy = held-out-magnification test set, orange = BACH. The distance of navy from grey is the
# magnification shift the model perceives; orange from navy is the institution shift.
FIG18_SETS = [('val', 'Validation (reference)', '#9A9A9A'), ('test', 'Held-out test', '#1F3A5F'),
              ('bach', 'BACH (other institution)', '#E69F00')]

if not globals().get('OOD_SCORES'):
    print('Skipped Fig 18 - per-run OOD scores not available.')
else:
    _meta18 = OOD_RUN_TABLE.set_index('run_id')
    fig, axes = plt.subplots(len(MAGS), len(ORDER), figsize=(3.5 * len(ORDER), 2.55 * len(MAGS)), squeeze=False)
    for r, ho in enumerate(MAGS):
        for c, method in enumerate(ORDER):
            ax = axes[r, c]
            run_ids = [rid for rid in OOD_SCORES if _meta18.loc[rid, 'method'] == method
                       and int(_meta18.loc[rid, 'held_out_mag']) == int(ho)]
            if not run_ids:
                ax.axis('off'); continue
            vals = {s: np.concatenate([OOD_SCORES[rid]['rne'][s] for rid in run_ids]) for s, _, _ in FIG18_SETS}
            lo, hi = np.percentile(np.concatenate(list(vals.values())), [0.5, 99.5])
            bins = np.linspace(lo, hi, 32)
            for s, label, color in FIG18_SETS:
                v = np.clip(vals[s], lo, hi)
                if s != 'val':                                   # reference drawn as an outline only
                    ax.hist(v, bins=bins, density=True, histtype='stepfilled', alpha=0.14, color=color)
                ax.hist(v, bins=bins, density=True, histtype='step', linewidth=1.5 if s != 'val' else 1.2,
                        color=color, label=label if (r == 0 and c == 0) else None)
                ax.axvline(np.median(vals[s]), color=color, linewidth=1.0, linestyle=':')
            au = ood_per_run[(ood_per_run.pair == 'bach_vs_test') & (ood_per_run.score == 'rne')
                             & ood_per_run.run_id.isin(run_ids)].auroc
            ax.set_ylim(0, ax.get_ylim()[1] * 1.35)                 # headroom for the annotation box
            ax.text(0.98, 0.95, f'AUROC {au.mean():.2f}\nBACH-test {np.mean(vals["bach"]) - np.mean(vals["test"]):+.2f} SD',
                    transform=ax.transAxes, ha='right', va='top', fontsize=8,
                    bbox=dict(boxstyle='round,pad=0.25', facecolor='white', edgecolor='0.8', alpha=0.9))
            if r == 0:
                ax.set_title(method, fontweight='bold', fontsize=10.5)
            if c == 0:
                ax.set_ylabel(f'held-out {ho}×\ndensity')
            if r == len(MAGS) - 1:
                ax.set_xlabel('RNE (validation SDs)')
            ax.set_yticks([])
            ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.tight_layout()
    fig.legend(*axes[0, 0].get_legend_handles_labels(), loc='upper center', ncol=3, frameon=False,
               bbox_to_anchor=(0.5, 1.025), fontsize=10)
    plt.suptitle('Reference-Normalized Energy on the reference, held-out and external sets (seeds pooled; dotted = median)',
                 y=1.05, fontweight='bold')
    save_fig(fig, 'fig18_rne_distributions_by_magnification')
    plt.show()

In [ ]:
# ---------- Fig 12b: feature space at every held-out magnification (UMAP of cached penultimate features) ----------
# Rows = held-out magnification, columns = encoder; seed = SEEDS[0]. Features are the input of each
# model's final linear classifier on its held-out test set, taken from the OOD cache, so no extra GPU
# pass is needed. Up to 400 images per panel, class-balanced. Two silhouette scores are shown: in the
# original feature space (cosine) and in the 2-D embedding, which UMAP itself can inflate.
UMAPB_SEED = SEEDS[0] if SEEDS else 13
UMAPB_MAX_N = 400

try:
    import umap
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'umap-learn'], check=False)
    try:
        import umap
    except Exception:
        umap = None

if umap is None:
    print('Skipped Fig 12b - umap-learn not available.')
elif not globals().get('OOD_CACHE'):
    print('Skipped Fig 12b - run the OOD extraction cell first (it caches the features).')
else:
    from sklearn.metrics import silhouette_score
    fig, axes = plt.subplots(len(MAGS), len(ORDER), figsize=(3.4 * len(ORDER), 3.25 * len(MAGS)), squeeze=False)
    _sil_rows = []
    for r, ho in enumerate(MAGS):
        for c, method in enumerate(ORDER):
            ax = axes[r, c]
            ax.set_xticks([]); ax.set_yticks([])
            run_id = f'{PREFIX_MAP.get(method)}_ho{ho}_frac1.0_seed{UMAPB_SEED}'
            if run_id not in OOD_CACHE:
                ax.text(0.5, 0.5, 'no cached features', ha='center', va='center', transform=ax.transAxes, color='0.5')
                continue
            cache = load_ood_cache(OOD_CACHE[run_id])
            X, y = cache['test_feats'].astype(np.float64), cache['test_labels'].astype(int)     # (N, D), (N,)
            rng = np.random.default_rng(UMAPB_SEED)
            keep = np.concatenate([rng.choice(np.flatnonzero(y == lab), size=min((y == lab).sum(), UMAPB_MAX_N // 2),
                                              replace=False) for lab in (0, 1)])
            keep.sort()
            Xk, yk = X[keep], y[keep]                                                             # (n, D), (n,)
            emb = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric='cosine',
                            random_state=UMAPB_SEED).fit_transform(Xk)                            # (n, 2)
            sil_feat = float(silhouette_score(Xk, yk, metric='cosine'))
            sil_emb = float(silhouette_score(emb, yk))
            _sil_rows.append({'method': method, 'held_out_mag': int(ho), 'seed': UMAPB_SEED, 'n': len(yk),
                              'silhouette_features_cosine': sil_feat, 'silhouette_umap_2d': sil_emb})
            for lab, name, col in [(0, 'Benign', '#4C72B0'), (1, 'Malignant', '#C44E52')]:
                m = yk == lab
                ax.scatter(emb[m, 0], emb[m, 1], s=9, c=col, alpha=0.75, edgecolor='white', linewidth=0.2,
                           label=name if (r == 0 and c == 0) else None)
            ax.text(0.02, 0.02, f'sil feat {sil_feat:.2f} | 2-D {sil_emb:.2f}', transform=ax.transAxes, fontsize=7.5,
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='0.85', alpha=0.9))
            if r == 0:
                ax.set_title(method, fontweight='bold', fontsize=10.5)
            if c == 0:
                ax.set_ylabel(f'held-out {ho}×', fontsize=10.5, fontweight='bold')
    plt.tight_layout()
    fig.legend(*axes[0, 0].get_legend_handles_labels(), loc='upper center', ncol=2, frameon=False,
               bbox_to_anchor=(0.5, 1.02), markerscale=2.5)
    plt.suptitle(f'Penultimate feature space of the held-out test set (UMAP, cosine; seed {UMAPB_SEED})',
                 y=1.045, fontweight='bold')
    save_fig(fig, 'fig12b_umap_all_mags', dpi=200)
    plt.show()
    umap_silhouettes = pd.DataFrame(_sil_rows)
    umap_silhouettes.to_csv(f'{OUTPUT_DIR}/tables/umap_silhouette_by_magnification.csv', index=False)
    print(umap_silhouettes.pivot(index='method', columns='held_out_mag', values='silhouette_features_cosine')
          .reindex(ORDER).round(3).rename_axis(columns='silhouette in feature space (cosine)').to_string())

## 2h. 3D interpretability figures (Fig 19–22)

Four 3D figures in the style of the segmentation papers, but built only from quantities this study actually has. BreakHis gives patch-level labels with **no lesion mask and no follow-up images**, so a wound-area trajectory, a severity-zone exploded view or a radial profile around a lesion centroid cannot be computed here — anything drawn that way would be invented. Each figure below keeps the visual idea and changes the quantity to one that exists:

| Segmentation-paper figure | What it needs | Figure here instead |
|---|---|---|
| GradCAM layer stack | encoder stages | **Fig 19** — Grad-CAM++ at all four encoder stages, stacked in 3D, with Attention Rollout at four transformer depths for Phikon |
| Radial attribution profile around the wound centroid | a lesion mask | **Fig 20** — attribution as a function of distance and direction from the **image centre**, averaged over test images; measures the centre-vs-tile-border bias the thesis reported |
| Wound healing trajectory over time | repeat visits | **Fig 21** — the same patient at 40×, 100×, 200× and 400×, each panel scored by the encoder for which that magnification was unseen (the seed fixes one test-patient set, so one patient really does appear at all four) |
| Severity-zone exploded view | severity zones | **Fig 22** — selective-risk landscape: coverage × held-out magnification × risk, the surface behind the risk–coverage curves |

Fig 19–21 need the GPU, BreakHis and the Fig 11 cell (model builders); Fig 22 reads only the stored predictions. Every attribution figure reports three numbers per map, so the maps are not left as decoration: the share of attribution in the top 10% of pixels, the border-to-mean ratio (above 1 means the map leans on the tile border) and the offset of the attribution centroid from the image centre.

In [ ]:
# ---------- 3D interpretability utilities: multi-stage saliency + surface plotting ----------
# Plotting helpers are plain numpy/matplotlib; the saliency extractors need torch and the model
# builders from the Fig 11 cell. Nothing here draws anything by itself.
#
# Two guards live here because the first real run needed them:
#   * Grad-CAM++ on an intermediate stage can come out identically zero (every gradient negative) or
#     as a single spike. Such a map carries no information, so map_is_degenerate() flags it, the
#     extractor retries with plain Grad-CAM, and the figures label whichever method produced the map.
#   * Ratios of means over a mask explode when the outside-mask mean is ~0 (a 2e+08 "nucleus focus"
#     in the first run). attribution_enrichment() uses mass shares instead and is bounded by 1/area.
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the '3d' projection)

SAL3D_SEED = int(SEEDS[0]) if SEEDS else 13
SAL3D_MAG = int(MAGS[-1]) if MAGS else 400
SAL3D_N_SAMPLES = 12                     # images averaged in Fig 20 / Fig 24
STAGE_LABELS = ['Stage 1\n(texture)', 'Stage 2\n(edges)', 'Stage 3\n(parts)', 'Stage 4\n(semantics)']
PHIKON_DEPTHS = (3, 6, 9, 12)            # rollout up to this many transformer blocks
STAGE_CMAPS = ['Blues', 'Greens', 'YlOrBr', 'Reds']
SAL3D_MAP_SIZE = 56                      # every map is resampled to this grid before plotting


def resample_map(m, size=SAL3D_MAP_SIZE):
    """Bilinear-resample a (h, w) map onto a (size, size) grid, then scale to [0, 1]."""
    m = np.asarray(m, dtype=np.float64)                                  # (h, w)
    h, w = m.shape
    yi = np.linspace(0, h - 1, size)
    xi = np.linspace(0, w - 1, size)
    y0 = np.clip(np.floor(yi).astype(int), 0, h - 1); y1 = np.clip(y0 + 1, 0, h - 1)
    x0 = np.clip(np.floor(xi).astype(int), 0, w - 1); x1 = np.clip(x0 + 1, 0, w - 1)
    wy = (yi - y0)[:, None]; wx = (xi - x0)[None, :]
    top = m[np.ix_(y0, x0)] * (1 - wx) + m[np.ix_(y0, x1)] * wx          # (size, size)
    bot = m[np.ix_(y1, x0)] * (1 - wx) + m[np.ix_(y1, x1)] * wx          # (size, size)
    out = top * (1 - wy) + bot * wy                                       # (size, size)
    return (out - out.min()) / (out.max() - out.min() + 1e-12)


def map_is_degenerate(cam, top10_limit=0.99):
    """(flag, reason) for a map that carries no usable spatial information."""
    cam = np.asarray(cam, dtype=np.float64)
    if not np.isfinite(cam).all():
        return True, 'non-finite'
    if cam.max() <= 1e-8:
        return True, 'identically zero'
    a = cam / cam.sum()
    flat = np.sort(a.ravel())[::-1]
    if flat[:max(1, len(flat) // 10)].sum() >= top10_limit:
        return True, 'single spike'
    return False, ''


def attribution_stats(cam):
    """Three numbers that keep an attribution map honest. cam: (h, w) non-negative."""
    cam = np.asarray(cam, dtype=np.float64)
    total = cam.sum()
    if total <= 0:
        return {'top10_share': float('nan'), 'border_ratio': float('nan'), 'centroid_offset': float('nan')}
    a = cam / total                                                       # (h, w) sums to 1
    flat = np.sort(a.ravel())[::-1]
    top10 = float(flat[:max(1, len(flat) // 10)].sum())                   # concentration
    h, w = cam.shape
    band = max(1, int(round(0.1 * min(h, w))))
    frame = np.zeros((h, w), bool)
    frame[:band, :] = frame[-band:, :] = True
    frame[:, :band] = frame[:, -band:] = True
    border = float(cam[frame].mean() / (cam.mean() + 1e-12))              # >1 = leans on the tile border
    ys, xs = np.mgrid[0:h, 0:w]
    cy, cx = float((a * ys).sum()), float((a * xs).sum())
    offset = float(np.hypot(cy - (h - 1) / 2, cx - (w - 1) / 2) / (np.hypot(h, w) / 2))
    return {'top10_share': top10, 'border_ratio': border, 'centroid_offset': offset}


def mass_area_fraction(cam, frac=0.5):
    """Smallest area (fraction of the tile) holding `frac` of the attribution mass.
    Comparable across map types: a peaked map needs a small area, a flat map needs ~`frac`."""
    a = np.asarray(cam, dtype=np.float64).ravel()
    total = a.sum()
    if total <= 0:
        return float('nan')
    csum = np.cumsum(np.sort(a)[::-1] / total)
    return float((np.searchsorted(csum, frac) + 1) / a.size)


def attribution_enrichment(cam, mask):
    """(attribution mass inside mask / total mass) / (mask area / tile area).
    1.0 = indifferent, above 1 = concentrated on the mask, bounded above by 1/area share."""
    cam = np.asarray(cam, dtype=np.float64)
    total = cam.sum()
    if total <= 0 or not np.any(mask):
        return float('nan')
    return float((cam[mask].sum() / total) / mask.mean())


def surface3d(ax, Z, cmap='viridis', zlabel='', zmax=None, offset=0.0, alpha=0.95, stride=1):
    """Plot a (n, n) map as a 3D surface on the unit square, optionally lifted by `offset`."""
    Z = np.asarray(Z, dtype=float)                                        # (n, n)
    n = Z.shape[0]
    X, Y = np.meshgrid(np.linspace(0, 1, n), np.linspace(0, 1, n))        # (n, n) each
    surf = ax.plot_surface(X, Y, Z + offset, cmap=cmap, rstride=stride, cstride=stride,
                           linewidth=0, antialiased=True, alpha=alpha, vmin=offset, vmax=offset + 1.0)
    ax.set_xlabel('Lateral', labelpad=-4, fontsize=8)
    ax.set_ylabel('Axial', labelpad=-4, fontsize=8)
    ax.set_zlabel(zlabel, labelpad=-6, fontsize=8)
    ax.tick_params(labelsize=7, pad=-2)
    if zmax is not None:
        ax.set_zlim(0, zmax)
    ax.view_init(elev=30, azim=-58)
    return surf


def polar_profile(cam, n_r=22, n_theta=60):
    """Mean attribution on a (radius, angle) grid around the image centre.
    Returns X, Y, Z for a polar surface plus the radial mean curve."""
    cam = np.asarray(cam, dtype=np.float64)                               # (h, w)
    h, w = cam.shape
    ys, xs = np.mgrid[0:h, 0:w]
    dy = (ys - (h - 1) / 2) / ((h - 1) / 2)
    dx = (xs - (w - 1) / 2) / ((w - 1) / 2)
    r = np.hypot(dy, dx)                                                  # (h, w), 1.0 at the edge mid-points
    th = np.mod(np.arctan2(dy, dx), 2 * np.pi)                            # (h, w) in [0, 2pi)
    r_edges = np.linspace(0, 1.0, n_r + 1)
    th_edges = np.linspace(0, 2 * np.pi, n_theta + 1)
    ri = np.clip(np.digitize(r, r_edges) - 1, 0, n_r - 1)
    ti = np.clip(np.digitize(th, th_edges) - 1, 0, n_theta - 1)
    inside = r <= 1.0
    total = np.zeros((n_r, n_theta)); count = np.zeros((n_r, n_theta))
    np.add.at(total, (ri[inside], ti[inside]), cam[inside])
    np.add.at(count, (ri[inside], ti[inside]), 1.0)
    ring_mean = total.sum(axis=1) / np.maximum(count.sum(axis=1), 1.0)     # (n_r,) pixel-weighted
    # Near the centre a ring holds fewer pixels than there are angular sectors (32 of 60 are empty in
    # the innermost ring at 224 px). An empty sector takes its ring's mean; filling it with 0 dragged
    # the innermost ring down to 0.47 on a perfectly flat map and put a false dip at the centre.
    Z = np.where(count > 0, total / np.maximum(count, 1.0), ring_mean[:, None])   # (n_r, n_theta)
    rc = 0.5 * (r_edges[:-1] + r_edges[1:])
    thc = 0.5 * (th_edges[:-1] + th_edges[1:])
    TH, RC = np.meshgrid(thc, rc)                                          # (n_r, n_theta)
    # Repeat the first angular column so the polar surface closes instead of leaving a wedge open.
    TH = np.hstack([TH, TH[:, :1] + 2 * np.pi]); RC = np.hstack([RC, RC[:, :1]])
    Zc = np.hstack([Z, Z[:, :1]])                                          # (n_r, n_theta + 1)
    return RC * np.cos(TH), RC * np.sin(TH), Zc, rc, ring_mean


def build_saliency_backend():
    """Model builders + per-stage saliency extractors. Returns None with a message if unavailable."""
    need = ['build_from_scratch', 'to_nchw', 'IMAGE_TRANSFORM', 'GradCAMPlusPlus', 'ClassifierOutputTarget']
    missing = [n for n in need if n not in globals()]
    if missing:
        print('  3D saliency unavailable - run the Fig 11 (Grad-CAM++) cell first; missing:', missing)
        return None
    import torch
    import torch.nn.functional as F
    if not torch.cuda.is_available():
        print('  3D saliency unavailable - no CUDA GPU.')
        return None
    dev = torch.device('cuda')
    try:
        from pytorch_grad_cam import GradCAM as _GradCAM        # fallback when Grad-CAM++ degenerates
    except Exception:
        _GradCAM = None

    def stage_modules(model, model_name):
        if model_name == 'convnext_tiny':
            return list(model.stages)[:4]
        if model_name in ('swin_tiny', 'medmamba'):
            return list(model.layers)[:4]
        if model_name == 'mambavision_tiny_1k':
            return list(getattr(model, 'levels', getattr(model, 'stages', [])))[:4]
        raise KeyError(model_name)

    def load_encoder(method, ho, seed):
        prefix = PREFIX_MAP.get(method)
        ckpt = f'runs/{prefix}_ho{int(ho)}_frac1.0_seed{int(seed)}/best.pt'
        if not reader.exists(ckpt):
            return None, None
        model_name = {v: k for k, v in DISPLAY.items()}.get(method, method)
        model = build_from_scratch(model_name)
        model.load_state_dict(torch.load(io.BytesIO(reader.open_bytes(ckpt)), map_location='cpu'))
        return model.to(dev).eval(), model_name

    def stage_cams(model, model_name, tensor, target_class=None):
        """Saliency at each of the four stages for one image. tensor: (1, 3, 224, 224).
        Returns (maps, predicted class, probability, method names). Grad-CAM++ first; if a stage's
        map is degenerate, plain Grad-CAM is tried and the name records which one was used."""
        with torch.no_grad():
            probs = torch.softmax(model(tensor), dim=1)[0]                # (2,)
        cls = int(probs.argmax()) if target_class is None else int(target_class)

        def one(cam_class, stage):
            cam = cam_class(model=model, target_layers=[stage], reshape_transform=to_nchw)
            try:
                return cam(input_tensor=tensor, targets=[ClassifierOutputTarget(cls)])[0]   # (224, 224)
            finally:
                del cam

        maps, methods = [], []
        for stage in stage_modules(model, model_name):
            m = one(GradCAMPlusPlus, stage)
            used = 'Grad-CAM++'
            if map_is_degenerate(m)[0] and _GradCAM is not None:
                alt = one(_GradCAM, stage)
                if not map_is_degenerate(alt)[0]:
                    m, used = alt, 'Grad-CAM'
            maps.append(m); methods.append(used)
        return maps, cls, float(probs[cls]), methods

    def phikon_depth_maps(fm, head, tensor, depths=PHIKON_DEPTHS):
        """Attention Rollout truncated at several depths. Returns (maps, class, probability, methods)."""
        with torch.no_grad():
            out = fm(pixel_values=tensor, output_attentions=True)
            logits = head(out.last_hidden_state[:, 0])                     # (1, 2)
            probs = torch.softmax(logits, dim=1)[0]                        # (2,)
        if not getattr(out, 'attentions', None):
            raise RuntimeError('Phikon returned no attentions; load it with attn_implementation="eager"')
        maps = []
        for depth in depths:
            result = None
            for A in out.attentions[:depth]:
                A = A.mean(dim=1)[0]                                       # (N, N)
                A = 0.5 * A + 0.5 * torch.eye(A.size(-1), device=A.device)
                A = A / A.sum(dim=-1, keepdim=True)
                result = A if result is None else A @ result
            m = result[0, 1:]                                              # (196,)
            g = int(round(m.numel() ** 0.5))
            m = m[:g * g].reshape(1, 1, g, g)                              # (1, 1, 14, 14)
            m = F.interpolate(m, size=(224, 224), mode='bilinear', align_corners=False)[0, 0]
            maps.append(((m - m.min()) / (m.max() - m.min() + 1e-8)).float().cpu().numpy())
        return maps, int(probs.argmax()), float(probs.max()), ['Attention Rollout'] * len(maps)

    def load_phikon():
        from transformers import AutoModel
        try:
            fm = AutoModel.from_pretrained('owkin/phikon', add_pooling_layer=False,
                                           attn_implementation='eager').to(dev).eval()
        except TypeError:
            fm = AutoModel.from_pretrained('owkin/phikon', add_pooling_layer=False).to(dev).eval()
        for p in fm.parameters():
            p.requires_grad_(False)
        return fm

    def load_probe(ho, seed):
        path = f'runs/{PREFIX_MAP["Phikon (frozen FM)"]}_ho{int(ho)}_frac1.0_seed{int(seed)}/probe.pt'
        if not reader.exists(path):
            return None
        saved = torch.load(io.BytesIO(reader.open_bytes(path)), map_location=dev)
        head = torch.nn.Linear(int(saved['feature_dim']), int(saved['num_classes'])).to(dev).eval()
        head.load_state_dict({k.replace('head.', ''): v for k, v in saved['state_dict'].items()})
        return head

    def to_tensor(pil_image):
        return IMAGE_TRANSFORM(pil_image).unsqueeze(0).to(dev)             # (1, 3, 224, 224)

    return {'torch': torch, 'device': dev, 'load_encoder': load_encoder, 'stage_cams': stage_cams,
            'load_phikon': load_phikon, 'load_probe': load_probe, 'phikon_depth_maps': phikon_depth_maps,
            'to_tensor': to_tensor, 'stage_modules': stage_modules, 'gradcam_fallback': _GradCAM is not None}


SAL3D = build_saliency_backend()
print('3D saliency backend:', 'ready' if SAL3D else 'not available (Fig 19-26 will skip; Fig 22 still runs)')
if SAL3D and not SAL3D['gradcam_fallback']:
    print('  note: plain Grad-CAM not importable, so degenerate Grad-CAM++ stages stay degenerate')

In [ ]:
# ---------- Fig 19: saliency layer stack in 3D, one figure per encoder ----------
# Task-trained encoders: Grad-CAM++ at each of the four stages. Phikon: Attention Rollout truncated
# at four transformer depths. Row 1 = input and the four maps as overlays; row 2 = the stack in 3D
# (oblique and top-down), the two deepest maps as surfaces, and the three attribution statistics.
FIG19_SAMPLE_LABEL = 1        # 1 = malignant, 0 = benign; the first test image of that class is used


def _depth_tick(label):
    """Short tick for a layer: 'Stage 3' for a CNN / Swin / Mamba stage, 'Block 9' for a Phikon depth
    (the first line of a rollout label is just 'Rollout', which made all four ticks identical)."""
    import re
    m = re.search(r'block\s*(\d+)', label, flags=re.IGNORECASE)
    return f'Block {m.group(1)}' if m else label.split('\n')[0]

if SAL3D is None:
    print('Skipped Fig 19 - 3D saliency backend not available.')
else:
    _frame19 = reconstruct_test_paths(SAL3D_MAG, SAL3D_SEED)
    if _frame19 is None or len(_frame19) == 0:
        print('Skipped Fig 19 - test set could not be rebuilt.')
    else:
        _idx19 = _frame19.index[_frame19.coarse_label == FIG19_SAMPLE_LABEL][0]
        _row19 = _frame19.loc[_idx19]
        _pil19 = PILImage.open(_row19['path']).convert('RGB')
        _show19 = np.asarray(_pil19.resize((224, 224)), dtype=np.float32) / 255.0      # (224, 224, 3)
        _tensor19 = SAL3D['to_tensor'](_pil19)                                          # (1, 3, 224, 224)
        _true19 = int(_row19['coarse_label'])
        layer_stack_stats = []

        for method in ORDER:
            try:
                if method == 'Phikon (frozen FM)':
                    fm = SAL3D['load_phikon']()
                    head = SAL3D['load_probe'](SAL3D_MAG, SAL3D_SEED)
                    if head is None:
                        print('  Phikon: probe.pt missing - skipped'); del fm; continue
                    maps, pred, prob, cam_methods = SAL3D['phikon_depth_maps'](fm, head, _tensor19)
                    labels = [f'Rollout\nto block {d}' for d in PHIKON_DEPTHS]
                    del fm, head
                else:
                    model, model_name = SAL3D['load_encoder'](method, SAL3D_MAG, SAL3D_SEED)
                    if model is None:
                        print(f'  {method}: best.pt missing - skipped'); continue
                    maps, pred, prob, cam_methods = SAL3D['stage_cams'](model, model_name, _tensor19)
                    labels = STAGE_LABELS
                    del model
                SAL3D['torch'].cuda.empty_cache()
            except Exception as exc:
                print(f'  {method}: FAILED - {repr(exc)[:200]}'); continue

            grids = [resample_map(m) for m in maps]                                    # 4 x (56, 56)
            stats = [attribution_stats(m) for m in maps]
            flags = [map_is_degenerate(m) for m in maps]                               # (bad, reason) per depth
            for lab, st, (bad, why), used in zip(labels, stats, flags, cam_methods):
                layer_stack_stats.append({'method': method, 'layer': lab.replace('\n', ' '),
                                          'saliency': used, 'degenerate': bool(bad), 'reason': why, **st})

            fig = plt.figure(figsize=(19, 7.4))
            gs = fig.add_gridspec(2, 5, height_ratios=[1.0, 1.25], hspace=0.22, wspace=0.3)

            ax = fig.add_subplot(gs[0, 0])
            ax.imshow(_show19); ax.set_xticks([]); ax.set_yticks([])
            ax.set_title(f'Input - true {CLASS_NAMES[_true19]}', fontsize=10.5, fontweight='bold')
            for k, (m, lab) in enumerate(zip(maps, labels)):
                ax = fig.add_subplot(gs[0, k + 1])
                ax.imshow(show_cam_on_image(_show19, m, use_rgb=True)); ax.set_xticks([]); ax.set_yticks([])
                ax.set_title(f'{lab}\n{cam_methods[k]}', fontsize=10, fontweight='bold')
                bad, why = flags[k]
                ax.set_xlabel(f'map {why} - not interpretable' if bad else
                              f"top10 {stats[k]['top10_share']:.2f} | border {stats[k]['border_ratio']:.2f}",
                              fontsize=8.5, color='#d62728' if bad else '0.3',
                              fontweight='bold' if bad else 'normal')

            for col, (elev, azim, title) in enumerate([(26, -58, 'Layer stack - oblique'),
                                                       (78, -90, 'Layer stack - top-down')]):
                ax = fig.add_subplot(gs[1, col], projection='3d')
                for k, g in enumerate(grids):
                    surface3d(ax, 0.85 * g, cmap=STAGE_CMAPS[k], zlabel='Layer depth', offset=float(k),
                              alpha=0.85, stride=2)
                if elev < 60:                                   # the top-down view has no readable depth axis
                    ax.set_zticks(range(len(grids)))
                    ax.set_zticklabels([_depth_tick(l) for l in labels], fontsize=7.5)
                    ax.set_zlabel('')                           # the tick labels already name each layer
                else:
                    ax.set_zticks([]); ax.set_zlabel('')
                ax.set_zlim(0, len(grids) + 0.1); ax.view_init(elev=elev, azim=azim)
                ax.set_title(title, fontsize=10.5, fontweight='bold')
            for col, k in enumerate([2, 3], start=2):
                ax = fig.add_subplot(gs[1, col], projection='3d')
                surface3d(ax, grids[k], cmap=STAGE_CMAPS[k], zlabel='Attribution', zmax=1.0, stride=1)
                ax.set_title(labels[k].replace('\n', ' '), fontsize=10.5, fontweight='bold')

            ax = fig.add_subplot(gs[1, 4])
            _x = np.arange(len(labels)); _w = 0.27
            for j, (key, lbl, col) in enumerate([('top10_share', 'Top-10% share', '#0072B2'),
                                                 ('border_ratio', 'Border / mean', '#D55E00'),
                                                 ('centroid_offset', 'Centroid offset', '#009E73')]):
                ax.bar(_x + (j - 1) * _w, [s[key] for s in stats], width=_w, color=col, edgecolor='black',
                       linewidth=0.4, label=lbl)
            ax.axhline(1.0, color='0.4', linestyle='--', linewidth=0.8)
            ax.set_xticks(_x); ax.set_xticklabels([_depth_tick(l) for l in labels], fontsize=8, rotation=12)
            ax.set_title('Attribution statistics', fontsize=10.5, fontweight='bold')
            # head-room above the tallest bar so the legend sits in empty space, not on the bars
            _top = max(max(s[k] for s in stats for k in ('top10_share', 'border_ratio', 'centroid_offset')), 1.0)
            ax.set_ylim(0, 1.6 * _top)
            ax.legend(fontsize=7.5, frameon=False, loc='upper left', ncol=1)
            ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

            plt.suptitle(f'{method} - saliency by depth (BreakHis, held-out {SAL3D_MAG}×, seed {SAL3D_SEED}; '
                         f'predicted {CLASS_NAMES[pred]} p={prob:.2f}, true {CLASS_NAMES[_true19]})',
                         y=1.0, fontsize=12.5, fontweight='bold')
            save_fig(fig, f"fig19_layer_stack_{method.split(' ')[0].replace('-', '')}", dpi=170)
            plt.show()

        if layer_stack_stats:
            layer_stack_stats = pd.DataFrame(layer_stack_stats)
            layer_stack_stats.to_csv(f'{OUTPUT_DIR}/tables/saliency_layer_statistics.csv', index=False)
            print('\nAttribution statistics per encoder and depth '
                  '(top10 = share in the top 10% of pixels, border = border-to-mean ratio):')
            print(layer_stack_stats.round(3).to_string(index=False))

In [ ]:
# ---------- Fig 20: radial attribution profile around the image centre ----------
# BreakHis has no lesion mask, so the profile is taken around the image centre: r = distance from
# the centre (1.0 at the edge mid-points), theta = direction, z = mean attribution of the deepest
# saliency map, averaged over the same test images for every encoder. The lower row gives the radial
# mean with a +/-1 SD band across images and the border-to-mean ratio, which is what "the map leans
# on the tile border" means numerically.
if SAL3D is None:
    print('Skipped Fig 20 - 3D saliency backend not available.')
else:
    _frame20 = reconstruct_test_paths(SAL3D_MAG, SAL3D_SEED)
    if _frame20 is None or len(_frame20) == 0:
        print('Skipped Fig 20 - test set could not be rebuilt.')
    else:
        _picks20 = []
        for lab in (0, 1):
            _picks20 += _frame20.index[_frame20.coarse_label == lab].tolist()[:SAL3D_N_SAMPLES // 2]
        _pils20 = [PILImage.open(_frame20.loc[i, 'path']).convert('RGB') for i in _picks20]
        print(f'Fig 20: averaging over {len(_picks20)} test images at held-out {SAL3D_MAG}×, seed {SAL3D_SEED}')
        radial_rows, radial_curves, radial_surfaces = [], {}, {}

        for method in ORDER:
            try:
                maps = []
                if method == 'Phikon (frozen FM)':
                    fm = SAL3D['load_phikon']()
                    head = SAL3D['load_probe'](SAL3D_MAG, SAL3D_SEED)
                    if head is None:
                        print('  Phikon: probe.pt missing - skipped'); del fm; continue
                    for pil in _pils20:
                        m, _, _, _ = SAL3D['phikon_depth_maps'](fm, head, SAL3D['to_tensor'](pil), depths=(12,))
                        if not map_is_degenerate(m[0])[0]:
                            maps.append(m[0])
                    del fm, head
                else:
                    model, model_name = SAL3D['load_encoder'](method, SAL3D_MAG, SAL3D_SEED)
                    if model is None:
                        print(f'  {method}: best.pt missing - skipped'); continue
                    for pil in _pils20:
                        stage_maps, _, _, _ = SAL3D['stage_cams'](model, model_name, SAL3D['to_tensor'](pil))
                        if not map_is_degenerate(stage_maps[-1])[0]:
                            maps.append(stage_maps[-1])
                    del model
                SAL3D['torch'].cuda.empty_cache()
            except Exception as exc:
                print(f'  {method}: FAILED - {repr(exc)[:200]}'); continue

            curves = []
            for m in maps:
                _, _, _, rc, prof = polar_profile(m)
                curves.append(prof)                                              # (n_r,)
                radial_rows.append({'method': method, **attribution_stats(m)})
            mean_map = np.mean(np.stack(maps), axis=0)                            # (224, 224)
            X, Y, Z, rc, _ = polar_profile(mean_map)
            radial_surfaces[method] = (X, Y, Z)
            radial_curves[method] = (rc, np.mean(np.stack(curves), axis=0), np.std(np.stack(curves), axis=0))
            print(f'  {method}: {len(maps)} usable maps of {len(_pils20)}')

        if not radial_surfaces:
            print('No saliency maps computed; Fig 20 skipped.')
        else:
            methods20 = [m for m in ORDER if m in radial_surfaces]
            radial_stats = pd.DataFrame(radial_rows).groupby('method').mean(numeric_only=True).reindex(methods20)
            fig = plt.figure(figsize=(4.4 * len(methods20), 8.4))
            gs = fig.add_gridspec(2, len(methods20), height_ratios=[1.25, 1.0], hspace=0.28, wspace=0.22)
            for c, method in enumerate(methods20):
                X, Y, Z = radial_surfaces[method]
                ax = fig.add_subplot(gs[0, c], projection='3d')
                ax.plot_surface(X, Y, Z, cmap='inferno', linewidth=0, antialiased=True,
                                vmin=float(Z.min()), vmax=float(Z.max()))
                ax.set_xlabel('cos θ', fontsize=8, labelpad=-4); ax.set_ylabel('sin θ', fontsize=8, labelpad=-4)
                ax.set_zlabel('Attribution', fontsize=8, labelpad=-6)
                ax.tick_params(labelsize=7, pad=-2); ax.view_init(elev=30, azim=-58)
                ax.set_title(method, fontsize=11, fontweight='bold')

                ax = fig.add_subplot(gs[1, c])
                rc, mean_c, sd_c = radial_curves[method]
                ax.plot(rc, mean_c, color=COLORS[method], linewidth=2.2)
                ax.fill_between(rc, mean_c - sd_c, mean_c + sd_c, color=COLORS[method], alpha=0.18, linewidth=0)
                ax.axvline(0.9, color='0.45', linestyle='--', linewidth=0.9)
                ax.text(0.9, ax.get_ylim()[1], ' outer 10%', fontsize=8, color='0.35', va='top')
                ax.set_xlabel('Distance from image centre (1.0 = edge)'); ax.set_xlim(0, 1)
                if c == 0:
                    ax.set_ylabel('Mean attribution')
                ax.set_title(f"border/mean {radial_stats.loc[method, 'border_ratio']:.2f}   "
                             f"top-10% {radial_stats.loc[method, 'top10_share']:.2f}", fontsize=9.5)
                ax.grid(True, linewidth=0.4); ax.set_axisbelow(True)
                ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
            plt.suptitle(f'Radial attribution profile around the image centre '
                         f'(deepest saliency map, mean over {len(_picks20)} test images, held-out {SAL3D_MAG}×)',
                         y=0.98, fontsize=12.5, fontweight='bold')
            save_fig(fig, 'fig20_radial_attribution_profile', dpi=170)
            plt.show()
            radial_stats.round(3).to_csv(f'{OUTPUT_DIR}/tables/radial_attribution_stats.csv')
            print('\nAttribution statistics averaged over images (border/mean > 1 = border-heavy map):')
            print(radial_stats.round(3).to_string())

In [ ]:
# ---------- Fig 21: one patient across 40x, 100x, 200x and 400x ----------
# The patient split is drawn from the seed alone, so a fixed seed gives the same test patients at
# every held-out magnification: one patient really can be followed across all four. Each panel is
# scored by the encoder run for which that magnification was held out, i.e. the magnification in the
# panel was unseen by the model that produced the map. This replaces a healing trajectory, which
# BreakHis cannot support - there are no repeat visits.
if SAL3D is None:
    print('Skipped Fig 21 - 3D saliency backend not available.')
else:
    _sets21 = {}
    for ho in MAGS:
        f = reconstruct_test_paths(int(ho), SAL3D_SEED)
        if f is None or len(f) == 0:
            _sets21 = {}; break
        _sets21[int(ho)] = f
    if not _sets21:
        print('Skipped Fig 21 - test sets could not be rebuilt.')
    else:
        _common = set.intersection(*[set(f.patient_id) for f in _sets21.values()])
        print(f'Patients in the test set at all {len(_sets21)} magnifications (seed {SAL3D_SEED}): {len(_common)}')
        if not _common:
            print('Skipped Fig 21 - no patient is a test patient at every magnification.')
        else:
            # Deterministic choice: the malignant patient with the most test images, else any patient.
            _counts = {p: sum(int((f.patient_id == p).sum()) for f in _sets21.values()) for p in sorted(_common)}
            _labels = {p: int(_sets21[MAGS[0]].loc[_sets21[MAGS[0]].patient_id == p, 'coarse_label'].iloc[0])
                       for p in _counts}
            _malignant = [p for p in _counts if _labels[p] == 1]
            _patient = max(_malignant or list(_counts), key=lambda p: _counts[p])
            _true21 = _labels[_patient]
            print(f'Chosen patient: {_patient} ({CLASS_NAMES[_true21]}), {_counts[_patient]} test images in total')

            _pils21, _shows21 = {}, {}
            for ho, f in _sets21.items():
                row = f[f.patient_id == _patient].iloc[0]
                _pils21[ho] = PILImage.open(row['path']).convert('RGB')
                _shows21[ho] = np.asarray(_pils21[ho].resize((224, 224)), dtype=np.float32) / 255.0
            trajectory_rows = []

            for method in ORDER:
                panels = {}
                try:
                    if method == 'Phikon (frozen FM)':
                        fm = SAL3D['load_phikon']()
                        for ho in _sets21:
                            head = SAL3D['load_probe'](ho, SAL3D_SEED)
                            if head is None:
                                continue
                            m, pred, prob, used = SAL3D['phikon_depth_maps'](
                                fm, head, SAL3D['to_tensor'](_pils21[ho]), depths=(12,))
                            panels[ho] = (m[0], pred, prob, used[0])
                            del head
                        del fm
                    else:
                        for ho in _sets21:
                            model, model_name = SAL3D['load_encoder'](method, ho, SAL3D_SEED)
                            if model is None:
                                continue
                            maps, pred, prob, used = SAL3D['stage_cams'](model, model_name,
                                                                        SAL3D['to_tensor'](_pils21[ho]))
                            panels[ho] = (maps[-1], pred, prob, used[-1])
                            del model
                            SAL3D['torch'].cuda.empty_cache()
                    SAL3D['torch'].cuda.empty_cache()
                except Exception as exc:
                    print(f'  {method}: FAILED - {repr(exc)[:200]}'); continue
                if not panels:
                    print(f'  {method}: no checkpoints for this patient - skipped'); continue

                hos = [ho for ho in MAGS if int(ho) in panels]
                fig = plt.figure(figsize=(4.0 * len(hos), 7.6))
                gs = fig.add_gridspec(2, len(hos), height_ratios=[1.0, 1.2], hspace=0.06, wspace=0.14)
                for c, ho in enumerate(hos):
                    cam, pred, prob, used = panels[int(ho)]
                    st = attribution_stats(cam)
                    trajectory_rows.append({'method': method, 'held_out_mag': int(ho), 'patient': _patient,
                                            'true': CLASS_NAMES[_true21], 'predicted': CLASS_NAMES[pred],
                                            'probability': prob, **st})
                    ax = fig.add_subplot(gs[0, c])
                    ax.imshow(show_cam_on_image(_shows21[int(ho)], cam, use_rgb=True))
                    ax.set_xticks([]); ax.set_yticks([])
                    ok = pred == _true21
                    for s in ax.spines.values():
                        s.set_visible(True); s.set_linewidth(2.2); s.set_edgecolor('#2ca02c' if ok else '#d62728')
                    _bad21, _why21 = map_is_degenerate(cam)
                    ax.set_title(f'{int(ho)}× (unseen by this run)\n{used}'
                                 + (f' - map {_why21}' if _bad21 else ''), fontsize=10.5, fontweight='bold')
                    ax.set_xlabel(f"pred {CLASS_NAMES[pred]} p={prob:.2f}   border {st['border_ratio']:.2f}",
                                  fontsize=9, color='#2ca02c' if ok else '#d62728', fontweight='bold')
                    ax = fig.add_subplot(gs[1, c], projection='3d')
                    surface3d(ax, resample_map(cam), cmap='magma', zlabel='Attribution', zmax=1.0)
                plt.suptitle(f'{method} - patient {_patient} ({CLASS_NAMES[_true21]}) across magnifications, '
                             f'each panel from the run that held that magnification out (seed {SAL3D_SEED})',
                             y=0.99, fontsize=12, fontweight='bold')
                save_fig(fig, f"fig21_magnification_trajectory_{method.split(' ')[0].replace('-', '')}", dpi=170)
                plt.show()

            if trajectory_rows:
                magnification_trajectory = pd.DataFrame(trajectory_rows)
                magnification_trajectory.to_csv(f'{OUTPUT_DIR}/tables/magnification_trajectory.csv', index=False)
                print('\nOne patient, four magnifications (probability = predicted-class confidence, T = 1):')
                print(magnification_trajectory.pivot(index='method', columns='held_out_mag', values='probability')
                      .reindex(ORDER).round(3).to_string())
                print('\nBorder-to-mean ratio of the same maps:')
                print(magnification_trajectory.pivot(index='method', columns='held_out_mag', values='border_ratio')
                      .reindex(ORDER).round(2).to_string())

In [ ]:
# ---------- Fig 22: selective-risk landscape (coverage x held-out magnification x risk) ----------
# The surface behind Fig 5: for every encoder, the cumulative selective risk as a function of
# coverage at each held-out magnification, averaged over seeds. Reads the stored predictions only -
# no GPU, no images. Ribbons, one per magnification, because the magnification axis has four ordinal
# levels and nothing exists between them.
FIG22_COVERAGE = np.linspace(0.30, 1.0, 71)          # (71,) coverage grid

_risk_cube, _missing22 = {}, []
for method in ORDER:
    prefix = PREFIX_MAP.get(method)
    rows = []
    for ho in MAGS:
        per_seed = []
        for seed in SEEDS:
            npz_path = f'runs/{prefix}_ho{int(ho)}_frac1.0_seed{int(seed)}/test_outputs.npz'
            if not (prefix and reader.exists(npz_path)):
                continue
            d = np.load(io.BytesIO(reader.open_bytes(npz_path)))
            probs, labels = d['probs'], d['labels']                       # (N, 2), (N,)
            conf = probs.max(axis=1); pred = probs.argmax(axis=1)         # (N,), (N,)
            order = np.argsort(-conf)
            err = (pred != labels).astype(float)[order]                   # (N,) ranked most-confident first
            risk = np.cumsum(err) / np.arange(1, len(err) + 1)            # (N,) risk at each coverage
            cov = np.arange(1, len(err) + 1) / len(err)                   # (N,)
            per_seed.append(np.interp(FIG22_COVERAGE, cov, risk))         # (71,)
        if per_seed:
            rows.append(np.mean(np.stack(per_seed), axis=0))              # (71,) mean over seeds
        else:
            rows.append(np.full(len(FIG22_COVERAGE), np.nan)); _missing22.append((method, int(ho)))
    _risk_cube[method] = np.stack(rows)                                   # (n_mags, 71)

if _missing22:
    print('  no stored predictions for:', _missing22)
if all(np.isnan(v).all() for v in _risk_cube.values()):
    print('Skipped Fig 22 - no stored predictions found.')
else:
    _zmax = float(np.nanmax([np.nanmax(v) for v in _risk_cube.values()]) * 1.05)
    fig = plt.figure(figsize=(4.6 * len(ORDER), 4.9))
    for i, method in enumerate(ORDER):
        ax = fig.add_subplot(1, len(ORDER), i + 1, projection='3d')
        Z = _risk_cube[method]                                            # (n_mags, 71)
        for j, ho in enumerate(MAGS):
            if np.isnan(Z[j]).all():
                continue
            ax.plot(FIG22_COVERAGE, np.full_like(FIG22_COVERAGE, j, dtype=float), Z[j],
                    color=COLORS[method], linewidth=2.0, alpha=0.95)
            verts_y = np.full_like(FIG22_COVERAGE, j, dtype=float)
            ax.plot_surface(np.vstack([FIG22_COVERAGE, FIG22_COVERAGE]),
                            np.vstack([verts_y, verts_y + 0.34]),
                            np.vstack([Z[j], Z[j]]), color=COLORS[method], alpha=0.30, linewidth=0,
                            shade=False)
        ax.set_xlabel('Coverage', labelpad=-2, fontsize=9)
        ax.set_ylabel('Held-out mag', labelpad=2, fontsize=9)
        ax.set_zlabel('Selective risk', labelpad=7, fontsize=9)
        ax.set_yticks(range(len(MAGS))); ax.set_yticklabels([f'{int(m)}×' for m in MAGS], fontsize=8)
        ax.set_zlim(0, _zmax); ax.set_xlim(FIG22_COVERAGE[0], 1.0)
        # at most 5 round-number ticks, not 9 (they collided with the axis label), pushed off the axis line
        from matplotlib.ticker import MaxNLocator, FormatStrFormatter
        ax.zaxis.set_major_locator(MaxNLocator(nbins=4, steps=[1, 2, 2.5, 5, 10]))
        ax.zaxis.set_major_formatter(FormatStrFormatter('%.2f'))
        ax.tick_params(labelsize=8, pad=-1)
        ax.tick_params(axis='z', pad=3)
        ax.view_init(elev=24, azim=-62)
        ax.set_title(method, fontsize=11, fontweight='bold')
    plt.suptitle('Selective-risk landscape: error among the most confident images, per held-out magnification '
                 '(mean over seeds)', y=1.02, fontsize=12.5, fontweight='bold')
    save_fig(fig, 'fig22_selective_risk_landscape', dpi=170)
    plt.show()

    _at = {c: [] for c in (0.8, 0.9, 1.0)}
    for method in ORDER:
        for c in _at:
            k = int(np.argmin(np.abs(FIG22_COVERAGE - c)))
            _at[c].append(np.nanmean(_risk_cube[method][:, k]))
    risk_landscape_table = pd.DataFrame(_at, index=ORDER).rename(columns=lambda c: f'risk @ {int(c * 100)}% coverage')
    risk_landscape_table.to_csv(f'{OUTPUT_DIR}/tables/risk_landscape_summary.csv')
    print('Mean over held-out magnifications (matches T5 at 80/90% coverage):')
    print(risk_landscape_table.round(3).to_string())

In [ ]:
# ---------- Fig 23: attention zones marked on the tile, with a haematoxylin-rich (nuclei) proxy ----------
# The zone map of the segmentation papers needs an annotated lesion; BreakHis has none. Two things
# can still be drawn ON the tile without inventing ground truth:
#   1. attribution zones - the top 10% / 25% / 50% of the model's own saliency, as contours
#      (core / mid / peripheral). These describe the model, not the tissue.
#   2. a nuclei proxy - the NUCLEI_QUANTILE most haematoxylin-rich pixels after Ruifrok colour
#      deconvolution, opened and de-speckled. A fixed quantile rather than a per-tile threshold such
#      as Otsu, so the proxy covers the same share of every tile and the enrichment ceiling
#      (1 / area share) is the same for every tile and encoder. The deconvolution is checked below on
#      pure-stain pixels before it is used. It stays an automatic approximation, never a
#      pathologist's annotation.
# The number that follows from it is an enrichment: what share of the attribution mass lands on the
# proxy, divided by the proxy's area share. 1.0 means indifferent; the ceiling is 1 / area share.
from scipy import ndimage as _ndi

ZONE_QUANTILES = [(0.90, 'Core (top 10%)', '#8B0000'), (0.75, 'Mid (top 25%)', '#E4572E'),
                  (0.50, 'Peripheral (top 50%)', '#F2C14E')]
NUCLEI_QUANTILE = 0.25        # the 25% most haematoxylin-rich pixels
NUCLEI_MIN_SIZE = 25          # pixels, at 224 x 224
_RUIFROK_HE = np.array([[0.650, 0.704, 0.286],      # haematoxylin
                        [0.072, 0.990, 0.105],      # eosin
                        [0.268, 0.570, 0.776]])     # residual


_STAINS = _RUIFROK_HE / np.linalg.norm(_RUIFROK_HE, axis=1, keepdims=True)   # rows = stain OD vectors


def stain_concentrations(rgb01):
    """Ruifrok-Johnston: a pixel's optical density is OD = C @ M (rows of M = stain vectors), so the
    concentrations are C = OD @ inv(M). Returns (H, W, 3) = haematoxylin, eosin, residual."""
    rgb = np.clip(np.asarray(rgb01, dtype=np.float64), 1e-6, 1.0)          # (H, W, 3)
    od = -np.log10(rgb)                                                     # (H, W, 3) optical density
    return (od.reshape(-1, 3) @ np.linalg.inv(_STAINS)).reshape(rgb.shape)  # (H, W, 3)


# Self-check: a pixel of pure stain must come back as that stain alone. The first version multiplied by
# inv(M).T instead of inv(M); it returned 0.35 haematoxylin for pure haematoxylin and ranked white
# background above cytoplasm, so the "nuclei" proxy outlined nuclei AND empty lumen.
for _k, _name in enumerate(('haematoxylin', 'eosin')):
    _pure = stain_concentrations(10.0 ** (-_STAINS[_k])[None, None, :])[0, 0]
    assert np.allclose(_pure, np.eye(3)[_k], atol=1e-6), f'colour deconvolution broken for pure {_name}: {_pure}'
_h_nucleus, _h_white = (stain_concentrations(np.array(c, float)[None, None] / 255.0)[0, 0, 0]
                        for c in ((80, 55, 140), (238, 232, 228)))
assert _h_nucleus > 10 * max(_h_white, 1e-3), (_h_nucleus, _h_white)


def haematoxylin_channel(rgb01):
    """Haematoxylin concentration per pixel, scaled to [0, 1] between its 1st and 99th percentile."""
    h = stain_concentrations(rgb01)[..., 0]                                 # (H, W)
    lo, hi = np.percentile(h, [1, 99])
    return np.clip((h - lo) / (hi - lo + 1e-12), 0, 1)


def otsu_threshold(x, bins=256):
    counts, edges = np.histogram(np.asarray(x).ravel(), bins=bins, range=(0.0, 1.0))
    p = counts / max(1, counts.sum())
    centres = 0.5 * (edges[:-1] + edges[1:])
    w0 = np.cumsum(p); w1 = 1.0 - w0
    m0 = np.cumsum(p * centres) / np.maximum(w0, 1e-12)
    m1 = (np.sum(p * centres) - np.cumsum(p * centres)) / np.maximum(w1, 1e-12)
    between = w0 * w1 * (m0 - m1) ** 2                                      # (bins,)
    return float(centres[int(np.argmax(between))])


def nuclei_proxy_mask(rgb01, quantile=NUCLEI_QUANTILE, min_size=NUCLEI_MIN_SIZE, use_otsu=False):
    """The most haematoxylin-rich pixels, opened and de-speckled. Fixed quantile by default."""
    h = haematoxylin_channel(rgb01)                                         # (H, W)
    thr = otsu_threshold(h) if use_otsu else float(np.quantile(h, 1.0 - quantile))
    mask = h >= thr                                                         # (H, W) bool
    mask = _ndi.binary_opening(mask, structure=np.ones((3, 3)))
    labels, n = _ndi.label(mask)
    if n:
        sizes = _ndi.sum(mask, labels, index=np.arange(1, n + 1))
        keep = np.zeros(n + 1, bool); keep[1:] = sizes >= min_size
        mask = keep[labels]
    return mask


def attention_zones(cam, smooth=2.0):
    """Boolean zone masks at the attribution quantiles in ZONE_QUANTILES.
    The map is Gaussian-smoothed first so the contours are closed curves rather than speckle;
    smoothing changes where a contour sits by a pixel or two, never which regions are ranked highest."""
    cam = np.asarray(cam, dtype=np.float64)
    if smooth:
        cam = _ndi.gaussian_filter(cam, sigma=smooth)
    return [(label, colour, cam >= np.quantile(cam, q)) for q, label, colour in ZONE_QUANTILES]


def nucleus_enrichment(cam, nuclei):
    """Attribution mass share on the nuclei proxy / the proxy's area share (1.0 = indifferent)."""
    return attribution_enrichment(cam, nuclei)


if SAL3D is None:
    print('Skipped Fig 23 - 3D saliency backend not available.')
else:
    _frame23 = reconstruct_test_paths(SAL3D_MAG, SAL3D_SEED)
    if _frame23 is None or len(_frame23) == 0:
        print('Skipped Fig 23 - test set could not be rebuilt.')
    else:
        _idx23 = _frame23.index[_frame23.coarse_label == 1][0]
        _pil23 = PILImage.open(_frame23.loc[_idx23, 'path']).convert('RGB')
        _show23 = np.asarray(_pil23.resize((224, 224)), dtype=np.float32) / 255.0     # (224, 224, 3)
        _nuclei23 = nuclei_proxy_mask(_show23)                                         # (224, 224) bool
        _otsu23 = nuclei_proxy_mask(_show23, use_otsu=True)
        print(f'Nuclei proxy (top {int(100 * NUCLEI_QUANTILE)}% haematoxylin): '
              f'{100 * _nuclei23.mean():.1f}% of the tile, {int(_ndi.label(_nuclei23)[1])} blobs   '
              f'[Otsu would take {100 * _otsu23.mean():.1f}%]')
        _panels23 = {}
        for method in ORDER:
            try:
                if method == 'Phikon (frozen FM)':
                    fm = SAL3D['load_phikon']()
                    head = SAL3D['load_probe'](SAL3D_MAG, SAL3D_SEED)
                    if head is None:
                        print('  Phikon: probe.pt missing - skipped'); del fm; continue
                    maps, pred, prob, used = SAL3D['phikon_depth_maps'](fm, head, SAL3D['to_tensor'](_pil23),
                                                                        depths=(12,))
                    del fm, head
                else:
                    model, model_name = SAL3D['load_encoder'](method, SAL3D_MAG, SAL3D_SEED)
                    if model is None:
                        print(f'  {method}: best.pt missing - skipped'); continue
                    maps, pred, prob, used = SAL3D['stage_cams'](model, model_name, SAL3D['to_tensor'](_pil23))
                    del model
                SAL3D['torch'].cuda.empty_cache()
                bad, why = map_is_degenerate(maps[-1])
                if bad:
                    print(f'  {method}: deepest map is {why} - column left out'); continue
                _panels23[method] = (maps[-1], pred, prob, used[-1])
            except Exception as exc:
                print(f'  {method}: FAILED - {repr(exc)[:200]}')

        if not _panels23:
            print('No usable saliency maps; Fig 23 skipped.')
        else:
            methods23 = [m for m in ORDER if m in _panels23]
            fig = plt.figure(figsize=(4.3 * len(methods23), 8.6))
            gs = fig.add_gridspec(2, len(methods23), height_ratios=[1.0, 1.05], hspace=0.12, wspace=0.12)
            for c, method in enumerate(methods23):
                cam, pred, prob, used = _panels23[method]
                zones = attention_zones(cam)
                enr = nucleus_enrichment(cam, _nuclei23)
                stats = attribution_stats(cam)

                ax = fig.add_subplot(gs[0, c])
                ax.imshow(_show23)
                ax.contour(_nuclei23.astype(float), levels=[0.5], colors='#1B9E77', linewidths=0.6)
                for label, colour, mask in zones:
                    ax.contour(mask.astype(float), levels=[0.5], colors=colour, linewidths=1.8)
                ax.set_xticks([]); ax.set_yticks([])
                ax.set_title(f'{method}\n{used}', fontsize=10.5, fontweight='bold')
                ax.set_xlabel(f'nuclei enrichment {enr:.2f}×   border {stats["border_ratio"]:.2f}\n'
                              f'pred {CLASS_NAMES[pred]}   p={prob:.2f}', fontsize=8.5)

                ax = fig.add_subplot(gs[1, c], projection='3d')
                cam_g = resample_map(cam)                                              # (56, 56)
                X, Y = np.meshgrid(np.linspace(0, 1, cam_g.shape[0]), np.linspace(0, 1, cam_g.shape[0]))
                ax.plot_surface(X, Y, np.zeros_like(cam_g), color='0.82', alpha=0.55, linewidth=0, shade=False)
                for k, (label, colour, mask) in enumerate(reversed(zones)):
                    mg = resample_map(mask.astype(float)) > 0.5                        # (56, 56)
                    Z = np.where(mg, cam_g * 0.8 + 0.55 * (k + 1), np.nan)             # lifted zone slab
                    ax.plot_surface(X, Y, Z, color=colour, alpha=0.9, linewidth=0, shade=True)
                    ax.text(0.02, 0.02, 0.55 * (k + 1) + 0.35, label.split(' (')[0][:6], color=colour,
                            fontsize=8, fontweight='bold')
                nz = resample_map(_nuclei23.astype(float)) > 0.5
                ax.plot_surface(X, Y, np.where(nz, 0.02, np.nan), color='#1B9E77', alpha=0.5,
                                linewidth=0, shade=False)
                ax.set_zlim(0, 0.55 * len(zones) + 1.0); ax.set_zticks([])
                ax.set_xlabel('Lateral', fontsize=8, labelpad=-4); ax.set_ylabel('Axial', fontsize=8, labelpad=-4)
                ax.tick_params(labelsize=7, pad=-2); ax.view_init(elev=26, azim=-58)

            plt.suptitle(f'Attention zones marked on the tile (held-out {SAL3D_MAG}×, seed {SAL3D_SEED})\n'
                         'contours: model attribution quantiles;  green outline: the '
                         f'{int(100 * NUCLEI_QUANTILE)}% most haematoxylin-rich pixels (automatic proxy, '
                         'not an annotation)', y=0.99, fontsize=12, fontweight='bold')
            save_fig(fig, 'fig23_attention_zone_map', dpi=170)
            plt.show()

In [ ]:
# ---------- Fig 24: does the attribution land on nuclei? enrichment by depth ----------
# The segmentation papers report "wound focus" per stage against the annotated wound. Here the
# reference is the automatic haematoxylin-rich proxy from the previous cell, and the statistic is an
# enrichment rather than a ratio of means:
#     enrichment = (attribution mass on the proxy / total mass) / (proxy area / tile area)
# 1.0 = indifferent, and the ceiling is 1 / area share (about 4 at a 25% proxy). The first run used a
# ratio of means and produced 2e+08 for a stage whose map was zero almost everywhere; this cannot.
# Degenerate maps (identically zero or a single spike) are counted and excluded.
if SAL3D is None:
    print('Skipped Fig 24 - 3D saliency backend not available.')
else:
    _frame24 = reconstruct_test_paths(SAL3D_MAG, SAL3D_SEED)
    if _frame24 is None or len(_frame24) == 0:
        print('Skipped Fig 24 - test set could not be rebuilt.')
    else:
        _picks24 = []
        for lab in (0, 1):
            _picks24 += _frame24.index[_frame24.coarse_label == lab].tolist()[:SAL3D_N_SAMPLES // 2]
        _pils24 = [PILImage.open(_frame24.loc[i, 'path']).convert('RGB') for i in _picks24]
        _shows24 = [np.asarray(p.resize((224, 224)), dtype=np.float32) / 255.0 for p in _pils24]
        _nuclei24 = [nuclei_proxy_mask(s) for s in _shows24]
        _areas24 = np.array([m.mean() for m in _nuclei24])
        print(f'Fig 24: {len(_picks24)} test images at held-out {SAL3D_MAG}×; nuclei proxy covers '
              f'{100 * _areas24.mean():.1f}% of a tile ({100 * _areas24.min():.1f}-{100 * _areas24.max():.1f}%), '
              f'so enrichment can reach at most {1 / _areas24.mean():.1f}×')
        focus_rows = []

        for method in ORDER:
            try:
                per_image, border_per_image, bad_count = [], [], 0
                if method == 'Phikon (frozen FM)':
                    fm = SAL3D['load_phikon']()
                    head = SAL3D['load_probe'](SAL3D_MAG, SAL3D_SEED)
                    if head is None:
                        print('  Phikon: probe.pt missing - skipped'); del fm; continue
                    labels24 = [f'Depth {d}' for d in PHIKON_DEPTHS]
                    for pil, nuc in zip(_pils24, _nuclei24):
                        maps, _, _, _ = SAL3D['phikon_depth_maps'](fm, head, SAL3D['to_tensor'](pil))
                        ok = [not map_is_degenerate(m)[0] for m in maps]
                        bad_count += sum(1 for o in ok if not o)
                        per_image.append([nucleus_enrichment(m, nuc) if o else np.nan for m, o in zip(maps, ok)])
                        border_per_image.append([attribution_stats(m)['border_ratio'] if o else np.nan
                                                 for m, o in zip(maps, ok)])
                    del fm, head
                else:
                    model, model_name = SAL3D['load_encoder'](method, SAL3D_MAG, SAL3D_SEED)
                    if model is None:
                        print(f'  {method}: best.pt missing - skipped'); continue
                    labels24 = [l.split('\n')[0] for l in STAGE_LABELS]
                    for pil, nuc in zip(_pils24, _nuclei24):
                        maps, _, _, _ = SAL3D['stage_cams'](model, model_name, SAL3D['to_tensor'](pil))
                        ok = [not map_is_degenerate(m)[0] for m in maps]
                        bad_count += sum(1 for o in ok if not o)
                        per_image.append([nucleus_enrichment(m, nuc) if o else np.nan for m, o in zip(maps, ok)])
                        border_per_image.append([attribution_stats(m)['border_ratio'] if o else np.nan
                                                 for m, o in zip(maps, ok)])
                    del model
                SAL3D['torch'].cuda.empty_cache()
            except Exception as exc:
                print(f'  {method}: FAILED - {repr(exc)[:200]}'); continue

            F = np.array(per_image, dtype=float)                     # (n_images, n_depths)
            B = np.array(border_per_image, dtype=float)              # (n_images, n_depths)
            for k, lab in enumerate(labels24):
                usable = int(np.isfinite(F[:, k]).sum())
                focus_rows.append({'method': method, 'depth': lab, 'depth_index': k,
                                   'n_images': int(F.shape[0]), 'n_usable': usable,
                                   'enrichment_mean': float(np.nanmean(F[:, k])) if usable else float('nan'),
                                   'enrichment_sd': float(np.nanstd(F[:, k], ddof=1)) if usable > 1 else 0.0,
                                   'border_ratio_mean': float(np.nanmean(B[:, k])) if usable else float('nan')})
            print(f'  {method}: {F.shape[0]} images x {F.shape[1]} depths'
                  + (f'   ({bad_count} degenerate maps excluded)' if bad_count else ''))

        if not focus_rows:
            print('No maps computed; Fig 24 skipped.')
        else:
            nucleus_focus_table = pd.DataFrame(focus_rows)
            nucleus_focus_table.to_csv(f'{OUTPUT_DIR}/tables/nucleus_focus_by_depth.csv', index=False)
            methods24 = [m for m in ORDER if m in set(nucleus_focus_table.method)]
            _ceiling = float(1 / _areas24.mean())
            fig, axes = plt.subplots(1, 2, figsize=(14.5, 4.6))
            for j, (col, sd_col, title, ylab) in enumerate([
                    ('enrichment_mean', 'enrichment_sd', 'Attribution mass on nuclei vs everywhere else',
                     'Enrichment (×; 1.0 = indifferent)'),
                    ('border_ratio_mean', None, 'Attribution on the tile border vs the mean',
                     'Border / mean (>1 = border-heavy)')]):
                ax = axes[j]
                n_depth = int(nucleus_focus_table.depth_index.max()) + 1     # 4 probed depths per encoder
                width = 0.8 / max(1, len(methods24))
                for i, method in enumerate(methods24):
                    sub = nucleus_focus_table[nucleus_focus_table.method == method].sort_values('depth_index')
                    x = sub.depth_index.to_numpy() - 0.4 + width * (i + 0.5)
                    ax.bar(x, sub[col].to_numpy(), width=width * 0.92, color=COLORS[method],
                           edgecolor='black', linewidth=0.4, label=method if j == 0 else None)
                    if sd_col:
                        ax.errorbar(x, sub[col].to_numpy(), yerr=sub[sd_col].to_numpy(), fmt='none',
                                    ecolor='black', elinewidth=0.8, capsize=2.5)
                    for xx, n_ok, n_all in zip(x, sub.n_usable, sub.n_images):
                        if n_ok < n_all:
                            ax.text(xx, 0.02, f'{n_all - n_ok} excl.', ha='center', va='bottom', fontsize=6.5,
                                    color='#d62728', rotation=90)
                ax.axhline(1.0, color='0.35', linestyle='--', linewidth=0.9)
                if j == 0:
                    ax.axhline(_ceiling, color='0.55', linestyle=':', linewidth=1.0)
                    ax.text(n_depth - 0.55, _ceiling, f' ceiling {_ceiling:.1f}×', fontsize=8, color='0.4',
                            va='bottom', ha='right')
                ax.set_xticks(range(n_depth)); ax.set_xticklabels([f'{k + 1}' for k in range(n_depth)], fontsize=9)
                ax.set_xlabel('Probed depth: 1-4 = encoder stages 1-4, and for Phikon transformer blocks '
                              + '/'.join(str(d) for d in PHIKON_DEPTHS))
                ax.set_ylabel(ylab); ax.set_title(title, fontweight='bold', fontsize=11)
                ax.grid(True, axis='y', linewidth=0.4); ax.set_axisbelow(True)
                ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
            axes[0].legend(fontsize=8.5, frameon=True, framealpha=0.95, edgecolor='0.7')
            plt.suptitle(f'Where the attribution falls, by depth (mean over {len(_picks24)} test images, '
                         f'held-out {SAL3D_MAG}×; proxy = top {int(100 * NUCLEI_QUANTILE)}% haematoxylin; '
                         'excluded = degenerate maps)', y=1.03, fontweight='bold', fontsize=12)
            plt.tight_layout()
            save_fig(fig, 'fig24_nucleus_focus_by_depth')
            plt.show()
            print('\nEnrichment (1.0 = indifferent) and border ratio:')
            print(nucleus_focus_table.round(3).to_string(index=False))

In [ ]:
# ---------- Fig 25: the region the model relies on, marked on the slice, ranked by its size ----------
# A classifier has no segmentation output, so "the detected part" has to be defined from the saliency
# map. The marked region is the smallest set of pixels that holds half of the map's attribution mass.
# That definition is used instead of the common half-maximum rule (saliency >= 50% of the peak, as in
# CAM bounding boxes) because the half-maximum rule is not comparable across saliency methods: an
# attention-rollout map is far peakier than a Grad-CAM++ map, so its half-maximum region is a speck
# (about 1% of the tile for Phikon here, against about 66% for Swin-T) by construction, not because the
# model looks at less. The mass rule gives every map type the same meaning - "where half of the
# evidence sits" - so the regions, their ranking and the graphs in Fig 26 compare like with like.
# The half-maximum area is still reported next to it, as the contrast. Row 1 marks the region on the
# slice with its area; row 2 is the attribution surface inside that region, as a 3D relief.
from matplotlib.colors import to_rgba

REGION_MASS = 0.5             # region = the smallest set of pixels holding 50% of the attribution mass
HALFMAX_LEVEL = 0.5           # reference only: saliency >= 50% of the map's maximum
HALFMAX_SMOOTH = 2.0          # Gaussian sigma used by the half-maximum reference rule
# A blob counts as a separate focus only if it covers at least this share of the tile. Without it the
# count measures the saliency grid, not the model: thresholding a bilinearly upsampled 14x14 rollout
# grid leaves ~18 slivers against ~5 for a 7x7 Grad-CAM grid; at 1% the two give comparable counts.
REGION_MIN_BLOB = 0.01
REGION_N_SAMPLES = 40         # tiles scored for the graphs in the next cell (20 benign + 20 malignant)
REGION_N_PANELS = 5           # tiles shown in this figure, spread over the size range


def detected_region(cam, mass=REGION_MASS):
    """Smallest high-attribution region holding `mass` of the total, and its area fraction.
    Computed on the map itself, so the drawn area equals the reported mass-50 area."""
    c = np.asarray(cam, dtype=np.float64)                              # (H, W)
    total = c.sum()
    if total <= 0:
        return np.zeros(c.shape, dtype=bool), 0.0
    flat = np.sort(c.ravel())[::-1]                                    # (H*W,) descending
    k = int(np.searchsorted(np.cumsum(flat) / total, mass))
    mask = c >= flat[min(k, flat.size - 1)]                            # (H, W) bool
    return mask, float(mask.mean())


def halfmax_region(cam, level=HALFMAX_LEVEL, smooth=HALFMAX_SMOOTH):
    """The half-maximum rule, kept only to report how different it is across saliency methods."""
    c = np.asarray(cam, dtype=np.float64)
    if smooth:
        c = _ndi.gaussian_filter(c, sigma=smooth)
    mask = c >= level * c.max()                                        # (H, W) bool
    return mask, float(mask.mean())


def region_stats(cam, mask, nuclei=None):
    """Measurements of the marked (mass-50) region, plus the half-maximum area for contrast."""
    cam = np.asarray(cam, dtype=np.float64)
    lab, n = _ndi.label(mask)
    sizes = _ndi.sum(mask, lab, index=np.arange(1, n + 1)) if n else np.array([])
    return {'area_pct': 100.0 * float(mask.mean()),
            'mass50_area_pct': 100.0 * mass_area_fraction(cam, 0.5),
            'halfmax_area_pct': 100.0 * halfmax_region(cam)[1],
            'components': int((sizes >= REGION_MIN_BLOB * mask.size).sum()) if n else 0,
            'largest_component_pct': 100.0 * float(sizes.max()) / mask.size if n else 0.0,
            'mean_attribution_in_region': float(cam[mask].mean()) if mask.any() else float('nan'),
            'nuclei_share_of_region': float(nuclei[mask].mean()) if (nuclei is not None and mask.any()) else float('nan'),
            'nuclei_enrichment': nucleus_enrichment(cam, nuclei) if nuclei is not None else float('nan')}


if SAL3D is None:
    print('Skipped Fig 25 - 3D saliency backend not available.')
else:
    _frame25 = reconstruct_test_paths(SAL3D_MAG, SAL3D_SEED)
    if _frame25 is None or len(_frame25) == 0:
        print('Skipped Fig 25 - test set could not be rebuilt.')
    else:
        _picks25 = []
        for lab in (0, 1):
            _picks25 += _frame25.index[_frame25.coarse_label == lab].tolist()[:REGION_N_SAMPLES // 2]
        _pils25 = {i: PILImage.open(_frame25.loc[i, 'path']).convert('RGB') for i in _picks25}
        _shows25 = {i: np.asarray(p.resize((224, 224)), dtype=np.float32) / 255.0 for i, p in _pils25.items()}
        _nuclei25 = {i: nuclei_proxy_mask(s) for i, s in _shows25.items()}
        print(f'Fig 25/26: {len(_picks25)} test tiles at held-out {SAL3D_MAG}×, seed {SAL3D_SEED}')

        # ---- one saliency map per tile per encoder (deepest stage), then the region and its stats ----
        REGION_RECORDS = []                       # rows for the graphs in the next cell
        REGION_MAPS = {}                          # (method, tile index) -> (cam, mask, pred, prob)
        for method in ORDER:
            try:
                if method == 'Phikon (frozen FM)':
                    fm = SAL3D['load_phikon']()
                    head = SAL3D['load_probe'](SAL3D_MAG, SAL3D_SEED)
                    if head is None:
                        print('  Phikon: probe.pt missing - skipped'); del fm; continue
                    for i in _picks25:
                        maps, pred, prob, used = SAL3D['phikon_depth_maps'](
                            fm, head, SAL3D['to_tensor'](_pils25[i]), depths=(12,))
                        REGION_MAPS[(method, i)] = (maps[0], pred, prob, used[0])
                    del fm, head
                else:
                    model, model_name = SAL3D['load_encoder'](method, SAL3D_MAG, SAL3D_SEED)
                    if model is None:
                        print(f'  {method}: best.pt missing - skipped'); continue
                    for i in _picks25:
                        maps, pred, prob, used = SAL3D['stage_cams'](model, model_name,
                                                                    SAL3D['to_tensor'](_pils25[i]))
                        REGION_MAPS[(method, i)] = (maps[-1], pred, prob, used[-1])
                    del model
                SAL3D['torch'].cuda.empty_cache()
            except Exception as exc:
                print(f'  {method}: FAILED - {repr(exc)[:200]}'); continue
            for i in _picks25:
                if (method, i) not in REGION_MAPS:
                    continue
                cam, pred, prob, used = REGION_MAPS[(method, i)]
                bad, why = map_is_degenerate(cam)
                if bad:
                    continue                                  # a zero or single-spike map has no region
                mask, _ = detected_region(cam)
                REGION_RECORDS.append({'method': method, 'tile': int(i), 'held_out_mag': int(SAL3D_MAG),
                                       'saliency': used,
                                       'true': int(_frame25.loc[i, 'coarse_label']), 'predicted': int(pred),
                                       'correct': bool(pred == int(_frame25.loc[i, 'coarse_label'])),
                                       'probability': float(prob),
                                       **region_stats(cam, mask, _nuclei25[i])})
            _kept = sum(1 for r in REGION_RECORDS if r['method'] == method)
            print(f'  {method}: {_kept} usable of {sum(1 for k in REGION_MAPS if k[0] == method)} tiles')

        if not REGION_RECORDS:
            print('No regions computed; Fig 25 skipped.')
        else:
            region_table = pd.DataFrame(REGION_RECORDS)
            region_table.to_csv(f'{OUTPUT_DIR}/tables/detected_region_stats.csv', index=False)
            methods25 = [m for m in ORDER if m in set(region_table.method)]
            # The same tiles for every encoder, spread evenly over the mean region size so the panels
            # go from the largest relied-on area to the smallest, as in the wound-progression figures.
            _order = region_table.groupby('tile').area_pct.mean().sort_values(ascending=False)
            _sel = [int(_order.index[int(round(k))]) for k in
                    np.linspace(0, len(_order) - 1, min(REGION_N_PANELS, len(_order)))]
            print('Panel tiles (largest to smallest mean region):', _sel)

            for method in methods25:
                _main_method = region_table.loc[region_table.method == method, 'saliency'].mode().iat[0]
                fig = plt.figure(figsize=(3.55 * len(_sel), 7.4))
                gs = fig.add_gridspec(2, len(_sel), height_ratios=[1.0, 1.15], hspace=0.08, wspace=0.1)
                for c, tile in enumerate(_sel):
                    cam, pred, prob, used = REGION_MAPS[(method, tile)]
                    mask, _ = detected_region(cam)
                    st = region_stats(cam, mask, _nuclei25[tile])
                    true = int(_frame25.loc[tile, 'coarse_label'])
                    ok = pred == true

                    ax = fig.add_subplot(gs[0, c])
                    ax.imshow(_shows25[tile])
                    overlay = np.zeros((*mask.shape, 4))
                    overlay[mask] = to_rgba(COLORS[method], alpha=0.45)
                    ax.imshow(overlay)
                    ax.contour(mask.astype(float), levels=[0.5], colors=COLORS[method], linewidths=2.0)
                    ax.contour(_nuclei25[tile].astype(float), levels=[0.5], colors='#1B9E77', linewidths=0.5)
                    ax.set_xticks([]); ax.set_yticks([])
                    # The saliency method goes in the suptitle; a panel names it only when it fell back.
                    _fallback = '' if used == _main_method else f'  [{used}]'
                    ax.set_title(f"Region {st['area_pct']:.1f}% of the tile{_fallback}\n"
                                 f"{st['components']} blob(s) · {100 * st['nuclei_share_of_region']:.0f}% nuclei "
                                 f"· {st['nuclei_enrichment']:.2f}×",
                                 fontsize=9, fontweight='bold')
                    for s in ax.spines.values():
                        s.set_visible(True); s.set_linewidth(2.0); s.set_edgecolor('#2ca02c' if ok else '#d62728')
                    # Initials keep the label inside its own panel: B = benign, M = malignant.
                    ax.set_xlabel(f"true {CLASS_NAMES[true][0]} / pred {CLASS_NAMES[pred][0]}   p={prob:.2f}",
                                  fontsize=9, color='#2ca02c' if ok else '#d62728', fontweight='bold')

                    ax = fig.add_subplot(gs[1, c], projection='3d')
                    cam_g = resample_map(cam)                                            # (56, 56)
                    mask_g = resample_map(mask.astype(float)) > 0.5                       # (56, 56)
                    X, Y = np.meshgrid(np.linspace(0, 1, cam_g.shape[0]), np.linspace(0, 1, cam_g.shape[0]))
                    ax.plot_surface(X, Y, np.zeros_like(cam_g), color='0.85', alpha=0.5, linewidth=0, shade=False)
                    ax.plot_surface(X, Y, np.where(mask_g, cam_g, np.nan), cmap='inferno', vmin=0, vmax=1,
                                    linewidth=0, antialiased=True, alpha=0.95)
                    ax.set_zlim(0, 1.05); ax.set_zlabel('Attribution', fontsize=8, labelpad=-6)
                    ax.set_xlabel('Lateral', fontsize=8, labelpad=-4); ax.set_ylabel('Axial', fontsize=8, labelpad=-4)
                    ax.tick_params(labelsize=7, pad=-2); ax.view_init(elev=28, azim=-58)

                plt.suptitle(f'{method} ({_main_method}) - the region the model relies on, marked on the slice\n'
                             f'(smallest region holding {int(100 * REGION_MASS)}% of the attribution mass; '
                             f'tiles ordered by region size; held-out {SAL3D_MAG}×, seed {SAL3D_SEED}; '
                             f'green outline = nuclei proxy)',
                             y=1.0, fontsize=12, fontweight='bold')
                save_fig(fig, f"fig25_detected_region_{method.split(' ')[0].replace('-', '')}", dpi=170)
                plt.show()

            print('\nMarked region per encoder = smallest area holding half the attribution mass (% of tile):')
            print(region_table.groupby('method').area_pct.describe()[['mean', 'std', 'min', 'max']]
                  .reindex(methods25).round(2).to_string())
            _gap = (region_table.area_pct - region_table.mass50_area_pct).abs().max()
            print(f'  drawn area vs reported mass-50 area: max difference {_gap:.2f} percentage points')
            print('\nThe same, split by the saliency method each tile actually used (Grad-CAM = fallback):')
            print(region_table.groupby(['method', 'saliency']).area_pct.agg(['count', 'mean', 'std'])
                  .round(2).to_string())
            print('\nFor contrast only - the half-maximum rule (saliency >= 50% of the peak):')
            print(region_table.groupby('method').halfmax_area_pct.agg(['mean', 'std', 'min', 'max'])
                  .reindex(methods25).round(2).to_string())
            print('  not used for the figure: a rollout map is far peakier than a Grad-CAM++ map, so its')
            print('  half-maximum region is small by construction, not because the model looks at less.')

In [ ]:
# ---------- Fig 26: graphs derived from the marked regions ----------
# Everything here comes from the regions drawn in Fig 25, which are the smallest areas holding half the
# attribution mass. All four panels therefore use one region definition that means the same thing for
# Grad-CAM++ and for attention rollout. (The half-maximum rule would not: on the real data it gave
# Phikon's rollout maps about 1% of the tile and Swin-T's Grad-CAM++ maps about 66%, by construction.)
#   (a) how concentrated the attribution is, (b) whether a broader region goes with higher confidence,
#   (c) whether the region sits on nuclei (enrichment, 1.0 = indifferent), (d) one focus or several.
from scipy.stats import spearmanr as _spearman

if globals().get('region_table') is None or len(globals().get('region_table', [])) == 0:
    print('Skipped Fig 26 - run the Fig 25 cell first (it builds region_table).')
else:
    methods26 = [m for m in ORDER if m in set(region_table.method)]
    _sal = region_table.groupby('method').saliency.agg(lambda s: s.mode().iat[0])
    _tick = [f"{SHORT_NAME.get(m, m)}\n({_sal.get(m, '')})" for m in methods26]
    fig, axes = plt.subplots(1, 4, figsize=(19, 4.8))

    # (a) how concentrated: area holding half the attribution mass
    ax = axes[0]
    for i, method in enumerate(methods26):
        v = region_table.loc[region_table.method == method, 'mass50_area_pct'].to_numpy()   # (n_tiles,)
        ax.boxplot([v], positions=[i], widths=0.55, showfliers=False,
                   medianprops=dict(color='black', linewidth=1.4),
                   boxprops=dict(facecolor=COLORS[method], alpha=0.45, edgecolor='black'), patch_artist=True)
        jitter = np.random.default_rng(0).uniform(-0.14, 0.14, len(v))
        ax.scatter(np.full(len(v), i) + jitter, v, s=11, color=COLORS[method], edgecolor='black',
                   linewidth=0.2, alpha=0.85, zorder=3)
    ax.axhline(50, color='0.4', linestyle='--', linewidth=0.9)
    ax.text(len(methods26) - 0.5, 50, ' flat map', fontsize=8, color='0.35', va='bottom', ha='right')
    ax.set_xticks(range(len(methods26))); ax.set_xticklabels(_tick, fontsize=8.5)
    ax.set_ylabel('Area holding half the attribution mass (% of tile)')
    ax.set_title('(a) How concentrated is the attribution?', fontsize=11, fontweight='bold')

    # (b) concentration vs confidence
    ax = axes[1]
    for method in methods26:
        sub = region_table[region_table.method == method]
        ax.scatter(sub.mass50_area_pct, sub.probability, s=22, color=COLORS[method], edgecolor='black',
                   linewidth=0.3, alpha=0.8, label=SHORT_NAME.get(method, method))
    rho, p_rho = _spearman(region_table.mass50_area_pct, region_table.probability)
    ax.set_xlabel('Area holding half the mass (% of tile)'); ax.set_ylabel('Confidence of the predicted class')
    ax.set_title(f'(b) Concentration vs confidence\nSpearman $\\rho$ = {rho:+.2f} (p = {p_rho:.3f})',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, frameon=True, framealpha=0.95, edgecolor='0.7')

    # (c) does the region sit on nuclei
    ax = axes[2]
    for i, method in enumerate(methods26):
        v = region_table.loc[region_table.method == method, 'nuclei_enrichment'].to_numpy()
        ax.bar(i, np.nanmean(v), width=0.62, color=COLORS[method], edgecolor='black', linewidth=0.4)
        ax.errorbar(i, np.nanmean(v), yerr=np.nanstd(v, ddof=1), fmt='none', ecolor='black',
                    elinewidth=0.9, capsize=4)
    _area26 = float(np.mean([m.mean() for m in _nuclei25.values()]))
    ax.axhline(1.0, color='0.35', linestyle='--', linewidth=0.9)
    ax.axhline(1 / _area26, color='0.55', linestyle=':', linewidth=1.0)
    ax.text(len(methods26) - 0.5, 1 / _area26, f' ceiling {1 / _area26:.1f}×', fontsize=8, color='0.4',
            va='bottom', ha='right')
    ax.set_xticks(range(len(methods26))); ax.set_xticklabels(_tick, fontsize=8.5)
    ax.set_ylabel('Nuclei enrichment (×)')
    ax.set_title(f'(c) Is the attribution on nuclei?\n(proxy covers {100 * _area26:.0f}% of a tile; '
                 '1.0 = indifferent)', fontsize=11, fontweight='bold')

    # (d) is the region one blob or several
    ax = axes[3]
    for i, method in enumerate(methods26):
        sub = region_table[region_table.method == method]
        ax.bar(i, sub.components.mean(), width=0.62, color=COLORS[method], edgecolor='black', linewidth=0.4)
        ax.errorbar(i, sub.components.mean(), yerr=sub.components.std(ddof=1), fmt='none', ecolor='black',
                    elinewidth=0.9, capsize=4)
        # at the base of the bar, clear of the error bar above it
        ax.text(i, 0.04, f'largest\n{sub.largest_component_pct.mean():.1f}%', transform=ax.get_xaxis_transform(),
                fontsize=7.5, va='bottom', ha='center', color='white', fontweight='bold')
    ax.set_xticks(range(len(methods26))); ax.set_xticklabels(_tick, fontsize=8.5)
    ax.set_ylabel(f'Blobs in the marked region (each ≥ {100 * REGION_MIN_BLOB:g}% of the tile)')
    ax.set_title('(d) One focus or many?', fontsize=11, fontweight='bold')

    for ax in axes:
        ax.grid(True, axis='y', linewidth=0.4); ax.set_axisbelow(True)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.suptitle(f'Measurements taken from the marked regions ({int(region_table.tile.nunique())} test tiles, '
                 f'held-out {SAL3D_MAG}×, seed {SAL3D_SEED})', y=1.04, fontsize=12.5, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, 'fig26_region_measurements')
    plt.show()

    _summary26 = region_table.groupby('method').agg(
        saliency=('saliency', lambda s: ', '.join(f'{k} x{v}' for k, v in s.value_counts().items())),
        mass50_pct_mean=('mass50_area_pct', 'mean'), mass50_pct_sd=('mass50_area_pct', 'std'),
        halfmax_pct_mean=('halfmax_area_pct', 'mean'),
        nuclei_enrichment_mean=('nuclei_enrichment', 'mean'), nuclei_share_mean=('nuclei_share_of_region', 'mean'),
        blobs_mean=('components', 'mean'), confidence_mean=('probability', 'mean'),
        accuracy=('correct', 'mean'), n_tiles=('tile', 'nunique')).reindex(methods26)
    _summary26.to_csv(f'{OUTPUT_DIR}/tables/region_measurement_summary.csv')
    print('Region measurements per encoder:')
    print(_summary26.round(3).to_string())
    print('\nArea holding half the mass, correct vs wrong predictions (% of tile):')
    print(region_table.pivot_table(index='method', columns='correct', values='mass50_area_pct', aggfunc='mean')
          .reindex(methods26).round(2).to_string())

## 2i. Capability summary (Fig 27)

One figure that puts every axis of the study side by side: accuracy, calibration, selective risk,
cross-institution detection, compute and seed stability. Nothing new is computed here - each axis reads
a number that a table above already reports - so this cell is a view, not a result.

Radar charts are easy to misuse, so two rules are enforced in the cell:

- **The scale of every axis is fixed before the data is read.** `RADAR_AXES` declares a floor and a
  ceiling per axis, both printed under the axis name on the figure. Min-max normalisation over the four
  encoders is deliberately *not* used: it would force the weakest encoder to zero on every axis and
  invent a gap that the numbers do not contain.
- **No claim is attached to the enclosed area.** The area of a radar polygon depends on the order of the
  axes, which is arbitrary, so the caption states that the figure shows the shape of each encoder's
  profile and that the quantitative claims live in T3-T7 with their confidence intervals.

Panel (b) is the same figure restricted to the encoder with the best accuracy and the encoder with the
best calibration, which is the paper's central point when those two are not the same model. Both
encoders are picked from the data, not named in the code. The axes that need section 2g (cross-institution)
or the compute report are dropped automatically when those inputs are missing, and the cell says so.

In [ ]:
# ---------- Fig 27: capability radar, every axis of the study on one figure ----------
# Each axis is scaled between a floor and a ceiling declared BEFORE the data is read, and both numbers
# are printed under the axis name, so the outer ring means "the stated better end", never "perfect".
# Min-max normalisation across the four encoders is avoided on purpose: it pins the weakest encoder to
# zero on every axis and manufactures a gap the measurements do not contain.
RADAR_AXES = [   # (key, label, floor = worse end, ceiling = better end, source)
    ('f1',      'Accuracy\nmacro-F1',                 0.80, 0.90, 'summary'),
    ('ece',     'Calibration\nECE',                   0.12, 0.05, 'summary'),
    ('risk80',  'Selective risk\nrisk @ 80% cov.',    0.12, 0.04, 'summary'),
    ('maha',    'Cross-institution\nMahalanobis AUROC', 0.70, 0.90, 'ood'),
    ('flops',   'Compute\nGFLOPs @ 224',              18.0, 2.0,  'compute'),
    ('seed_sd', 'Seed stability\nSD of macro-F1',     0.08, 0.03, 'summary'),
]
PHIKON_FLOPS_G = 17.60          # frozen ViT-B/16 forward pass, the same constant T7 prints


def radar_to_unit(value, floor, ceiling):
    """Map a measurement onto [0, 1] where 1 is the stated ceiling. Handles lower-is-better axes."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return np.nan
    return float(np.clip((value - floor) / (ceiling - floor), 0.0, 1.0))


# ---- 1. collect one value per (encoder, axis) from tables that already exist ----
_rad = {}
for method in ORDER:
    sub = summary[summary.method == method]
    if not len(sub):
        continue
    _rad[method] = {'f1': sub.test_macro_f1_mean.mean(), 'ece': sub.test_ece_mean.mean(),
                    'risk80': sub.test_risk_at_coverage_80_mean.mean(),
                    'seed_sd': sub.test_macro_f1_sd.mean()}

_have = {'summary'}
if globals().get('ood_summary') is not None and len(ood_summary):
    _m = ood_summary[(ood_summary.pair == 'bach_vs_test') & (ood_summary.score == 'maha')]
    if len(_m):
        for method, value in _m.groupby('method').auroc_mean.mean().items():
            if method in _rad:
                _rad[method]['maha'] = float(value)
        _have.add('ood')
if globals().get('compute_report') is not None:
    _flops = {DISPLAY.get(r['model'], r['model']): float(r['flops_g_at_224'])
              for _, r in compute_report.iterrows() if not np.isnan(r['flops_g_at_224'])}
    # compute_report only covers the trained encoders; the frozen FM's forward cost is the T7 constant.
    _flops.setdefault(DISPLAY.get('owkin/phikon', 'Phikon (frozen FM)'), PHIKON_FLOPS_G)
    if any(m in _flops for m in _rad):
        for method in _rad:
            if method in _flops:
                _rad[method]['flops'] = _flops[method]
        _have.add('compute')

# an axis survives only if every encoder on the figure has a number for it
RADAR_USED = [a for a in RADAR_AXES
              if a[4] in _have and all(not np.isnan(_rad[m].get(a[0], np.nan)) for m in _rad)]
_dropped = [a[1].replace('\n', ' ') for a in RADAR_AXES if a not in RADAR_USED]
if _dropped:
    print('axes dropped (input missing):', '; '.join(_dropped))

if len(_rad) < 2 or len(RADAR_USED) < 3:
    print('Skipped Fig 27 - need at least 2 encoders and 3 usable axes '
          f'(have {len(_rad)} encoders, {len(RADAR_USED)} axes).')
    radar_table = None
else:
    _methods = [m for m in ORDER if m in _rad]
    radar_table = pd.DataFrame([{'method': m, **{k: _rad[m].get(k, np.nan) for k, *_ in RADAR_USED}}
                                for m in _methods]).set_index('method')
    radar_scaled = pd.DataFrame(
        [[radar_to_unit(_rad[m].get(k, np.nan), lo, hi) for k, _, lo, hi, _s in RADAR_USED]
         for m in _methods], index=_methods, columns=[k for k, *_ in RADAR_USED])
    print('Fig 27 raw values (each already reported in T3-T7 / T11):')
    print(radar_table.round(4).to_string())
    print('\nFig 27 after scaling to [0, 1] (1 = the ceiling printed on the axis):')
    print(radar_scaled.round(3).to_string())
    _out = radar_table.join(radar_scaled, rsuffix='_scaled')
    _out.to_csv(f'{OUTPUT_DIR}/tables/fig27_radar_values.csv')
    print(f'  wrote tables/fig27_radar_values.csv')

    # ---- 2. panel (b): best accuracy vs best calibration, chosen from the data ----
    _best_f1 = radar_table.f1.idxmax()
    _best_cal = radar_table.ece.idxmin()
    if _best_cal == _best_f1:                      # one encoder wins both: contrast with selective risk
        _best_cal = radar_table.risk80.drop(index=_best_f1).idxmin()
        _pair_title = '(b) best accuracy vs lowest selective risk'
    else:
        _pair_title = '(b) the accuracy winner vs the calibration winner'
    _pair = [m for m in _methods if m in (_best_f1, _best_cal)]
    print(f'\npanel (b): {_best_f1} (best macro-F1) vs {_best_cal} '
          f"({'best ECE' if _pair_title.endswith('winner') else 'lowest risk@80%'})")

    # ---- 3. draw ----
    _n = len(RADAR_USED)
    # Angles stay inside [0, 2pi) and the offset turns the first axis to the top. Adding pi/2 to the
    # angles instead pushed the last tick past 2pi, and set_xticks then widened the polar theta limits,
    # which drew the radar as a narrow wedge with its labels scattered across the figure.
    _theta = np.linspace(0, 2 * np.pi, _n, endpoint=False)
    _closed = np.concatenate([_theta, _theta[:1]])
    fig, axes = plt.subplots(1, 2, figsize=(14.6, 8.0), subplot_kw=dict(projection='polar'))
    for ax, (title, subset) in zip(axes, [('(a) all encoders', _methods), (_pair_title, _pair)]):
        ax.set_theta_offset(np.pi / 2); ax.set_theta_direction(-1)
        ax.set_ylim(0, 1.0)
        ax.set_yticks([0.25, 0.5, 0.75, 1.0]); ax.set_yticklabels([])
        ax.set_xticks(_theta)
        ax.set_xlim(0, 2 * np.pi)                              # the full circle, whatever the ticks did
        ax.set_xticklabels([f'{lab}\n{lo:g} → {hi:g}' for _k, lab, lo, hi, _s in RADAR_USED],
                           fontsize=9)
        ax.tick_params(axis='x', pad=14)
        ax.grid(color='0.75', linewidth=0.6); ax.set_axisbelow(True)
        ax.spines['polar'].set_color('0.7')
        for method in subset:
            vals = radar_scaled.loc[method].to_numpy(dtype=float)
            vc = np.concatenate([vals, vals[:1]])
            colour = COLORS.get(method, '#444444')
            ax.plot(_closed, vc, color=colour, linewidth=2.2, zorder=3)
            ax.fill(_closed, vc, color=colour, alpha=0.13, zorder=2)
            ax.scatter(_theta, vals, color=colour, s=34, edgecolor='white', linewidth=0.8, zorder=4)
        ax.set_title(title, fontsize=11.5, fontweight='bold', pad=62)   # clear of the 3-line top axis label
    _handles = [plt.Line2D([], [], color=COLORS.get(m, '#444444'), linewidth=2.4, marker='o',
                           markersize=6, markeredgecolor='white', label=m) for m in _methods]
    fig.legend(handles=_handles, loc='lower center', bbox_to_anchor=(0.5, 0.075),
               ncol=min(4, len(_methods)), frameon=False, fontsize=10.5, columnspacing=2.4)
    fig.suptitle('Capability profile under magnification shift: higher is better on every axis',
                 fontsize=13.5, fontweight='bold', y=0.975)
    fig.text(0.5, 0.042,
             'Each axis is linearly scaled between the two values printed under its name (inner = worse, '
             'outer = better); the outer ring is the stated better end, not a perfect score.\n'
             'Accuracy, calibration, selective risk and seed stability are means over the four held-out '
             'magnifications; cross-institution detection is the Mahalanobis AUROC on BACH.',
             ha='center', fontsize=9, color='0.32', linespacing=1.5)
    fig.subplots_adjust(top=0.76, bottom=0.255, left=0.07, right=0.93, wspace=0.42)
    save_fig(fig, 'fig27_capability_radar')
    plt.show()

    # ---- 4. the sentence the Results section can quote ----
    # Encoders within RADAR_TIE of the best on the scaled axis share it: a 0.003 AUROC lead is not a win.
    RADAR_TIE = 0.02
    print(f'\nper-axis leader (encoders within {RADAR_TIE} of the best on the scaled axis are tied):')
    _leaders = set()
    for k, lab, lo, hi, _s in RADAR_USED:
        best = radar_scaled[k].max()
        tied = [m for m in _methods if radar_scaled.loc[m, k] >= best - RADAR_TIE]
        _leaders.update(tied)
        raw_vals = ', '.join(f'{m} {radar_table.loc[m, k]:.4g}' for m in tied)
        print(f"  {lab.split(chr(10))[0].lower():<18} {'tie: ' if len(tied) > 1 else ''}{raw_vals}")
    print(f'{len(_leaders)} of {len(_methods)} encoders lead or share the lead on at least one of '
          f'{len(RADAR_USED)} axes' + (': no single encoder dominates the profile.' if len(_leaders) > 1
                                        else '.'))

## 2j. Ranking statistics and failure-detection checks (T5b, Fig 28–29)

Three checks that the related literature uses and that a reviewer is likely to ask for. None needs a GPU;
they read the per-run tables built above.

- **T5b, excess AURC (E-AURC).** AURC mixes two things: how many errors an encoder makes and how well its
  confidence ranks them. E-AURC subtracts the lowest AURC reachable at the same error rate
  (Geifman et al., ICLR 2019), which leaves only the ranking quality. Jaeger et al. (ICLR 2023) recommend
  AURC-based evaluation for failure detection.
- **Fig 28, accuracy against cross-institution AUROC, one point per run.** On natural images, closed-set
  accuracy predicts open-set detection almost perfectly (Vaze et al., ICLR 2022, correlation 0.95). This
  figure checks whether that holds for magnification-shifted histopathology encoders facing another
  institution.
- **Fig 29, critical-difference diagrams.** A Friedman test runs over the 12 (held-out magnification × seed)
  runs, with the Nemenyi critical difference at α = 0.05 (Demšar, JMLR 2006). Encoders joined by a bar are
  not significantly different. The paper's wording must follow this figure: where a bar joins two encoders,
  the claim is "best mean rank", not "significantly better".

In [ ]:
# ---------- T5b: excess AURC (E-AURC), the confidence-ranking part of AURC ----------
# For a classifier with error rate r, the lowest reachable area under the risk-coverage curve is
#     AURC* = r + (1 - r) * ln(1 - r)          (Geifman et al., ICLR 2019)
# which is attained when every error is ranked below every correct prediction. E-AURC = AURC - AURC*
# removes the part of AURC that is only the error rate, so encoders of different accuracy can be compared
# on how well their confidence orders their own mistakes. AURC comes from the training records
# (test_risk_coverage_auc), so nothing is recomputed.
def aurc_optimal(error_rate):
    r = np.clip(np.asarray(error_rate, dtype=np.float64), 0.0, 1.0 - 1e-12)
    return r + (1.0 - r) * np.log(1.0 - r)


def _fmt_ci5b(mean, lo, hi, digits=3):
    return '--' if np.isnan(mean) else f'{mean:.{digits}f} $_{{[{lo:.{digits}f}, {hi:.{digits}f}]}}$'


_need5b = {'test_accuracy', 'test_risk_coverage_auc', 'method', 'held_out_mag', 'seed', 'run_id'}
_lomo_src = raw[raw.branch.isin(['from_scratch', 'fm_linear_probe'])] if 'branch' in raw.columns else raw
if not _need5b.issubset(_lomo_src.columns):
    print('Skipped T5b - the run table lacks', sorted(_need5b - set(_lomo_src.columns)))
    lomo_reliability_runs = e_aurc_table = None
else:
    lomo_reliability_runs = _lomo_src.copy()          # one row per LOMO run; Fig 29 reads it too
    lomo_reliability_runs['aurc_optimal'] = aurc_optimal(1.0 - lomo_reliability_runs.test_accuracy)
    lomo_reliability_runs['e_aurc'] = (lomo_reliability_runs.test_risk_coverage_auc
                                       - lomo_reliability_runs.aurc_optimal)
    # AURC* is the continuous-coverage bound; on a finite test set a near-perfect ranking can sit a hair
    # below it. Report such runs instead of hiding them.
    _neg = lomo_reliability_runs[lomo_reliability_runs.e_aurc < 0]
    if len(_neg):
        print(f'  note: {len(_neg)} run(s) have E-AURC slightly below 0 (finite-sample effect), '
              f'min {lomo_reliability_runs.e_aurc.min():.4f}')

    _rows5b = []
    for (method, ho), g in lomo_reliability_runs.groupby(['method', 'held_out_mag']):
        a, alo, ahi = bootstrap_ci(g.test_risk_coverage_auc.values)
        e, elo, ehi = bootstrap_ci(g.e_aurc.values)
        _rows5b.append({'method': method, 'held_out_mag': int(ho), 'n_seeds': len(g),
                        'accuracy_mean': g.test_accuracy.mean(),
                        'aurc_mean': a, 'aurc_ci_low': alo, 'aurc_ci_high': ahi,
                        'e_aurc_mean': e, 'e_aurc_ci_low': elo, 'e_aurc_ci_high': ehi})
    e_aurc_table = pd.DataFrame(_rows5b)
    e_aurc_table.to_csv(f'{OUTPUT_DIR}/tables/e_aurc_per_cell.csv', index=False)

    _overall5b = (lomo_reliability_runs.groupby('method')
                  [['test_accuracy', 'test_risk_coverage_auc', 'aurc_optimal', 'e_aurc']].mean()
                  .reindex([m for m in ORDER if m in set(lomo_reliability_runs.method)]))
    _overall5b.columns = ['accuracy', 'AURC', 'AURC* (optimal)', 'E-AURC']
    print('AURC and E-AURC, mean over each encoder\'s 12 LOMO runs (lower is better):')
    print(_overall5b.round(4).to_string())
    print('rank by AURC  :', _overall5b['AURC'].rank().astype(int).to_dict())
    print('rank by E-AURC:', _overall5b['E-AURC'].rank().astype(int).to_dict())
    _same = (_overall5b['AURC'].rank() == _overall5b['E-AURC'].rank()).all()
    print('-> the ordering ' + ('does not change' if _same else 'CHANGES') +
          ' once the accuracy part is removed from AURC.')

    # T5b (LaTeX): AURC and E-AURC with 95% bootstrap intervals, per encoder and held-out magnification
    _lines5b = [r'\begin{tabular}{llcc}', r'\toprule',
                r'Encoder & Held-out & AURC & E-AURC \\', r'\midrule']
    for method in [m for m in ORDER if m in set(e_aurc_table.method)]:
        for _, r_ in e_aurc_table[e_aurc_table.method == method].sort_values('held_out_mag').iterrows():
            _lines5b.append(f"{method} & {int(r_.held_out_mag)}$\\times$ & "
                            f"{_fmt_ci5b(r_.aurc_mean, r_.aurc_ci_low, r_.aurc_ci_high)} & "
                            f"{_fmt_ci5b(r_.e_aurc_mean, r_.e_aurc_ci_low, r_.e_aurc_ci_high)}" + r' \\')
        _lines5b.append(r'\midrule' if method != ORDER[-1] else r'\bottomrule')
    if _lines5b[-1] != r'\bottomrule':
        _lines5b[-1] = r'\bottomrule'
    _lines5b.append(r'\end{tabular}')
    Path(f'{OUTPUT_DIR}/tables/T5b_e_aurc.tex').write_text('\n'.join(_lines5b) + '\n', encoding='utf-8')
    print('  wrote tables/T5b_e_aurc.tex and tables/e_aurc_per_cell.csv')

In [ ]:
# ---------- Fig 28: does a more accurate encoder flag another institution better? ----------
# One point per LOMO run: macro-F1 on its held-out magnification against its BACH-vs-own-test AUROC.
# Vaze et al. (ICLR 2022) found closed-set accuracy and open-set AUROC correlated at 0.95 on natural
# images, so "train a better classifier" would be enough there. A weak correlation here means accuracy
# is not a usable proxy for whether an encoder will notice images from another institution.
from scipy.stats import spearmanr as _spearman28, pearsonr as _pearson28

FIG28_SCORES = [('msp', 'MSP'), ('energy', 'Energy (= RNE within a run)'),
                ('maha', 'Mahalanobis'), ('knn', f"kNN (k = {globals().get('KNN_K', 10)})")]

if globals().get('ood_per_run') is None or not len(ood_per_run):
    print('Skipped Fig 28 - run the RNE / OOD cells in section 2g first.')
    accuracy_vs_ood = None
else:
    _acc28 = (raw[raw.branch.isin(['from_scratch', 'fm_linear_probe'])] if 'branch' in raw.columns else raw)
    _acc28 = _acc28[['run_id', 'method', 'held_out_mag', 'seed', 'test_macro_f1']]
    _bach28 = (ood_per_run[ood_per_run.pair == 'bach_vs_test']
               .pivot_table(index='run_id', columns='score', values='auroc'))
    _j28 = _acc28.merge(_bach28, left_on='run_id', right_index=True, how='inner')
    _scores28 = [(s, n) for s, n in FIG28_SCORES if s in _j28.columns]
    print(f'Fig 28: {len(_j28)} LOMO runs with both macro-F1 and BACH AUROC')

    _rows28 = []
    fig, axes = plt.subplots(1, len(_scores28), figsize=(4.3 * len(_scores28), 4.5), sharey=True, squeeze=False)
    for ax, (score, name) in zip(axes[0], _scores28):
        for method in ORDER:
            s = _j28[_j28.method == method]
            if not len(s):
                continue
            ax.scatter(s.test_macro_f1, s[score], s=48, marker=MARKERS.get(method, 'o'),
                       color=COLORS.get(method, '#666666'), edgecolor='black', linewidth=0.5, alpha=0.9,
                       label=method, zorder=3)
        rho, p_rho = _spearman28(_j28.test_macro_f1, _j28[score])
        r_p, p_r = _pearson28(_j28.test_macro_f1, _j28[score])
        _rows28.append({'score': score, 'n_runs': len(_j28), 'spearman_rho': rho, 'spearman_p': p_rho,
                        'pearson_r': r_p, 'pearson_p': p_r,
                        'auroc_min': _j28[score].min(), 'auroc_max': _j28[score].max()})
        ax.axhline(0.5, color='0.4', linestyle='--', linewidth=0.8, zorder=1)
        ax.set_title(f'{name}\nSpearman $\\rho$ = {rho:+.2f} (p = {p_rho:.2g})', fontsize=10.5, fontweight='bold')
        ax.set_xlabel('Macro-F1 on the held-out magnification')
        ax.grid(True, linewidth=0.4, alpha=0.5); ax.set_axisbelow(True)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    axes[0][0].set_ylabel('AUROC, BACH vs own held-out test set')
    _h28, _l28 = axes[0][0].get_legend_handles_labels()
    fig.legend(_h28, _l28, loc='lower center', ncol=len(_l28), frameon=False, fontsize=9.5,
               bbox_to_anchor=(0.5, -0.02))
    fig.suptitle(f'Does a more accurate encoder flag another institution better? '
                 f'(one point per LOMO run, n = {len(_j28)}; dashed line = chance)',
                 fontsize=12, fontweight='bold', y=1.0)
    fig.tight_layout(rect=(0, 0.07, 1, 0.97))
    save_fig(fig, 'fig28_accuracy_vs_ood')
    plt.show()

    accuracy_vs_ood = pd.DataFrame(_rows28)
    accuracy_vs_ood.to_csv(f'{OUTPUT_DIR}/tables/accuracy_vs_ood_correlation.csv', index=False)
    print('\nCorrelation between per-run macro-F1 and BACH AUROC (Vaze et al. report 0.95 on natural images):')
    print(accuracy_vs_ood.round(3).to_string(index=False))

In [ ]:
# ---------- Fig 29: critical-difference diagrams (Friedman test + Nemenyi post-hoc) ----------
# Blocks are the 12 (held-out magnification x seed) LOMO runs; within each block the encoders are ranked
# (1 = best). The Friedman test asks whether any encoder differs; the Nemenyi critical difference (CD) says
# which pairs do. Encoders joined by a bar are NOT significantly different at alpha = 0.05 (Demsar, JMLR
# 2006). With 4 encoders and 12 blocks the test has little power, so this figure sets the paper's
# wording: "best mean rank" where a bar joins two encoders, "significantly better" only where none does.
from scipy.stats import friedmanchisquare as _friedman29

NEMENYI_Q05 = {2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949, 8: 3.031}   # Demsar 2006, Tab. 5
FIG29_METRICS = [('test_macro_f1', 'Macro-F1', True), ('test_ece', 'ECE', False),
                 ('test_risk_at_coverage_80', 'Risk @ 80% coverage', False),
                 ('e_aurc', 'E-AURC (confidence ranking)', False),
                 ('maha', 'BACH Mahalanobis AUROC', True)]

_runs29 = globals().get('lomo_reliability_runs')
if _runs29 is None:
    _runs29 = raw[raw.branch.isin(['from_scratch', 'fm_linear_probe'])] if 'branch' in raw.columns else raw
_runs29 = _runs29.copy()
if globals().get('ood_per_run') is not None and len(ood_per_run):
    _m29 = ood_per_run[(ood_per_run.pair == 'bach_vs_test') & (ood_per_run.score == 'maha')][['run_id', 'auroc']]
    _runs29 = _runs29.merge(_m29.rename(columns={'auroc': 'maha'}), on='run_id', how='left')
_methods29 = [m for m in ORDER if m in set(_runs29.method)]
_metrics29 = [t for t in FIG29_METRICS if t[0] in _runs29.columns and _runs29[t[0]].notna().any()]

if len(_methods29) < 2 or not _metrics29:
    print('Skipped Fig 29 - need at least 2 encoders and one metric.')
    cd_ranks = None
else:
    _k = len(_methods29)
    _rows29 = []
    fig, axes = plt.subplots(len(_metrics29), 1, figsize=(8.6, 1.62 * len(_metrics29) + 0.8), squeeze=False)
    for ax, (col, label, higher) in zip(axes[:, 0], _metrics29):
        block = _runs29.pivot_table(index=['held_out_mag', 'seed'], columns='method', values=col)[_methods29].dropna()
        n = len(block)
        ranks = block.rank(axis=1, ascending=not higher)                   # (n, k), 1 = best
        avg = ranks.mean()
        p = _friedman29(*[block[m] for m in _methods29]).pvalue if n >= 2 else float('nan')
        cd = NEMENYI_Q05[_k] * np.sqrt(_k * (_k + 1) / (6.0 * n))
        pairs = [(a, b) for i, a in enumerate(_methods29) for b in _methods29[i + 1:]
                 if abs(avg[a] - avg[b]) > cd]
        _rows29.append({'metric': label, 'n_blocks': n, 'friedman_p': p, 'critical_difference': cd,
                        **{f'rank_{m}': avg[m] for m in _methods29},
                        'significant_pairs': '; '.join(f'{a} vs {b}' for a, b in pairs) or 'none'})

        # ---- draw: axis 1..k (best on the right), markers at the mean ranks, bars for cliques ----
        ax.set_xlim(0.75, _k + 0.25); ax.set_ylim(-0.3, 1.0); ax.invert_xaxis()
        ax.hlines(0.72, 1, _k, color='0.3', linewidth=1)
        for t in range(1, _k + 1):
            ax.vlines(t, 0.68, 0.76, color='0.3', linewidth=1)
            ax.text(t, 0.84, str(t), ha='center', fontsize=8.5)
        for m in _methods29:
            ax.plot(avg[m], 0.72, marker=MARKERS.get(m, 'o'), color=COLORS.get(m, '#666666'),
                    markersize=9, markeredgecolor='black', linestyle='', zorder=4)
        srt = avg.sort_values()
        cliques = []
        for i in range(_k):
            j = max(jj for jj in range(i, _k) if srt.iloc[jj] - srt.iloc[i] < cd)
            if j > i and not any(a <= i and j <= b for a, b in cliques):
                cliques.append((i, j))
        y = 0.50
        for a, b in cliques:
            ax.hlines(y, srt.iloc[a] - 0.03, srt.iloc[b] + 0.03, color='black', linewidth=3.2)
            y -= 0.16
        ax.hlines(-0.17, 1, 1 + cd, color='#B00020', linewidth=2)
        ax.text(1 + cd / 2, -0.11, f'CD = {cd:.2f}', ha='center', fontsize=7.5, color='#B00020')
        ax.set_title(f'{label}    (Friedman p = {p:.2g}; '
                     f"{'no pair separated' if not pairs else str(len(pairs)) + ' pair(s) separated'})",
                     fontsize=10, fontweight='bold', loc='left')
        ax.axis('off')
    _h29 = [plt.Line2D([], [], marker=MARKERS.get(m, 'o'), color=COLORS.get(m, '#666666'), linestyle='',
                       markersize=8, markeredgecolor='black', label=m) for m in _methods29]
    fig.legend(handles=_h29, loc='lower center', ncol=_k, frameon=False, fontsize=9, bbox_to_anchor=(0.5, 0.0))
    fig.suptitle(f'Mean rank over {n} LOMO runs per encoder (1 = best, right); '
                 f'a bar joins encoders that are not significantly different',
                 fontsize=11, fontweight='bold', y=0.995)
    fig.tight_layout(rect=(0, 0.045, 1, 0.965))
    save_fig(fig, 'fig29_critical_difference')
    plt.show()

    cd_ranks = pd.DataFrame(_rows29)
    cd_ranks.to_csv(f'{OUTPUT_DIR}/tables/cd_ranks.csv', index=False)
    print('Friedman + Nemenyi over the LOMO runs (rank 1 = best):')
    for _, r_ in cd_ranks.iterrows():
        ranks_txt = '  '.join(f"{m.split(' ')[0]} {r_[f'rank_{m}']:.2f}" for m in _methods29)
        print(f"  {r_.metric:<28} {ranks_txt}   p = {r_.friedman_p:.2g}   CD = {r_.critical_difference:.2f}"
              f"   separated: {r_.significant_pairs}")

## 3. LaTeX tables

Table snippets written to `analysis_outputs/tables/T*.tex`, ready for `\input{}` in a LaTeX paper.

In [ ]:
# ---------- LaTeX tables T3–T7 ----------
def fmt_ci(mean, lo, hi, digits=3):
    if np.isnan(mean): return '--'
    return f'{mean:.{digits}f} $_{{[{lo:.{digits}f}, {hi:.{digits}f}]}}$'

def write_tex(name, content):
    with open(f'{OUTPUT_DIR}/tables/{name}.tex', 'w', encoding='utf-8') as f: f.write(content)
    print(f'  wrote tables/{name}.tex')

# T3: accuracy family (Balanced Acc, Macro-F1, AUC)
# T4: calibration (ECE, Brier)
# T5: risk-coverage (@80, @90, @95)
for tname, metric_pairs in [
    ('T3_accuracy', [('Balanced Acc','test_balanced_accuracy'),
                      ('Macro-F1','test_macro_f1'),
                      ('AUC','test_auc')]),
    ('T4_calibration', [('ECE','test_ece'), ('Brier','test_brier')]),
    ('T5_risk_coverage', [('Risk@80\\%','test_risk_at_coverage_80'),
                           ('Risk@90\\%','test_risk_at_coverage_90'),
                           ('Risk@95\\%','test_risk_at_coverage_95')]),
]:
    cols = 'l' + 'c' * (1 + len(metric_pairs))
    header = 'Method & Held-out mag & ' + ' & '.join(h for h,_ in metric_pairs) + r' \\'
    lines = [r'\begin{tabular}{' + cols + '}', r'\toprule', header, r'\midrule']
    for method in ORDER:
        for ho in MAGS:
            row = summary[(summary.method == method) & (summary.held_out_mag == ho)]
            if len(row) == 0: continue
            r = row.iloc[0]
            cells = [fmt_ci(r[f'{m}_mean'], r[f'{m}_ci_low'], r[f'{m}_ci_high']) for _, m in metric_pairs]
            lines.append(f'{method} & {ho}x & ' + ' & '.join(cells) + r' \\')
    lines += [r'\bottomrule', r'\end{tabular}']
    write_tex(tname, '\n'.join(lines))

# T6: BACH OOD scores (standard energy sign; Delta = BACH minus the run's own test set)
if bach_ood is not None and len(bach_ood) > 0:
    _gap = 'energy_gap' in bach_ood.columns and bach_ood['energy_gap'].notna().any()
    lines = [r'\begin{tabular}{lccc' + ('c' if _gap else '') + '}', r'\toprule',
             r'Method & Held-out mag & MSP mean & Energy mean' + (r' & $\Delta$Energy vs test' if _gap else '') + r' \\',
             r'\midrule']
    for method in ORDER:
        for ho in MAGS:
            sub = bach_ood[(bach_ood.method == method) & (bach_ood.held_out_mag == ho)]
            if len(sub) == 0: continue
            row = (f'{method} & {ho}x & '
                   f'{sub.msp_mean.mean():.3f} $\\pm$ {sub.msp_std.mean():.3f} & '
                   f'{sub.energy_mean.mean():.3f} $\\pm$ {sub.energy_std.mean():.3f}')
            if _gap:
                row += f' & {sub.energy_gap.mean():+.3f}'
            lines.append(row + ' \\\\')
    lines += [r'\bottomrule', r'\end{tabular}']
    write_tex('T6_bach_ood', '\n'.join(lines))

# ---- Guard: verify the parameter counts before T7 quotes them ----
# Part A/B's cell-7 computed params as sum(parameter_count(model).values()). fvcore keys
# that dict by module prefix and each entry already includes its submodules, so every
# parameter got counted once per level of the module tree. Part A's CSV shows
# convnext_tiny=219.08M and swin_tiny=216.99M against published ~28.6M / ~28.3M -- the
# same 7.66x / 7.67x inflation in two unrelated architectures. Uncorrected, T7 would
# claim the from-scratch encoders are LARGER than the 86M Phikon when they are roughly
# 3x smaller, which inverts the compute-matched argument. FLOPs are unaffected, since
# FlopCountAnalysis().total() returns a scalar.
KNOWN_PARAMS_M = {            # published reference values, used only to audit the CSV
    'convnext_tiny': 28.6,
    'swin_tiny': 28.3,
    'mambavision_tiny_1k': 31.8,
}
PARAM_INFLATION_TOL = 1.5     # flag anything above 1.5x the published value

if compute_report is not None:
    _suspect = []
    for _, _row in compute_report.iterrows():
        _ref = KNOWN_PARAMS_M.get(_row['model'])
        if _ref is not None and _row['parameters'] / 1e6 > PARAM_INFLATION_TOL * _ref:
            _suspect.append((_row['model'], _row['parameters'] / 1e6, _ref))
    if _suspect:
        print('  WARNING: from_scratch_compute_report.csv parameter counts look inflated.')
        for _m, _got, _ref in _suspect:
            print(f'    {_m}: CSV says {_got:.1f}M, published is ~{_ref:.1f}M  ({_got/_ref:.2f}x)')
        print('    T7 below prints the CSV values VERBATIM. Regenerate the CSV with the')
        print('    corrected counter (Part C cell c5 has the fix) before using T7 in the')
        print('    paper, or the compute-matched claim will read backwards.')
        _msg = ['Parameter counts in from_scratch_compute_report.csv appear inflated:']
        for _m, _got, _ref in _suspect:
            _msg.append(f'  {_m}: {_got:.1f}M vs published ~{_ref:.1f}M ({_got/_ref:.2f}x)')
        _msg += ['',
                 'Cause: sum(parameter_count(model).values()) double-counts across the module tree.',
                 "Fix:   parameter_count(model)[''] , or sum(p.numel() for p in model.parameters()).",
                 '',
                 'FLOPs in the same CSV are correct and unaffected.']
        Path(f'{OUTPUT_DIR}/tables/T7_PARAMS_SUSPECT.txt').write_text(
            '\n'.join(_msg) + '\n', encoding='utf-8')
        print(f'    wrote tables/T7_PARAMS_SUSPECT.txt')
    else:
        print('  Parameter-count audit passed (no value exceeds '
              f'{PARAM_INFLATION_TOL}x its published reference).')

# T7: compute report
if compute_report is not None:
    lines = [r'\begin{tabular}{lcc}', r'\toprule',
             r'Method & Parameters (M) & FLOPs @ 224 (G) \\', r'\midrule']
    for _, row in compute_report.iterrows():
        method = DISPLAY.get(row['model'], row['model'])
        params_m = row['parameters'] / 1e6
        flops = row['flops_g_at_224']
        line = f'{method} & {params_m:.1f} & {flops:.2f} \\\\' if not np.isnan(flops) else f'{method} & {params_m:.1f} & -- \\\\'
        lines.append(line)
    lines.append(r'Phikon (frozen ViT-B/16) & 86.0 & 17.60 \\')
    lines += [r'\bottomrule', r'\end{tabular}']
    write_tex('T7_compute', '\n'.join(lines))

print('\nDone. LaTeX tables in', f'{OUTPUT_DIR}/tables/')

In [ ]:
# ---------- T8: Grad-CAM++ Faithfulness (Deletion / Insertion AUC) ----------
# Deletion: progressively remove top-k% most salient pixels; measure predicted-class prob.
#   Lower AUC = better (removing salient pixels crashes prob quickly = CAM is faithful).
# Insertion: start from black; progressively add top-k% most salient pixels back.
#   Higher AUC = better (adding salient pixels rises prob quickly = CAM is faithful).
# Reference: Petsiuk, Das & Saenko, 2018 (RISE).
# numpy 2.x renamed trapz -> trapezoid and deprecated the old name, which otherwise
# emits a DeprecationWarning on every metric computation and buries the run log.
# The two are numerically identical, so this alias changes no results.
_trapz = getattr(np, 'trapezoid', None) or np.trapz

if not HAS_IMAGES:
    print('Skipped T8 - BreakHis not attached.')
else:
    import torch
    from torchvision import transforms as _tf
    try:
        from pytorch_grad_cam import GradCAMPlusPlus
        from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    except ImportError:
        print('Skipped T8 - grad-cam not installed.'); GradCAMPlusPlus = None

    if GradCAMPlusPlus is not None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        FAITH_HELD_OUT = MAGS[-1] if MAGS else 400
        FAITH_SEED     = SEEDS[0] if SEEDS else 13
        FAITH_N        = 20
        FAITH_STEPS    = 20   # 5% increments

        test_paths_frame = reconstruct_test_paths(FAITH_HELD_OUT, FAITH_SEED)
        if test_paths_frame is None or len(test_paths_frame) == 0:
            print('Skipped T8 - no test paths.')
        elif 'build_from_scratch' not in globals():
            print('Skipped T8 - build_from_scratch missing (run Fig 11 cell first).')
        else:
            rng = np.random.default_rng(FAITH_SEED)
            keep = []
            for lab in [0, 1]:
                idxs = test_paths_frame.index[test_paths_frame.coarse_label == lab].tolist()
                keep.extend(rng.choice(idxs, size=min(len(idxs), FAITH_N // 2), replace=False).tolist())
            faith_frame = test_paths_frame.loc[sorted(keep)].reset_index(drop=True)

            TF = _tf.Compose([_tf.Resize((224,224)), _tf.ToTensor(),
                              _tf.Normalize((.485,.456,.406),(.229,.224,.225))])

            def compute_curves(model, model_name, target_layers):
                cam = GradCAMPlusPlus(model=model, target_layers=target_layers,
                                      reshape_transform=globals().get('to_nchw'))   # layout detected per tensor
                del_aucs, ins_aucs = [], []
                for _, row in faith_frame.iterrows():
                    img = PILImage.open(row['path']).convert('RGB').resize((224,224))
                    img_t = TF(img).unsqueeze(0).to(device)
                    with torch.no_grad():
                        p0 = torch.softmax(model(img_t), dim=1)
                        pc = int(p0.argmax(1).item()); p0v = float(p0[0, pc].item())
                    gcam = cam(input_tensor=img_t, targets=[ClassifierOutputTarget(pc)])[0]
                    order = np.argsort(-gcam.flatten())
                    n_pix = gcam.size
                    del_probs, ins_probs = [p0v], [0.0]
                    baseline = torch.zeros_like(img_t)
                    for step in range(1, FAITH_STEPS + 1):
                        k = int(n_pix * step / FAITH_STEPS)
                        mfl = np.zeros(n_pix, dtype=bool); mfl[order[:k]] = True
                        m2d = torch.from_numpy(mfl.reshape(gcam.shape)).to(device)
                        del_img = img_t.clone(); del_img[0, :, m2d] = baseline[0, :, m2d]
                        with torch.no_grad():
                            del_probs.append(float(torch.softmax(model(del_img), dim=1)[0, pc].item()))
                        ins_img = baseline.clone(); ins_img[0, :, m2d] = img_t[0, :, m2d]
                        with torch.no_grad():
                            ins_probs.append(float(torch.softmax(model(ins_img), dim=1)[0, pc].item()))
                    xs = np.linspace(0, 1, FAITH_STEPS + 1)
                    del_aucs.append(float(_trapz(del_probs, xs)))
                    ins_aucs.append(float(_trapz(ins_probs, xs)))
                del cam
                return (float(np.mean(del_aucs)), float(np.std(del_aucs)),
                        float(np.mean(ins_aucs)), float(np.std(ins_aucs)))

            reverse_display = {v: k for k, v in DISPLAY.items()}
            results = {}
            for method_name in ORDER:
                if method_name == 'Phikon (frozen FM)':
                    continue  # ViT: use Attention Rollout instead, not covered by this CAM-based metric
                model_name = reverse_display.get(method_name)
                prefix = PREFIX_MAP.get(method_name)
                ckpt_path = f'runs/{prefix}_ho{FAITH_HELD_OUT}_frac1.0_seed{FAITH_SEED}/best.pt'
                if not reader.exists(ckpt_path):
                    print(f'  {method_name}: no checkpoint; skipping'); continue
                try:
                    model = build_from_scratch(model_name)
                    sd = torch.load(io.BytesIO(reader.open_bytes(ckpt_path)), map_location=device)
                    model.load_state_dict(sd); model = model.to(device).eval()
                    target_layers = get_target_layer(model, model_name)
                    da, ds, ia, is_ = compute_curves(model, model_name, target_layers)
                    results[method_name] = (da, ds, ia, is_)
                    print(f'  {method_name:<20} Deletion AUC = {da:.3f} \u00b1 {ds:.3f}   Insertion AUC = {ia:.3f} \u00b1 {is_:.3f}')
                    del model
                    if torch.cuda.is_available(): torch.cuda.empty_cache()
                except Exception as e:
                    print(f'  {method_name}: FAILED - {e}')

            if results:
                lines = [r'\begin{tabular}{lcc}', r'\toprule',
                         r'Method & Deletion AUC $\downarrow$ & Insertion AUC $\uparrow$ \\', r'\midrule']
                for m, (da, ds, ia, is_) in results.items():
                    lines.append(f'{m} & {da:.3f} $\\pm$ {ds:.3f} & {ia:.3f} $\\pm$ {is_:.3f} \\\\')
                lines += [r'\bottomrule', r'\end{tabular}']
                write_tex('T8_faithfulness', '\n'.join(lines))
                print('\n' + '=' * 66)
                print(f'T8: Grad-CAM++ faithfulness (ho = {FAITH_HELD_OUT}x, seed = {FAITH_SEED}, N = {len(faith_frame)})')
                print('=' * 66)
                print(f'{"Method":<25} Deletion (lower is better)   Insertion (higher is better)')
                for m, (da, ds, ia, is_) in results.items():
                    print(f'  {m:<25} {da:.3f}                        {ia:.3f}')


In [ ]:
# ---------- T9: ablation table,  T10: cross-validation table ----------
# Written with the same fmt_ci / write_tex helpers as T3-T7, so the output drops into the
# paper via \input{} identically.
_has_branch = 'raw_all' in globals() and 'branch' in raw_all.columns

# Compute report for the ablated variants, written by the Part C ablation task.
ablation_compute = None
if 'reader' in globals() and reader.exists('ablation_compute_report.csv'):
    try:
        ablation_compute = pd.read_csv(io.BytesIO(reader.open_bytes('ablation_compute_report.csv')))
        print('Loaded ablation_compute_report.csv:', list(ablation_compute.model))
    except Exception as exc:
        print('Could not read ablation_compute_report.csv:', repr(exc)[:120])
else:
    print('ablation_compute_report.csv not found - T9 will omit the params/FLOPs columns.')

T9_NAMES = {
    'dsamamba':           'DSA-Mamba (baseline)',
    'dsamamba_no_ssm':    'w/o SS2D ($\\rightarrow$ depthwise conv)',
    'dsamamba_no_decomp': 'w/o trend decomposition',
    'medmamba':           'MedMamba (baseline)',
    'medmamba_no_ssm':    'w/o SS2D ($\\rightarrow$ depthwise conv)',
    'medmamba_shallow':   'Shallower (depths $1,1,2,1$)',
    'medmamba_narrow':    'Narrower (dims $48$--$384$)',
}

# ---------------- T9 ----------------
_abl = raw_all[raw_all.branch == 'ablation'] if _has_branch else None
if _abl is None or len(_abl) == 0:
    print('Skipped T9 - no runs with branch="ablation".')
else:
    _ho = int(_abl.held_out_mag.mode().iloc[0])
    _base_model = sorted({v.split('_')[0] for v in _abl.model.unique()})[0]
    _abl = _abl[(_abl.held_out_mag == _ho) & _abl.model.str.startswith(_base_model + '_')]
    _ref = raw_all[(raw_all.branch == 'baseline_ref') & (raw_all.model == _base_model)
                   & (raw_all.held_out_mag == _ho)]
    _lomo = raw[(raw.model == _base_model) & (raw.held_out_mag == _ho)]
    _base = _ref if len(_ref) else _lomo
    _base_src = 'same-session baseline' if len(_ref) else 'LOMO runs, same seeds and split'
    if len(_base) == 0:
        print(f'Skipped T9 - no {_base_model} runs at ho={_ho} to measure the deltas against.')
    else:
        _order = ([_base_model] + [k for k in T9_NAMES if k in set(_abl.model)]
                  + sorted(set(_abl.model) - set(T9_NAMES)))
        _base_f1 = _base.test_macro_f1.mean()
        _use_compute = ablation_compute is not None
        cols = 'l' + ('cc' if _use_compute else '') + 'ccc'
        header = ('Variant & ' + ('Params (M) & FLOPs (G) & ' if _use_compute else '')
                  + r'Macro-F1 & ECE & $\Delta$ Macro-F1 \\')
        lines = [r'\begin{tabular}{' + cols + '}', r'\toprule', header, r'\midrule']
        printed = []
        for mdl in _order:
            sub = _base if mdl == _base_model else _abl[_abl.model == mdl]
            if len(sub) == 0:
                continue
            f1_m, f1_lo, f1_hi = bootstrap_ci(sub.test_macro_f1.values)
            ec_m, ec_lo, ec_hi = bootstrap_ci(sub.test_ece.values)
            cells = [T9_NAMES.get(mdl, mdl)]
            if _use_compute:
                row = ablation_compute[ablation_compute.model == mdl]
                if len(row) and row.parameters.iloc[0] > 0:
                    cells.append(f'{row.parameters.iloc[0] / 1e6:.1f}')
                    fl = row.flops_g_at_224.iloc[0]
                    cells.append(f'{fl:.2f}' if fl == fl else '--')
                else:
                    cells += ['--', '--']
            cells.append(fmt_ci(f1_m, f1_lo, f1_hi))
            cells.append(fmt_ci(ec_m, ec_lo, ec_hi))
            cells.append('--' if mdl == _base_model else f'${f1_m - _base_f1:+.3f}$')
            lines.append(' & '.join(cells) + r' \\')
            printed.append((T9_NAMES.get(mdl, mdl), f1_m, None if mdl == _base_model else f1_m - _base_f1, len(sub)))
        lines += [r'\bottomrule', r'\end{tabular}']
        write_tex('T9_ablation', '\n'.join(lines))
        print(f'\nT9 (ablation @ ho={_ho}; baseline = {_base_src}):')
        for nm, f1, d, n in printed:
            print(f'  {nm:<44} macro-F1={f1:.4f}  '
                  + ('baseline' if d is None else f'delta={d:+.4f}') + f'  (n={n})')

        # The code audit explains why T9 has no cross-attention variant; write it as a
        # caption note so the paper states the reason rather than leaving the gap unexplained.
        if 'reader' in globals() and reader.exists('dsamamba_code_audit.json'):
            _audit = json.loads(reader.open_text('dsamamba_code_audit.json'))
            _ct = _audit.get('cross_attention_param_tensors') or 0
            _cg = _audit.get('cross_attention_tensors_with_gradient')
            if _ct and _cg == 0:
                _ca = (f"the decoder cross-attention output is overwritten before use, so it does not "
                       f"influence the output: none of its {_ct} parameter tensors "
                       f"({_audit.get('cross_attention_params', 0)/1e6:.2f}M parameters) receives a "
                       f"gradient, and bypassing it changes the output by at most "
                       f"{_audit.get('max_abs_output_diff_without_cross_attention')}.")
            else:
                _ca = (f"AUDIT DID NOT CONFIRM that cross-attention is inactive ({_cg}/{_ct} tensors "
                       f"receive a gradient); re-check before writing a caption.")
            _dc = ("series_decomp returns the raw input rather than the residual, so only its "
                   "moving-average trend affects the output."
                   if _audit.get('decomp_first_output_is_raw_input') and not _audit.get('decomp_first_output_is_residual')
                   else "AUDIT DID NOT CONFIRM the series_decomp behaviour; re-check before writing a caption.")
            _note = ('DSA-Mamba code audit (Part C, no training), for the T9 caption or a footnote.\n'
                     f"In the public implementation (commit {_audit.get('dsamamba_commit', 'unknown')}), "
                     f"{_ca}\n{_dc}\n")
            Path(f'{OUTPUT_DIR}/tables/T9_dsamamba_audit.txt').write_text(_note, encoding='utf-8')
            print('\n' + _note + '  wrote tables/T9_dsamamba_audit.txt')

# ---------------- T10 ----------------
T10_NAMES = {'owkin/phikon': 'Phikon (frozen)', 'dsamamba': 'DSA-Mamba', 'medmamba': 'MedMamba',
             'convnext_tiny': 'ConvNeXt-T', 'swin_tiny': 'Swin-T'}
T10_ORDER = ['owkin/phikon', 'dsamamba', 'medmamba', 'convnext_tiny', 'swin_tiny']
_cv = raw_all[raw_all.branch == 'cv'] if _has_branch else None
if _cv is None or len(_cv) == 0:
    print('\nSkipped T10 - no runs with branch="cv".')
else:
    _metrics = [(h, c) for h, c in (('F1', 'test_macro_f1'), ('BAcc', 'test_balanced_accuracy'),
                                    ('AUC', 'test_auc')) if c in _cv.columns]
    _k = int(_cv['fold_idx'].nunique()) if 'fold_idx' in _cv.columns else None
    _rank = {m: i for i, m in enumerate(T10_ORDER)}
    _groups = sorted(_cv.groupby(['model', 'held_out_mag']),
                     key=lambda g: (_rank.get(g[0][0], len(_rank)), g[0][1]))

    def _ms(v):
        v = v.dropna()
        if len(v) == 0:
            return '--'
        return f'{v.mean():.3f}' + (f' $\\pm$ {v.std(ddof=1):.3f}' if len(v) > 1 else '')

    n = len(_metrics)
    _cv_head = (f'{_k}-fold CV' if _k else 'CV') + r' (mean $\pm$ SD)'
    lines = [r'\begin{tabular}{ll' + 'c' * (2 * n) + '}', r'\toprule',
             r'Model & Held-out & \multicolumn{' + str(n) + '}{c}{' + _cv_head + '} & '
             + r'\multicolumn{' + str(n) + r'}{c}{LOMO seeds (mean $\pm$ SD)} \\',
             r'\cmidrule(lr){3-' + str(2 + n) + r'}\cmidrule(lr){' + str(3 + n) + '-' + str(2 + 2 * n) + '}',
             ' & & ' + ' & '.join(h for h, _ in _metrics) + ' & ' + ' & '.join(h for h, _ in _metrics) + r' \\',
             r'\midrule']
    print(f'\nT10 ({_k}-fold patient-level CV vs LOMO seeds):')
    for (mdl, ho), sub in _groups:
        lomo = raw[(raw.model == mdl) & (raw.held_out_mag == ho)]
        cells = [T10_NAMES.get(mdl, mdl), f'{int(ho)}$\\times$']
        cells += [_ms(sub[c]) for _, c in _metrics]
        cells += [_ms(lomo[c]) if len(lomo) else '--' for _, c in _metrics]
        lines.append(' & '.join(cells) + r' \\')
        print(f'  {T10_NAMES.get(mdl, mdl):<16} {int(ho):>3}x  CV '
              + '  '.join(f'{h}={sub[c].mean():.4f}+/-{sub[c].std(ddof=1):.4f}' for h, c in _metrics)
              + ('   | LOMO ' + '  '.join(f'{h}={lomo[c].mean():.4f}' for h, c in _metrics)
                 if len(lomo) else '   | LOMO n/a'))
        if 'fold_idx' in sub.columns:
            for _, r in sub.sort_values('fold_idx').iterrows():
                print(f'      fold {int(r.fold_idx)}: ' + '  '.join(f'{h}={r[c]:.4f}' for h, c in _metrics))
    lines += [r'\bottomrule', r'\end{tabular}']
    write_tex('T10_cross_validation', '\n'.join(lines))

print(f'\nDone. Part C tables in {OUTPUT_DIR}/tables/')


In [ ]:
# ---------- T10b: CV reliability,  T11: per-run OOD separability,  T12: pooled AUROC and threshold ----------
# Uses fmt_ci / write_tex from the T3-T7 cell. T11 reports mean over seeds with the 95% bootstrap
# interval; Energy and RNE share a column because their per-run values are identical by construction.

# T10b: the reliability metrics of Fig 14b, patient-level CV folds next to the LOMO seeds.
if globals().get('cv_reliability') is not None and len(globals().get('cv_reliability', [])):
    _c = cv_reliability[cv_reliability.convention == 'scaled']
    _cells = [(m, h) for (m, h), g in _c.groupby(['method', 'held_out_mag']) if g.design.nunique() == 2]
    _cols10b = [('ece', 'ECE'), ('brier', 'Brier'), ('risk80', 'Risk@80\\%'), ('aurc', 'AURC')]
    lines = [r'\begin{tabular}{llcccc}', r'\toprule',
             r'\multicolumn{6}{l}{Reliability under 5-fold patient-level CV vs the three LOMO seeds '
             r'(mean $\pm$ SD; lower is better)} \\', r'\midrule',
             r'Encoder & Design & ' + ' & '.join(h for _, h in _cols10b) + r' \\', r'\midrule']
    for method, ho in _cells:
        lines.append(f'\\multicolumn{{6}}{{l}}{{\\textit{{held-out {ho}$\\times$}}}} \\\\')
        for design in ('CV fold', 'LOMO seed'):
            sub = _c[(_c.method == method) & (_c.held_out_mag == ho) & (_c.design == design)]
            if not len(sub):
                continue
            cells = [f'{sub[c].mean():.3f} $\\pm$ {sub[c].std(ddof=1):.3f}' if len(sub) > 1
                     else f'{sub[c].mean():.3f}' for c, _ in _cols10b]
            lines.append(f'{method} & {design} ($n={len(sub)}$) & ' + ' & '.join(cells) + r' \\')
        lines.append(r'\midrule')
    lines[-1] = r'\bottomrule'
    lines.append(r'\end{tabular}')
    write_tex('T10b_cv_reliability', '\n'.join(lines))
if globals().get('ood_summary') is None:
    print('Skipped T11/T12 - run the RNE / OOD cells in section 2g first.')
else:
    _t11_cols = [('msp', 'MSP'), ('energy', 'Energy = RNE'), ('maha', 'Mahalanobis'), ('knn', f'$k$NN ($k={KNN_K}$)')]
    for tname, metric, note in [('T11_ood_auroc', 'auroc', 'AUROC (higher is better)'),
                                ('T11b_ood_fpr95', 'fpr95', 'FPR at 95\\% TPR (lower is better)')]:
        lines = [r'\begin{tabular}{ll' + 'c' * len(_t11_cols) + '}', r'\toprule',
                 r'\multicolumn{' + str(2 + len(_t11_cols)) + r'}{l}{BACH vs.\ own held-out test set; ' + note + r'} \\',
                 r'\midrule',
                 'Encoder & Held-out & ' + ' & '.join(h for _, h in _t11_cols) + r' \\', r'\midrule']
        for method in ORDER:
            for ho in MAGS:
                cells = []
                for score, _ in _t11_cols:
                    r_ = ood_summary[(ood_summary.method == method) & (ood_summary.held_out_mag == ho)
                                     & (ood_summary.score == score) & (ood_summary.pair == 'bach_vs_test')]
                    cells.append(fmt_ci(r_.iloc[0][f'{metric}_mean'], r_.iloc[0][f'{metric}_ci_low'],
                                        r_.iloc[0][f'{metric}_ci_high']) if len(r_) else '--')
                lines.append(f'{method} & {ho}$\\times$ & ' + ' & '.join(cells) + r' \\')
            lines.append(r'\midrule' if method != ORDER[-1] else r'\bottomrule')
        lines.append(r'\end{tabular}')
        write_tex(tname, '\n'.join(lines))

    if globals().get('ood_pooled') is not None:
        _hdr = [('msp', 'MSP'), ('energy', 'Energy'), ('maha', 'Mahal.'), ('knn', '$k$NN')]
        lines = [r'\begin{tabular}{l' + 'cc' * len(_hdr) + '}', r'\toprule',
                 'Pool & ' + ' & '.join(r'\multicolumn{2}{c}{' + h + '}' for _, h in _hdr) + r' \\',
                 ' & ' + ' & '.join(['raw & RN'] * len(_hdr)) + r' \\', r'\midrule']
        for _, row in ood_pooled.iterrows():
            cells = []
            for base, _ in _hdr:
                raw_v, rn_v = row[f'{base}_auroc'], row[f'{RN_NAME[base]}_auroc']
                cells += [f'{raw_v:.3f}', (r'\textbf{' + f'{rn_v:.3f}' + '}') if rn_v > raw_v + 0.005 else f'{rn_v:.3f}']
            pool_tex = row['pool'].replace('×', r'$\times$')          # no backslash inside an f-string (Python < 3.12)
            lines.append(f"{pool_tex} ({int(row['n_runs'])} runs) & " + ' & '.join(cells) + r' \\')
        lines += [r'\bottomrule', r'\end{tabular}']
        write_tex('T12_ood_pooled_auroc', '\n'.join(lines))

    if globals().get('ood_threshold') is not None:
        _t = ood_threshold[ood_threshold.pool == 'All encoders, all magnifications']
        lines = [r'\begin{tabular}{llcc}', r'\toprule',
                 r'Score & Scale & False alarms on held-out test & BACH flagged \\',
                 r' & & mean $\pm$ SD [min, max] over runs & mean $\pm$ SD [min, max] \\', r'\midrule']
        for _, r_ in _t.iterrows():
            lines.append(f"{SCORE_LABELS.get(r_.base, r_.base)} & {'reference-normalised' if r_.normalised else 'raw'} & "
                         f"{r_.fpr_test_mean:.3f} $\\pm$ {r_.fpr_test_sd:.3f} [{r_.fpr_test_min:.3f}, {r_.fpr_test_max:.3f}] & "
                         f"{r_.tpr_bach_mean:.3f} $\\pm$ {r_.tpr_bach_sd:.3f} [{r_.tpr_bach_min:.3f}, {r_.tpr_bach_max:.3f}]" + r' \\')
        lines += [r'\bottomrule', r'\end{tabular}']
        write_tex('T12b_ood_fixed_threshold', '\n'.join(lines))

## 4. Headline findings

Auto-generated one-page summary you can quote directly in the paper's abstract / conclusion.

In [ ]:
# ---------- Headline findings printout ----------
print('=' * 72)
print('HEADLINE FINDINGS  (CNN / Transformer / Mamba encoders vs frozen Phikon, BreakHis LOMO)')
print('=' * 72)

# 1. Overall winner by macro-F1
overall_f1 = summary.groupby('method').test_macro_f1_mean.mean().sort_values(ascending=False)
print('\n1. Overall macro-F1 (mean across all held-out mags):')
for i, (m, v) in enumerate(overall_f1.items(), 1):
    print(f'   {i}. {m:<25} {v:.3f}')

# 2. Per-magnification winner
print('\n2. Best method per held-out magnification (by macro-F1):')
for ho in MAGS:
    row = summary[summary.held_out_mag == ho].sort_values('test_macro_f1_mean', ascending=False).head(1)
    if len(row):
        r = row.iloc[0]
        print(f'   ho={ho}x  {r.method:<25} macro-F1 = {r.test_macro_f1_mean:.3f}  '
              f'[{r.test_macro_f1_ci_low:.3f}, {r.test_macro_f1_ci_high:.3f}]')

# 3. Calibration (lowest ECE)
overall_ece = summary.groupby('method').test_ece_mean.mean().sort_values()
print('\n3. Best-calibrated methods (lowest post-TS ECE, averaged over mags):')
for i, (m, v) in enumerate(overall_ece.items(), 1):
    print(f'   {i}. {m:<25} ECE = {v:.4f}')

# 4. Selective prediction (risk @ 80% coverage)
print('\n4. Selective prediction — risk at 80% coverage (mean over mags):')
risk80 = summary.groupby('method').test_risk_at_coverage_80_mean.mean().sort_values()
for i, (m, v) in enumerate(risk80.items(), 1):
    print(f'   {i}. {m:<25} risk@80% = {v:.4f}')

# 5. Seed stability
print('\n5. Seed stability (SD of macro-F1 across seeds, averaged over mags):')
stability = raw.groupby(['method','held_out_mag']).test_macro_f1.std().groupby('method').mean().sort_values()
for i, (m, v) in enumerate(stability.items(), 1):
    print(f'   {i}. {m:<25} SD = {v:.4f}')

# 6. BACH OOD (standard energy sign; see the load cell)
if bach_ood is not None and len(bach_ood) > 0:
    if 'energy_gap' in bach_ood.columns and bach_ood['energy_gap'].notna().any():
        print('\n6. BACH energy gap vs own test set (higher = more shift-aware):')
        _col, _lab = 'energy_gap', 'gap'
    else:
        print('\n6. BACH mean energy (higher = less confident on BACH):')
        _col, _lab = 'energy_mean', 'Energy'
    energy_avg = bach_ood.groupby('method')[_col].mean().sort_values(ascending=False)
    for i, (m, v) in enumerate(energy_avg.items(), 1):
        print(f'   {i}. {m:<25} {_lab} = {v:+.3f}')

print('\n' + '=' * 72)
print(f'Based on {len(raw)} runs across {len(SEEDS)} seeds and {len(MAGS)} held-out magnifications.')
print('=' * 72)

In [ ]:
# ---------- Headline findings: RNE and OOD separability ----------
print('=' * 72)
print('RNE / OOD SEPARABILITY  (ID = own held-out BreakHis test set, OOD = BACH)')
print('=' * 72)
if globals().get('ood_summary') is None:
    print('Not available - section 2g did not run (needs GPU, BreakHis and BACH attached).')
else:
    _m = ood_summary[(ood_summary.pair == 'bach_vs_test') & ood_summary.score.isin(BASE_SCORES)]
    _by = _m.groupby(['score', 'method']).auroc_mean.mean().unstack().reindex(index=BASE_SCORES, columns=ORDER)
    print('\n1. Per-run AUROC averaged over held-out magnifications and seeds (energy = RNE inside a run):')
    print(_by.round(3).to_string())
    best = _by.mean(axis=1).sort_values(ascending=False)
    print('   Mean over encoders:', ', '.join(f'{SCORE_LABELS.get(s, s)} {v:.3f}' for s, v in best.items()))
    if globals().get('ood_pooled') is not None:
        _all = ood_pooled.iloc[0]
        print(f'\n2. Pooled over all {int(_all.n_runs)} runs (raw -> reference-normalised):')
        for base in BASE_SCORES:
            print(f'   {SCORE_LABELS.get(base, base):<14} {_all[f"{base}_auroc"]:.3f} -> {_all[f"{RN_NAME[base]}_auroc"]:.3f}')
    if globals().get('ood_threshold') is not None:
        _t = ood_threshold[(ood_threshold.pool == 'All encoders, all magnifications') & (ood_threshold.base == 'energy')]
        print('\n3. One threshold for every run (energy vs RNE): SD of false alarms across runs')
        for _, r in _t.iterrows():
            print(f"   {r.score:<7} false alarms {r.fpr_test_mean:.3f} +/- {r.fpr_test_sd:.3f}, BACH flagged {r.tpr_bach_mean:.3f}")
    if globals().get('ood_gaps') is not None:
        print('\n4. Mean RNE, in validation SDs (magnification shift on test; institution shift on BACH):')
        print(ood_gaps.groupby('method')[['rne_mean_test', 'rne_mean_bach', 'rne_gap']].mean().reindex(ORDER).round(2).to_string())
print('=' * 72)

## 4b. Subtype extension: detection, then subtype classification (Fig 32–37, T14–T17)

Same question as the rest of the study, on a harder task: how accurate **and how reliable** are the four encoders,
and the proposed MagScan-Mamba (section 4c), at a magnification they never saw in training, when they must name
one of the eight BreakHis subtypes? The runs come from `subtype_hierarchical.ipynb`, which trains three models per
encoder, held-out magnification and seed:

* **Stage 1:** benign vs malignant.
* **Stage 2:** a 4-class benign model (A, F, PT, TA) and a 4-class malignant model (DC, LC, MC, PC).
  Stage 1's prediction decides which one runs.

Every stage stored the logits of every test image with its fitted temperature, so this section re-scores the
whole pipeline from those files (CPU only, no images):

* **Prediction** = stage 1's branch, then that branch's subtype (hard routing). Accuracy, balanced accuracy,
  macro-F1, per-subtype F1 and the 8 × 8 confusion use it.
* **Confidence** = the path probability of the predicted subtype, P(branch) × P(subtype | branch). ECE, AURC,
  E-AURC and risk at 80% coverage use it, as for the binary runs. The path probabilities of all eight subtypes
  sum to 1 and give the Brier score.
* **Error sources:** a benign image sent to the malignant branch, a malignant image sent to the benign branch,
  or the right branch but the wrong subtype. The three add up to 1 − accuracy.

| Output | Content |
|---|---|
| Fig 32 | End-to-end accuracy (top) and reliability (bottom) at every held-out magnification |
| Fig 33 | Where the pipeline's errors come from, per encoder and held-out magnification |
| Fig 34 | F1 of each subtype at each held-out magnification |
| Fig 35 | 8 × 8 confusion per encoder, summed over held-out magnifications and seeds |
| Fig 36 | Do the accuracy and reliability rankings agree? Mean ranks (Friedman + Nemenyi) and macro-F1 against ECE per run |
| Fig 37 | SS2D ablation: MedMamba against MedMamba with every SS2D block replaced by a depthwise convolution |
| T14–T17 | Each stage on its own, the pipeline per held-out magnification, per-subtype F1, the ablation |

**Inputs.** Attach every session's output of `subtype_hierarchical.ipynb` (or one dataset holding them all),
including the MagScan-Mamba session (`subtype_ho40-100-200-400_image_none_magscan`). They are found automatically
(`SUBTYPE_INPUT_PATHS` in the configuration cell) and kept apart from the binary runs. Without them this section
prints a note and skips.

**Same test images for every encoder.** The sessions ran separately (one per account, MagScan-Mamba in one
session for all four magnifications), so the load cell checks that every encoder was tested on the same images
with the same labels at each held-out magnification and seed, and that the train / validation / test sizes of
every stage agree.

**Split.** The headline setting is LOMO with an image-level split: the test images are unseen, but the same
patients appear at the other magnifications in training. These numbers therefore measure robustness to
magnification, not generalisation to new patients, and are not directly comparable with the patient-disjoint
binary results above.

In [ ]:
# ---------- Subtype extension: load every stage run and score the two-stage pipeline end to end ----------
# Reads the outputs of subtype_hierarchical.ipynb (SUBTYPE_INPUT_PATHS, configuration cell). A pipeline is one
# (encoder, protocol, split mode, held-out magnification, seed) with all three stage runs present; each is
# re-scored from the stored logits of every test image under each stage's fitted temperature, with the same
# pipeline_scores() the subtype notebook uses. ECE is also computed at T = 1 (ece_T1), since in the binary
# study temperature scaling alone moved one encoder's ECE ranking at 400x.
from sklearn.metrics import f1_score

# ---- pipeline scoring: the same code in subtype_hierarchical.ipynb and in the analysis notebook ----
import warnings as _pl_warnings
from sklearn.metrics import (balanced_accuracy_score as _pl_bacc, f1_score as _pl_f1, roc_auc_score as _pl_auc,
                             precision_recall_fscore_support as _pl_prfs, confusion_matrix as _pl_cm)
_pl_trapz = getattr(np, 'trapezoid', None) or np.trapz


def pl_softmax(logits, temperature=1.0):
    z = np.asarray(logits, dtype=np.float64) / float(temperature)
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def pl_ece(conf, correct, n_bins=15):
    """15 equal-width confidence bins, (lo, hi], as in the training notebooks."""
    edges, ece = np.linspace(0, 1, n_bins + 1), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(ece)


def pl_risk_coverage(conf, correct):
    order = np.argsort(-conf, kind='stable')
    err = (~correct[order]).astype(float)
    k = np.arange(1, len(err) + 1)
    coverage, risk = k / len(err), np.cumsum(err) / k
    return coverage, risk, float(_pl_trapz(risk, coverage))


def pipeline_scores(p1, pb, pm, y8):
    """The two-stage pipeline on N test images.
    p1 (N, 2): stage-1 probabilities (benign, malignant); pb, pm (N, 4): benign- and malignant-subtype
    probabilities; y8 (N,): true subtype, 0-3 benign (A, F, PT, TA) and 4-7 malignant (DC, LC, MC, PC).
    Prediction: stage 1 picks the branch and that branch's model picks the subtype (hard routing).
    Confidence: the path probability of the predicted subtype, P(branch) x P(subtype | branch).
    The path probabilities of all 8 subtypes sum to 1 and give the Brier score and one-vs-rest AUC.
    Returns (scores, per-class arrays, 8 x 8 confusion, predictions)."""
    y8 = np.asarray(y8, dtype=int)
    n = len(y8)
    coarse, true_coarse = p1.argmax(axis=1), (y8 >= 4).astype(int)
    pred = np.where(coarse == 0, pb.argmax(axis=1), 4 + pm.argmax(axis=1))
    p8 = np.hstack([p1[:, :1] * pb, p1[:, 1:] * pm])
    conf, correct = p8[np.arange(n), pred], pred == y8
    _, risk, aurc = pl_risk_coverage(conf, correct)
    err = 1.0 - correct.mean()
    routed = coarse == true_coarse
    with _pl_warnings.catch_warnings():
        _pl_warnings.simplefilter('ignore')            # a subtype absent from a small test set
        bacc = _pl_bacc(y8, pred)
        try:
            auc = _pl_auc(y8, p8, multi_class='ovr', average='macro', labels=list(range(8)))
        except ValueError:
            auc = float('nan')
    prec, rec, f1, sup = _pl_prfs(y8, pred, labels=list(range(8)), zero_division=0)
    scores = {
        'n_test': n, 'accuracy': float(correct.mean()), 'balanced_accuracy': float(bacc),
        'macro_f1': float(_pl_f1(y8, pred, average='macro', labels=list(range(8)), zero_division=0)),
        'ece': pl_ece(conf, correct), 'brier': float(((p8 - np.eye(8)[y8]) ** 2).sum(axis=1).mean()),
        'auc_ovr': float(auc), 'aurc': aurc,
        # E-AURC = AURC minus the lowest AURC reachable at this error rate (Geifman et al., ICLR 2019)
        'e_aurc': float(aurc - (err + (1 - err) * np.log(1 - err))) if err < 1 else float('nan'),
        'risk_at_coverage_80': float(risk[int(0.8 * n) - 1]) if n >= 2 else float('nan'),
        'stage1_accuracy': float(routed.mean()),
        # the three error sources add up to 1 - accuracy
        'error_benign_to_malignant': float(((true_coarse == 0) & (coarse == 1)).mean()),
        'error_malignant_to_benign': float(((true_coarse == 1) & (coarse == 0)).mean()),
        'error_wrong_subtype': float((routed & ~correct).mean()),
        'missed_malignant_rate': float(((true_coarse == 1) & (coarse == 0)).sum() / max((true_coarse == 1).sum(), 1)),
        'subtype_accuracy_when_routed_correctly': float(correct[routed].mean()) if routed.any() else float('nan'),
        'accuracy_soft_argmax': float((p8.argmax(axis=1) == y8).mean())}
    per_class = {'precision': prec, 'recall': rec, 'f1': f1, 'support': sup}
    return scores, per_class, _pl_cm(y8, pred, labels=list(range(8))), pred

SUB_CODES = ['A', 'F', 'PT', 'TA', 'DC', 'LC', 'MC', 'PC']          # global labels 0..7; 0-3 benign
SUB_NAMES = {'A': 'Adenosis', 'F': 'Fibroadenoma', 'PT': 'Phyllodes tumour', 'TA': 'Tubular adenoma',
             'DC': 'Ductal carcinoma', 'LC': 'Lobular carcinoma', 'MC': 'Mucinous carcinoma',
             'PC': 'Papillary carcinoma'}
SUB_STAGES = ['detect', 'benign_subtype', 'malignant_subtype']
SUB_STAGE_LABELS = {'detect': 'Stage 1: benign vs malignant', 'benign_subtype': 'Stage 2: benign subtypes',
                    'malignant_subtype': 'Stage 2: malignant subtypes'}
SUB_DISPLAY = {**DISPLAY, 'medmamba_no_ssm': 'MedMamba w/o SS2D', 'magscan_mamba': 'MagScan-Mamba'}
SUB_COLORS = {**COLORS, 'MedMamba w/o SS2D': '#8C8C8C', 'MagScan-Mamba': '#7B2D8E'}
SUB_MARKERS = {**MARKERS, 'MedMamba w/o SS2D': 'v', 'MagScan-Mamba': '*'}
SUB_ABLATION = ('MedMamba', 'MedMamba w/o SS2D')                  # (baseline, SS2D replaced)
SUB_MAIN = ('lomo', 'image')                                      # headline (protocol, split mode)
_SUB_BASE_ORDER = ['MagScan-Mamba', 'Phikon (frozen FM)', 'ConvNeXt-T', 'Swin-T', 'MedMamba', 'MedMamba w/o SS2D']
Path(f'{OUTPUT_DIR}/tables').mkdir(parents=True, exist_ok=True)


def _subtype_roots(paths):
    """Folders to search: attached folders as they are, a .zip extracted next to (not into) OUTPUT_DIR."""
    roots = []
    for p in paths or []:
        p = Path(p)
        if p.is_file() and p.suffix == '.zip':
            dst = Path(OUTPUT_DIR).parent / 'subtype_inputs' / p.stem
            if not dst.exists():
                zipfile.ZipFile(p).extractall(dst)
            roots.append(dst)
        elif p.is_dir():
            roots.append(p)
        else:
            print('  subtype input not found, skipped:', p)
    return roots


_sub_recs, _sub_dir = [], {}
for _root in _subtype_roots(globals().get('SUBTYPE_INPUT_PATHS')):
    for _rp in sorted(_root.rglob('record.json')):
        if _rp.parent.parent.name != 'runs':
            continue
        try:
            _rec = json.loads(_rp.read_text(encoding='utf-8'))
        except Exception as exc:
            print('  unreadable record, skipped:', _rp, repr(exc)[:120]); continue
        _rid = _rec.get('run_id')
        if not str(_rec.get('branch', '')).startswith('subtype_') or _rid in _sub_dir:
            continue                                   # not a subtype run, or already found in another input
        if not (_rp.parent / 'test_outputs.npz').exists():
            print('  record without test_outputs.npz, skipped:', _rid); continue
        _sub_dir[_rid] = _rp.parent
        _sub_recs.append(_rec)

SUB_OK = bool(_sub_recs)
sub_runs = sub_e2e = sub_class = sub_main = sub_main_runs = None
sub_confusion = {}
SUB_IMAGES = {}      # (method, protocol, split, held-out, seed) -> per-image arrays; section 4d reads them
if not SUB_OK:
    print('Skipped section 4b - no subtype runs found. Attach the outputs of subtype_hierarchical.ipynb '
          '(Add Input) or set SUBTYPE_INPUT_PATHS in the configuration cell.')
else:
    sub_runs = pd.DataFrame(_sub_recs)
    sub_runs['method'] = sub_runs.model.map(lambda m: SUB_DISPLAY.get(m, m))
    sub_runs['held_out'] = sub_runs.held_out_mag.map(lambda h: 'pooled' if pd.isna(h) else str(int(h)))
    _methods = list(sub_runs.method.unique())
    SUB_ORDER = [m for m in _SUB_BASE_ORDER if m in _methods] + sorted(set(_methods) - set(_SUB_BASE_ORDER))

    e2e_rows, class_rows, _incomplete = [], [], []
    _test_sets, _test_mismatch = {}, []     # (protocol, split, held-out, seed) -> the first encoder's test set
    _group_cols = ['model', 'method', 'protocol', 'split_mode', 'held_out', 'seed']
    for gkey, g in sub_runs.groupby(_group_cols, sort=False):
        info = dict(zip(_group_cols, gkey))
        by_stage = dict(zip(g.stage, g.run_id))
        if set(by_stage) != set(SUB_STAGES):
            _incomplete.append((info['method'], info['held_out'], info['seed'], sorted(set(SUB_STAGES) - set(by_stage))))
            continue
        arr = {s: np.load(_sub_dir[by_stage[s]] / 'test_outputs.npz') for s in SUB_STAGES}
        idx = arr['detect']['full_test_index']
        if not all(np.array_equal(idx, arr[s]['full_test_index']) for s in SUB_STAGES):
            print(f"  WARNING: stage outputs not aligned, skipped: {info['method']} ho={info['held_out']} "
                  f"seed={info['seed']}"); continue
        if 'full_test_subtype_label' in arr['detect'].files:
            y8 = arr['detect']['full_test_subtype_label'].astype(int)
        else:                                          # older outputs: labels from the session's own manifest
            _man = pd.read_csv(_sub_dir[by_stage['detect']].parent.parent / 'breakhis_manifest.csv')
            y8 = _man.subtype.map({c: i for i, c in enumerate(SUB_CODES)}).to_numpy()[idx].astype(int)
        # every encoder must be tested on the same images with the same labels (sessions ran separately)
        _tkey = (info['protocol'], info['split_mode'], info['held_out'], info['seed'])
        if _tkey not in _test_sets:
            _test_sets[_tkey] = (idx, y8, info['method'])
        elif not (np.array_equal(_test_sets[_tkey][0], idx) and np.array_equal(_test_sets[_tkey][1], y8)):
            _test_mismatch.append((info['method'], _test_sets[_tkey][2]) + _tkey)
        temps = {s: float(arr[s]['temperature'][0]) for s in SUB_STAGES}
        probs = {s: pl_softmax(arr[s]['full_test_logits'], temps[s]) for s in SUB_STAGES}
        assert [probs[s].shape[1] for s in SUB_STAGES] == [2, 4, 4], 'unexpected number of classes'
        scores, per_class, cm, _pred8 = pipeline_scores(probs['detect'], probs['benign_subtype'],
                                                        probs['malignant_subtype'], y8)
        scores_t1, _, _, _ = pipeline_scores(*[pl_softmax(arr[s]['full_test_logits'], 1.0) for s in SUB_STAGES], y8)
        ho = info['held_out']
        SUB_IMAGES[(info['method'], info['protocol'], info['split_mode'], ho, int(info['seed']))] = {
            'index': idx, 'y8': y8, 'pred': np.asarray(_pred8, dtype=int),
            'p8': np.hstack([probs['detect'][:, :1] * probs['benign_subtype'],          # path probabilities
                             probs['detect'][:, 1:] * probs['malignant_subtype']]).astype(np.float32),
            'model': info['model'], 'run_dirs': {s: _sub_dir[by_stage[s]] for s in SUB_STAGES}}
        e2e_rows.append({**info, 'held_out_mag': np.nan if ho == 'pooled' else int(ho), **scores,
                         'ece_T1': scores_t1['ece'], 'risk_at_coverage_80_T1': scores_t1['risk_at_coverage_80'],
                         **{f'temperature_{s}': temps[s] for s in SUB_STAGES}})
        for c, code in enumerate(SUB_CODES):
            class_rows.append({**info, 'subtype': code, **{k: float(v[c]) for k, v in per_class.items()}})
        ck = (info['method'], info['protocol'], info['split_mode'], ho)
        sub_confusion[ck] = sub_confusion.get(ck, np.zeros((8, 8), dtype=int)) + cm

    sub_e2e = pd.DataFrame(e2e_rows)
    sub_class = pd.DataFrame(class_rows)
    _stage_cols = ['run_id', 'branch', 'method', 'model', 'stage', 'protocol', 'split_mode', 'held_out', 'seed',
                   'n_train', 'n_val', 'n_test', 'test_accuracy', 'test_balanced_accuracy', 'test_macro_f1',
                   'test_ece', 'test_brier', 'test_risk_coverage_auc', 'test_risk_at_coverage_80',
                   'test_temperature', 'wallclock_s']
    sub_runs[[c for c in _stage_cols if c in sub_runs.columns]].to_csv(
        f'{OUTPUT_DIR}/tables/subtype_stage_runs.csv', index=False)
    if len(sub_e2e):
        sub_e2e.to_csv(f'{OUTPUT_DIR}/tables/subtype_end_to_end_runs.csv', index=False)
        sub_class.to_csv(f'{OUTPUT_DIR}/tables/subtype_per_class_runs.csv', index=False)
        np.savez_compressed(f'{OUTPUT_DIR}/tables/subtype_confusion.npz',
                            **{'__'.join(map(str, k)).replace(' ', '_').replace('/', '_'): v
                               for k, v in sub_confusion.items()})

    # ---- the headline setting: LOMO, image-level split (or whatever setting the attached runs have) ----
    _settings = (sub_e2e.groupby(['protocol', 'split_mode']).size() if len(sub_e2e)
                 else pd.Series(dtype=int))
    if len(_settings) and SUB_MAIN not in _settings.index:
        SUB_MAIN = _settings.idxmax()
        print(f'  NOTE: no LOMO image-level pipelines; the figures use {SUB_MAIN} instead')
    SUB_SETTING_TXT = ('LOMO' if SUB_MAIN[0] == 'lomo' else 'pooled magnifications') + f', {SUB_MAIN[1]}-level split'
    if len(sub_e2e):
        sub_main = sub_e2e[(sub_e2e.protocol == SUB_MAIN[0]) & (sub_e2e.split_mode == SUB_MAIN[1])].copy()
    else:
        sub_main = sub_e2e
    sub_main_runs = sub_runs[(sub_runs.protocol == SUB_MAIN[0]) & (sub_runs.split_mode == SUB_MAIN[1])].copy()
    SUB_ENCODERS = [m for m in SUB_ORDER if m != SUB_ABLATION[1] and len(sub_main) and m in set(sub_main.method)]
    SUB_MAGS = sorted(int(h) for h in sub_main.held_out_mag.dropna().unique()) if len(sub_main) else []

    print(f'Subtype runs: {len(sub_runs)} stage runs from {len(set(map(str, _sub_dir.values())))} run folders; '
          f'{len(sub_e2e)} complete pipelines')
    for (pr, sm), n in _settings.items():
        print(f'  {pr:<6} {sm:<7} split: {n} pipelines' + ('   <- headline setting' if (pr, sm) == SUB_MAIN else ''))
    # ---- same test set and same split sizes for every encoder ----
    _multi = (int((sub_e2e.groupby(['protocol', 'split_mode', 'held_out', 'seed']).method.nunique() > 1).sum())
              if len(sub_e2e) else 0)
    if _test_mismatch:
        print(f'  WARNING: {len(_test_mismatch)} pipeline(s) were tested on different images or labels than the '
              f'first encoder of their setting; do not compare them:')
        for m, m0, pr, sm, h, s_ in _test_mismatch[:12]:
            print(f'    {m} vs {m0}: {pr} {sm} held-out {h} seed {s_}')
    else:
        print(f'  test images and labels identical across encoders in all {_multi} (held-out, seed) settings '
              f'that more than one encoder ran')
    _size_cols = [c for c in ('n_train', 'n_val', 'n_test') if c in sub_runs.columns]
    if _size_cols:
        _nsz = sub_runs.groupby(['protocol', 'split_mode', 'held_out', 'seed', 'stage'])[_size_cols].nunique()
        _nsz = _nsz[(_nsz > 1).any(axis=1)]
        print('  train / val / test sizes of every stage ' + ('agree across encoders' if _nsz.empty else
              f'DIFFER across encoders in {len(_nsz)} (setting, stage) cell(s), e.g. {list(_nsz.index[:3])}'))
    SUB_TEST_MISMATCH = _test_mismatch
    if _incomplete:
        print(f'  {len(_incomplete)} pipeline(s) incomplete (a stage missing), not scored:')
        for m, h, s, miss in _incomplete[:12]:
            print(f'    {m:<20} ho={h:<6} seed={s}: missing {miss}')
    if len(sub_main):
        _cover = sub_main.pivot_table(index='method', columns='held_out', values='seed', aggfunc='count', fill_value=0)
        _cover = _cover.reindex(index=[m for m in SUB_ORDER if m in _cover.index],
                                columns=sorted(_cover.columns, key=lambda c: (c == 'pooled', int(c) if c != 'pooled' else 0)))
        print(f'\nPipelines (seeds) per encoder and held-out magnification, {SUB_SETTING_TXT}:')
        print(_cover.to_string())
        _short = [(m, h) for m in _cover.index for h in _cover.columns
                  if _cover.loc[m, h] < 3 and m != SUB_ABLATION[1]]
        if _short:
            print('  fewer than 3 seeds (a session not attached or still running?):', _short)
        _summ = (sub_main.groupby('method')[['accuracy', 'macro_f1', 'stage1_accuracy', 'ece', 'ece_T1',
                                             'risk_at_coverage_80']].mean().reindex(SUB_ORDER).dropna(how='all'))
        print('\nEnd-to-end pipeline, mean over every held-out magnification and seed:')
        print(_summ.round(3).to_string())
SUB_HAS_E2E = bool(SUB_OK and sub_main is not None and len(sub_main))

In [ ]:
# ---------- Fig 32: the subtype pipeline at every held-out magnification, accuracy above reliability ----------
# Top row: how often the pipeline names the right subtype. Bottom row: how far its confidence can be trusted.
# Mean over seeds; bars are one SD. The SS2D ablation is left out (it runs at one magnification; Fig 37).
FIG32_PANELS = [('macro_f1', 'Macro-F1', True), ('balanced_accuracy', 'Balanced accuracy', True),
                ('stage1_accuracy', 'Stage-1 accuracy (benign vs malignant)', True),
                ('ece', 'ECE', False), ('brier', 'Brier score (8 subtypes)', False),
                ('risk_at_coverage_80', 'Risk @ 80% coverage', False)]

if not globals().get('SUB_HAS_E2E'):
    print('Skipped Fig 32 - no complete subtype pipelines (section 4b needs the subtype outputs).')
else:
    _d32 = sub_main[sub_main.method.isin(SUB_ENCODERS)]
    _x32 = {m: i for i, m in enumerate(SUB_MAGS)}
    _off32 = {m: (i - (len(SUB_ENCODERS) - 1) / 2) * 0.09 for i, m in enumerate(SUB_ENCODERS)}
    fig, axes = plt.subplots(2, 3, figsize=(15, 8.4))
    for ax, (col, label, higher) in zip(axes.ravel(), FIG32_PANELS):
        agg = _d32.groupby(['method', 'held_out_mag'])[col].agg(['mean', 'std', 'count']).reset_index()
        for m in SUB_ENCODERS:
            a = agg[agg.method == m].sort_values('held_out_mag')
            if a.empty:
                continue
            xs = [_x32[int(h)] + _off32[m] for h in a.held_out_mag]
            ax.errorbar(xs, a['mean'], yerr=a['std'].fillna(0), color=SUB_COLORS.get(m, '#666666'),
                        marker=SUB_MARKERS.get(m, 'o'), markersize=7, markeredgecolor='black', markeredgewidth=0.6,
                        linewidth=1.8, capsize=3, label=m)
        ax.set_xticks(range(len(SUB_MAGS)))
        ax.set_xticklabels([f'{h}$\\times$' for h in SUB_MAGS])
        ax.set_xlim(-0.5, len(SUB_MAGS) - 0.5)
        ax.set_xlabel('Held-out (test) magnification')
        ax.set_title(f"{label}\n({'higher' if higher else 'lower'} is better)", fontsize=11, fontweight='bold')
        ax.grid(alpha=0.3)
    axes[0, 0].set_ylabel('Accuracy', fontsize=12, fontweight='bold')
    axes[1, 0].set_ylabel('Reliability', fontsize=12, fontweight='bold')
    _h32, _l32 = axes[0, 0].get_legend_handles_labels()
    fig.legend(_h32, _l32, loc='lower center', ncol=len(_l32), frameon=False, fontsize=10,
               bbox_to_anchor=(0.5, -0.01))
    fig.suptitle(f'Two-stage subtype pipeline under magnification shift ({SUB_SETTING_TXT}): 8 subtypes, '
                 f'mean $\\pm$ SD over seeds', fontsize=13, fontweight='bold')
    fig.tight_layout(rect=(0, 0.04, 1, 0.97))
    save_fig(fig, 'fig32_subtype_by_magnification')
    plt.show()

    print('Mean over seeds (rows: encoder; columns: held-out magnification):')
    for col, label, higher in FIG32_PANELS:
        _t32 = _d32.pivot_table(index='method', columns='held_out_mag', values=col, aggfunc='mean')
        _t32 = _t32.reindex([m for m in SUB_ENCODERS if m in _t32.index])
        _t32.columns = [f'{int(c)}x' for c in _t32.columns]
        print(f"\n  {label} ({'higher' if higher else 'lower'} is better)")
        print(_t32.round(3).to_string())

In [ ]:
# ---------- Fig 33: where the pipeline's errors come from ----------
# Every wrong subtype has one of three causes: stage 1 sent a benign image to the malignant branch, stage 1
# sent a malignant image to the benign branch (a missed cancer), or the branch was right and the subtype model
# was wrong. As shares of all test images the three add up to the pipeline's error rate (1 - accuracy).
FIG33_PARTS = [('error_malignant_to_benign', 'malignant sent to the benign branch (missed cancer)', '#B00020'),
               ('error_benign_to_malignant', 'benign sent to the malignant branch', '#E69F00'),
               ('error_wrong_subtype', 'right branch, wrong subtype', '#7F9CB8')]

if not globals().get('SUB_HAS_E2E'):
    print('Skipped Fig 33 - no complete subtype pipelines.')
else:
    _d33 = sub_main[sub_main.method.isin(SUB_ENCODERS)]
    _m33 = _d33.groupby(['held_out_mag', 'method'])[[c for c, _, _ in FIG33_PARTS] + ['accuracy']].mean()
    fig, axes = plt.subplots(1, len(SUB_MAGS), figsize=(3.6 * len(SUB_MAGS) + 1.2, 4.9), sharey=True, squeeze=False)
    _ymax33 = float((_m33[[c for c, _, _ in FIG33_PARTS]].sum(axis=1)).max()) * 100
    for ax, ho in zip(axes[0], SUB_MAGS):
        encs = [m for m in SUB_ENCODERS if (ho, m) in _m33.index]
        bottom = np.zeros(len(encs))
        for col, label, color in FIG33_PARTS:
            vals = np.array([_m33.loc[(ho, m), col] for m in encs]) * 100
            ax.bar(range(len(encs)), vals, bottom=bottom, color=color, edgecolor='white', linewidth=0.6,
                   width=0.72, label=label)
            bottom += vals
        for i, tot in enumerate(bottom):
            ax.text(i, tot + _ymax33 * 0.015, f'{tot:.1f}', ha='center', va='bottom', fontsize=8.5)
        ax.set_xticks(range(len(encs)))
        ax.set_xticklabels([m.replace(' (frozen FM)', '\n(frozen FM)') for m in encs], fontsize=8.5)
        ax.set_title(f'held-out {ho}$\\times$', fontsize=11, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)
        ax.set_ylim(0, _ymax33 * 1.12 + 1e-9)
    axes[0, 0].set_ylabel('Share of test images misclassified (%)')
    _h33, _l33 = axes[0, 0].get_legend_handles_labels()
    fig.legend(_h33, _l33, loc='lower center', ncol=3, frameon=False, fontsize=9.5, bbox_to_anchor=(0.5, -0.02))
    fig.suptitle(f'Where the two-stage pipeline goes wrong ({SUB_SETTING_TXT}; mean over seeds; '
                 f'number on top = total error %)', fontsize=12, fontweight='bold')
    fig.tight_layout(rect=(0, 0.07, 1, 0.96))
    save_fig(fig, 'fig33_subtype_error_sources')
    plt.show()

    _share33 = _d33.groupby('method')[[c for c, _, _ in FIG33_PARTS]].mean()
    _share33 = _share33.div(_share33.sum(axis=1), axis=0).reindex([m for m in SUB_ENCODERS if m in _share33.index])
    _share33.columns = ['missed cancer', 'benign -> malignant branch', 'wrong subtype']
    print('Share of each encoder\'s errors by cause (mean over held-out magnifications and seeds):')
    print((100 * _share33).round(1).to_string())
    _miss33 = _d33.groupby('method').missed_malignant_rate.agg(['mean', 'std']).reindex(_share33.index)
    print('\nMissed-cancer rate (malignant test images sent to the benign branch), mean and SD over runs:')
    print((100 * _miss33).round(2).to_string())

In [ ]:
# ---------- Fig 34: F1 of every subtype at every held-out magnification ----------
# One panel per encoder; rows are the eight subtypes (benign above the line), columns the held-out
# magnification, each cell the end-to-end F1 averaged over seeds. The right-hand column is the mean over
# magnifications, so rare or look-alike subtypes that stay hard at every magnification stand out.
if not globals().get('SUB_HAS_E2E'):
    print('Skipped Fig 34 - no complete subtype pipelines.')
else:
    _c34 = sub_class[(sub_class.protocol == SUB_MAIN[0]) & (sub_class.split_mode == SUB_MAIN[1])
                     & sub_class.method.isin(SUB_ENCODERS)].copy()
    _c34['held_out_mag'] = _c34.held_out.astype(int)
    _n34 = len(SUB_ENCODERS)
    fig, axes = plt.subplots(1, _n34, figsize=(2.3 + 0.62 * (len(SUB_MAGS) + 1) * _n34 + 0.9 * _n34, 5.2),
                             sharey=True, squeeze=False)
    for j, (ax, m) in enumerate(zip(axes[0], SUB_ENCODERS)):
        piv = _c34[_c34.method == m].pivot_table(index='subtype', columns='held_out_mag', values='f1', aggfunc='mean')
        piv = piv.reindex(index=SUB_CODES, columns=SUB_MAGS)
        mat = np.column_stack([piv.to_numpy(), np.nanmean(piv.to_numpy(), axis=1)])
        im = ax.imshow(mat, cmap='YlGnBu', vmin=0, vmax=1, aspect='auto')
        for r in range(mat.shape[0]):
            for c in range(mat.shape[1]):
                if np.isfinite(mat[r, c]):
                    ax.text(c, r, f'{mat[r, c]:.2f}', ha='center', va='center', fontsize=8,
                            color='white' if mat[r, c] > 0.62 else 'black',
                            fontweight='bold' if c == mat.shape[1] - 1 else 'normal')
        ax.axhline(3.5, color='#B00020', linewidth=1.3)
        ax.axvline(len(SUB_MAGS) - 0.5, color='black', linewidth=1.0)
        ax.set_xticks(range(len(SUB_MAGS) + 1))
        ax.set_xticklabels([f'{h}$\\times$' for h in SUB_MAGS] + ['mean'], fontsize=9)
        ax.set_yticks(range(8))
        ax.set_yticklabels([f'{c}  {SUB_NAMES[c]}' for c in SUB_CODES] if j == 0 else SUB_CODES, fontsize=9)
        ax.set_xlabel('Held-out magnification')
        ax.set_title(m, fontsize=11, fontweight='bold', color=SUB_COLORS.get(m, 'black'))
        ax.grid(False)
    cb = fig.colorbar(im, ax=axes[0].tolist(), fraction=0.025, pad=0.015)
    cb.set_label('F1 (end to end)')
    fig.suptitle(f'Per-subtype F1 of the two-stage pipeline ({SUB_SETTING_TXT}; mean over seeds). '
                 f'Benign above the red line, malignant below.', fontsize=12, fontweight='bold')
    save_fig(fig, 'fig34_subtype_f1_heatmap')
    plt.show()

    _f34 = _c34.groupby(['subtype', 'method']).f1.mean().unstack('method')
    _f34 = _f34.reindex(index=SUB_CODES, columns=[m for m in SUB_ENCODERS if m in _f34.columns])
    print('Per-subtype F1, mean over held-out magnifications and seeds:')
    print(_f34.round(3).to_string())
    _hard34 = _f34.mean(axis=1).sort_values()
    print(f'\nHardest subtypes (mean over encoders): '
          + ', '.join(f'{c} ({SUB_NAMES[c]}) {v:.2f}' for c, v in _hard34.head(3).items()))

In [ ]:
# ---------- Fig 35: 8 x 8 confusion per encoder, summed over held-out magnifications and seeds ----------
# Row-normalised: each row shows where the images of one true subtype went. The red lines split benign
# (A, F, PT, TA) from malignant (DC, LC, MC, PC), so the two off-diagonal blocks are stage-1 routing errors
# and everything inside a diagonal block is a subtype confusion within the right branch.
if not globals().get('SUB_HAS_E2E'):
    print('Skipped Fig 35 - no complete subtype pipelines.')
else:
    _n35 = len(SUB_ENCODERS)
    _cols35 = 2 if _n35 == 4 else _n35
    _rows35 = int(np.ceil(_n35 / _cols35))
    fig, axes = plt.subplots(_rows35, _cols35, figsize=(5.4 * _cols35, 5.0 * _rows35), squeeze=False)
    for ax in axes.ravel()[_n35:]:
        ax.axis('off')
    for ax, m in zip(axes.ravel(), SUB_ENCODERS):
        cm = sum((v for (mm, pr, sm, _h), v in sub_confusion.items()
                  if mm == m and (pr, sm) == tuple(SUB_MAIN)), np.zeros((8, 8), dtype=int))
        cmn = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
        ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
        for r in range(8):
            for c in range(8):
                ax.text(c, r, f'{cmn[r, c]:.2f}' if cmn[r, c] >= 0.005 else '·', ha='center', va='center',
                        fontsize=7.5, color='white' if cmn[r, c] > 0.6 else 'black')
        ax.axhline(3.5, color='#B00020', linewidth=1.3); ax.axvline(3.5, color='#B00020', linewidth=1.3)
        ax.set_xticks(range(8)); ax.set_xticklabels(SUB_CODES)
        ax.set_yticks(range(8)); ax.set_yticklabels(SUB_CODES)
        ax.set_xlabel('Predicted subtype'); ax.set_ylabel('True subtype')
        _routing = (cm[:4, 4:].sum() + cm[4:, :4].sum()) / max(cm.sum(), 1)
        ax.set_title(f'{m}\n(n = {cm.sum():,}; {100 * _routing:.1f}% in the off-diagonal blocks)',
                     fontsize=10.5, fontweight='bold', color=SUB_COLORS.get(m, 'black'))
        ax.grid(False)
    fig.suptitle(f'Where each subtype goes ({SUB_SETTING_TXT}; all held-out magnifications and seeds, '
                 f'row-normalised)', fontsize=12.5, fontweight='bold')
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    save_fig(fig, 'fig35_subtype_confusion')
    plt.show()

    print('Most frequent confusions (share of the true subtype\'s images), per encoder:')
    for m in SUB_ENCODERS:
        cm = sum((v for (mm, pr, sm, _h), v in sub_confusion.items()
                  if mm == m and (pr, sm) == tuple(SUB_MAIN)), np.zeros((8, 8), dtype=int))
        cmn = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
        off = [(cmn[r, c], SUB_CODES[r], SUB_CODES[c]) for r in range(8) for c in range(8) if r != c]
        top = sorted(off, reverse=True)[:3]
        print(f'  {m:<20} ' + '; '.join(f'{t} -> {p} {100 * v:.1f}%' for v, t, p in top))

In [ ]:
# ---------- Fig 36: do the accuracy ranking and the reliability ranking agree for subtypes? ----------
# (a) Blocks are the (held-out magnification x seed) pipelines that every encoder completed. Within a block the
#     encoders are ranked per metric (1 = best) and the ranks averaged. Friedman tests whether any encoder
#     differs; two encoders whose mean ranks are further apart than the Nemenyi critical difference (CD,
#     alpha = 0.05; Demsar, JMLR 2006) differ significantly. A line that changes height between the accuracy
#     metrics (left) and the reliability metrics (right) is an encoder whose reliability rank differs from
#     its accuracy rank, the study's central question asked again for eight classes.
# (b) Every pipeline run: macro-F1 against ECE, with each encoder's mean as a large marker.
from scipy.stats import friedmanchisquare as _friedman36, spearmanr as _spearman36

FIG36_METRICS = [('macro_f1', 'Macro-F1', True), ('balanced_accuracy', 'Balanced\naccuracy', True),
                 ('ece', 'ECE', False), ('brier', 'Brier', False),
                 ('risk_at_coverage_80', 'Risk @ 80%\ncoverage', False), ('e_aurc', 'E-AURC', False)]
_NEMENYI36 = {2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949, 8: 3.031}   # Demsar 2006, Tab. 5
subtype_cd_ranks = None

if not globals().get('SUB_HAS_E2E') or len(SUB_ENCODERS) < 2:
    print('Skipped Fig 36 - needs complete subtype pipelines for at least two encoders.')
else:
    _d36 = sub_main[sub_main.method.isin(SUB_ENCODERS)]
    _k36 = len(SUB_ENCODERS)
    _rows36, _avg36 = [], {}
    for col, label, higher in FIG36_METRICS:
        block = _d36.pivot_table(index=['held_out_mag', 'seed'], columns='method', values=col)[SUB_ENCODERS].dropna()
        n = len(block)
        if n == 0:
            continue
        ranks = block.rank(axis=1, ascending=not higher)
        avg = ranks.mean()
        p = _friedman36(*[block[m] for m in SUB_ENCODERS]).pvalue if (n >= 2 and _k36 >= 3) else float('nan')
        cd = _NEMENYI36.get(_k36, float('nan')) * np.sqrt(_k36 * (_k36 + 1) / (6.0 * n))
        pairs = [(a, b) for i, a in enumerate(SUB_ENCODERS) for b in SUB_ENCODERS[i + 1:] if abs(avg[a] - avg[b]) > cd]
        _avg36[col] = avg
        _rows36.append({'metric': label.replace('\n', ' '), 'column': col, 'n_blocks': n, 'friedman_p': p,
                        'critical_difference': cd, **{f'rank_{m}': avg[m] for m in SUB_ENCODERS},
                        'significant_pairs': '; '.join(f'{a} vs {b}' for a, b in pairs) or 'none'})
    subtype_cd_ranks = pd.DataFrame(_rows36)
    subtype_cd_ranks.to_csv(f'{OUTPUT_DIR}/tables/subtype_cd_ranks.csv', index=False)

    fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(15.5, 5.6), gridspec_kw={'width_ratios': [1.35, 1]})
    _mets36 = [t for t in FIG36_METRICS if t[0] in _avg36]
    _xs36 = np.arange(len(_mets36))
    for m in SUB_ENCODERS:
        ys = [_avg36[c][m] for c, _, _ in _mets36]
        ax_a.plot(_xs36, ys, color=SUB_COLORS.get(m, '#666666'), marker=SUB_MARKERS.get(m, 'o'), markersize=9,
                  markeredgecolor='black', markeredgewidth=0.6, linewidth=2.2, label=m, zorder=3)
    _n_acc36 = sum(1 for _, _, h in _mets36 if h)
    ax_a.axvline(_n_acc36 - 0.5, color='0.35', linestyle='--', linewidth=1)
    ax_a.set_ylim(_k36 + 0.45, 0.2)
    ax_a.set_yticks(range(1, _k36 + 1))
    ax_a.set_ylabel('Mean rank (1 = best)')
    ax_a.set_xticks(_xs36)
    _p_lookup36 = dict(zip(subtype_cd_ranks.column, subtype_cd_ranks.friedman_p))
    ax_a.set_xticklabels([f'{lab}\np = {_p_lookup36[c]:.2g}' if np.isfinite(_p_lookup36[c]) else lab
                          for c, lab, _ in _mets36], fontsize=9)
    ax_a.set_xlim(-0.6, len(_mets36) - 0.4)
    ax_a.text((_n_acc36 - 1) / 2, 0.47, 'accuracy', ha='center', va='center', fontsize=10, fontweight='bold',
              color='0.25')
    ax_a.text(_n_acc36 + (len(_mets36) - _n_acc36 - 1) / 2, 0.47, 'reliability', ha='center', va='center',
              fontsize=10, fontweight='bold', color='0.25')
    _cd36 = float(subtype_cd_ranks.critical_difference.iloc[0])
    if np.isfinite(_cd36) and _cd36 < _k36:
        ax_a.annotate('', xy=(-0.45, 1), xytext=(-0.45, 1 + _cd36),
                      arrowprops=dict(arrowstyle='|-|', color='#B00020', linewidth=1.6, shrinkA=0, shrinkB=0))
        ax_a.text(-0.38, 1 + _cd36 / 2, f'CD = {_cd36:.2f}', color='#B00020', fontsize=8.5, va='center', rotation=90)
    ax_a.grid(axis='y', alpha=0.3)
    ax_a.legend(loc='lower center', bbox_to_anchor=(0.5, -0.36), ncol=_k36, frameon=False, fontsize=9)
    ax_a.set_title(f'(a) Mean rank over {int(subtype_cd_ranks.n_blocks.iloc[0])} held-out magnification x seed '
                   f'blocks\n(p = Friedman test; ranks further apart than CD differ, Nemenyi $\\alpha$ = 0.05)',
                   fontsize=10.5, fontweight='bold')

    for m in SUB_ENCODERS:
        g = _d36[_d36.method == m]
        ax_b.scatter(g.macro_f1, g.ece, color=SUB_COLORS.get(m, '#666666'), marker=SUB_MARKERS.get(m, 'o'), s=34,
                     alpha=0.55, edgecolor='none')
        ax_b.scatter(g.macro_f1.mean(), g.ece.mean(), color=SUB_COLORS.get(m, '#666666'),
                     marker=SUB_MARKERS.get(m, 'o'), s=230, edgecolor='black', linewidth=1.1, zorder=4, label=m)
    _rho36, _prho36 = _spearman36(_d36.macro_f1, _d36.ece)
    ax_b.set_xlabel('Macro-F1 (higher is better)')
    ax_b.set_ylabel('ECE (lower is better)')
    ax_b.grid(alpha=0.3)
    ax_b.set_title(f'(b) Every pipeline run (small) and each encoder\'s mean (large)\n'
                   f'Spearman $\\rho$(macro-F1, ECE) = {_rho36:.2f} (p = {_prho36:.2g}, n = {len(_d36)})',
                   fontsize=10.5, fontweight='bold')
    fig.suptitle(f'Accuracy and reliability rankings of the subtype pipeline ({SUB_SETTING_TXT})',
                 fontsize=12.5, fontweight='bold')
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    save_fig(fig, 'fig36_subtype_accuracy_vs_reliability_ranks')
    plt.show()

    print('Friedman + Nemenyi over the subtype pipelines (rank 1 = best):')
    for _, r_ in subtype_cd_ranks.iterrows():
        ranks_txt = '  '.join(f"{m.split(' ')[0]} {r_[f'rank_{m}']:.2f}" for m in SUB_ENCODERS)
        print(f"  {r_.metric:<22} {ranks_txt}   p = {r_.friedman_p:.2g}   CD = {r_.critical_difference:.2f}"
              f"   separated: {r_.significant_pairs}")
    _best_acc36 = min(SUB_ENCODERS, key=lambda m: _avg36['macro_f1'][m])
    _best_rel36 = {c: min(SUB_ENCODERS, key=lambda m: _avg36[c][m]) for c, _, h in _mets36 if not h}
    print(f'\nBest mean rank by macro-F1: {_best_acc36}; by reliability metric: {_best_rel36}')

In [ ]:
# ---------- Fig 37: SS2D ablation on the subtype task ----------
# MedMamba against MedMamba with every SS2D (2-D selective scan) block replaced by a depthwise-conv block of
# the same width, at the same held-out magnification, seeds and split. Each stage on its own test images
# (from the training records) and the whole pipeline (re-scored above). Lines join the two models' runs with
# the same seed; the panel title gives the mean paired difference (w/o SS2D minus MedMamba).
FIG37_METRICS = [('macro_f1', 'Macro-F1', True), ('ece', 'ECE', False), ('risk_at_coverage_80', 'Risk @ 80% coverage', False)]
subtype_ablation = None

_abl_ok37 = (globals().get('SUB_OK') and sub_main_runs is not None
             and set(SUB_ABLATION) <= set(sub_main_runs.method))
if not _abl_ok37:
    print('Skipped Fig 37 - needs the subtype runs of both MedMamba and MedMamba w/o SS2D '
          '(run guide: session C of subtype_hierarchical.ipynb).')
else:
    _abl37 = sub_main_runs[sub_main_runs.method == SUB_ABLATION[1]]
    ABL_HO37 = _abl37.held_out.mode().iloc[0]
    _rows37 = []
    # each stage on its own: test_* fields of the stage records
    for stage in SUB_STAGES:
        for m in SUB_ABLATION:
            g = sub_main_runs[(sub_main_runs.method == m) & (sub_main_runs.held_out == ABL_HO37)
                              & (sub_main_runs.stage == stage)]
            for _, r_ in g.iterrows():
                _rows37.append({'part': stage, 'method': m, 'seed': int(r_.seed),
                                **{c: r_.get(f'test_{c}', np.nan) for c, _, _ in FIG37_METRICS}})
    # the pipeline end to end
    if SUB_HAS_E2E:
        for m in SUB_ABLATION:
            g = sub_main[(sub_main.method == m) & (sub_main.held_out == ABL_HO37)]
            for _, r_ in g.iterrows():
                _rows37.append({'part': 'pipeline', 'method': m, 'seed': int(r_.seed),
                                **{c: r_[c] for c, _, _ in FIG37_METRICS}})
    subtype_ablation = pd.DataFrame(_rows37)
    _parts37 = [p for p in SUB_STAGES + ['pipeline'] if p in set(subtype_ablation.part)]
    _part_label37 = {**SUB_STAGE_LABELS, 'pipeline': 'Whole pipeline (8 subtypes)'}

    fig, axes = plt.subplots(len(FIG37_METRICS), len(_parts37), figsize=(3.7 * len(_parts37), 3.0 * len(FIG37_METRICS)),
                             squeeze=False)
    _delta_rows37 = []
    for i, (col, label, higher) in enumerate(FIG37_METRICS):
        for j, part in enumerate(_parts37):
            ax = axes[i, j]
            d = subtype_ablation[subtype_ablation.part == part].pivot_table(index='seed', columns='method', values=col)
            d = d.reindex(columns=list(SUB_ABLATION))
            for k, m in enumerate(SUB_ABLATION):
                ax.bar(k, d[m].mean(), color=SUB_COLORS.get(m, '#666666'), width=0.6, alpha=0.85,
                       edgecolor='black', linewidth=0.6)
            for s, row in d.iterrows():
                ax.plot([0, 1], row.values, color='0.25', linewidth=0.9, marker='o', markersize=4, zorder=3)
            paired = d.dropna()
            delta = (paired[SUB_ABLATION[1]] - paired[SUB_ABLATION[0]])
            _delta_rows37.append({'part': part, 'metric': label, 'n_pairs': len(paired),
                                  'medmamba_mean': d[SUB_ABLATION[0]].mean(), 'no_ss2d_mean': d[SUB_ABLATION[1]].mean(),
                                  'delta_mean': delta.mean(), 'delta_sd': delta.std() if len(delta) > 1 else np.nan,
                                  'better_without_ss2d': int(((delta > 0) if higher else (delta < 0)).sum())})
            ax.set_xticks([0, 1]); ax.set_xticklabels(['MedMamba', 'w/o SS2D'], fontsize=9)
            ax.set_title(f'{_part_label37[part]}\n$\\Delta$ = {delta.mean():+.3f}' if i == 0 else
                         f'$\\Delta$ = {delta.mean():+.3f}', fontsize=9.5, fontweight='bold')
            if j == 0:
                ax.set_ylabel(f"{label}\n({'higher' if higher else 'lower'} is better)", fontsize=10)
            ax.grid(axis='y', alpha=0.3)
    fig.suptitle(f'SS2D ablation on the subtype task (held-out {ABL_HO37}$\\times$, {SUB_SETTING_TXT}; '
                 f'bars = mean, lines join runs with the same seed)', fontsize=12, fontweight='bold')
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    save_fig(fig, 'fig37_subtype_ss2d_ablation')
    plt.show()

    subtype_ablation_delta = pd.DataFrame(_delta_rows37)
    subtype_ablation_delta.to_csv(f'{OUTPUT_DIR}/tables/subtype_ablation_delta.csv', index=False)
    print(f'SS2D ablation at held-out {ABL_HO37}x: w/o SS2D minus MedMamba (paired by seed)')
    print(subtype_ablation_delta.round(4).to_string(index=False))

In [ ]:
# ---------- T14-T17: subtype extension tables (LaTeX + CSV) ----------
# T14 each stage on its own test images; T15 the pipeline per held-out magnification (best encoder per
# magnification in bold); T16 per-subtype F1; T17 the SS2D ablation. Values are mean with SD over runs as a
# subscript, in the headline setting (SUB_SETTING_TXT).
def _pm(mean, sd, digits=3, bold=False):
    if mean is None or not np.isfinite(mean):
        return '--'
    core = f'{mean:.{digits}f}'
    if bold:
        core = f'\\mathbf{{{core}}}'
    sd_txt = f'_{{\\pm {sd:.{digits}f}}}' if sd is not None and np.isfinite(sd) else ''
    return f'${core}{sd_txt}$'


def _write_tex(name, lines):
    Path(f'{OUTPUT_DIR}/tables/{name}').write_text('\n'.join(lines) + '\n', encoding='utf-8')
    print(f'  wrote tables/{name}')


if not globals().get('SUB_OK'):
    print('Skipped T14-T17 - no subtype runs.')
else:
    # ---- T14: every stage on its own (records of the stage runs) ----
    _st = sub_main_runs[sub_main_runs.method.isin(SUB_ENCODERS)]
    t14 = (_st.groupby(['method', 'stage'])[['test_accuracy', 'test_macro_f1', 'test_ece', 'test_risk_at_coverage_80']]
           .agg(['mean', 'std']))
    t14.to_csv(f'{OUTPUT_DIR}/tables/T14_subtype_stages.csv')
    _l14 = [r'\begin{tabular}{l' + 'cc' * len(SUB_STAGES) + '}', r'\toprule',
            ' & ' + ' & '.join(f'\\multicolumn{{2}}{{c}}{{{SUB_STAGE_LABELS[s]}}}' for s in SUB_STAGES) + r' \\',
            ''.join(f'\\cmidrule(lr){{{2 + 2 * i}-{3 + 2 * i}}}' for i in range(len(SUB_STAGES))),
            'Encoder' + ' & Macro-F1 & ECE' * len(SUB_STAGES) + r' \\', r'\midrule']
    for m in SUB_ENCODERS:
        cells = []
        for s in SUB_STAGES:
            for c in ('test_macro_f1', 'test_ece'):
                cells.append(_pm(t14.loc[(m, s), (c, 'mean')], t14.loc[(m, s), (c, 'std')])
                             if (m, s) in t14.index else '--')
        _l14.append(f'{m} & ' + ' & '.join(cells) + r' \\')
    _l14 += [r'\bottomrule', r'\end{tabular}']
    _write_tex('T14_subtype_stages.tex', _l14)

    if globals().get('SUB_HAS_E2E'):
        # ---- T15: the pipeline end to end, per held-out magnification ----
        T15_COLS = [('accuracy', 'Acc.', True), ('balanced_accuracy', 'Bal. acc.', True), ('macro_f1', 'Macro-F1', True),
                    ('ece', 'ECE', False), ('brier', 'Brier', False), ('risk_at_coverage_80', 'Risk@80', False),
                    ('e_aurc', 'E-AURC', False)]
        _e = sub_main[sub_main.method.isin(SUB_ENCODERS)]
        t15 = _e.groupby(['held_out_mag', 'method'])[[c for c, _, _ in T15_COLS]].agg(['mean', 'std'])
        t15_all = _e.groupby('method')[[c for c, _, _ in T15_COLS]].agg(['mean', 'std'])
        t15.to_csv(f'{OUTPUT_DIR}/tables/T15_subtype_pipeline.csv')
        t15_all.to_csv(f'{OUTPUT_DIR}/tables/T15_subtype_pipeline_all.csv')
        _l15 = [r'\begin{tabular}{ll' + 'c' * len(T15_COLS) + '}', r'\toprule',
                'Held-out & Encoder & ' + ' & '.join(lab for _, lab, _ in T15_COLS) + r' \\', r'\midrule']
        for ho in SUB_MAGS + ['all']:
            tab = t15.loc[ho] if ho != 'all' else t15_all
            encs = [m for m in SUB_ENCODERS if m in tab.index]
            best = {c: (tab[(c, 'mean')].idxmax() if h else tab[(c, 'mean')].idxmin()) for c, _, h in T15_COLS}
            for i, m in enumerate(encs):
                head = (f'{ho}$\\times$' if ho != 'all' else 'All') if i == 0 else ''
                _l15.append(f'{head} & {m} & ' + ' & '.join(
                    _pm(tab.loc[m, (c, 'mean')], tab.loc[m, (c, 'std')], bold=(best[c] == m)) for c, _, _ in T15_COLS)
                    + r' \\')
            _l15.append(r'\midrule' if ho != 'all' else r'\bottomrule')
        _l15.append(r'\end{tabular}')
        _write_tex('T15_subtype_pipeline.tex', _l15)

        # ---- T16: per-subtype F1 ----
        _c16 = sub_class[(sub_class.protocol == SUB_MAIN[0]) & (sub_class.split_mode == SUB_MAIN[1])
                         & sub_class.method.isin(SUB_ENCODERS)]
        t16 = _c16.groupby(['subtype', 'method']).f1.agg(['mean', 'std']).unstack('method')
        _sup16 = _c16.groupby('subtype').support.mean()
        t16.to_csv(f'{OUTPUT_DIR}/tables/T16_subtype_f1.csv')
        _l16 = [r'\begin{tabular}{llr' + 'c' * len(SUB_ENCODERS) + '}', r'\toprule',
                'Subtype & & Test images & ' + ' & '.join(SUB_ENCODERS) + r' \\', r'\midrule']
        for i, code in enumerate(SUB_CODES):
            if code not in t16.index:
                continue
            best = max(SUB_ENCODERS, key=lambda m: t16.loc[code, ('mean', m)] if np.isfinite(t16.loc[code, ('mean', m)]) else -1)
            _l16.append(f'{SUB_NAMES[code]} & {code} & {_sup16[code]:.0f} & ' + ' & '.join(
                _pm(t16.loc[code, ('mean', m)], t16.loc[code, ('std', m)], bold=(m == best)) for m in SUB_ENCODERS)
                + r' \\')
            if i == 3:
                _l16.append(r'\midrule')
        _l16 += [r'\bottomrule', r'\end{tabular}']
        _write_tex('T16_subtype_f1.tex', _l16)
        print('  (T16 "Test images" = mean number per held-out magnification)')

    # ---- T17: the SS2D ablation ----
    _abl17 = globals().get('subtype_ablation_delta')
    if subtype_ablation is None or _abl17 is None or not len(_abl17):
        print('  T17 skipped - no subtype ablation runs')
    else:
        _part17 = {**SUB_STAGE_LABELS, 'pipeline': 'Whole pipeline'}
        _mets17 = list(dict.fromkeys(_abl17.metric))
        _mets17_tex = [m.replace('%', r'\%') for m in _mets17]          # a bare % starts a LaTeX comment
        _l17 = [r'\begin{tabular}{l' + 'ccc' * len(_mets17) + '}', r'\toprule',
                ' & ' + ' & '.join(f'\\multicolumn{{3}}{{c}}{{{m}}}' for m in _mets17_tex) + r' \\',
                ''.join(f'\\cmidrule(lr){{{2 + 3 * i}-{4 + 3 * i}}}' for i in range(len(_mets17))),
                'Part' + ' & MedMamba & w/o SS2D & $\\Delta$' * len(_mets17) + r' \\', r'\midrule']
        for part in dict.fromkeys(_abl17.part):
            cells = []
            for met in _mets17:
                r_ = _abl17[(_abl17.part == part) & (_abl17.metric == met)].iloc[0]
                cells += [f'{r_.medmamba_mean:.3f}', f'{r_.no_ss2d_mean:.3f}', _pm(r_.delta_mean, r_.delta_sd)]
            _l17.append(f'{_part17.get(part, part)} & ' + ' & '.join(cells) + r' \\')
        _l17 += [r'\bottomrule', r'\end{tabular}']
        _write_tex('T17_subtype_ablation.tex', _l17)

In [ ]:
# ---------- Headline findings: subtype extension ----------
if not globals().get('SUB_HAS_E2E'):
    print('Subtype extension: no complete pipelines attached, nothing to report.')
else:
    _e46 = sub_main[sub_main.method.isin(SUB_ENCODERS)]
    _m46 = _e46.groupby('method')[['macro_f1', 'accuracy', 'stage1_accuracy', 'ece', 'ece_T1', 'brier',
                                   'risk_at_coverage_80', 'e_aurc', 'missed_malignant_rate']].mean()
    _m46 = _m46.reindex([m for m in SUB_ENCODERS if m in _m46.index])
    print('=' * 100)
    print(f'SUBTYPE EXTENSION - two-stage pipeline, 8 subtypes ({SUB_SETTING_TXT})')
    print('=' * 100)
    _cnt46 = _e46.groupby('method').size()
    print(f"Pipelines per encoder: {', '.join(f'{m} {_cnt46[m]}' for m in _m46.index)} "
          f"(held-out magnifications {SUB_MAGS})")
    _best_f1 = _m46.macro_f1.idxmax()
    _best_ece = _m46.ece.idxmin()
    _best_risk = _m46.risk_at_coverage_80.idxmin()
    print(f'\n1. Accuracy: highest mean macro-F1 = {_best_f1} ({_m46.macro_f1.max():.3f}); '
          f'lowest = {_m46.macro_f1.idxmin()} ({_m46.macro_f1.min():.3f}).')
    print(f'2. Reliability: lowest ECE = {_best_ece} ({_m46.ece.min():.3f}); lowest risk at 80% coverage = '
          f'{_best_risk} ({_m46.risk_at_coverage_80.min():.3f}).')
    print('   -> ' + ('the most accurate encoder is also the best calibrated.' if _best_f1 == _best_ece else
                      f'the accuracy leader ({_best_f1}) is NOT the calibration leader ({_best_ece}).'))
    _rk_ece = _m46.ece.rank().astype(int).to_dict()
    _rk_ece1 = _m46.ece_T1.rank().astype(int).to_dict()
    if _rk_ece != _rk_ece1:
        print(f'   note: the ECE ranking depends on temperature scaling (with T: {_rk_ece}; at T = 1: {_rk_ece1}).')
    _cd46 = globals().get('subtype_cd_ranks')
    if _cd46 is not None and len(_cd46):
        _sig = _cd46[_cd46.significant_pairs != 'none']
        print('3. Significance (Friedman + Nemenyi): '
              + ('no encoder pair is separated on any metric -> report "best mean rank", not "better".'
                 if _sig.empty else
                 '; '.join(f'{r_.metric}: {r_.significant_pairs}' for _, r_ in _sig.iterrows())))
    _hard = _e46.groupby('held_out_mag').macro_f1.mean()
    print(f'4. Hardest held-out magnification (mean macro-F1 over encoders): {int(_hard.idxmin())}x '
          f'({_hard.min():.3f}); easiest {int(_hard.idxmax())}x ({_hard.max():.3f}).')
    _err = _e46[['error_malignant_to_benign', 'error_benign_to_malignant', 'error_wrong_subtype']].mean()
    print(f'5. Error sources (share of all errors): wrong subtype within the right branch '
          f'{100 * _err.error_wrong_subtype / _err.sum():.1f}%, stage-1 routing '
          f'{100 * (_err.error_malignant_to_benign + _err.error_benign_to_malignant) / _err.sum():.1f}% '
          f'(missed cancers {100 * _err.error_malignant_to_benign / _err.sum():.1f}%).')
    print(f"   missed-cancer rate per encoder: "
          + ', '.join(f'{m} {100 * v:.2f}%' for m, v in _m46.missed_malignant_rate.items()))
    _c46 = sub_class[(sub_class.protocol == SUB_MAIN[0]) & (sub_class.split_mode == SUB_MAIN[1])
                     & sub_class.method.isin(SUB_ENCODERS)].groupby('subtype').f1.mean().reindex(SUB_CODES)
    print(f'6. Hardest subtypes (F1, mean over encoders): '
          + ', '.join(f'{SUB_NAMES[c]} {v:.2f}' for c, v in _c46.sort_values().head(3).items())
          + f'; easiest: {SUB_NAMES[_c46.idxmax()]} {_c46.max():.2f}.')
    _abl46 = globals().get('subtype_ablation_delta')
    if _abl46 is not None and len(_abl46):
        _p46 = _abl46[_abl46.part == 'pipeline'] if 'pipeline' in set(_abl46.part) else _abl46
        print('7. SS2D ablation (w/o SS2D minus MedMamba, paired by seed): '
              + '; '.join(f'{r_.part} {r_.metric} {r_.delta_mean:+.3f}' for _, r_ in _p46.iterrows()))
    print('\nCaveat for the paper: image-level split, so test patients also appear at the training '
          'magnifications; compare with the patient-disjoint binary results only qualitatively.')
    print('=' * 100)

## 4c. The proposed model: MagScan-Mamba (Fig 38–41, T18–T23)

**The model.** MagScan-Mamba keeps Phikon (ViT-B/16) frozen and reads its patch tokens instead of only the CLS
token: the 14 × 14 token grid is average-pooled to 7 × 7, the CLS token is added, and a small two-layer head
(width 256, about 1.6 M parameters) mixes the 50 tokens with one Mamba (S6) selective scan shared across four
scan orders (rows and columns, forward and backward). Mean pooling and a linear layer give the class, and a
temperature fitted on the validation set calibrates it. A Mahalanobis gate on the frozen CLS feature flags a
tile that is far from the training magnifications. The runs come from `magscan_mamba_full.ipynb`, on the
benchmark's own splits: the LOMO patient splits (seeds 13, 42, 2025) and Part C's five patient folds.

**How the design was fixed.** Two pilots chose the design on held-out 400× and 40×. Held-out 100× and 200× were
kept back as a confirmation with criteria written down before those runs (the figures shade them). Where the
confirmation failed, this section says so; the paper's claims follow these numbers, not the pilots.

| Output | Content |
|---|---|
| Fig 38 / T18 | MagScan-Mamba next to the four benchmark encoders at every held-out magnification: accuracy (top), reliability (bottom) |
| Fig 39 / T19 | Mean ranks over the 12 (held-out, seed) runs: Friedman + Nemenyi for all pairs, and Wilcoxon signed-rank tests with Holm correction for MagScan-Mamba against each encoder |
| Fig 40 / T20 | Ablations: the token mixer (Mamba scan, attention, gated convolution without a state-space model), the CLS-only probe and the three-zoom variants, as paired differences |
| T21 | Patient-level 5-fold CV against Part C's Phikon probe on the same folds |
| Fig 41 / T22–T23 | Compute (parameters, FLOPs, throughput) and the reliability gate: which feature flags the unseen magnification and BACH |

The subtype pipeline with MagScan-Mamba is in section 4b (Fig 32–36), which picks it up automatically.

**Checks before any comparison.** Every MagScan-Mamba run's stored test labels must equal those of the benchmark's
Phikon probe on the same split (LOMO seed or CV fold). Its metrics are recomputed from its stored probabilities
with the benchmark's definitions, and all its runs must share one configuration hash.

**Inputs.** Attach the output of `magscan_mamba_full.ipynb` (`magscan_full`). The pilots' outputs are optional,
because their runs are already inside it. The output is found automatically (`MAGSCAN_INPUT_PATHS` in the
configuration cell) and kept apart from the binary runs. CPU only; without it this section prints a note and
skips.

In [ ]:
# ---------- MagScan-Mamba: load the runs of magscan_mamba_full.ipynb and check them against the benchmark ----------
# Three checks before anything is compared:
#   1. split   : every run's stored test labels equal those of the benchmark's Phikon probe on the same split
#                (runs/fm_owkin_phikon_... for a LOMO seed, runs/cv_owkin_phikon_..._fold<f> for a CV fold);
#   2. metrics : macro-F1, ECE, Brier, AURC and risk at 80% coverage are recomputed from the stored
#                probabilities with the benchmark's definitions and compared with the record;
#   3. config  : all runs share one configuration hash (the pilots' restored runs included).
# A run whose split differs is left out of every comparison.
from sklearn.metrics import f1_score as _f1_4c
from collections import Counter as _Counter4c

MAGSCAN_LABEL = 'MagScan-Mamba'
MAGSCAN_PROPOSED = 'magscan_single'
MAGSCAN_DESIGN_MAGS = (400, 40)            # the pilots chose the design on these held-out magnifications
MAGSCAN_CONFIRM_MAGS = (100, 200)          # held back: the pre-registered confirmation
MAGSCAN_VARIANTS = {                       # variant -> label; the first three differ only in the token mixer
    'magscan_single': 'MagScan-Mamba (Mamba scan)',
    'attn_single': 'attention mixer',
    'noscan_single': 'gated conv, no SSM',
    'cls_probe': 'CLS linear probe',
    'cls3_probe': '3-zoom CLS probe',
    'magscan': '3-zoom, Mamba scan',
    'attn': '3-zoom, attention',
    'noscan': '3-zoom, gated conv'}
MAGSCAN_COLOR = '#7B2D8E'
COLORS[MAGSCAN_LABEL] = MAGSCAN_COLOR
MARKERS[MAGSCAN_LABEL] = '*'
# fvcore FLOPs of the heads counted on the CPU with the reference scan (the fixed compute cell of
# magscan_mamba_full.ipynb); used only where the Kaggle report has no head count (its GPU trace failed)
MAGSCAN_HEAD_GFLOPS_OFFLINE = {'magscan_single': 0.212, 'magscan': 0.635}
MS_METRICS = [('test_macro_f1', 'Macro-F1', True), ('test_balanced_accuracy', 'Balanced accuracy', True),
              ('test_auc', 'AUROC', True), ('test_ece', 'ECE', False), ('test_brier', 'Brier', False),
              ('test_risk_at_coverage_80', 'Risk @ 80% coverage', False),
              ('test_risk_coverage_auc', 'AURC', False), ('e_aurc', 'E-AURC', False)]
_ms_trapz = getattr(np, 'trapezoid', None) or np.trapz


def _ms_aurc_optimal(error_rate):
    if 'aurc_optimal' in globals():
        return aurc_optimal(error_rate)
    r = np.clip(np.asarray(error_rate, dtype=np.float64), 0.0, 1.0 - 1e-12)
    return r + (1.0 - r) * np.log(1.0 - r)


def _ms_metrics(probs, labels, n_bins=15):
    """The training notebooks' definitions (compute_all_metrics), from stored probabilities."""
    probs = np.asarray(probs, dtype=np.float64)
    labels = np.asarray(labels).astype(int)
    conf, pred = probs.max(axis=1), probs.argmax(axis=1)
    acc = (pred == labels).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum():
            ece += m.mean() * abs(acc[m].mean() - conf[m].mean())
    order = np.argsort(-conf)
    risk = np.cumsum((1.0 - acc)[order]) / np.arange(1, len(acc) + 1)
    cov = np.arange(1, len(acc) + 1) / len(acc)
    return {'test_accuracy': float(acc.mean()), 'test_macro_f1': float(_f1_4c(labels, pred, average='macro')),
            'test_ece': float(ece), 'test_brier': float(((probs - np.eye(probs.shape[1])[labels]) ** 2).sum(axis=1).mean()),
            'test_risk_coverage_auc': float(_ms_trapz(risk, cov)),
            'test_risk_at_coverage_80': float(risk[int(0.8 * len(cov)) - 1])}


def _ms_roots(paths):
    roots = []
    for p in paths or []:
        p = Path(p)
        if p.is_file() and p.suffix == '.zip':
            dst = Path(OUTPUT_DIR).parent / 'magscan_inputs' / p.stem
            if not dst.exists():
                zipfile.ZipFile(p).extractall(dst)
            roots.append(dst)
        elif p.is_dir():
            roots.append(p)
        else:
            print('  MagScan input not found, skipped:', p)
    return roots


_ms_roots_found = _ms_roots(globals().get('MAGSCAN_INPUT_PATHS'))
_ms_cands = {}                                      # run_id -> [(record, folder), ...] over every input
for _root in _ms_roots_found:
    for _rp in sorted(_root.rglob('record.json')):
        if _rp.parent.parent.name != 'runs' or not _rp.parent.name.startswith('magscan_'):
            continue
        if not (_rp.parent / 'test_outputs.npz').exists():
            continue
        try:
            _r = json.loads(_rp.read_text(encoding='utf-8'))
        except Exception as exc:
            print('  unreadable record, skipped:', _rp, repr(exc)[:120]); continue
        _ms_cands.setdefault(_r['run_id'], []).append((_r, _rp.parent))

MS_OK = bool(_ms_cands)
ms_runs = ms_lomo = ms_cv = ms_main = ms_combo = ms_gate = ms_compute = None
if not MS_OK:
    print('Skipped section 4c - no MagScan-Mamba runs found. Attach the output of magscan_mamba_full.ipynb '
          '(Add Input) or set MAGSCAN_INPUT_PATHS in the configuration cell.')
else:
    # ---- one record per run id, all from the configuration hash of the proposed model's runs ----
    _hash_votes = _Counter4c(r.get('config_hash') for c in _ms_cands.values() for r, _ in c
                             if r.get('variant') == MAGSCAN_PROPOSED)
    MS_CONFIG_HASH = _hash_votes.most_common(1)[0][0] if _hash_votes else None
    _chosen, _dropped_hash = {}, 0
    for rid, cands in _ms_cands.items():
        same = [(r, d) for r, d in cands if r.get('config_hash') == MS_CONFIG_HASH]
        if same:
            _chosen[rid] = same[0]
        else:
            _dropped_hash += 1

    _rows = []
    for rid, (r, d) in _chosen.items():
        z = np.load(d / 'test_outputs.npz')
        labels, probs = z['labels'].astype(int), z['probs']
        again = _ms_metrics(probs, labels)
        diff = max(abs(v - float(r[k])) for k, v in again.items() if r.get(k) is not None and pd.notna(r.get(k)))
        fold = r.get('fold_idx')
        is_cv = fold is not None and not (isinstance(fold, float) and np.isnan(fold))
        frac = float(r.get('patient_fraction', 1.0))
        brid = (f"cv_owkin_phikon_ho{int(r['held_out_mag'])}_frac{frac}_seed{int(r['seed'])}_fold{int(fold)}" if is_cv
                else f"fm_owkin_phikon_ho{int(r['held_out_mag'])}_frac{frac}_seed{int(r['seed'])}")
        bpath = f'runs/{brid}/test_outputs.npz'
        if reader.exists(bpath):
            blabels = np.load(io.BytesIO(reader.open_bytes(bpath)))['labels'].astype(int)
            split = 'identical' if np.array_equal(blabels, labels) else 'DIFFERS'
        else:
            split = 'benchmark run not attached'
        branch = 'magscan_cv' if is_cv else 'magscan_lomo'           # the pilots wrote 'magscan_pilot'
        _rows.append({**{k: v for k, v in r.items() if not isinstance(v, (list, dict))},
                      'branch': branch, 'fold_idx': int(fold) if is_cv else np.nan,
                      'held_out_mag': int(r['held_out_mag']), 'seed': int(r['seed']),
                      'split_status': split, 'metric_check_max_diff': diff, 'source_dir': str(d)})
    ms_runs = pd.DataFrame(_rows)
    ms_runs['e_aurc'] = ms_runs.test_risk_coverage_auc - _ms_aurc_optimal(1.0 - ms_runs.test_accuracy)
    ms_runs['variant_label'] = ms_runs.variant.map(MAGSCAN_VARIANTS).fillna(ms_runs.variant)
    ms_runs.to_csv(f'{OUTPUT_DIR}/tables/magscan_runs.csv', index=False)

    _split_counts = ms_runs.split_status.value_counts().to_dict()
    _bad = ms_runs[ms_runs.split_status == 'DIFFERS']
    print(f'MagScan-Mamba: {len(ms_runs)} runs from {len(_ms_roots_found)} input(s) '
          f'({(ms_runs.branch == "magscan_lomo").sum()} LOMO, {(ms_runs.branch == "magscan_cv").sum()} CV); '
          f'config hash {MS_CONFIG_HASH}' + (f', {_dropped_hash} run id(s) only with another hash were left out'
                                             if _dropped_hash else ', every run has it'))
    print(f'  split check against the benchmark Phikon runs: {_split_counts}')
    if len(_bad):
        print(f'  WARNING: {len(_bad)} run(s) were tested on other labels than the benchmark and are left out:',
              _bad.run_id.tolist()[:6])
    _n_unchecked = int((ms_runs.split_status != 'identical').sum()) - len(_bad)
    if _n_unchecked:
        print(f'  note: {_n_unchecked} run(s) could not be checked (benchmark run not attached); attach the four '
              f'LOMO outputs and the Part C CV output')
    _maxdiff = float(ms_runs.metric_check_max_diff.max())
    print(f'  metrics recomputed from the stored probabilities: largest difference from the records {_maxdiff:.1e}'
          + ('' if _maxdiff < 1e-6 else '   <- CHECK: records and stored outputs disagree'))
    ms_runs = ms_runs[ms_runs.split_status != 'DIFFERS'].reset_index(drop=True)

    ms_lomo = ms_runs[ms_runs.branch == 'magscan_lomo'].copy()
    ms_cv = ms_runs[ms_runs.branch == 'magscan_cv'].copy()
    ms_main = ms_lomo[ms_lomo.variant == MAGSCAN_PROPOSED].assign(method=MAGSCAN_LABEL)

    # ---- the benchmark encoders' LOMO runs on the same splits, with E-AURC ----
    _b = (lomo_reliability_runs if globals().get('lomo_reliability_runs') is not None
          else raw[raw.branch.isin(LOMO_BRANCHES)]).copy()
    if 'e_aurc' not in _b.columns:
        _b['e_aurc'] = _b.test_risk_coverage_auc - _ms_aurc_optimal(1.0 - _b.test_accuracy)
    _keep = ['run_id', 'method', 'held_out_mag', 'seed', 'test_accuracy'] + [c for c, _, _ in MS_METRICS]
    ms_combo = pd.concat([ms_main[[c for c in _keep if c in ms_main.columns]],
                          _b[[c for c in _keep if c in _b.columns]]], ignore_index=True)
    ms_combo['held_out_mag'] = ms_combo.held_out_mag.astype(int)
    ms_combo['seed'] = ms_combo.seed.astype(int)
    MS_ORDER = [MAGSCAN_LABEL] + [m for m in ORDER if m in set(ms_combo.method)]
    MS_MAGS = sorted(ms_combo.held_out_mag.unique())
    ms_combo.to_csv(f'{OUTPUT_DIR}/tables/magscan_vs_benchmark_runs.csv', index=False)

    # ---- the gate and compute tables written by the training notebook ----
    def _ms_find(name):
        hits = [p for r in _ms_roots_found for p in r.rglob(name)]
        return max(hits, key=lambda p: len(pd.read_csv(p))) if hits else None
    _gp, _cp = _ms_find('reliability_gate.csv'), _ms_find('magscan_compute_report.csv')
    ms_gate = pd.read_csv(_gp) if _gp is not None else None
    ms_compute = pd.read_csv(_cp) if _cp is not None else None
    if ms_compute is not None:
        _miss = ms_compute.head_gflops.isna() & ms_compute.variant.isin(list(MAGSCAN_HEAD_GFLOPS_OFFLINE))
        ms_compute['head_gflops_source'] = np.where(ms_compute.head_gflops.notna(), 'this run', 'not counted')
        ms_compute.loc[_miss, 'head_gflops'] = ms_compute.loc[_miss, 'variant'].map(MAGSCAN_HEAD_GFLOPS_OFFLINE)
        ms_compute.loc[_miss, 'head_gflops_source'] = 'offline CPU count'
        ms_compute['flops_g_at_224'] = ms_compute.backbone_gflops + ms_compute.head_gflops
        if _miss.any():
            print(f'  compute: head FLOPs of {ms_compute.loc[_miss, "variant"].tolist()} filled from the offline CPU '
                  f'count (the Kaggle run could not trace them on the GPU)')

    _cover = ms_lomo.pivot_table(index='variant', columns='held_out_mag', values='seed', aggfunc='count', fill_value=0)
    _cover = _cover.reindex([v for v in MAGSCAN_VARIANTS if v in _cover.index])
    print('\nLOMO runs (seeds) per variant and held-out magnification:')
    print(_cover.to_string())
    if len(ms_cv):
        print('\nCV runs (folds) per variant and held-out magnification:')
        print(ms_cv.pivot_table(index='variant', columns='held_out_mag', values='seed', aggfunc='count',
                                fill_value=0).reindex([v for v in MAGSCAN_VARIANTS if v in set(ms_cv.variant)]).to_string())
    _blocks = ms_combo.groupby(['held_out_mag', 'seed']).method.nunique()
    print(f'\n(held-out, seed) blocks with all {len(MS_ORDER)} methods: {int((_blocks == len(MS_ORDER)).sum())} '
          f'of {len(_blocks)}; gate table: {"found" if ms_gate is not None else "missing"}; '
          f'compute report: {"found" if ms_compute is not None else "missing"}')
MS_HAS_MAIN = bool(MS_OK and ms_main is not None and len(ms_main))

In [ ]:
# ---------- Fig 38: MagScan-Mamba against the four benchmark encoders at every held-out magnification ----------
# Same patient splits and seeds for all five. Top row accuracy, bottom row reliability; mean over seeds, bars are
# one SD. Shaded columns are the confirmation magnifications (100x, 200x), which were not used to choose the design.
FIG38_PANELS = [('test_macro_f1', 'Macro-F1', True), ('test_balanced_accuracy', 'Balanced accuracy', True),
                ('test_auc', 'AUROC', True), ('test_ece', 'ECE', False),
                ('test_risk_at_coverage_80', 'Risk @ 80% coverage', False), ('e_aurc', 'E-AURC', False)]

if not globals().get('MS_HAS_MAIN'):
    print('Skipped Fig 38 - no MagScan-Mamba runs (attach the magscan_mamba_full.ipynb output).')
else:
    _x38 = {h: i for i, h in enumerate(MS_MAGS)}
    _off38 = {m: (i - (len(MS_ORDER) - 1) / 2) * 0.1 for i, m in enumerate(MS_ORDER)}
    fig, axes = plt.subplots(2, 3, figsize=(15, 8.6))
    for ax, (col, label, higher) in zip(axes.ravel(), FIG38_PANELS):
        agg = ms_combo.groupby(['method', 'held_out_mag'])[col].agg(['mean', 'std']).reset_index()
        for h in MAGSCAN_CONFIRM_MAGS:
            if h in _x38:
                ax.axvspan(_x38[h] - 0.45, _x38[h] + 0.45, color='#F6EFD2', zorder=0, linewidth=0)
        for m in MS_ORDER:
            a = agg[agg.method == m].sort_values('held_out_mag')
            if a.empty:
                continue
            big = m == MAGSCAN_LABEL
            ax.errorbar([_x38[h] + _off38[m] for h in a.held_out_mag], a['mean'], yerr=a['std'].fillna(0),
                        color=COLORS.get(m, '#666666'), marker=MARKERS.get(m, 'o'), markersize=13 if big else 7,
                        markeredgecolor='black', markeredgewidth=0.6, linewidth=2.4 if big else 1.3,
                        capsize=3, label=m, zorder=4 if big else 3)
        ax.set_xticks(range(len(MS_MAGS)))
        ax.set_xticklabels([f'{h}$\\times$' + ('\n(confirm.)' if h in MAGSCAN_CONFIRM_MAGS else '') for h in MS_MAGS])
        ax.set_xlim(-0.5, len(MS_MAGS) - 0.5)
        ax.set_title(f"{label} ({'higher' if higher else 'lower'} is better)", fontsize=11, fontweight='bold')
        ax.grid(alpha=0.3)
    axes[0, 0].set_ylabel('Accuracy', fontsize=12, fontweight='bold')
    axes[1, 0].set_ylabel('Reliability', fontsize=12, fontweight='bold')
    for ax in axes[1]:
        ax.set_xlabel('Held-out (test) magnification')
    _h38, _l38 = axes[0, 0].get_legend_handles_labels()
    fig.legend(_h38, _l38, loc='lower center', ncol=len(_l38), frameon=False, fontsize=10, bbox_to_anchor=(0.5, -0.01))
    fig.suptitle('MagScan-Mamba against the benchmark encoders under an unseen magnification (patient-disjoint LOMO, '
                 'mean $\\pm$ SD over 3 seeds)\nshaded: confirmation magnifications, not used to choose the design',
                 fontsize=12.5, fontweight='bold')
    fig.tight_layout(rect=(0, 0.04, 1, 0.95))
    save_fig(fig, 'fig38_magscan_vs_benchmark')
    plt.show()

    # ---- the numbers behind the figure, and who is best in each cell ----
    ms_per_mag = ms_combo.groupby(['held_out_mag', 'method'])[[c for c, _, _ in MS_METRICS]].agg(['mean', 'std'])
    ms_per_mag.to_csv(f'{OUTPUT_DIR}/tables/magscan_vs_benchmark_per_magnification.csv')
    ms_overall = (ms_combo.groupby('method')[[c for c, _, _ in MS_METRICS]].mean().reindex(MS_ORDER))
    ms_overall.to_csv(f'{OUTPUT_DIR}/tables/magscan_vs_benchmark_overall.csv')
    print('Mean over every held-out magnification and seed (12 runs per method):')
    print(ms_overall.rename(columns={c: l for c, l, _ in MS_METRICS}).round(3).to_string())
    _best_rows = []
    for c, lab, higher in MS_METRICS:
        for h in MS_MAGS:
            s = ms_per_mag.loc[h][(c, 'mean')].dropna()
            if s.empty:
                continue
            best = s.idxmax() if higher else s.idxmin()
            rank = int(s.rank(ascending=not higher, method='min')[MAGSCAN_LABEL]) if MAGSCAN_LABEL in s else None
            _best_rows.append({'metric': lab, 'held_out_mag': h, 'best': best, 'best_value': s[best],
                               'MagScan-Mamba': s.get(MAGSCAN_LABEL, np.nan), 'MagScan rank': rank})
    ms_best_cells = pd.DataFrame(_best_rows)
    ms_best_cells.to_csv(f'{OUTPUT_DIR}/tables/magscan_best_per_cell.csv', index=False)
    print('\nMagScan-Mamba\'s rank among the 5 methods (1 = best) per metric and held-out magnification:')
    print(ms_best_cells.pivot_table(index='metric', columns='held_out_mag', values='MagScan rank')
          .reindex([l for _, l, _ in MS_METRICS]).astype('Int64').to_string())
    print(f'  ranked first in {int((ms_best_cells["MagScan rank"] == 1).sum())} of {len(ms_best_cells)} '
          f'(metric, magnification) cells')

In [ ]:
# ---------- Fig 39 / T19: is MagScan-Mamba's lead significant? Ranks over the 12 (held-out, seed) runs ----------
# Two tests over the same blocks:
#   * Friedman + Nemenyi (Demsar, JMLR 2006): all pairs of the five methods; a bar joins methods whose mean ranks
#     are closer than the critical difference (CD), i.e. not significantly different at alpha = 0.05.
#   * Wilcoxon signed-rank, MagScan-Mamba against each encoder on the paired runs, Holm-corrected over the four
#     comparisons (the comparison with a control recommended by Benavoli et al., JMLR 2016). It uses the size of
#     each paired difference, not only its rank, so it has more power than Nemenyi with 12 blocks.
from scipy.stats import friedmanchisquare as _friedman39, wilcoxon as _wilcoxon39

_NEMENYI39 = {2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949, 8: 3.031}   # Demsar 2006, Tab. 5
FIG39_DRAW = ['test_macro_f1', 'test_ece', 'test_brier', 'test_risk_at_coverage_80', 'e_aurc']
ms_ranks = None


def _holm(pvals):
    """Holm step-down adjusted p-values, in the input order."""
    p = np.asarray(pvals, dtype=float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    running = 0.0
    for i, j in enumerate(order):
        running = max(running, (len(p) - i) * p[j])
        adj[j] = min(1.0, running)
    return adj


if not globals().get('MS_HAS_MAIN'):
    print('Skipped Fig 39 - no MagScan-Mamba runs.')
else:
    _rows39, _avg39 = [], {}
    for col, label, higher in MS_METRICS:
        block = ms_combo.pivot_table(index=['held_out_mag', 'seed'], columns='method', values=col)
        block = block[[m for m in MS_ORDER if m in block.columns]].dropna()
        k, n = block.shape[1], len(block)
        if k < 3 or n < 2 or MAGSCAN_LABEL not in block.columns:
            continue
        avg = block.rank(axis=1, ascending=not higher).mean()
        p = _friedman39(*[block[m] for m in block.columns]).pvalue
        cd = _NEMENYI39[k] * np.sqrt(k * (k + 1) / (6.0 * n))
        pairs = [(a, b) for i, a in enumerate(block.columns) for b in block.columns[i + 1:] if abs(avg[a] - avg[b]) > cd]
        others = [m for m in block.columns if m != MAGSCAN_LABEL]
        diffs = {m: block[MAGSCAN_LABEL] - block[m] for m in others}          # > 0: MagScan higher
        raw_p = []
        for m in others:
            d = diffs[m]
            raw_p.append(1.0 if np.allclose(d, 0) else float(_wilcoxon39(d).pvalue))
        adj = _holm(raw_p)
        row = {'metric': label, 'column': col, 'n_blocks': n, 'friedman_p': p, 'critical_difference': cd,
               **{f'rank {m}': avg[m] for m in block.columns},
               'nemenyi_separated': '; '.join(f'{a} vs {b}' for a, b in pairs) or 'none'}
        for m, pr, pa in zip(others, raw_p, adj):
            d = diffs[m]
            better = int(((d > 0) if higher else (d < 0)).sum())
            row.update({f'delta vs {m}': float(d.mean()), f'wins vs {m}': f'{better}/{n}',
                        f'holm p vs {m}': float(pa)})
        _rows39.append(row)
        _avg39[col] = (avg, cd, p, pairs, label)
    ms_ranks = pd.DataFrame(_rows39)
    ms_ranks.to_csv(f'{OUTPUT_DIR}/tables/magscan_ranks_and_tests.csv', index=False)

    _draw39 = [c for c in FIG39_DRAW if c in _avg39]
    fig, axes = plt.subplots(len(_draw39), 1, figsize=(8.8, 1.7 * len(_draw39) + 0.9), squeeze=False)
    for ax, col in zip(axes[:, 0], _draw39):
        avg, cd, p, pairs, label = _avg39[col]
        k = len(avg)
        ax.set_xlim(0.75, k + 0.25); ax.set_ylim(-0.3, 1.0); ax.invert_xaxis()
        ax.hlines(0.72, 1, k, color='0.3', linewidth=1)
        for t in range(1, k + 1):
            ax.vlines(t, 0.68, 0.76, color='0.3', linewidth=1)
            ax.text(t, 0.84, str(t), ha='center', fontsize=8.5)
        for m in avg.index:
            ax.plot(avg[m], 0.72, marker=MARKERS.get(m, 'o'), color=COLORS.get(m, '#666666'),
                    markersize=15 if m == MAGSCAN_LABEL else 9, markeredgecolor='black', linestyle='', zorder=4)
        srt = avg.sort_values()
        cliques = []
        for i in range(k):
            j = max(jj for jj in range(i, k) if srt.iloc[jj] - srt.iloc[i] < cd)
            if j > i and not any(a <= i and j <= b for a, b in cliques):
                cliques.append((i, j))
        y = 0.50
        for a, b in cliques:
            ax.hlines(y, srt.iloc[a] - 0.03, srt.iloc[b] + 0.03, color='black', linewidth=3.2)
            y -= 0.16
        ax.hlines(-0.17, 1, 1 + cd, color='#B00020', linewidth=2)
        ax.text(1 + cd / 2, -0.11, f'CD = {cd:.2f}', ha='center', fontsize=7.5, color='#B00020')
        _r = ms_ranks[ms_ranks.column == col].iloc[0]
        _sig = [m for m in avg.index if m != MAGSCAN_LABEL and _r[f'holm p vs {m}'] < 0.05]
        ax.set_title(f"{label}   (Friedman p = {p:.2g}; Nemenyi: "
                     f"{'no pair' if not pairs else str(len(pairs)) + ' pair(s)'}; Wilcoxon-Holm vs "
                     f"{', '.join(m.split(' ')[0] for m in _sig) or 'none'} p < 0.05)",
                     fontsize=9.5, fontweight='bold', loc='left')
        ax.axis('off')
    _h39 = [plt.Line2D([], [], marker=MARKERS.get(m, 'o'), color=COLORS.get(m, '#666666'), linestyle='',
                       markersize=11 if m == MAGSCAN_LABEL else 8, markeredgecolor='black', label=m) for m in MS_ORDER]
    fig.legend(handles=_h39, loc='lower center', ncol=len(MS_ORDER), frameon=False, fontsize=9, bbox_to_anchor=(0.5, 0.0))
    fig.suptitle(f'Mean rank over {int(ms_ranks.n_blocks.iloc[0])} LOMO runs (1 = best, right); a bar joins methods '
                 f'that are not significantly different (Nemenyi)', fontsize=11, fontweight='bold', y=0.995)
    fig.tight_layout(rect=(0, 0.045, 1, 0.965))
    save_fig(fig, 'fig39_magscan_critical_difference')
    plt.show()

    print('Mean rank (1 = best), Friedman p, Nemenyi CD; then MagScan-Mamba minus each encoder '
          '(mean paired difference, runs where MagScan is better, Holm-corrected Wilcoxon p):')
    for _, r_ in ms_ranks.iterrows():
        ranks_txt = '  '.join(f"{m.split(' ')[0]} {r_[f'rank {m}']:.2f}" for m in MS_ORDER if f'rank {m}' in r_)
        print(f"\n  {r_.metric:<20} {ranks_txt}   Friedman p = {r_.friedman_p:.2g}   CD = {r_.critical_difference:.2f}"
              f"   Nemenyi separated: {r_.nemenyi_separated}")
        for m in MS_ORDER[1:]:
            if f'delta vs {m}' in r_:
                print(f"      vs {m:<20} {r_[f'delta vs {m}']:+.4f}   better in {r_[f'wins vs {m}']:<6} "
                      f"Holm p = {r_[f'holm p vs {m}']:.3g}" + ('   *' if r_[f'holm p vs {m}'] < 0.05 else ''))

In [ ]:
# ---------- Fig 40 / T20: ablations - which part of MagScan-Mamba matters? ----------
# All variants read the same frozen Phikon tokens with the same training recipe and splits:
#   token mixer : Mamba scan (proposed) vs multi-head attention vs gated convolution with no state-space model
#   tokens      : the proposed head vs a linear probe on the CLS token alone (the benchmark's Phikon probe recipe)
#   zoom views  : three synthetic zoom views of the tile (full, centre 1/2, centre 1/4) instead of one
# (a) macro-F1 per held-out magnification; (b)-(d) paired differences, proposed minus variant, one dot per
# (held-out, seed) run; a dot above 0 in (b) or below 0 in (c)-(d) favours the proposed model.
FIG40_DELTAS = [('test_macro_f1', 'Macro-F1', True), ('test_ece', 'ECE', False),
                ('test_risk_at_coverage_80', 'Risk @ 80% coverage', False)]
_VC40 = {'magscan_single': MAGSCAN_COLOR if 'MAGSCAN_COLOR' in globals() else '#7B2D8E', 'attn_single': '#0072B2',
         'noscan_single': '#555555', 'cls_probe': '#D55E00', 'cls3_probe': '#E69F00', 'magscan': '#C9A0DC',
         'attn': '#9CC9EA', 'noscan': '#BBBBBB'}
ms_ablation = ms_ablation_delta = None

if not globals().get('MS_HAS_MAIN'):
    print('Skipped Fig 40 - no MagScan-Mamba runs.')
else:
    _vars40 = [v for v in MAGSCAN_VARIANTS if v in set(ms_lomo.variant)]
    ms_ablation = ms_lomo.groupby(['variant', 'held_out_mag'])[[c for c, _, _ in MS_METRICS]].agg(['mean', 'std'])
    ms_ablation.to_csv(f'{OUTPUT_DIR}/tables/magscan_ablation_per_magnification.csv')
    _p40 = ms_lomo[ms_lomo.variant == MAGSCAN_PROPOSED].set_index(['held_out_mag', 'seed'])
    _drows = []
    for v in _vars40:
        if v == MAGSCAN_PROPOSED:
            continue
        o = ms_lomo[ms_lomo.variant == v].set_index(['held_out_mag', 'seed'])
        common = _p40.index.intersection(o.index)
        for c, lab, higher in MS_METRICS:
            d = (_p40.loc[common, c] - o.loc[common, c]).astype(float)
            _drows.append({'variant': v, 'label': MAGSCAN_VARIANTS[v], 'metric': lab, 'column': c, 'runs': len(common),
                           'delta_mean': d.mean(), 'delta_sd': d.std(),
                           'proposed_better_in': int(((d > 0) if higher else (d < 0)).sum()),
                           'values': d.to_dict()})
    ms_ablation_delta = pd.DataFrame(_drows)
    ms_ablation_delta.drop(columns='values').to_csv(f'{OUTPUT_DIR}/tables/magscan_ablation_paired_deltas.csv', index=False)

    fig, axes = plt.subplots(1, 4, figsize=(19, 5.2), gridspec_kw={'width_ratios': [1.25, 1, 1, 1]})
    ax = axes[0]
    _mags40 = sorted(ms_lomo.held_out_mag.unique())
    for j, v in enumerate(_vars40):
        g = ms_lomo[ms_lomo.variant == v].groupby('held_out_mag').test_macro_f1.agg(['mean', 'std']).reindex(_mags40)
        prop = v == MAGSCAN_PROPOSED
        ax.errorbar(np.arange(len(_mags40)) + (j - (len(_vars40) - 1) / 2) * 0.07, g['mean'], yerr=g['std'].fillna(0),
                    color=_VC40.get(v, '#999999'), marker='*' if prop else ('o' if v.endswith('single') or v == 'cls_probe' else 's'),
                    markersize=13 if prop else 6, linewidth=2.4 if prop else 1.1,
                    linestyle='-' if (v.endswith('single') or 'probe' in v) else ':', capsize=2,
                    markeredgecolor='black', markeredgewidth=0.5, label=MAGSCAN_VARIANTS[v], zorder=4 if prop else 3)
    ax.set_xticks(range(len(_mags40))); ax.set_xticklabels([f'{int(h)}$\\times$' for h in _mags40])
    ax.set_xlabel('Held-out (test) magnification'); ax.set_ylabel('Macro-F1 (mean $\\pm$ SD over seeds)')
    ax.set_title('(a) Every variant, every held-out magnification', fontsize=10.5, fontweight='bold')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.16), ncol=2, frameon=False)
    _others40 = [v for v in _vars40 if v != MAGSCAN_PROPOSED]
    for ax, (c, lab, higher), tag in zip(axes[1:], FIG40_DELTAS, 'bcd'):
        for i, v in enumerate(_others40):
            r_ = ms_ablation_delta[(ms_ablation_delta.variant == v) & (ms_ablation_delta.column == c)].iloc[0]
            vals = np.array(list(r_['values'].values()), dtype=float)
            jitter = np.linspace(-0.18, 0.18, len(vals)) if len(vals) > 1 else np.zeros(1)
            ax.scatter(np.full(len(vals), i) + jitter, vals, s=16, color=_VC40.get(v, '#999999'), alpha=0.75,
                       edgecolor='none', zorder=3)
            ax.errorbar(i, r_.delta_mean, yerr=r_.delta_sd, fmt='D', color='black', markersize=6, capsize=4, zorder=4)
            ax.text(i, 1.02, f'{r_.proposed_better_in}/{r_.runs}', transform=ax.get_xaxis_transform(),
                    ha='center', fontsize=8, color='0.25')
        ax.axhline(0, color='#B00020', linewidth=1)
        ax.set_xticks(range(len(_others40)))
        ax.set_xticklabels([MAGSCAN_VARIANTS[v] for v in _others40], rotation=40, ha='right', fontsize=8.5)
        ax.set_title(f'({tag}) {lab}: proposed minus variant\n'
                     f"({'above' if higher else 'below'} 0 favours the proposed model; top: runs it wins)",
                     fontsize=10, fontweight='bold', pad=16)
        ax.grid(axis='y', alpha=0.3)
    fig.suptitle('Ablations of MagScan-Mamba on the 12 LOMO runs: token mixer, CLS-only probe, three zoom views',
                 fontsize=12.5, fontweight='bold')
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    save_fig(fig, 'fig40_magscan_ablation')
    plt.show()

    print('Macro-F1 per variant and held-out magnification (mean over seeds):')
    _t40 = ms_lomo.pivot_table(index='variant', columns='held_out_mag', values='test_macro_f1', aggfunc='mean')
    _t40['mean'] = ms_lomo.groupby('variant').test_macro_f1.mean()
    _t40['ECE'] = ms_lomo.groupby('variant').test_ece.mean()
    _t40['risk@80'] = ms_lomo.groupby('variant').test_risk_at_coverage_80.mean()
    _t40.index = [MAGSCAN_VARIANTS.get(v, v) for v in _t40.index]
    print(_t40.reindex([MAGSCAN_VARIANTS[v] for v in _vars40]).round(3).to_string())
    print('\nProposed minus variant, paired by (held-out, seed):')
    _show40 = ms_ablation_delta[ms_ablation_delta.column.isin([c for c, _, _ in FIG40_DELTAS])]
    for _, r_ in _show40.iterrows():
        print(f'  vs {r_.label:<22} {r_.metric:<20} {r_.delta_mean:+.4f} +/- {r_.delta_sd:.4f}   '
              f'proposed better in {r_.proposed_better_in}/{r_.runs}')

In [ ]:
# ---------- Fig 41: cross-validation, compute and the reliability gate of MagScan-Mamba ----------
# CV      : Part C's five patient folds (seed 13) at every held-out magnification, the same folds on which Part C
#           cross-validated the Phikon probe; paired by fold.
# Compute : parameters, fvcore FLOPs at 224 x 224 and T4 throughput. MagScan-Mamba and the Phikon probe share the
#           frozen backbone (FLOPs from the MagScan run's own fvcore count); the trained encoders come from the
#           benchmark's compute report. Fig 3 plots Phikon at the commonly quoted 17.6 GFLOPs for ViT-B/16; fvcore,
#           which skips softmax / normalisation, counts 16.9, so the two figures differ slightly for Phikon.
# Gate    : AUROC for telling the held-out magnification from the validation set (training magnifications) and
#           BACH from the held-out test set: Mahalanobis on the frozen CLS feature (the gate), and the head's own
#           Mahalanobis and confidence (MSP) scores.
import re as _re41

ms_cv_table = ms_cv_pairs = ms_compute_table = ms_gate_table = None
if not globals().get('MS_HAS_MAIN'):
    print('Skipped Fig 41 - no MagScan-Mamba runs.')
else:
    # ---- cross-validation ----
    _pc = raw_all[(raw_all.branch == 'cv')].copy() if 'branch' in raw_all.columns else raw_all.iloc[0:0]
    if len(_pc):
        _pc['fold_idx'] = _pc.run_id.str.extract(r'_fold(\d+)$')[0].astype(float)
    if len(ms_cv):
        _cvr = []
        for (v, h), g in ms_cv.groupby(['variant', 'held_out_mag']):
            _cvr.append({'method': MAGSCAN_VARIANTS.get(v, v), 'variant': v, 'held_out_mag': int(h), 'folds': len(g),
                         'macro_f1_mean': g.test_macro_f1.mean(), 'macro_f1_sd': g.test_macro_f1.std(),
                         'ece_mean': g.test_ece.mean(), 'risk80_mean': g.test_risk_at_coverage_80.mean()})
        for (m, h), g in _pc.groupby(['method', 'held_out_mag']):
            _cvr.append({'method': f'{m} (Part C)', 'variant': None, 'held_out_mag': int(h), 'folds': len(g),
                         'macro_f1_mean': g.test_macro_f1.mean(), 'macro_f1_sd': g.test_macro_f1.std(),
                         'ece_mean': g.test_ece.mean(), 'risk80_mean': g.test_risk_at_coverage_80.mean()})
        ms_cv_table = pd.DataFrame(_cvr)
        ms_cv_table.to_csv(f'{OUTPUT_DIR}/tables/magscan_cv_summary.csv', index=False)
        _pairs41 = []
        _phk = _pc[_pc.model == 'owkin/phikon'].set_index(['held_out_mag', 'fold_idx'])
        _msp = ms_cv[ms_cv.variant == MAGSCAN_PROPOSED].set_index(['held_out_mag', 'fold_idx'])
        for key in _msp.index.intersection(_phk.index):
            _pairs41.append({'held_out_mag': int(key[0]), 'fold': int(key[1]),
                             **{f'{c}_magscan': float(_msp.loc[key, c]) for c in ('test_macro_f1', 'test_ece', 'test_risk_at_coverage_80')},
                             **{f'{c}_phikon': float(_phk.loc[key, c]) for c in ('test_macro_f1', 'test_ece', 'test_risk_at_coverage_80')}})
        ms_cv_pairs = pd.DataFrame(_pairs41)
        print('5-fold patient CV, macro-F1 mean +/- SD over folds:')
        _show = ms_cv_table.assign(v=lambda t: t.macro_f1_mean.map('{:.3f}'.format) + ' +/- ' + t.macro_f1_sd.map('{:.3f}'.format))
        print(_show.pivot_table(index='method', columns='held_out_mag', values='v', aggfunc='first').to_string())
        if len(ms_cv_pairs):
            ms_cv_pairs.to_csv(f'{OUTPUT_DIR}/tables/magscan_cv_paired_with_phikon.csv', index=False)
            _d = ms_cv_pairs.test_macro_f1_magscan - ms_cv_pairs.test_macro_f1_phikon
            _de = ms_cv_pairs.test_ece_magscan - ms_cv_pairs.test_ece_phikon
            print(f'\nMagScan-Mamba minus the Phikon probe on the same {len(ms_cv_pairs)} folds: macro-F1 {_d.mean():+.3f} '
                  f'(better in {int((_d > 0).sum())}/{len(_d)}), ECE {_de.mean():+.3f} (lower in {int((_de < 0).sum())}/{len(_de)})')
            print(ms_cv_pairs.assign(dF1=_d).groupby('held_out_mag').dF1.agg(['mean', 'min', 'max']).round(3).to_string())
    else:
        print('No MagScan-Mamba CV runs found.')

    # ---- compute ----
    _perf = ms_combo.groupby('method')[['test_macro_f1', 'test_ece']].mean()
    _crow = []
    if ms_compute is not None:
        for v, name in ((MAGSCAN_PROPOSED, MAGSCAN_LABEL), ('cls_probe', 'Phikon (frozen FM)')):
            r_ = ms_compute[ms_compute.variant == v]
            if len(r_):
                r_ = r_.iloc[0]
                _crow.append({'method': name, 'parameters_M': r_.parameters / 1e6, 'trainable_M': r_.trainable_params / 1e6,
                              'gflops': r_.flops_g_at_224, 'tiles_per_s_T4': r_.tiles_per_s,
                              'flops_source': 'MagScan run (fvcore)' + ('' if r_.head_gflops_source == 'this run'
                                                                       else f', head: {r_.head_gflops_source}')})
    if compute_report is not None:
        for _, r_ in compute_report.iterrows():
            name = DISPLAY.get(r_.model, r_.model)
            if name in _perf.index:
                _crow.append({'method': name, 'parameters_M': r_.parameters / 1e6, 'trainable_M': r_.parameters / 1e6,
                              'gflops': r_.flops_g_at_224, 'tiles_per_s_T4': np.nan, 'flops_source': 'benchmark report (fvcore)'})
    ms_compute_table = pd.DataFrame(_crow)
    if len(ms_compute_table):
        ms_compute_table = ms_compute_table.merge(_perf.reset_index(), on='method', how='left')
        ms_compute_table['_o'] = ms_compute_table.method.map({m: i for i, m in enumerate(MS_ORDER)})
        ms_compute_table = ms_compute_table.sort_values('_o').drop(columns='_o').reset_index(drop=True)
        ms_compute_table.to_csv(f'{OUTPUT_DIR}/tables/magscan_compute.csv', index=False)
        print('\nCompute (parameters in millions, GFLOPs at 224 x 224, tiles per second on a T4) and mean LOMO accuracy:')
        print(ms_compute_table.round(3).to_string(index=False))

    # ---- the gate ----
    _grow = []
    for h in MS_MAGS:
        g = ms_main[ms_main.held_out_mag == h]
        row = {'held_out_mag': h}
        for pair in ('test_vs_val', 'bach_vs_test'):
            for sc in ('maha', 'msp', 'energy'):
                c = f'{sc}_{pair}_auroc'
                row[f'head {sc} {pair}'] = g[c].mean() if c in g.columns else np.nan
        if ms_gate is not None:
            gg = ms_gate[ms_gate.held_out_mag == h]
            for sc in ('maha', 'knn'):
                for pair in ('test_vs_val', 'bach_vs_test'):
                    c = f'{sc}_{pair}'
                    row[f'gate {sc} {pair}'] = gg[c].mean() if c in gg.columns else np.nan
        _o2g = globals().get('ood_per_run')
        if _o2g is not None and len(_o2g):
            for pair in ('test_vs_val', 'bach_vs_test'):
                s = _o2g[(_o2g.method == 'Phikon (frozen FM)') & (_o2g.held_out_mag == h) & (_o2g.score == 'maha') & (_o2g.pair == pair)]
                row[f'2g Phikon maha {pair}'] = s.auroc.mean() if len(s) else np.nan
        _grow.append(row)
    ms_gate_table = pd.DataFrame(_grow)
    ms_gate_table.to_csv(f'{OUTPUT_DIR}/tables/magscan_gate.csv', index=False)
    print('\nReliability gate, AUROC (mean over seeds; 0.5 = chance):')
    print(ms_gate_table.set_index('held_out_mag').T.round(3).to_string())

    # ---- figure ----
    fig, axes = plt.subplots(1, 4, figsize=(20, 4.9))
    for ax, col, lab in ((axes[0], 'test_macro_f1', 'Macro-F1 (higher is better)'), (axes[1], 'test_ece', 'ECE (lower is better)')):
        if ms_compute_table is None or not len(ms_compute_table):
            ax.text(0.5, 0.5, 'compute report missing', ha='center', transform=ax.transAxes); continue
        for _, r_ in ms_compute_table.iterrows():
            big = r_.method == MAGSCAN_LABEL
            ax.scatter(r_.gflops, r_[col], s=420 if big else 200, color=COLORS.get(r_.method, '#666666'),
                       marker=MARKERS.get(r_.method, 'o'), edgecolor='black', linewidth=1, zorder=4 if big else 3)
            ax.annotate(r_.method, (r_.gflops, r_[col]), xytext=(12, 10 if big else -15), textcoords='offset points',
                        fontsize=9, fontweight='bold' if big else 'normal')
        ax.set_xlabel('Inference GFLOPs at 224 $\\times$ 224 (fvcore)')
        ax.set_ylabel(f'{lab.split(" (")[0]}, mean of 12 LOMO runs')
        ax.set_title(f'({"a" if col == "test_macro_f1" else "b"}) {lab} against compute', fontsize=10.5, fontweight='bold')
        ax.grid(alpha=0.3)
        _xmax = float(np.nanmax(ms_compute_table.gflops))
        ax.set_xlim(0, _xmax * 1.25)
        _yv = ms_compute_table[col].astype(float)
        _pad = 0.15 * max(float(_yv.max() - _yv.min()), 0.01)
        ax.set_ylim(float(_yv.min()) - _pad, float(_yv.max()) + _pad)
    _series41 = [('gate maha', 'gate: Mahalanobis, frozen CLS', MAGSCAN_COLOR, '*', '-'),
                 ('head maha', 'head features: Mahalanobis', '#B39DDB', 'o', '--'),
                 ('head msp', 'head confidence (MSP)', '#888888', 's', ':'),
                 ('2g Phikon maha', 'Phikon probe, section 2g', COLORS.get('Phikon (frozen FM)', '#D55E00'), 'D', 'none')]
    for ax, pair, tag, title in ((axes[2], 'test_vs_val', 'c', 'unseen magnification vs validation'),
                                 (axes[3], 'bach_vs_test', 'd', 'BACH vs held-out test set')):
        xs = np.arange(len(MS_MAGS))
        for key, lab, col, mk, ls in _series41:
            c = f'{key} {pair}'
            if c not in ms_gate_table.columns or ms_gate_table[c].isna().all():
                continue
            ax.plot(xs, ms_gate_table[c], color=col, marker=mk, linestyle=ls, markersize=11 if mk == '*' else 7,
                    markeredgecolor='black', markeredgewidth=0.5, linewidth=2 if mk == '*' else 1.4, label=lab)
        ax.axhline(0.5, color='0.4', linestyle='--', linewidth=1)
        ax.set_xticks(xs); ax.set_xticklabels([f'{h}$\\times$' for h in MS_MAGS])
        ax.set_ylim(0.3, 1.02); ax.set_xlabel('Held-out (test) magnification'); ax.set_ylabel('AUROC (0.5 = chance)')
        ax.set_title(f'({tag}) Detecting the shift: {title}', fontsize=10.5, fontweight='bold')
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8, loc='lower left', frameon=True, framealpha=0.9)
    fig.suptitle('MagScan-Mamba: cost against benefit, and which score flags a shifted input',
                 fontsize=12.5, fontweight='bold')
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    save_fig(fig, 'fig41_magscan_compute_and_gate')
    plt.show()

In [ ]:
# ---------- T18-T23: MagScan-Mamba tables (LaTeX + CSV) ----------
# T18 main comparison per held-out magnification, T19 ranks and tests, T20 ablations, T21 cross-validation,
# T22 compute, T23 the reliability gate. The best value in each column (per magnification) is bold.
def _pm4c(mean, sd, digits=3, bold=False):
    if mean is None or not np.isfinite(mean):
        return '--'
    s = f'{mean:.{digits}f}' + (f' $\\pm$ {sd:.{digits}f}' if sd is not None and np.isfinite(sd) else '')
    return f'\\textbf{{{s}}}' if bold else s


def _write_tex4c(name, lines):
    Path(f'{OUTPUT_DIR}/tables/{name}').write_text('\n'.join(lines) + '\n', encoding='utf-8')
    print('  wrote tables/' + name)


if not globals().get('MS_HAS_MAIN'):
    print('Skipped T18-T23 - no MagScan-Mamba runs.')
else:
    # ---- T18: five methods x held-out magnification ----
    _t18 = [('test_macro_f1', 'Macro-F1', True), ('test_balanced_accuracy', 'Bal. acc.', True),
            ('test_ece', 'ECE', False), ('test_brier', 'Brier', False),
            ('test_risk_at_coverage_80', 'Risk@80', False), ('e_aurc', 'E-AURC', False)]
    L = [r'\begin{tabular}{ll' + 'c' * len(_t18) + '}', r'\toprule',
         'Held-out & Method & ' + ' & '.join(l for _, l, _ in _t18) + r' \\', r'\midrule']
    for h in MS_MAGS + ['all']:
        sub = ms_combo if h == 'all' else ms_combo[ms_combo.held_out_mag == h]
        agg = sub.groupby('method')[[c for c, _, _ in _t18]].agg(['mean', 'std'])
        best = {c: (agg[(c, 'mean')].idxmax() if hi else agg[(c, 'mean')].idxmin()) for c, _, hi in _t18}
        tag = 'All' if h == 'all' else f'{h}$\\times$' + ('$^{\\dagger}$' if h in MAGSCAN_CONFIRM_MAGS else '')
        for i, m in enumerate([m for m in MS_ORDER if m in agg.index]):
            name = f'\\textbf{{{m}}}' if m == MAGSCAN_LABEL else m
            cells = [_pm4c(agg.loc[m, (c, 'mean')], agg.loc[m, (c, 'std')], bold=(best[c] == m)) for c, _, _ in _t18]
            L.append(f"{tag if i == 0 else ''} & {name} & " + ' & '.join(cells) + r' \\')
        L.append(r'\midrule' if h != 'all' else r'\bottomrule')
    L += [r'\end{tabular}', r'% dagger: confirmation magnification, not used to choose the design. Mean $\pm$ SD over 3 seeds'
          r' (All: over 12 runs). Bold: best in the column at that magnification.']
    _write_tex4c('T18_magscan_vs_benchmark.tex', L)

    # ---- T19: ranks and tests ----
    if ms_ranks is not None and len(ms_ranks):
        _others = [m for m in MS_ORDER[1:] if f'rank {m}' in ms_ranks.columns]
        L = [r'\begin{tabular}{l' + 'c' * (len(MS_ORDER) + 2) + '}', r'\toprule',
             r' & \multicolumn{' + str(len(MS_ORDER)) + r'}{c}{Mean rank (1 = best)} & Friedman & \\',
             'Metric & ' + ' & '.join(MS_ORDER) + r' & $p$ & Holm-Wilcoxon: MagScan vs \\', r'\midrule']
        for _, r_ in ms_ranks.iterrows():
            ranks = [r_[f'rank {m}'] for m in MS_ORDER]
            cells = [f'\\textbf{{{v:.2f}}}' if v == min(ranks) else f'{v:.2f}' for v in ranks]
            sig = ', '.join(f"{m.split(' ')[0]} ({r_[f'holm p vs {m}']:.3f})" for m in _others
                            if r_[f'holm p vs {m}'] < 0.05) or 'none'
            L.append(f'{r_.metric} & ' + ' & '.join(cells) + f' & {r_.friedman_p:.3f} & {sig}' + r' \\')
        L += [r'\bottomrule', r'\end{tabular}',
              f'% Nemenyi CD = {ms_ranks.critical_difference.iloc[0]:.2f} over {int(ms_ranks.n_blocks.iloc[0])} blocks; '
              'last column: encoders MagScan-Mamba differs from (Holm-adjusted Wilcoxon p < 0.05).']
        _write_tex4c('T19_magscan_ranks.tex', L)

    # ---- T20: ablations ----
    if ms_ablation_delta is not None and len(ms_ablation_delta):
        _v20 = [v for v in MAGSCAN_VARIANTS if v in set(ms_lomo.variant)]
        _f20 = ms_lomo.pivot_table(index='variant', columns='held_out_mag', values='test_macro_f1', aggfunc='mean')
        _m20 = ms_lomo.groupby('variant')[['test_macro_f1', 'test_ece', 'test_risk_at_coverage_80']].mean()
        L = [r'\begin{tabular}{l' + 'c' * len(MS_MAGS) + 'ccccc}', r'\toprule',
             r' & \multicolumn{' + str(len(MS_MAGS)) + r'}{c}{Macro-F1 at held-out} & \multicolumn{3}{c}{Mean of 12 runs} & '
             r'\multicolumn{2}{c}{Proposed minus variant} \\',
             'Variant & ' + ' & '.join(f'{h}$\\times$' for h in MS_MAGS) + r' & F1 & ECE & Risk@80 & $\Delta$F1 & wins \\',
             r'\midrule']
        for v in _v20:
            name = MAGSCAN_VARIANTS[v]
            name = f'\\textbf{{{name}}}' if v == MAGSCAN_PROPOSED else name
            d = ms_ablation_delta[(ms_ablation_delta.variant == v) & (ms_ablation_delta.column == 'test_macro_f1')]
            dcell = '--' if d.empty else f'{d.delta_mean.iloc[0]:+.3f}'
            wcell = '--' if d.empty else f'{int(d.proposed_better_in.iloc[0])}/{int(d.runs.iloc[0])}'
            L.append(f'{name} & ' + ' & '.join(f'{_f20.loc[v, h]:.3f}' if h in _f20.columns and pd.notna(_f20.loc[v, h]) else '--'
                                               for h in MS_MAGS)
                     + f' & {_m20.loc[v, "test_macro_f1"]:.3f} & {_m20.loc[v, "test_ece"]:.3f} & '
                       f'{_m20.loc[v, "test_risk_at_coverage_80"]:.3f} & {dcell} & {wcell}' + r' \\')
        L += [r'\bottomrule', r'\end{tabular}']
        _write_tex4c('T20_magscan_ablation.tex', L)

    # ---- T21: cross-validation ----
    if ms_cv_table is not None and len(ms_cv_table):
        _mags21 = sorted(ms_cv_table.held_out_mag.unique())
        _rows21 = [MAGSCAN_VARIANTS[v] for v in MAGSCAN_VARIANTS if v in set(ms_cv_table.variant.dropna())]
        _rows21 += sorted(m for m in ms_cv_table.method.unique() if m.endswith('(Part C)'))
        L = [r'\begin{tabular}{l' + 'c' * len(_mags21) + '}', r'\toprule',
             'Method (5 patient folds) & ' + ' & '.join(f'{h}$\\times$' for h in _mags21) + r' \\', r'\midrule']
        for m in _rows21:
            cells = []
            for h in _mags21:
                s = ms_cv_table[(ms_cv_table.method == m) & (ms_cv_table.held_out_mag == h)]
                cells.append('--' if s.empty else _pm4c(s.macro_f1_mean.iloc[0], s.macro_f1_sd.iloc[0]))
            L.append(f'{m} & ' + ' & '.join(cells) + r' \\')
        L += [r'\bottomrule', r'\end{tabular}', '% macro-F1, mean $\\pm$ SD over folds; Part C rows: the benchmark\'s own CV on the same folds.']
        _write_tex4c('T21_magscan_cv.tex', L)

    # ---- T22: compute ----
    if ms_compute_table is not None and len(ms_compute_table):
        L = [r'\begin{tabular}{lrrrrcc}', r'\toprule',
             r'Method & Params (M) & Trainable (M) & GFLOPs & Tiles/s (T4) & Macro-F1 & ECE \\', r'\midrule']
        for _, r_ in ms_compute_table.iterrows():
            tp = '--' if not np.isfinite(r_.tiles_per_s_T4) else f'{r_.tiles_per_s_T4:.0f}'
            L.append(f'{r_.method} & {r_.parameters_M:.1f} & {r_.trainable_M:.2f} & {r_.gflops:.2f} & {tp} & '
                     f'{r_.test_macro_f1:.3f} & {r_.test_ece:.3f}' + r' \\')
        L += [r'\bottomrule', r'\end{tabular}', '% FLOPs counted with fvcore at 224 x 224 (one fused multiply-add = one FLOP).']
        _write_tex4c('T22_magscan_compute.tex', L)

    # ---- T23: the gate ----
    if ms_gate_table is not None and len(ms_gate_table):
        _cols23 = [('gate maha test_vs_val', 'Gate (frozen CLS)'), ('head maha test_vs_val', 'Head Mahal.'),
                   ('head msp test_vs_val', 'Head MSP'), ('gate maha bach_vs_test', 'Gate (frozen CLS)'),
                   ('head maha bach_vs_test', 'Head Mahal.'), ('head msp bach_vs_test', 'Head MSP')]
        _cols23 = [(c, l) for c, l in _cols23 if c in ms_gate_table.columns]
        _n_mag = sum(1 for c, _ in _cols23 if 'test_vs_val' in c)
        L = [r'\begin{tabular}{l' + 'c' * len(_cols23) + '}', r'\toprule',
             r' & \multicolumn{' + str(_n_mag) + r'}{c}{Unseen magnification vs validation} & \multicolumn{'
             + str(len(_cols23) - _n_mag) + r'}{c}{BACH vs test} \\',
             'Held-out & ' + ' & '.join(l for _, l in _cols23) + r' \\', r'\midrule']
        for _, r_ in ms_gate_table.iterrows():
            L.append(f'{int(r_.held_out_mag)}$\\times$ & ' + ' & '.join('--' if not np.isfinite(r_[c]) else f'{r_[c]:.3f}'
                                                                       for c, _ in _cols23) + r' \\')
        L += [r'\bottomrule', r'\end{tabular}', '% AUROC, mean over seeds; 0.5 = chance.']
        _write_tex4c('T23_magscan_gate.tex', L)

In [ ]:
# ---------- Headline findings: the proposed model (MagScan-Mamba) ----------
# Every sentence is derived from the tables above, including the ones that go against the model.
if not globals().get('MS_HAS_MAIN'):
    print('MagScan-Mamba: no runs attached, nothing to report.')
else:
    _ov = ms_overall
    print('=' * 100)
    print('PROPOSED MODEL - MagScan-Mamba (frozen Phikon patch tokens + Mamba scan head), binary LOMO, same splits')
    print('=' * 100)
    _f1 = _ov.test_macro_f1.sort_values(ascending=False)
    print(f'1. Accuracy (mean macro-F1 over 12 runs): ' + ', '.join(f'{m} {v:.3f}' for m, v in _f1.items()))
    print(f'   MagScan-Mamba ranks {list(_f1.index).index(MAGSCAN_LABEL) + 1} of {len(_f1)}'
          + (f'; the gap to {_f1.index[0]} is {_f1[MAGSCAN_LABEL] - _f1.iloc[0]:+.3f}' if _f1.index[0] != MAGSCAN_LABEL
             else f'; next is {_f1.index[1]} at {_f1.iloc[1] - _f1[MAGSCAN_LABEL]:+.3f}'))
    _rel = [(c, l) for c, l, hi in MS_METRICS if not hi]
    _lead = [l for c, l in _rel if _ov[c].idxmin() == MAGSCAN_LABEL]
    _lag = [f'{l} ({_ov[c].idxmin()})' for c, l in _rel if _ov[c].idxmin() != MAGSCAN_LABEL]
    print(f'2. Reliability: MagScan-Mamba has the lowest mean {", ".join(_lead) or "- none -"}'
          + (f'; not the lowest on {", ".join(_lag)}' if _lag else ''))
    for c, l in _rel:
        print(f'     {l:<20} ' + ', '.join(f'{m.split(" ")[0]} {v:.3f}' for m, v in _ov[c].sort_values().items()))
    # ---- the pre-registered confirmation ----
    for h in MAGSCAN_CONFIRM_MAGS:
        if h not in MS_MAGS:
            continue
        mm = ms_per_mag.loc[h]
        f1_ok = all(mm.loc[MAGSCAN_LABEL, ('test_macro_f1', 'mean')] >= mm.loc[m, ('test_macro_f1', 'mean')] for m in mm.index)
        rel_ok = all(mm.loc[MAGSCAN_LABEL, (c, 'mean')] <= mm.loc[m, (c, 'mean')] for m in mm.index
                     for c in ('test_ece', 'test_risk_at_coverage_80'))
        top = mm[('test_macro_f1', 'mean')].idxmax()
        print(f'3. Confirmation at {h}x (not used for the design): accuracy {"PASS" if f1_ok else "FAIL"}'
              + ('' if f1_ok else f' (best {top} {mm.loc[top, ("test_macro_f1", "mean")]:.3f} vs '
                                  f'{mm.loc[MAGSCAN_LABEL, ("test_macro_f1", "mean")]:.3f})')
              + f'; lowest ECE and risk@80 {"PASS" if rel_ok else "FAIL"} (ECE best: {mm[("test_ece", "mean")].idxmin()}, '
                f'risk@80 best: {mm[("test_risk_at_coverage_80", "mean")].idxmin()})')
    if ms_ranks is not None and len(ms_ranks):
        _sig = []
        for _, r_ in ms_ranks.iterrows():
            s = [m for m in MS_ORDER[1:] if f'holm p vs {m}' in r_ and r_[f'holm p vs {m}'] < 0.05]
            if s:
                better = [m for m in s if (r_[f'delta vs {m}'] > 0) == any(r_.column == c for c, _, hi in MS_METRICS if hi)]
                _sig.append(f"{r_.metric}: differs from {', '.join(s)} (better than {', '.join(better) or 'none'})")
        print('4. Significance (Wilcoxon signed-rank, Holm, 12 paired runs): '
              + ('no difference to any encoder on any metric -> write "comparable", not "better"' if not _sig
                 else '\n     ' + '\n     '.join(_sig)))
        _nem = ms_ranks[ms_ranks.nemenyi_separated.str.contains(MAGSCAN_LABEL, regex=False)]
        print('   Nemenyi (all pairs): ' + ('MagScan-Mamba is not separated from any encoder on any metric.' if _nem.empty
              else '; '.join(f'{r_.metric}: {r_.nemenyi_separated}' for _, r_ in _nem.iterrows())))
    if ms_ablation_delta is not None and len(ms_ablation_delta):
        _a = ms_ablation_delta[ms_ablation_delta.column == 'test_macro_f1'].set_index('variant')
        def _line(v):
            return (f'{_a.loc[v, "delta_mean"]:+.3f} F1 vs {MAGSCAN_VARIANTS[v]} (better in '
                    f'{int(_a.loc[v, "proposed_better_in"])}/{int(_a.loc[v, "runs"])})') if v in _a.index else None
        print('5. Ablations: ' + '; '.join(x for x in (_line('cls_probe'), _line('attn_single'), _line('noscan_single')) if x))
        _mix = [v for v in ('attn_single', 'noscan_single') if v in _a.index]
        if _mix and all(abs(_a.loc[v, 'delta_mean']) < _a.loc[v, 'delta_sd'] for v in _mix):
            print('   -> the token mixer makes no difference beyond seed noise: the gain comes from reading the frozen '
                  'foundation-model patch tokens, not from the state-space scan.')
    if ms_cv_pairs is not None and len(ms_cv_pairs):
        _d = ms_cv_pairs.test_macro_f1_magscan - ms_cv_pairs.test_macro_f1_phikon
        print(f'6. Patient-level CV (same {len(_d)} folds as Part C): MagScan-Mamba minus Phikon probe, macro-F1 '
              f'{_d.mean():+.3f}, better in {int((_d > 0).sum())}/{len(_d)} folds.')
    if ms_gate_table is not None and 'gate maha test_vs_val' in ms_gate_table:
        g = ms_gate_table.set_index('held_out_mag')
        print('7. Gate (frozen CLS Mahalanobis), unseen magnification vs validation AUROC: '
              + ', '.join(f'{h}x {v:.2f}' for h, v in g['gate maha test_vs_val'].items())
              + (f'; the head\'s own features: ' + ', '.join(f'{h}x {v:.2f}' for h, v in g['head maha test_vs_val'].items())
                 if 'head maha test_vs_val' in g else ''))
    if ms_compute_table is not None and len(ms_compute_table):
        c = ms_compute_table.set_index('method')
        if MAGSCAN_LABEL in c.index and 'Phikon (frozen FM)' in c.index:
            print(f'8. Cost: {c.loc[MAGSCAN_LABEL, "gflops"]:.2f} GFLOPs vs {c.loc["Phikon (frozen FM)", "gflops"]:.2f} for the '
                  f'Phikon probe (+{100 * (c.loc[MAGSCAN_LABEL, "gflops"] / c.loc["Phikon (frozen FM)", "gflops"] - 1):.1f}%), '
                  f'{c.loc[MAGSCAN_LABEL, "trainable_M"]:.2f} M trainable parameters.')
    # ---- the subtype pipeline (section 4b) ----
    if globals().get('SUB_HAS_E2E') and MAGSCAN_LABEL in set(sub_main.method):
        _s = sub_main[sub_main.method.isin(SUB_ENCODERS)].groupby('method')[
            ['macro_f1', 'accuracy', 'stage1_accuracy', 'ece', 'risk_at_coverage_80', 'missed_malignant_rate']].mean()
        _sf = _s.macro_f1.sort_values(ascending=False)
        _other = _sf.drop(MAGSCAN_LABEL)
        print(f'9. Subtypes (two-stage pipeline, 8 classes, {SUB_SETTING_TXT}): MagScan-Mamba macro-F1 '
              f'{_s.loc[MAGSCAN_LABEL, "macro_f1"]:.3f}, rank {list(_sf.index).index(MAGSCAN_LABEL) + 1} of {len(_sf)} '
              f'(best other: {_other.index[0]} {_other.iloc[0]:.3f}); stage-1 accuracy {_s.loc[MAGSCAN_LABEL, "stage1_accuracy"]:.3f} '
              f'(rank {int(_s.stage1_accuracy.rank(ascending=False)[MAGSCAN_LABEL])}); ECE {_s.loc[MAGSCAN_LABEL, "ece"]:.3f} '
              f'(rank {int(_s.ece.rank()[MAGSCAN_LABEL])}); risk@80 {_s.loc[MAGSCAN_LABEL, "risk_at_coverage_80"]:.3f} '
              f'(rank {int(_s.risk_at_coverage_80.rank()[MAGSCAN_LABEL])}); missed cancers '
              f'{100 * _s.loc[MAGSCAN_LABEL, "missed_malignant_rate"]:.1f}% (rank {int(_s.missed_malignant_rate.rank()[MAGSCAN_LABEL])}).')
    print('\nWording for the paper: claim only what the lines above support. Where a comparison is not significant, '
          'write "comparable" or "best mean", and report the failed confirmation criteria as they are.')
    print('=' * 100)

## 4d. The subtype pipeline in 3D (Fig 42–47)

The binary study's 3D figures (Fig 16, 19, 21, 22 and the radar Fig 27), redrawn for the two-stage subtype
pipeline with all five encoders, MagScan-Mamba included. Each figure keeps the visual idea and uses only
quantities the subtype runs actually stored: per-image logits of every stage, the stage checkpoints and the
manifest. There are no masks and no follow-up images, as in the binary study.

| Binary figure | Subtype figure | Needs |
|---|---|---|
| Fig 16: accuracy and reliability as 3D bars | **Fig 42**: pipeline macro-F1 and ECE over encoder × held-out magnification, plus accuracy rank against reliability rank | stored logits |
| Fig 22: selective-risk landscape | **Fig 43**: the same surface for the pipeline, confidence = path probability of the predicted subtype | stored logits |
| (new) | **Fig 44**: error towers, the 8 × 8 confusion with the diagonal removed, coloured by error source (missed cancer, false alarm, wrong subtype) | stored logits |
| Fig 27: capability radar | **Fig 45**: subtype radar with fixed, printed floors and ceilings: accuracy, detection, calibration, selective risk, missed cancers, seed stability, pipeline compute | stored logits, compute reports |
| Fig 19: saliency layer stack | **Fig 46**: the malignant-subtype model's saliency by depth on one lobular carcinoma tile (the hardest subtype). Grad-CAM++ at four stages for ConvNeXt-T, Swin-T and MedMamba; Attention Rollout for Phikon; for MagScan-Mamba, rollout of its frozen backbone at two depths plus token Grad-CAM after each of its two Mamba layers | GPU, BreakHis, stage checkpoints |
| Fig 21: one patient across magnifications | **Fig 47**: one lobular carcinoma patient at 40×, 100×, 200× and 400×. Every panel is scored by the pipeline that never saw that magnification in training: probability of the true subtype (3D bars), share of the patient's tiles named correctly, and the most frequent prediction | stored logits; BreakHis for the tiles |

Fig 42–45 and Fig 47 (without its tile row) run on the CPU from section 4b's arrays. Fig 46 needs the
section 2h saliency backend (GPU) and BreakHis. The subtype checkpoints are read from the attached subtype
outputs (`best.pt`, `probe.pt`, `head.pt`).

In [ ]:
# ---------- Fig 42: subtype pipeline accuracy and reliability for every encoder x held-out magnification ----------
# Fig 16 for the subtype task: (a) end-to-end macro-F1 and (b) ECE of the path probability as 3D bars over the
# encoder x held-out magnification grid (means over seeds), (c) each encoder's rank by accuracy and by
# reliability, averaged over magnifications. Bars, not a surface: the encoder axis is categorical.
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from scipy.stats import spearmanr as _spearman42

SUB_SHORT = {'MagScan-Mamba': 'MagScan', 'Phikon (frozen FM)': 'Phikon', 'ConvNeXt-T': 'ConvNeXt-T',
             'Swin-T': 'Swin-T', 'MedMamba': 'MedMamba', 'MedMamba w/o SS2D': 'MedMamba w/o SS2D'}

if not globals().get('SUB_HAS_E2E'):
    print('Skipped Fig 42 - no complete subtype pipelines (section 4b).')
else:
    _d42 = sub_main[sub_main.method.isin(SUB_ENCODERS)]
    F42_F1 = _d42.pivot_table(index='method', columns='held_out_mag', values='macro_f1', aggfunc='mean').reindex(
        index=SUB_ENCODERS, columns=SUB_MAGS)
    F42_ECE = _d42.pivot_table(index='method', columns='held_out_mag', values='ece', aggfunc='mean').reindex(
        index=SUB_ENCODERS, columns=SUB_MAGS)

    def _bars42(ax, table, zlabel, zmax, title):
        n_enc, n_mag = table.shape
        xs, ys = np.meshgrid(np.arange(n_enc), np.arange(n_mag), indexing='ij')
        xs, ys = xs.ravel(), ys.ravel()
        dz = np.nan_to_num(table.to_numpy(dtype=float).ravel())
        ax.bar3d(xs - 0.28, ys - 0.28, np.zeros_like(dz), 0.56, 0.56, dz,
                 color=[SUB_COLORS.get(table.index[i], '#666666') for i in xs],
                 edgecolor='black', linewidth=0.3, alpha=0.92, shade=True)
        ax.set_xticks(np.arange(n_enc))
        ax.set_xticklabels([SUB_SHORT.get(m, m) for m in table.index], rotation=-42, ha='left', va='top', fontsize=7.5)
        ax.set_yticks(np.arange(n_mag)); ax.set_yticklabels([f'{m}×' for m in table.columns], fontsize=8.5)
        ax.set_zlim(0, zmax); ax.set_zlabel(zlabel, labelpad=6)
        ax.set_ylabel('Held-out magnification', labelpad=8)
        ax.view_init(elev=24, azim=-50)
        ax.set_title(title, fontweight='bold', fontsize=11, pad=2)

    fig = plt.figure(figsize=(18.5, 5.6))
    _gs42 = fig.add_gridspec(1, 3, width_ratios=[1.0, 1.0, 0.85], wspace=0.22)
    _bars42(fig.add_subplot(_gs42[0, 0], projection='3d'), F42_F1, 'Macro-F1 (higher is better)', 1.0,
            '(a) Accuracy: 8-subtype macro-F1')
    _bars42(fig.add_subplot(_gs42[0, 1], projection='3d'), F42_ECE, 'ECE (lower is better)',
            float(np.ceil(np.nanmax(F42_ECE.to_numpy()) * 1.15 * 50) / 50), '(b) Reliability: ECE of the path probability')

    ax = fig.add_subplot(_gs42[0, 2])
    _f1m, _ecem = F42_F1.mean(axis=1), F42_ECE.mean(axis=1)
    _ra = _f1m.rank(ascending=False, method='min').astype(int)
    _rr = _ecem.rank(ascending=True, method='min').astype(int)
    for m in SUB_ENCODERS:
        ax.plot([0, 1], [_ra[m], _rr[m]], color=SUB_COLORS.get(m, '#666666'), linewidth=2.6,
                marker=SUB_MARKERS.get(m, 'o'), markersize=10 if m == 'MagScan-Mamba' else 9,
                markeredgecolor='black', markeredgewidth=0.6)
        ax.text(-0.12, _ra[m], f'#{_ra[m]}  {SUB_SHORT.get(m, m)}  {_f1m[m]:.3f}', ha='right', va='center', fontsize=9)
        ax.text(1.12, _rr[m], f'{_ecem[m]:.3f}  {SUB_SHORT.get(m, m)}  #{_rr[m]}', ha='left', va='center', fontsize=9)
    ax.set_xlim(-1.3, 2.3); ax.set_ylim(len(SUB_ENCODERS) + 0.5, 0.5)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['Rank by\nmacro-F1', 'Rank by\nECE'], fontsize=10)
    ax.set_yticks([]); ax.grid(False)
    for side in ('top', 'right', 'bottom', 'left'):
        ax.spines[side].set_visible(False)
    ax.set_title('(c) Accuracy rank vs reliability rank\n(mean over held-out magnifications)', fontweight='bold', fontsize=11)
    plt.suptitle(f'Subtype pipeline: accuracy and reliability across encoders and held-out magnifications '
                 f'({SUB_SETTING_TXT})', y=1.02, fontweight='bold')
    save_fig(fig, 'fig42_subtype_accuracy_reliability_3d')
    plt.show()

    _ok42 = F42_F1.notna().to_numpy().ravel() & F42_ECE.notna().to_numpy().ravel()
    _rho42, _p42 = _spearman42(F42_F1.to_numpy().ravel()[_ok42], F42_ECE.to_numpy().ravel()[_ok42])
    _win42 = pd.DataFrame({'best macro-F1': F42_F1.idxmax(axis=0), 'lowest ECE': F42_ECE.idxmin(axis=0)})
    _win42['same encoder'] = _win42['best macro-F1'] == _win42['lowest ECE']
    print('Rank by macro-F1 :', _ra.sort_values().to_dict())
    print('Rank by ECE      :', _rr.sort_values().to_dict())
    print(f'Spearman rho(macro-F1, ECE) over {int(_ok42.sum())} (encoder, magnification) cells = {_rho42:+.3f} '
          f'(p = {_p42:.3f}); negative = more accurate cells tend to be better calibrated.')
    print('\nPer held-out magnification:')
    print(_win42.to_string())

In [ ]:
# ---------- Fig 43: selective-risk landscape of the subtype pipeline (coverage x held-out magnification x risk) ----------
# Fig 22 for the subtype task. Images are ranked by the path probability of the predicted subtype (the
# pipeline's confidence); the curve is the error rate among the most confident images as coverage grows,
# averaged over seeds. One ribbon per held-out magnification: the axis has four ordinal levels, nothing between.
FIG43_COVERAGE = np.linspace(0.30, 1.0, 71)                              # (71,)
subtype_risk_landscape = None

if not globals().get('SUB_HAS_E2E') or not globals().get('SUB_IMAGES'):
    print('Skipped Fig 43 - no per-image subtype predictions (section 4b).')
else:
    _cube43 = {}
    for m in SUB_ENCODERS:
        rows = []
        for ho in SUB_MAGS:
            curves = []
            for (mm, pr, sm, h, s), a in SUB_IMAGES.items():
                if mm != m or (pr, sm) != SUB_MAIN or h != str(ho):
                    continue
                conf = a['p8'][np.arange(len(a['pred'])), a['pred']]
                order = np.argsort(-conf, kind='stable')                    # as pipeline_scores ranks them
                err = (a['pred'] != a['y8']).astype(float)[order]
                k = np.arange(1, len(err) + 1)
                curves.append(np.interp(FIG43_COVERAGE, k / len(err), np.cumsum(err) / k))
            rows.append(np.mean(curves, axis=0) if curves else np.full(len(FIG43_COVERAGE), np.nan))
        _cube43[m] = np.stack(rows)                                         # (n_mags, 71)

    from matplotlib.ticker import MaxNLocator as _MaxN43, FormatStrFormatter as _Fmt43
    _zmax43 = float(np.nanmax([np.nanmax(v) for v in _cube43.values()]) * 1.05)
    fig = plt.figure(figsize=(4.6 * len(SUB_ENCODERS), 4.9))
    for i, m in enumerate(SUB_ENCODERS):
        ax = fig.add_subplot(1, len(SUB_ENCODERS), i + 1, projection='3d')
        Z = _cube43[m]
        for j, ho in enumerate(SUB_MAGS):
            if np.isnan(Z[j]).all():
                continue
            yj = np.full_like(FIG43_COVERAGE, j, dtype=float)
            ax.plot(FIG43_COVERAGE, yj, Z[j], color=SUB_COLORS.get(m, '#666666'), linewidth=2.0, alpha=0.95)
            ax.plot_surface(np.vstack([FIG43_COVERAGE, FIG43_COVERAGE]), np.vstack([yj, yj + 0.34]),
                            np.vstack([Z[j], Z[j]]), color=SUB_COLORS.get(m, '#666666'), alpha=0.30, linewidth=0,
                            shade=False)
        ax.set_xlabel('Coverage', labelpad=-2, fontsize=9)
        ax.set_ylabel('Held-out mag', labelpad=2, fontsize=9)
        ax.set_zlabel('Selective risk', labelpad=7, fontsize=9)
        ax.set_yticks(range(len(SUB_MAGS))); ax.set_yticklabels([f'{h}×' for h in SUB_MAGS], fontsize=8)
        ax.set_zlim(0, _zmax43); ax.set_xlim(FIG43_COVERAGE[0], 1.0)
        ax.zaxis.set_major_locator(_MaxN43(nbins=4, steps=[1, 2, 2.5, 5, 10]))
        ax.zaxis.set_major_formatter(_Fmt43('%.2f'))
        ax.tick_params(labelsize=8, pad=-1); ax.tick_params(axis='z', pad=3)
        ax.view_init(elev=24, azim=-62)
        ax.set_title(m, fontsize=11, fontweight='bold')
    plt.suptitle('Subtype pipeline: selective-risk landscape - error among the most confident images, per held-out '
                 'magnification (mean over seeds)', y=1.02, fontsize=12.5, fontweight='bold')
    save_fig(fig, 'fig43_subtype_selective_risk_landscape', dpi=170)
    plt.show()

    _at43 = {c: [] for c in (0.5, 0.8, 1.0)}
    for m in SUB_ENCODERS:
        for c in _at43:
            k = int(np.argmin(np.abs(FIG43_COVERAGE - c)))
            _at43[c].append(np.nanmean(_cube43[m][:, k]))
    subtype_risk_landscape = pd.DataFrame(_at43, index=SUB_ENCODERS).rename(
        columns=lambda c: f'risk @ {int(c * 100)}% coverage')
    subtype_risk_landscape.to_csv(f'{OUTPUT_DIR}/tables/subtype_risk_landscape_summary.csv')
    print('Mean over held-out magnifications (the 80% column matches T15 up to interpolation):')
    print(subtype_risk_landscape.round(3).to_string())

In [ ]:
# ---------- Fig 44: error towers - where the subtype pipeline's mistakes pile up ----------
# The 8 x 8 confusion of each encoder (summed over held-out magnifications and seeds), row-normalised so each true
# subtype sums to 1, with the diagonal removed: every tower is the share of a true subtype sent to one wrong
# subtype. Colour = error source, as in Fig 33:
#   red    malignant image sent to the benign branch (a missed cancer)
#   orange benign image sent to the malignant branch (a false alarm)
#   blue   right branch, wrong subtype
_ERR44 = {'missed': '#B00020', 'false_alarm': '#E69F00', 'subtype': '#0072B2'}
subtype_error_towers = None

if not globals().get('SUB_HAS_E2E') or not sub_confusion:
    print('Skipped Fig 44 - no subtype confusion matrices (section 4b).')
else:
    _cm44 = {}
    for (m, pr, sm, h), cm in sub_confusion.items():
        if (pr, sm) == SUB_MAIN and m in SUB_ENCODERS:
            _cm44[m] = _cm44.get(m, np.zeros((8, 8))) + cm
    _rows44 = []
    for m, cm in _cm44.items():
        r = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
        for t in range(8):
            for p in range(8):
                if t == p:
                    continue
                kind = 'missed' if (t >= 4 and p < 4) else 'false_alarm' if (t < 4 and p >= 4) else 'subtype'
                _rows44.append({'method': m, 'true': SUB_CODES[t], 'predicted': SUB_CODES[p], 'share': r[t, p],
                                'count': int(cm[t, p]), 'error_source': kind})
    subtype_error_towers = pd.DataFrame(_rows44)
    subtype_error_towers.to_csv(f'{OUTPUT_DIR}/tables/subtype_error_towers.csv', index=False)
    _zmax44 = max(0.05, float(subtype_error_towers.share.max()) * 1.08)
    _enc44 = [m for m in SUB_ENCODERS if m in _cm44]

    fig = plt.figure(figsize=(4.7 * len(_enc44), 5.4))
    for i, m in enumerate(_enc44):
        ax = fig.add_subplot(1, len(_enc44), i + 1, projection='3d')
        d = subtype_error_towers[subtype_error_towers.method == m]
        xs = d.predicted.map(SUB_CODES.index).to_numpy(dtype=float)
        ys = d.true.map(SUB_CODES.index).to_numpy(dtype=float)
        ax.bar3d(xs - 0.35, ys - 0.35, np.zeros(len(d)), 0.7, 0.7, d.share.to_numpy(),
                 color=[_ERR44[k] for k in d.error_source], edgecolor='black', linewidth=0.2, alpha=0.9, shade=True)
        ax.set_xticks(range(8)); ax.set_xticklabels(SUB_CODES, fontsize=7.5)
        ax.set_yticks(range(8)); ax.set_yticklabels(SUB_CODES, fontsize=7.5)
        ax.set_xlabel('Predicted', labelpad=-2, fontsize=9); ax.set_ylabel('True', labelpad=-2, fontsize=9)
        ax.set_zlim(0, _zmax44); ax.set_zlabel('Share of the true subtype', labelpad=4, fontsize=8.5)
        ax.tick_params(axis='z', labelsize=7.5, pad=1)
        ax.view_init(elev=30, azim=-128)
        top = d.sort_values('share', ascending=False).iloc[0]
        ax.set_title(f'{m}\nlargest: {top.true} -> {top.predicted} ({100 * top.share:.0f}%)', fontsize=10.5,
                     fontweight='bold')
    _h44 = [plt.Rectangle((0, 0), 1, 1, color=c) for c in _ERR44.values()]
    fig.legend(_h44, ['missed cancer (malignant -> benign branch)', 'false alarm (benign -> malignant branch)',
                      'wrong subtype in the right branch'], loc='lower center', ncol=3, frameon=False, fontsize=10)
    plt.suptitle('Subtype pipeline: error towers - row-normalised confusion without the diagonal '
                 f'(summed over held-out magnifications and seeds; {SUB_SETTING_TXT})', y=1.0, fontsize=12.5,
                 fontweight='bold')
    fig.subplots_adjust(left=0.02, right=0.98, bottom=0.1, top=0.86, wspace=0.08)
    save_fig(fig, 'fig44_subtype_error_towers', dpi=170)
    plt.show()

    print('Three largest error towers per encoder (share of the true subtype):')
    for m in _enc44:
        d = subtype_error_towers[subtype_error_towers.method == m].sort_values('share', ascending=False).head(3)
        print(f'  {m:<20} ' + '; '.join(f'{r_.true}->{r_.predicted} {100 * r_.share:.1f}% ({r_.error_source})'
                                         for _, r_ in d.iterrows()))
    _src44 = subtype_error_towers.groupby(['method', 'error_source'])['count'].sum().unstack(fill_value=0)
    print('\nMisclassified images per error source (all held-out magnifications and seeds):')
    print(_src44.reindex(_enc44).to_string())

In [ ]:
# ---------- Fig 45: subtype capability radar ----------
# Fig 27 for the subtype task. Every axis is scaled between a fixed floor (worse end) and ceiling (better end),
# printed under its name; no min-max scaling across encoders, which would pin the weakest one to zero.
# Pipeline compute counts what one image costs at inference: a task-trained encoder runs two full networks
# (stage 1, then the routed stage-2 model); Phikon and MagScan-Mamba run the frozen backbone once and two
# small heads on its output.
SUB_RADAR_AXES = [   # (key, label, floor, ceiling)
    ('f1',      'Accuracy\n8-subtype macro-F1',        0.40, 0.75),
    ('stage1',  'Detection\nstage-1 accuracy',         0.85, 0.97),
    ('ece',     'Calibration\nECE',                    0.20, 0.05),
    ('risk80',  'Selective risk\nrisk @ 80% cov.',     0.45, 0.10),
    ('missed',  'Missed cancers\nshare of malignant',  0.15, 0.01),
    ('seed_sd', 'Seed stability\nSD of macro-F1',      0.15, 0.01),
    ('flops',   'Pipeline compute\nGFLOPs per image',  20.0, 2.0),
]
subtype_radar_table = None


def _unit45(value, floor, ceiling):
    if value is None or not np.isfinite(value):
        return np.nan
    return float(np.clip((value - floor) / (ceiling - floor), 0.0, 1.0))


if not globals().get('SUB_HAS_E2E'):
    print('Skipped Fig 45 - no complete subtype pipelines (section 4b).')
else:
    _d45 = sub_main[sub_main.method.isin(SUB_ENCODERS)]
    _r45 = {}
    for m in SUB_ENCODERS:
        g = _d45[_d45.method == m]
        _r45[m] = {'f1': g.macro_f1.mean(), 'stage1': g.stage1_accuracy.mean(), 'ece': g.ece.mean(),
                   'risk80': g.risk_at_coverage_80.mean(), 'missed': g.missed_malignant_rate.mean(),
                   'seed_sd': g.groupby('held_out_mag').macro_f1.std().mean()}
    # ---- pipeline compute (GFLOPs at 224 x 224) ----
    _backbone45 = float(globals().get('PHIKON_FLOPS_G', 17.60))           # frozen ViT-B/16, Fig 27 / T7 constant
    _flops45 = {}
    if globals().get('compute_report') is not None:
        for _, r_ in compute_report.iterrows():
            _flops45[DISPLAY.get(r_['model'], r_['model'])] = 2.0 * float(r_['flops_g_at_224'])
    _flops45['Phikon (frozen FM)'] = _backbone45                            # two linear heads: negligible
    _mhead = globals().get('MAGSCAN_HEAD_GFLOPS_OFFLINE', {}).get('magscan_single', 0.212)
    _flops45['MagScan-Mamba'] = _backbone45 + 2 * _mhead
    for m in _r45:
        _r45[m]['flops'] = _flops45.get(m, np.nan)
    _used45 = [a for a in SUB_RADAR_AXES if all(np.isfinite(_r45[m].get(a[0], np.nan)) for m in _r45)]
    _drop45 = [a[1].replace('\n', ' ') for a in SUB_RADAR_AXES if a not in _used45]
    if _drop45:
        print('axes dropped (a value is missing for some encoder):', '; '.join(_drop45))

    _methods45 = [m for m in SUB_ENCODERS if m in _r45]
    subtype_radar_table = pd.DataFrame([{'method': m, **{k: _r45[m][k] for k, *_ in _used45}} for m in _methods45]
                                       ).set_index('method')
    _scaled45 = pd.DataFrame([[_unit45(_r45[m][k], lo, hi) for k, _, lo, hi in _used45] for m in _methods45],
                             index=_methods45, columns=[k for k, *_ in _used45])
    subtype_radar_table.join(_scaled45, rsuffix='_scaled').to_csv(f'{OUTPUT_DIR}/tables/fig45_subtype_radar_values.csv')
    print('Fig 45 raw values:')
    print(subtype_radar_table.round(4).to_string())
    print('\nscaled to [0, 1] (1 = the ceiling printed on the axis):')
    print(_scaled45.round(3).to_string())

    _best_f1 = subtype_radar_table.f1.idxmax()
    _best_cal = subtype_radar_table.ece.idxmin()
    if _best_cal == _best_f1:
        _best_cal = subtype_radar_table.risk80.drop(index=_best_f1).idxmin()
        _pair_title = '(b) best accuracy vs lowest selective risk'
    else:
        _pair_title = '(b) the accuracy winner vs the calibration winner'
    _pair45 = [m for m in _methods45 if m in (_best_f1, _best_cal)]
    if 'MagScan-Mamba' in _methods45 and 'MagScan-Mamba' not in _pair45:
        _pair45.append('MagScan-Mamba'); _pair_title += ', with MagScan-Mamba'

    _n45 = len(_used45)
    _th = np.linspace(0, 2 * np.pi, _n45, endpoint=False)
    _cl = np.concatenate([_th, _th[:1]])
    fig, axes = plt.subplots(1, 2, figsize=(15.2, 8.2), subplot_kw=dict(projection='polar'))
    for ax, (title, subset) in zip(axes, [('(a) all encoders', _methods45), (_pair_title, _pair45)]):
        ax.set_theta_offset(np.pi / 2); ax.set_theta_direction(-1)
        ax.set_ylim(0, 1.0); ax.set_yticks([0.25, 0.5, 0.75, 1.0]); ax.set_yticklabels([])
        ax.set_xticks(_th); ax.set_xlim(0, 2 * np.pi)
        ax.set_xticklabels([f'{lab}\n{lo:g} → {hi:g}' for _k, lab, lo, hi in _used45], fontsize=9)
        ax.tick_params(axis='x', pad=14)
        ax.grid(color='0.75', linewidth=0.6); ax.set_axisbelow(True)
        ax.spines['polar'].set_color('0.7')
        for m in subset:
            vals = _scaled45.loc[m].to_numpy(dtype=float)
            vc = np.concatenate([vals, vals[:1]])
            c = SUB_COLORS.get(m, '#444444')
            ax.plot(_cl, vc, color=c, linewidth=2.8 if m == 'MagScan-Mamba' else 2.0, zorder=3)
            ax.fill(_cl, vc, color=c, alpha=0.12, zorder=2)
            ax.scatter(_th, vals, color=c, s=40, edgecolor='white', linewidth=0.8, zorder=4)
        ax.set_title(title, fontsize=11.5, fontweight='bold', pad=62)
    _hd45 = [plt.Line2D([], [], color=SUB_COLORS.get(m, '#444444'), linewidth=2.4, marker='o', markersize=6,
                        markeredgecolor='white', label=m) for m in _methods45]
    fig.legend(handles=_hd45, loc='lower center', bbox_to_anchor=(0.5, 0.075), ncol=len(_methods45), frameon=False,
               fontsize=10.5, columnspacing=2.0)
    fig.suptitle(f'Subtype pipeline capability profile ({SUB_SETTING_TXT}): higher is better on every axis',
                 fontsize=13.5, fontweight='bold', y=0.975)
    fig.text(0.5, 0.042,
             'Each axis is linearly scaled between the two values printed under its name (inner = worse, outer = '
             'better); the outer ring is the stated better end, not a perfect score.\nMeans over held-out '
             'magnifications and seeds; seed stability = SD of macro-F1 over seeds, averaged over magnifications; '
             'compute = GFLOPs one image costs through the two stages.',
             ha='center', fontsize=9, color='0.32', linespacing=1.5)
    fig.subplots_adjust(top=0.76, bottom=0.255, left=0.07, right=0.93, wspace=0.42)
    save_fig(fig, 'fig45_subtype_capability_radar')
    plt.show()

    print('\nper-axis leader (within 0.02 of the best scaled value = tied):')
    for k, lab, lo, hi in _used45:
        best = _scaled45[k].max()
        tied = [m for m in _methods45 if _scaled45.loc[m, k] >= best - 0.02]
        print(f"  {lab.split(chr(10))[0].lower():<17} {'tie: ' if len(tied) > 1 else ''}"
              + ', '.join(f'{m} {subtype_radar_table.loc[m, k]:.4g}' for m in tied))

In [ ]:
# ---------- Fig 46: saliency by depth of the malignant-subtype models, one lobular carcinoma tile ----------
# Fig 19 for the subtype task. The tile is the first lobular carcinoma (LC, the hardest subtype) among the test
# images at held-out SUB3D_MAG for seed SUB3D_SEED; every encoder's stage-2 malignant model is explained for the
# TRUE class, so the maps show the evidence each model finds for lobular carcinoma (its own prediction is
# printed). Four depths per encoder:
#   ConvNeXt-T, Swin-T, MedMamba : Grad-CAM++ at the four encoder stages (plain Grad-CAM where ++ degenerates)
#   Phikon probe                 : Attention Rollout of the frozen backbone to blocks 3, 6, 9, 12. The backbone is
#                                  frozen, so these maps are the same for every task; the probe only reads CLS.
#   MagScan-Mamba                : rollout of its frozen backbone to blocks 6 and 12, then token Grad-CAM after
#                                  each of its two Mamba layers (7 x 7 token grid): the part the head learned.
# Needs the section 2h saliency backend (GPU, Fig 11 cell) and BreakHis.
SUB3D_MAG, SUB3D_SEED, SUB3D_STAGE, SUB3D_SUBTYPE = 400, 13, 'malignant_subtype', 'LC'
SUB3D_MODEL_KEYS = {'ConvNeXt-T': 'convnext_tiny', 'Swin-T': 'swin_tiny', 'MedMamba': 'medmamba'}
subtype_layer_stats = None

try:
    import torch
    from torch import nn
except ImportError:
    torch = None

if torch is not None:
    # ---- MagScan-Mamba's head exactly as subtype_hierarchical.ipynb defines it (cell 06b), pure-torch scan ----
    import math as _math46

    def _ms_selective_scan(u, delta, A, B, C, D=None, z=None, delta_bias=None, delta_softplus=False):
        u, delta = u.float(), delta.float()
        if delta_bias is not None:
            delta = delta + delta_bias[..., None].float()
        if delta_softplus:
            delta = torch.nn.functional.softplus(delta)
        b, d, l = u.shape
        B = B.float().repeat_interleave(d // B.shape[1], dim=1)
        C = C.float().repeat_interleave(d // C.shape[1], dim=1)
        dA = torch.exp(torch.einsum('bdl,dn->bdln', delta, A.float()))
        dBu = torch.einsum('bdl,bdnl,bdl->bdln', delta, B, u)
        h, ys = u.new_zeros(b, d, A.shape[1]), []
        for i in range(l):
            h = dA[:, :, i] * h + dBu[:, :, i]
            ys.append(torch.einsum('bdn,bdn->bd', h, C[:, :, :, i]))
        y = torch.stack(ys, dim=2)
        if D is not None:
            y = y + u * D.float()[..., None]
        return y

    class MSMambaMixer(nn.Module):
        def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
            super().__init__()
            self.d_inner, self.d_state = expand * d_model, d_state
            self.dt_rank = _math46.ceil(d_model / 16)
            self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=False)
            self.conv1d = nn.Conv1d(self.d_inner, self.d_inner, d_conv, groups=self.d_inner, padding=d_conv - 1)
            self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
            self.x_proj = nn.Linear(self.d_inner, self.dt_rank + 2 * d_state, bias=False)
            self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)
            self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)))
            self.D = nn.Parameter(torch.ones(self.d_inner))

        def forward(self, x):
            L = x.shape[1]
            u, z = self.in_proj(x).chunk(2, dim=-1)
            u = torch.nn.functional.silu(self.conv1d(u.transpose(1, 2))[..., :L])
            dt, Bm, Cm = torch.split(self.x_proj(u.transpose(1, 2)), [self.dt_rank, self.d_state, self.d_state], dim=-1)
            dt = (dt @ self.dt_proj.weight.t()).transpose(1, 2)
            u = _ms_selective_scan(u.float().contiguous(), dt.float().contiguous(), -torch.exp(self.A_log.float()),
                                   Bm.transpose(1, 2).unsqueeze(1).float().contiguous(),
                                   Cm.transpose(1, 2).unsqueeze(1).float().contiguous(),
                                   self.D.float(), z=None, delta_bias=self.dt_proj.bias.float(), delta_softplus=True)
            return self.out_proj(u.transpose(1, 2) * torch.nn.functional.silu(z))

    class MSScanMixer(nn.Module):
        def __init__(self, d_model, grid):
            super().__init__()
            self.core = MSMambaMixer(d_model)
            idx = torch.arange(grid * grid).reshape(grid, grid)
            for k, o in enumerate([idx.flatten(), idx.flatten().flip(0), idx.t().flatten(), idx.t().flatten().flip(0)]):
                self.register_buffer(f'order{k}', o, persistent=False)
                self.register_buffer(f'inverse{k}', torch.argsort(o), persistent=False)

        def forward(self, x):
            return sum(self.core(x[:, getattr(self, f'order{k}')])[:, getattr(self, f'inverse{k}')] for k in range(4)) / 4

    class MSMagScanHead(nn.Module):
        def __init__(self, in_dim, grid, num_classes, d=256, layers=2, dropout=0.1):
            super().__init__()
            self.inp = nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, d))
            self.scale_emb = nn.Parameter(torch.zeros(1, 1, 1, d))
            self.pos_emb = nn.Parameter(torch.zeros(1, 1, grid * grid, d))
            self.blocks = nn.ModuleList([nn.ModuleDict({'n1': nn.LayerNorm(d), 'mix': MSScanMixer(d, grid), 'n2': nn.LayerNorm(d),
                                                        'mlp': nn.Sequential(nn.Linear(d, 2 * d), nn.GELU(),
                                                                             nn.Dropout(dropout), nn.Linear(2 * d, d))})
                                         for _ in range(layers)])
            self.norm, self.drop, self.fc = nn.LayerNorm(d), nn.Dropout(dropout), nn.Linear(d, num_classes)

        def forward(self, tokens, keep=None):                    # tokens: (B, 1, 1 + grid^2, in_dim)
            x = (self.inp(tokens[:, :, 1:].float()) + self.scale_emb + self.pos_emb).flatten(1, 2)
            for blk in self.blocks:
                x = x + blk['mix'](blk['n1'](x))
                x = x + blk['mlp'](blk['n2'](x))
                if keep is not None:
                    x.retain_grad(); keep.append(x)
            return self.fc(self.drop(self.norm(x).mean(dim=1)))

    def magscan_tokens(fm, tensor, pool=2):
        """(1, 1, 1 + 49, 768) tokens as the subtype notebook built them: CLS + 14 x 14 patches pooled to 7 x 7."""
        with torch.no_grad():
            h = fm(pixel_values=tensor).last_hidden_state                               # (1, 197, 768)
        g = int(round((h.shape[1] - 1) ** 0.5))
        pooled = torch.nn.functional.avg_pool2d(h[:, 1:].reshape(1, g, g, -1).permute(0, 3, 1, 2), pool)
        return torch.cat([h[:, :1], pooled.flatten(2).transpose(1, 2)], dim=1).unsqueeze(1)

    def magscan_token_cams(head, tokens, target, size=224):
        """Grad-CAM on the token sequence after each Mamba layer: weights = mean gradient per channel,
        map = ReLU(sum over channels), reshaped to the 7 x 7 grid and upsampled. Returns (maps, probs)."""
        keep = []
        head.eval(); head.zero_grad(set_to_none=True)
        logits = head(tokens, keep=keep)                                                # (1, C)
        logits[0, int(target)].backward()
        maps = []
        for act in keep:
            w = act.grad[0].mean(dim=0)                                                 # (d,)
            cam = torch.relu((act[0] * w).sum(dim=-1)).detach()                         # (49,)
            g = int(round(cam.numel() ** 0.5))
            cam = torch.nn.functional.interpolate(cam.reshape(1, 1, g, g), size=(size, size), mode='bilinear',
                                                  align_corners=False)[0, 0]
            maps.append(((cam - cam.min()) / (cam.max() - cam.min() + 1e-8)).cpu().numpy())
        return maps, torch.softmax(logits.detach(), dim=1)[0].cpu().numpy()


def _draw_subtype_stack(name, show, maps, labels, used, subtitle, fname):
    """Fig 19's layout: input + four overlays on top; 3D stack (oblique, top-down), the two deepest maps as
    surfaces and the attribution statistics below."""
    grids = [resample_map(m) for m in maps]
    stats = [attribution_stats(m) for m in maps]
    flags = [map_is_degenerate(m) for m in maps]
    fig = plt.figure(figsize=(19, 7.4))
    gs = fig.add_gridspec(2, 5, height_ratios=[1.0, 1.25], hspace=0.22, wspace=0.3)
    ax = fig.add_subplot(gs[0, 0]); ax.imshow(show); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'Input - true {SUB_NAMES[SUB3D_SUBTYPE]}', fontsize=10.5, fontweight='bold')
    for k, (m, lab) in enumerate(zip(maps, labels)):
        ax = fig.add_subplot(gs[0, k + 1])
        ax.imshow(show_cam_on_image(show, m.astype(np.float32), use_rgb=True)); ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f'{lab}\n{used[k]}', fontsize=10, fontweight='bold')
        bad, why = flags[k]
        ax.set_xlabel(f'map {why} - not interpretable' if bad else
                      f"top10 {stats[k]['top10_share']:.2f} | border {stats[k]['border_ratio']:.2f}",
                      fontsize=8.5, color='#d62728' if bad else '0.3', fontweight='bold' if bad else 'normal')
    ticks = [l.split('\n')[0] for l in labels]
    for col, (elev, azim, title) in enumerate([(26, -58, 'Layer stack - oblique'), (78, -90, 'Layer stack - top-down')]):
        ax = fig.add_subplot(gs[1, col], projection='3d')
        for k, g in enumerate(grids):
            surface3d(ax, 0.85 * g, cmap=STAGE_CMAPS[k % len(STAGE_CMAPS)], zlabel='', offset=float(k), alpha=0.85, stride=2)
        if elev < 60:
            ax.set_zticks(range(len(grids))); ax.set_zticklabels(ticks, fontsize=7.5)
        else:
            ax.set_zticks([])
        ax.set_zlabel(''); ax.set_zlim(0, len(grids) + 0.1); ax.view_init(elev=elev, azim=azim)
        ax.set_title(title, fontsize=10.5, fontweight='bold')
    for col, k in enumerate([len(grids) - 2, len(grids) - 1], start=2):
        ax = fig.add_subplot(gs[1, col], projection='3d')
        surface3d(ax, grids[k], cmap=STAGE_CMAPS[k % len(STAGE_CMAPS)], zlabel='Attribution', zmax=1.0, stride=1)
        ax.set_title(labels[k].replace('\n', ' '), fontsize=10.5, fontweight='bold')
    ax = fig.add_subplot(gs[1, 4])
    _x = np.arange(len(labels)); _w = 0.27
    for j, (key, lbl, col) in enumerate([('top10_share', 'Top-10% share', '#0072B2'),
                                         ('border_ratio', 'Border / mean', '#D55E00'),
                                         ('centroid_offset', 'Centroid offset', '#009E73')]):
        ax.bar(_x + (j - 1) * _w, [s[key] for s in stats], width=_w, color=col, edgecolor='black', linewidth=0.4, label=lbl)
    ax.axhline(1.0, color='0.4', linestyle='--', linewidth=0.8)
    ax.set_xticks(_x); ax.set_xticklabels(ticks, fontsize=8, rotation=12)
    ax.set_title('Attribution statistics', fontsize=10.5, fontweight='bold')
    _top = max(max(np.nan_to_num(s[k]) for s in stats for k in ('top10_share', 'border_ratio', 'centroid_offset')), 1.0)
    ax.set_ylim(0, 1.6 * _top); ax.legend(fontsize=7.5, frameon=False, loc='upper left')
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.suptitle(f'{name} - {subtitle}', y=1.0, fontsize=12.5, fontweight='bold')
    save_fig(fig, fname, dpi=170)
    plt.show()
    return [{'method': name, 'layer': l.replace('\n', ' '), 'saliency': u, 'degenerate': bool(b), 'reason': w, **s}
            for l, u, (b, w), s in zip(labels, used, flags, stats)]


def _sub3d_tile(mag, seed, subtype):
    """(PIL image, file name) of the first test image of `subtype` at held-out `mag`, from section 4b's arrays."""
    keys = [k for k in SUB_IMAGES if (k[1], k[2]) == SUB_MAIN and k[3] == str(mag) and k[4] == seed]
    if not keys:
        return None, None
    a = SUB_IMAGES[keys[0]]
    rows = a['index'][a['y8'] == SUB_CODES.index(subtype)]
    if not len(rows):
        return None, None
    man = pd.read_csv(Path(a['run_dirs']['detect']).parent.parent / 'breakhis_manifest.csv')
    if '_SUB_LOCAL' not in globals():
        globals()['_SUB_LOCAL'] = {p.name: str(p) for p in Path(BREAKHIS_ROOT).rglob('SOB_*.png')}
    fname = Path(str(man.path.iloc[int(rows[0])])).name
    from PIL import Image as _PIL46
    return _PIL46.open(_SUB_LOCAL[fname]).convert('RGB'), fname


if not globals().get('SUB_HAS_E2E'):
    print('Skipped Fig 46 - no complete subtype pipelines (section 4b).')
elif globals().get('SAL3D') is None or torch is None or not globals().get('HAS_IMAGES'):
    print('Skipped Fig 46 - needs the section 2h saliency backend (GPU, Fig 11 cell) and BreakHis attached.')
else:
    _pil46, _fname46 = _sub3d_tile(SUB3D_MAG, SUB3D_SEED, SUB3D_SUBTYPE)
    if _pil46 is None:
        print(f'Skipped Fig 46 - no {SUB3D_SUBTYPE} test image at held-out {SUB3D_MAG}x, seed {SUB3D_SEED}.')
    else:
        _dev = SAL3D['device']
        _show46 = np.asarray(_pil46.resize((224, 224)), dtype=np.float32) / 255.0
        _tensor46 = SAL3D['to_tensor'](_pil46)
        _target46 = SUB_CODES.index(SUB3D_SUBTYPE) - 4                       # index inside the malignant model
        _mal = SUB_CODES[4:]
        print(f'Tile: {_fname46} (held-out {SUB3D_MAG}x, seed {SUB3D_SEED}); explained class: {SUB3D_SUBTYPE}')
        _stats46, _fm46 = [], None
        for m in SUB_ENCODERS:
            key = [k for k in SUB_IMAGES if k[0] == m and (k[1], k[2]) == SUB_MAIN and k[3] == str(SUB3D_MAG)
                   and k[4] == SUB3D_SEED]
            if not key:
                print(f'  {m}: no pipeline at held-out {SUB3D_MAG}x seed {SUB3D_SEED} - skipped'); continue
            rdir = Path(SUB_IMAGES[key[0]]['run_dirs'][SUB3D_STAGE])
            try:
                if m in SUB3D_MODEL_KEYS:
                    if not (rdir / 'best.pt').exists():
                        print(f'  {m}: best.pt missing in {rdir.name} - skipped'); continue
                    model = build_from_scratch(SUB3D_MODEL_KEYS[m], num_classes=4)
                    model.load_state_dict(torch.load(rdir / 'best.pt', map_location='cpu'))
                    model = model.to(_dev).eval()
                    with torch.no_grad():
                        probs = torch.softmax(model(_tensor46), dim=1)[0].cpu().numpy()
                    maps, _, _, used = SAL3D['stage_cams'](model, SUB3D_MODEL_KEYS[m], _tensor46, target_class=_target46)
                    labels = STAGE_LABELS
                    del model
                else:
                    if _fm46 is None:
                        _fm46 = SAL3D['load_phikon']()
                    if m == 'MagScan-Mamba':
                        saved = torch.load(rdir / 'head.pt', map_location='cpu')
                        head = MSMagScanHead(768, 7, int(saved['num_classes']))
                        head.load_state_dict(saved['state_dict'])
                        head = head.to(_dev)
                        roll, _, _, _ = SAL3D['phikon_depth_maps'](_fm46, lambda z: z[:, :2], _tensor46, depths=(6, 12))
                        tcams, probs = magscan_token_cams(head, magscan_tokens(_fm46, _tensor46), _target46)
                        maps = roll + tcams
                        labels = ['Backbone 6\n(frozen)', 'Backbone 12\n(frozen)', 'Mamba layer 1\n(head)',
                                  'Mamba layer 2\n(head)']
                        used = ['Attention Rollout', 'Attention Rollout', 'token Grad-CAM', 'token Grad-CAM']
                        del head
                    else:                                                           # Phikon probe
                        saved = torch.load(rdir / 'probe.pt', map_location='cpu')
                        head = torch.nn.Linear(int(saved['feature_dim']), int(saved['num_classes']))
                        head.load_state_dict({k.replace('head.', ''): v for k, v in saved['state_dict'].items()})
                        head = head.to(_dev).eval()
                        maps, _, _, used = SAL3D['phikon_depth_maps'](_fm46, head, _tensor46)
                        with torch.no_grad():
                            probs = torch.softmax(head(_fm46(pixel_values=_tensor46).last_hidden_state[:, 0]), dim=1)[0].cpu().numpy()
                        labels = [f'Block {d}\n(frozen)' for d in PHIKON_DEPTHS]
                        del head
                SAL3D['torch'].cuda.empty_cache()
            except Exception as exc:
                print(f'  {m}: FAILED - {repr(exc)[:200]}'); continue
            pred = _mal[int(np.argmax(probs))]
            sub = (f'saliency by depth for {SUB_NAMES[SUB3D_SUBTYPE]} (malignant-subtype model, held-out {SUB3D_MAG}×, '
                   f'seed {SUB3D_SEED}; its prediction {pred} p={float(np.max(probs)):.2f}, '
                   f'P({SUB3D_SUBTYPE})={float(probs[_target46]):.2f})')
            _stats46 += _draw_subtype_stack(m, _show46, maps, labels, used, sub,
                                            f"fig46_subtype_layer_stack_{m.split(' ')[0].replace('-', '')}")
        if _fm46 is not None:
            del _fm46; SAL3D['torch'].cuda.empty_cache()
        if _stats46:
            subtype_layer_stats = pd.DataFrame(_stats46)
            subtype_layer_stats.to_csv(f'{OUTPUT_DIR}/tables/subtype_saliency_layer_statistics.csv', index=False)
            print('\nAttribution statistics per encoder and depth:')
            print(subtype_layer_stats.round(3).to_string(index=False))

In [ ]:
# ---------- Fig 47: one lobular carcinoma patient across 40x, 100x, 200x and 400x ----------
# Fig 21 for the subtype task. With the image-level LOMO split every image at the held-out magnification is a
# test image, so one patient (slide) can be followed across all four; each panel is scored by the pipeline for
# which that magnification was held out, i.e. never seen in training. Per encoder and magnification, over the
# patient's tiles and the seeds: the mean path probability of the true subtype (3D bars), the share of tiles
# named correctly and the most frequent prediction (heat map). The tile row needs BreakHis; the rest is CPU.
SUB_TRAJ_SUBTYPE = 'LC'
subtype_patient_trajectory = None

if not globals().get('SUB_HAS_E2E') or not globals().get('SUB_IMAGES'):
    print('Skipped Fig 47 - no per-image subtype predictions (section 4b).')
else:
    _any47 = next(iter(SUB_IMAGES.values()))
    _man47 = pd.read_csv(Path(_any47['run_dirs']['detect']).parent.parent / 'breakhis_manifest.csv')
    _cand = (_man47[_man47.subtype == SUB_TRAJ_SUBTYPE].groupby('patient_id')
             .agg(n=('path', 'size'), mags=('magnification', 'nunique')))
    _cand = _cand[_cand.mags == len(SUB_MAGS)]
    if _cand.empty:
        print(f'Skipped Fig 47 - no {SUB_TRAJ_SUBTYPE} patient has images at all {len(SUB_MAGS)} magnifications.')
    else:
        _pat = _cand.n.idxmax()                                          # deterministic: the most images
        _rows_pat = set(_man47.index[_man47.patient_id == _pat])
        _ytrue = SUB_CODES.index(SUB_TRAJ_SUBTYPE)
        print(f'Patient {_pat} ({SUB_NAMES[SUB_TRAJ_SUBTYPE]}): '
              + ', '.join(f'{h}x {int(((_man47.patient_id == _pat) & (_man47.magnification == h)).sum())}'
                          for h in SUB_MAGS) + ' tiles')
        _tr = []
        for m in SUB_ENCODERS:
            for h in SUB_MAGS:
                p_true, correct, preds = [], [], []
                for (mm, pr, sm, hh, s), a in SUB_IMAGES.items():
                    if mm != m or (pr, sm) != SUB_MAIN or hh != str(h):
                        continue
                    sel = np.isin(a['index'], list(_rows_pat))
                    if not sel.any():
                        continue
                    p_true.append(float(a['p8'][sel, _ytrue].mean()))
                    correct.append(float((a['pred'][sel] == _ytrue).mean()))
                    preds += [SUB_CODES[k] for k in a['pred'][sel]]
                if p_true:
                    mode = pd.Series(preds).value_counts()
                    _tr.append({'method': m, 'held_out_mag': h, 'seeds': len(p_true), 'tiles_per_seed': int(len(preds) / len(p_true)),
                                'p_true_mean': float(np.mean(p_true)), 'share_correct': float(np.mean(correct)),
                                'most_frequent_prediction': mode.index[0], 'its_share': float(mode.iloc[0] / len(preds))})
        subtype_patient_trajectory = pd.DataFrame(_tr)
        subtype_patient_trajectory.to_csv(f'{OUTPUT_DIR}/tables/subtype_patient_trajectory.csv', index=False)

        _tiles47 = {}
        if globals().get('HAS_IMAGES'):
            from PIL import Image as _PIL47
            if '_SUB_LOCAL' not in globals():
                globals()['_SUB_LOCAL'] = {p.name: str(p) for p in Path(BREAKHIS_ROOT).rglob('SOB_*.png')}
            for h in SUB_MAGS:
                f = _man47[(_man47.patient_id == _pat) & (_man47.magnification == h)].path.iloc[0]
                local = _SUB_LOCAL.get(Path(str(f)).name)
                if local:
                    _tiles47[h] = np.asarray(_PIL47.open(local).convert('RGB').resize((336, 224)))
        _enc47 = [m for m in SUB_ENCODERS if m in set(subtype_patient_trajectory.method)]
        fig = plt.figure(figsize=(19, 10.2 if _tiles47 else 6.2))
        gs = fig.add_gridspec(2 if _tiles47 else 1, 4, height_ratios=[0.8, 1.35] if _tiles47 else [1],
                              hspace=0.12, wspace=0.25)
        r0 = 0
        if _tiles47:
            for j, h in enumerate(SUB_MAGS):
                ax = fig.add_subplot(gs[0, j])
                if h in _tiles47:
                    ax.imshow(_tiles47[h])
                ax.set_xticks([]); ax.set_yticks([])
                ax.set_title(f'{h}×  (scored by the pipeline that held {h}× out)', fontsize=10.5, fontweight='bold')
            r0 = 1
        ax = fig.add_subplot(gs[r0, 0:2], projection='3d')
        _P = subtype_patient_trajectory.pivot(index='method', columns='held_out_mag', values='p_true_mean').reindex(
            index=_enc47, columns=SUB_MAGS)
        xs, ys = np.meshgrid(np.arange(len(SUB_MAGS)), np.arange(len(_enc47)))
        dz = np.nan_to_num(_P.to_numpy(dtype=float)).ravel()
        ax.bar3d(xs.ravel() - 0.3, ys.ravel() - 0.3, np.zeros_like(dz), 0.6, 0.6, dz,
                 color=[SUB_COLORS.get(_enc47[i], '#666666') for i in ys.ravel()], edgecolor='black', linewidth=0.3,
                 alpha=0.92, shade=True)
        ax.set_xticks(range(len(SUB_MAGS))); ax.set_xticklabels([f'{h}×' for h in SUB_MAGS], fontsize=9)
        ax.set_yticks(range(len(_enc47))); ax.set_yticklabels([SUB_SHORT.get(m, m) for m in _enc47], fontsize=8.5)
        ax.set_zlim(0, 1); ax.set_zlabel(f'P({SUB_TRAJ_SUBTYPE}), mean over tiles and seeds', labelpad=6)
        ax.set_xlabel('Magnification (held out)', labelpad=8)
        ax.view_init(elev=24, azim=-58)
        ax.set_title(f'(a) Probability the pipeline gives the true subtype ({SUB_TRAJ_SUBTYPE})', fontsize=11,
                     fontweight='bold')
        ax = fig.add_subplot(gs[r0, 2:4])
        _S = subtype_patient_trajectory.pivot(index='method', columns='held_out_mag', values='share_correct').reindex(
            index=_enc47, columns=SUB_MAGS)
        _M = subtype_patient_trajectory.pivot(index='method', columns='held_out_mag', values='most_frequent_prediction').reindex(
            index=_enc47, columns=SUB_MAGS)
        im = ax.imshow(_S.to_numpy(dtype=float), cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
        for i in range(len(_enc47)):
            for j in range(len(SUB_MAGS)):
                v = _S.iloc[i, j]
                if np.isfinite(v):
                    ax.text(j, i, f'{100 * v:.0f}% right\nmost often: {_M.iloc[i, j]}', ha='center', va='center',
                            fontsize=9, color='black')
        ax.set_xticks(range(len(SUB_MAGS))); ax.set_xticklabels([f'{h}×' for h in SUB_MAGS])
        ax.set_yticks(range(len(_enc47))); ax.set_yticklabels(_enc47)
        ax.set_title('(b) Share of the patient\'s tiles named correctly, and the most frequent prediction',
                     fontsize=11, fontweight='bold')
        fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02, label='share correct')
        fig.suptitle(f'One {SUB_NAMES[SUB_TRAJ_SUBTYPE].lower()} patient ({_pat}) across magnifications, each scored by '
                     f'a pipeline that never saw that magnification ({SUB_SETTING_TXT})', fontsize=12.5,
                     fontweight='bold', y=0.995)
        save_fig(fig, 'fig47_subtype_patient_trajectory', dpi=170)
        plt.show()
        print(subtype_patient_trajectory.round(3).to_string(index=False))

## 5. Bundle everything into a ZIP for download

In [ ]:
# ---------- Archive all outputs into one ZIP ----------
archive = shutil.make_archive(f'{OUTPUT_DIR}_bundle', 'zip', OUTPUT_DIR)
print(f'Bundle: {archive}')
print(f'\nContents of {OUTPUT_DIR}:')
for root, _, files in os.walk(OUTPUT_DIR):
    for f in sorted(files):
        full = os.path.join(root, f); rel = os.path.relpath(full, OUTPUT_DIR)
        print(f'  {rel:<50} {os.path.getsize(full):>10} bytes')
print(f'\nDownload {archive} from the Kaggle Output panel and share with your team.')